# TAS v7 — CPU preparation / A100 experiments

Upload this notebook to Colab. Use **CPU → Run all** first, then select your
**A100 80 GB → Run all**. `PHASE="auto"` selects preparation on CPU and the
study on GPU. Models/data/caches/results live in the same persistent Drive
folder across sessions and future notebooks. CPU/GPU environments are
recreated locally; the notebook kernel is left intact.

Bring **only this notebook**. It clones the fixed public baseline, verifies
and restores its packaged experiment modules, then imports them normally.
The editable implementation remains in the experiment branch; the
compressed package is generated from its exact commit. No separate source
upload is required. The code snapshot is also saved to Drive with its hashes.

Automatic flow: blocking P0 audit → 30-image smoke → 240-image screen →
predeclared shortlist → confirmation (3 seeds + greedy) → controls → P0-D.
This implements the owner's explicit override of the earlier smoke review
pause. Final remainder and P3/P4/P5 remain disabled by default. A session
budget saves progress; the next Run all continues completed stages.
Shortlisting requires the predeclared coverage guard and a nondominated
observed hallucination improvement. If none qualifies, the study saves its
comparison and runs baseline perception diagnostics, then stops dependent
stages without forcing a finalist.

Full A100 weights have not been tested by the local CPU suite. P0 rejects
baseline mismatches before bulk generation. No scientific gains are assumed.


In [ ]:
# Edit these settings once. Reuse PERSISTENT_ROOT in future idea notebooks.
REPO_URL = 'https://github.com/RachalHys/Tag-And-Segment-AGLA.git'
BASELINE_COMMIT = '624f74501bdafe9b7d6a05ae739d12b422ce8737'
RESEARCH_COMMIT = 'e0e9414fa69ec780d9b2c8737b8a0263a92b9d1b'
PERSISTENT_ROOT = "/content/drive/MyDrive/TAS-Shared"
LOCAL_ROOT = "/content/tas-v7-work"
PHASE = "auto"             # auto / prepare / run / score / check
STAGE = "flow"             # flow / audit / smoke / screen / confirm / controls / diagnostics / final / external
MAX_SESSION_MINUTES = 180  # planned stop; run again to resume without changing the experiment
SEEDS = [0, 1, 2]
REQUIRE_A100_80GB = True
AUTO_SHUTDOWN = True       # flush Drive, then release runtime after this session's work
RUN_CONFIRMATION = True
RUN_CONTROLS = True
RUN_DIAGNOSTICS = True
RUN_FINAL_REMAINDER = False
RESERVE_POPE_IMAGES = True
DATASET = "coco"           # AMBER/POPE require STAGE=external and explicit verified manifests below
EXTERNAL_MANIFEST = None
EXTERNAL_IMAGE_ROOT = None
METHODS = []               # empty = exact initial 14; external runs should select frozen finalists explicitly
BATCH_SIZES = [1, 2, 4, 8]  # measured once, correctness checked, then policy locked
ATTENTION_BACKENDS = ["eager", "sdpa"]
LAST_POSITION_LOGITS = True
SPLIT_SEED = 20261021
COVERAGE_TOLERANCE_PP = 2.0  # approved absolute percentage-point loss vs harmonized TAG; freeze before outputs

def release_runtime():
    if AUTO_SHUTDOWN:
        from google.colab import drive, runtime
        print("Saving pending Drive writes, then releasing the Colab runtime.")
        drive.flush_and_unmount()
        runtime.unassign()


In [ ]:
# @title Versioned experiment source package — generated, do not edit
SOURCE_PACKAGE = {'schema': 'tas-private-source-v1', 'base_commit': '624f74501bdafe9b7d6a05ae739d12b422ce8737', 'source_commit': 'e0e9414fa69ec780d9b2c8737b8a0263a92b9d1b', 'sha256': '1bed642d85c0ea0940e6ac950d8a4999dc725d74ed4408159bb809c95182258f', 'files': {'AGENTS.md': 'a8a0d1cead72904a1e0ef0ae3853d578a77c3017deec0ee728dbb372e284ff33', 'docs/TAS_V7_COLAB.md': '881c139718ea1e73b0e42ec0eec5c4fe4b0f81a8b079b417df578295449bf4ff', 'experiments/__init__.py': 'fe7a0225cf031bed0d2204dbb763b431d530abac99b01ea33faa4d0f6b244526', 'experiments/tas_v7/__init__.py': '5ff13022425f78d78413600e40c3941e750ee983755c9b17a5d564609400edf7', 'experiments/tas_v7/audit.py': '86f0e44218e45f06dce1fcc0b172c51f8b5379554bed63a194090005d92bb7e4', 'experiments/tas_v7/cli.py': '438f6ff9c4194bb961857f7abfc94c12d1f963c849f0fbba36767266edf002b9', 'experiments/tas_v7/diagnostics.py': '2a1a56e626656d6172bf0c9355a5db5b60519f0ac82350f5003cf3416cd1f77a', 'experiments/tas_v7/evaluation.py': '33b1d34b7b8dd1a4d2930fbde872de14dec65adae04f4bb449fd52526b7fe90c', 'experiments/tas_v7/external.py': '7e9d9eb0f91f1d424c1bf34bcdbd45587d16faa8d2148eace5dec068cd97bc56', 'experiments/tas_v7/methods.py': '2913cc58e41e1a726d7e1ac8c7b2711a5259afa5dff4fa7e6f0d3fbf4d7ad9f5', 'experiments/tas_v7/prepared.py': '81b5d30c97e731f8e05bab40b944835333ba937d041c4cc135e601540d93fecc', 'experiments/tas_v7/report.py': 'ce3b357400fa88e3c86ec7f4b47a078bb32b02c9ca032ecebde112c39a279008', 'experiments/tas_v7/study.py': 'feecd5a06b58917ad5f4e19f9a50da6205dcdfbc0226bcb5a45526624a5742a3', 'experiments/tas_v7/study_diagnostics.py': '9ca7d8103abce847dfae98d6fb7491decce010dc931d97b90492afc017558d75', 'experiments/tas_v7/views.py': '30a082a37f0205fce49cf43b2b3002df0a40935ed89df667f8366fecd1d5114e', 'requirements/colab-amber.txt': '7ee7be8843a02ea16d1333e4271ca0b8174445aab6867efcd46e57739a95389a', 'requirements/colab-base.txt': '992b028d26a85bd15ccd29321a8192bbc81efb6696df38eb834820f06377a9a2', 'requirements/colab-perception.txt': '0e98ee13388aa50ad35045d981a278d347fb0c56f1634fca4b9e5567ca6aa03d', 'tas_runtime/__init__.py': 'ebfdfb5371eb85b8640ab449c246efc862a11ea70e142718bbef4c9f7b5deca5', 'tas_runtime/artifacts.py': '37e4bc9271a13ca334df828af92b58e425706d0164f9f30384c81e8dde7b2c37', 'tas_runtime/assets.py': '675315d0c9b6d195e7941829b0bc93d3101a34c8f5205d9f3aa6fb9e0eeb3538', 'tas_runtime/attention.py': '94585b4980179e09acba88e0b3492c2161d94904faa7783f6f5a8391c51854fc', 'tas_runtime/bootstrap.py': 'a679b0195a09af4f18afe5f5fc1bd948fb983490993225860ec3cb66c6f2f733', 'tas_runtime/config.py': 'a58903c262e490dc90a3b1cb9ded9b6bf1c18c018a1592f4c9c2bd19c6aedcf6', 'tas_runtime/data.py': '39cb6a954ddf713a622630c584087fc3f5e8e5cd4da0d9a2377b7a7aa1be2523', 'tas_runtime/decoding.py': 'a5a370f0345e5ed6fbdbb612f16b9b285ab287bbb6241dfeb39189663ebf88b5', 'tas_runtime/perception.py': 'bf7179deaa1bcc8421d6c6e989554399155c1e391fd16c0be57185e43871e025', 'tas_runtime/persistence.py': '9fef2d9912d19ae2f20964167e121f2874e92ae3e3e695f3c64e96cbf1956a1b', 'tas_runtime/profiling.py': '4d5139230aedd1ffbfe00dbdb7d8773c1b85eb10eb0d0b72467876f56eec2d94', 'tas_runtime/runtime.py': '4e25c196b9180831a11e5a73704d2512745cbdebc622ccc4ed1722a0960c17d7', 'tests/conftest.py': '5b20fdc9ab1817a64ddada47c22d695378148741eada4f98b1a745a49f7c73d7', 'tests/test_artifacts.py': 'b21d651d6110626e3e99471230b5d7e9386b91f30ebaf035ef8e9c5427d9f506', 'tests/test_assets.py': '6b51775e20a03a2c9384bc9922285075bd960131ae9368bfde5f73fec98b6753', 'tests/test_audit.py': 'c6fde3cbbc8ad8394a0456390e4b4f350871eda9fc7efa1fd2f30eb50bbb446c', 'tests/test_bootstrap.py': '8d599f15e50306e449ff55168cb740c2ea4cd847220bead6554d32522fbf4d27', 'tests/test_data.py': 'c55b36a369a9da72cfe17fc8c88b478cde2ae3d466bbbca67b60013ba18e92e3', 'tests/test_diagnostics.py': 'e5cb5776067150a2b1b6aff62102f7e881d0d8d88ee6a53485265478b2e02874', 'tests/test_evaluation.py': '4a8e77ae0c9e4b2723fb1fd1a40f42e3d5c4e15075ea2cc2e2d54dfd350da98d', 'tests/test_notebook.py': '4c6ad80f1e0811eab4fee9b07069268d5bdfb479f124b4ed6b3467c686e00d1c', 'tests/test_notebook_delivery.py': '377d9a8761c98b6ba6bcb2752d6f7549bb89f49feb0b109678ffb3ef5fbe85e6', 'tests/test_persistence.py': 'ae9b71a14af81561373fb273e58b773f8ced8fb8e4189a1206742cdbc682ffa5', 'tests/test_prepared.py': '1cb769c861c2f06cff94d17bd5027354a94f0cc8baf631ed3b8cc1019b797a73', 'tests/test_report.py': '65f0088060033ed91f2649124f7c11c7554154eedd3312b16cd7f3992113f60f', 'tests/test_runtime.py': '2feac10c8d88d362f05a7e0c59660a7717d1034deea0038cb895797fff40877e', 'tests/test_science.py': '7303f63cd2f15c550ce45d78c5b6e904fe148a3d58e59f103239d84bb9ef2ea0', 'tests/test_study.py': '46caf5aeb11f5061d5df4a7b0536e6d80867bc37181cd73b0849047935d600fb', 'tools/build_colab_notebook.py': 'b1334a6aa7710d6d99f68f64c5ca88348db72a1b3a9f92e2eb08aa20e86f44ef', 'tools/notebook_delivery.py': '64cb82c53439b1b91eefd867d31081f9a1e74a30b28186330aea0c029776c5cf'}, 'archive_b85': 'P)h>@6aWAK2mk;8Apj#0Q1xL0001Ti000R9002QpMNU*xE^TCOR84Q&Mi9O0R}9cYTEHT0^kAS#3aC!&7BN~Ok$ou&ak(6cTP=6j`ypHT^?gH1mJ{?Ma;cq}_ujmj?`RqQzpOKDi|LfpJe#e0rq$vz?XQWS14lEk%VZLDUKe@ij*O$faZEKXt{<+~H-FtNeyN7;pYQ+g>krlJkDO?E$h~)DhG1FpsBCbwX3t3yHd&>0EH0y9-H>t}HD>^+mcRGcb@O(Xf$+v#)spj>Tkm(0)JN1%v@y>D@arPl4}-}SePj+K;XazJamvmb-YdIC?z*|4>Cmdg*0>Xh!FOz_<Bf4fk|l~9qVI|}t>tPqo6~*bF{?>HCHfqx{#adI67Ft{<!>ogz@1Z$Dm1k55%DI?=#}IZeegmMfhj9%#|oCWN*{pWv6wg#6z#H~^mvnK$I-E^0O~<mv@|n4ncP$J=iTb#`om(icwYyV`#SqNfQX9R)y@VA6Xf|IZ=5p8Ponwbr~;5R=NphSfRcckKhi5wuiz)j-rF>#X)0Qc=?aumNH#!Bl}lt$QI_uR)WhPav}k_h2l})rsG@t;yWowag;uxAdtrFXj-xOXWnQcxbYH<8QPgcQwp(ivX??m`g#*p!lQrnw*N!=U{<*1;N+x1^tks<h1*Lat3-jh)<)g*Wfe_7JKpi~*QpMuZBw~aLK8`Uwkl%A0!3X0UQisx7qv>S8x57V3k0f6S)~iki(J^=2i2^TvUS7ULT>z^eILE^Ur4nQ=Mosydb(y9?)J)V`uXo06E4o?U0~;nsIwFHU`VLSP@T7k1!Tj6<Fpnl5q!Wv)OE{b(96ZSS5Kv3x;J9bnME`{y!GwU8Yj}iA2Zb?4oT_nZ6v#ZCz8=Ow(7ke77O_C0_g%Vx!$X-me<qO_1+&o!DzWGo`3a~PoVDhZfSG*B5-6a4i_Bjpr=%W|fS2?PfoLH`w1`?pc|HaGG~QJK<GV*|F#oDp*@FOvb`M@g<jIq2G4us>p@J6sAl=ES8?rtnLXlJw{dzV3{R9@3G>_W~O9)V2nmNt@C?7?`n^sUBPpAq2qmh6{)@D3VTU;mZ0%$3BizNqqpU?mmb$>L}v;!p$WClN;urdaPm4x{}TMP{w09j)q_tW;k)oS)1P)h>@6aWAK2mk;8Apkdoqicc+000gY000yK003lfV{<Q5K~rB=H(x_fOhG~}ZDf5|!E)O;620>)x@HcOxFk|`lF96NGgXS?IN5P*Rcz1JZfeVdNJzvOA{YRatvT&K>>uuz?CS<8DRbCMB8vpj{rdIm2YpYs(;M2nCa;55zO1QnQ7@c($g<m7ldeou3yTj=t@dWEn@FXrG?_rIq51ynBC0UnydIa%s>M7fUOnu^Z>!9zt|@En>CwcR6s>fly^6XTVas)k2!gKgvE7neQmkFjG~43;hSsj?tPVMS#PQZOI%E<1FU=EHwB5p*k`^jxYZ^_esW_eJ@DK582eJ1Xk?Nvbt7uAP?KaP@8@rWPcM&&QS6J<6p{Z-FQx#6apJ$w2o@L*EPoKMnOq0IJvg4fY<UN`Xn}h=3-fv#tr$wgSHjBB0he&l9fx*$y+3fD<h<w*XM3`qMIn5XeJmLKj9-i_NL|i*>{?PQ!n~j#&me#E5$cqEwTq0pf!YaSg84*?PvEh&a+$R*xhEabqa;a2QvhmgBjNx*fFBur@5!6PK<t^r)wsIb)GQO$txSHqLi=1v!3P(rN<H>}6n$X4DqoWa@-+UYUN}1-*bZA1v(^Q>{*&QFGn~m|V;VO;x<y#uAmUU965lJ#2o~Fx7_1koSEtQ;6m`(ERWiF+WgGntT7^jeT2nI~j4Wn@ezFEyM;E>M7AUl91rEVj?Rw^)6(BRm-?m*3)e$~2_n1jNhcs-a9nO5Gp$Xr-<u>*jn(rTA&09&Tkx<^uxtX6?RJ9!1XXz6udP719mHQ+)K<pl4#RSi-ezy9yP|8x8kzm15>ht`=UhOD(+*q5+RBBx8PyiZUtgTZ&N6Zn;0D!fD?cGZ>^UA5AY`!d!DBCApUz!_2K7Oi)nLO7+aS(?U#8XG-2{~+hAHE;bC2Sjt45@=D;Z=bNYYiX^vnb%)C10cNao7^<3o|+M*#m3UKva{=A+o7%Nu6Y1d^Xc8~_5IC<yW4lyf4_Qj>pMNCCDP@3B&<@(3}xToI~qhvvd}0?rM7mInWnT|#m5g!Xsv1*oZCadF*8DdqT~IfSAPJ>mP=C_WwS5yp*g!oA>~!`?D_m(&*~W4@N-eDAZFbnhk_L==T>M6-k^|wqjNSC3+ooe8f5HWNY9F4*LM3n&u(<=T1iYl>9P|JYVRG2qHJ&pfN#?O+A)wl3`6&$axHkza3_sA-%H;T6wI=>=%ojSV>Y4jFBI19LDRDr6EbVH(h!K_m$cT9nkqb(7fR5hp=T#ACnIUR09wft#cCR279U+g9K8_uLFqsn&+3XKu=Qk=NG3}`GN$ede^*=Cr_vj3UCS|MSQ<Sr<L_sa@jI%FS~WPul!2bTe$_|$yf4_&abG14K<czd-I!sX+y>~O4KtY}c<RkcSdkL;dSmn>wF)KA>AgJhnrl>HG0(O(j#;*5FN)dAV)kl8?=Qv|qpUVnrCIiI$OaK3#iSjqFo+(sQsu*ffnp^^(;*H>N1CRpMS)mb1Ef4MV4$uQC0U!un%{#;464Xc6Ok~buII71M!6v|6`(?j{LH!#ARPEB09dr*+BFyx6(;)yJ)56h|91X)dU1Y#dwq!yS7+z<v)P-Id@>)AS_EgisI#S4rIc%uPfni8nC)Abvm>_%oNh&DAEqBa8=#rVr+|7$MnbT}UT!z6s@xi=vck3w)7kLwW(cCQ2)il8K`g!dL(t6nE@tHfsBt?GI*CI+mOw;=wuG`mpPR}+|1^E=z`Uhl>U!M*R;fp}a{~vWen_U>oICxNubjldW*X?2iLkM4K|Czcr3lLIMU?jn%h8+Xlj8iCg+y3)d44fH`v-SrbZGjVu9$Q^9&*(AWc&du#vf$Ohw0}}*H<6^dH(J+V@t$A@~N0C!ybSUOuVCoWx-~QRu~o7$+3lzDaKX~)e1tz^fb@pWbYPgVJzc|bZaib``L?=Q+m{FBtqW<`11J2oE1Vw4)i`HH((k@*-AU8gvUt6ghnqgr@NNn@BNL3bQTL0OP2VBuGPk1vh`I>CPXl7vc2F5GFwb<{d@8#uI(F4MoH;>#r0xjOLm9D3TU|6hU5lOiqAa_ypEjAQW;>y{1DE<Y=Sl`=eIPp5PXGhR<VXonKefhU_w)YvQ`;RPIPCCaUMhTs_g{;(%d&cC|`Qh3QYP5%{0cqtYC&A@|5=ID{6ww%rb(vBmj*OXy+SIt_InlDna8-W;Z%yO$t*yN=qk=kwKkFNT=j1OIAWB>^M5?)yoC<XXyKb3~Gd7*QgDUVRN5n?+!YHiO5ZDc)BA-OiLG~IkL6XGe`-xXmR#9gfXOKmIK%>U;*qlMyyJuL(x|qasggkQ0NTwZo!;{FFI4HF{OuNqOV#n%&K#cGPhtF(0Y(rglBECH31ETZ!w{JDNobxjBh5rlS|gD)LAyBIqoL+{k$#aKl3d>+d}LUx<#PI2_sY$a?Oh%jWMQkM{M?5o5))A#xUni_t%QRmme5?-kYNV*M=@1F*#+hYS5+Ii_C!-$!@(@=!~uI#UR`d1`-Yd`@$bSu-q%YEMZ!3e;izFKuGt^4Td*OeqqiRwgPfLhkelqmOpax@}aS<{GSq&k8B?~m2@Yd(5MyzE28a_6_yL3IP=jr{<+uRoUWm}kKXVg7E8&NvycOX+({^;SErDC%pe&Q{ax=xemu#Ke>^av$(Y?bzMh`sFY{w6Fh}rL;wXmJ@n7&Ix9csK7N9X}5P)9hFJ9#*`OeZkXy0hdAy~Z>$k3>Kk#gyp!7>BEnD?VERd#-won3yMkyIu%7_~+mWMI=Kb(8clmgJ>ABk{&p4n+bEXr%1DRiT~^yplVwN$+t;%xk`5+~_nr_WpCWO{t^%ym{I?&vq*y<j$`OIN33X#;6u8w*(q!%$9&Vr+7ZfzI2-b;8lkyyWggAV4z}Ivw5hCZUcYX@W>2&0u~(XBN%@XJlQz2cn&^Rw&lUT-+A~=Ay2+nJhD(53$b`xEY;V%j%y18oV|xqB)x}(Oh3In|6I(jXXnF>mF73_Gv{x<8{qtIE|*QY1fdYU`)IlAV)nP2>#NH`6cUXS>#)18_Lif!xV(dhi1tzzI}jHLrQaSFP=b1`Jm26BZ&15yY|;+CGhClt=PBqP!afU(5fA!z!!>0vp;<2l$e5#M=eHC5xd1eV3TmInr#NRVW_l)tyX2d2KbCP<n$Izl>+I_C_SgG&)7$Brg0n56EBe@~0{u;?jx=8LC%!iDKQP%}v;C7qM~!<nzNZ7hJWv__L4?^Z%ZIi9SGZ_API5(m<wd$08@_q-X2Sgb=`Wm0^2jN)R>_Y+GX8oQ>VFe}oA}dDHNa2XrC5IUWO%Rce-qlfK>5*#+p%900O48{d`G0PMKDjJZyJ*8f`zdUB+uS=){gf#@T4h){PwXoFuY0%JuqO2Rf!h^8Ir@wx`y3u|GS1*t#udDX^DUk`eYF<W_N>aT6Fed_|**ipR(fY?gOEPfZz@ZQXNdkLw^pDDXl^wcM!_??3?&k3$XluP)h>@6aWAK2mk;8AprW+yJ1HF002$^000*N003opaAk67ZDnqBb1z?CX>MtBUtcb8c{Rzw4S+BR1<<=w5C(7omv9AQe=4!G4Uoq1J^8MWnNQax3Bo(!YUoK*u16~3AhkI>wO7hp;}7asaB4$Jkd1;CJ2P(|P)h>@6aWAK2mk;8Apni)N%*b+0080u0015U003opaAk67ZDnqBb1!sZb6<8hFJE72ZfSI1UoLQYMUJ};!Y~j7dw<2sHKe7YpokC!4?#e5=!g$u$@U@J3FPl_XsBm)v|8(Cb;7Vf<ir%)4%WXMdk&ehl~5_wbOphS=^dv(MmN4BPb|$4B%^g?eS$pTnNlCwOmdXrr&qvM3_JB^`|NT81%vhB3}^3=qAh8P3@2pDP=XtAA5r??54ZjF1&&6Jg~6~d^Ga#0m8!3Y`ncKFyXJPOkEhLkhb45TG#?gPeF0EQ0|XQR000O8001EXlkp53=@I|{sYn0-8vp<RWq5F9a%pX4Zgg`mbYXK}b~i6!b!2IDE^v9}Tw8PFMs|MZujs%J25b<LVrL(y;#Q5bnza+hGwa>*Zo*=rnq-rL1p-`Lq@h*%?>U!lpc@yG(j;4{3Li9Nqc5k=<vW*djOTgZrTO|H$#&x7wOHS)^+PFSwh>b7l`QK*ZG`%y)^(NS+4PX7$@(Y`Nt#z8&xBl8wM@m6N_O{E8F`*JnQV)EFXDJxS1>q^MY2ESMFn$ac_nd{Quo=&O0H8`mZ~(J3Ux^3T21uND%q<^^CK_T_j+h0%ebgBd>+Z7O13~qGl{J7eX@>!E%U73d!yF*26wAF`x615KVYl+a3JzRYNC!RUKKI}uIR=s(4aQ)p-|i8Q=I0zq|#3c=w58%QmIWi3Hs|EihP@-*1{@#Oo}|)tE`ImGE26qtU^&9VKrz@ep5V0v{%)AzG*hcKjYOgPBx*~)}=CJ2w+`ZCM#H_I>MCYgPsAXJxG+Jo<jeQw@Q+TR<c^(hh}Eoit{C(R2o01<76_~sIB2h&Rp{K&O))yvu(076NKWcQ0xle!*TvzrX_UP%TIBpp5iKhP+2+uDbG|eBd+aKrV3IDG1v1%ZJTB+kdwI*Vxi%uGqw9u<b}0D<US0WIV!b3U?14^-0E&MTYd9Gd~kYUgZuoU`~0DOzRBZK?hmQ6QqxF=2WF!A*A7#8$z+4n%ei5FL`|S7MPzpfgLdFSZkqOaf0+FkxpijZcG@DZ@^zlhS>}=X-S#MP<2<lbf3H=xRzUc9$~EnQ);Y~3#S%9vM?bXM(RZzS^c^_FUmg?nq`#tB>aSn%KaPe{OM`NA?;zI?YUAm-``QZ@KC_jxgB7iMn(eM|!pve|Lu7|g{G@Wf+DO4>J+p1xH>S?}wdC9`Y0Z!q$u7xcI_pD&+aEusvza_vsY=2R*jQJ}&e&km|CVgcBxQX-%xM)F*e#fajY+t$;!Hk@F<N}uPENx>OQ5BmtwOl6Z@#&DxX$;4nPnyq<$YdhI>8U90(D!fB-n2PZcRC=2L33WUq{#9YEPkm-tf=Hl)UfX{rn#ve){3R?*5=P4<!IS|3&LCvvU5IqHYxsIGw7@Z|DfbKh3G7Aw_qjm1P3le+Le?z#ZNdMPB&cd-7>h_G!Xw5Oxf_FU2O=!qkz+!hOn%vI3|E@_B2;4tc%4m)Q<ohNjUpV9!uo8`lYJKB#pC9?}F3NhW?v4lVhi7@`}8W^%8vyP0C<=*c!I$a{h+0fNq(UJD$yjJ9bmD?c#Hx~<McW`nRRpT2{SrEWegA*OnbAd;)n574LhLEz3ofeN^fvI2J>Nn&@rPV-Xv4xJ%u2L1_IEAWpym&;)B^&EI$h>7^2KfNKT^Q2Z|0M@tnJH&y)5G&_%;eGh|hd+AcbF|!MJyhrlR1?ta*T*3H(}y2dK%ua}#u!8}Ivs4bKcg+$SInoG)O!{f^XhQmj5WDh>3z_kIz@<gh&J;Io_9k^{@hvYMMv6<W0BW^TJ3$Fn`RGn1r#$+>q`5KSRq~Bj5WJOJkQlE=+Lp{wx=PAy>vUEL##bF=rPV%*%5(PaBI=DzG!fj^#%w9J&b9){m8P&pD-TPacl^n1mQNA*-?W8<Q;c%rwXx>rGg+^i5-HlsP0t;X^NW4YAUBVXNxc8t;p0Pj42bq%SvoznxdPICN$+cXecXq{Gf_V0aBIVyFi#Lh|=;bJ)&v=Ubeu=u31QQN#ZG~?g6Y|eJeQtA;Z=c1_>PLvG)R!*!2PEKp>836^zr8Pxi2{G7|44P`PoPObJ1jJPKY1QzsC{t^Tg@R48Q26{u(eXnT;=Jv`gX1eD1zj`f@~14@}lb1`wi?rE|z4?n_B=1r03h>_Ufx1wksMf(S6^-)&fLD*jbA}3`P=MVIg`*JcF6IFAb^h$W$WHUmdm`Y4kG5g{qbZ!9-_K5(eH>GdQ8WjrTTlEQ0K0txl&xjy^@HQlw*p@|Gp4+x%C2XL@L+P7u1sI-D#h7zy7_wMek4e1U#+}L`z$uQ<Zj^m-y<B+EC^1|1?9N(k;#96+3oyikbY-omaf&Jm3Z4soD&&UMPyG0!{N)`69s5J&*+#xjU@OTM5@!RLIOnqiXx@)^e|-0^|A}j5kh(l7ad&+SK=?FWDZnoUwva=dUjmeU`0n4P-(MM{;>3oB$O4Q5c9g8e-Hk;8Envk!xS?h`Q2Iv+Cj0*5H~)+gG3@aQo~|UL>6ON!TeNntOMqOF*VUn}?Cj23?8V+X=3d~cH4EA1t~k=X0xU8~NM3|sY#VieNFEU*--=)Jl|aibZVgTtM+pQAnIeF2YPV_3+~Il!gpurEoeXI($f!d@kWxs=f{4*Z+F}Ac@DUbCKOnyXrCEmk6gq;I#~A%!V3H%42mVHNR>Cm#oAR#%NXPuPK=PQZb#K#FDuCyzLIue)-y=}b5ur&NKn*Ey>SR0-HiN~M3F|6>Bn|vR`~{H83<p&}oJkcL6c+@*Gl2P&!ia}pHxlpASpdq(N-LC}jbH=-Jrhre`Vzpg25^PYpd{a_!PeA&ZXp$?RC^673hF89tWYVS>PFA^{d+wG@>dA~9WOxZKIL_~5y>tCl?TGV1oNtQNI2D4fH@4VOkT_~LDq0_t3@q4gLSJR^IqS!za?GQc-q<mrwC3=YSlRUaavz(tp2^;$@cUB@p~Qe+aE&No5+1&;boKexA42rAQLz4EN&O{mM%Isn+IcZx9`~qFmt66ovZf%F6IXu<~5pX?D_8c>h9(i!fwE6&>QxdtaMt}w4>4vz_#o!goz9(MolTdtkI>THnMr^h$0gIuGFF9);S$bjJy7#i%Zy>6KYriO>k;V5GVf?JT`vC9{c1s-u`5uUNLCBz;~*e&;W2&Zl0EG*o`2LX2SRGuDwur0Q7;ITrj+1rezg0{dTk)ky9zMbR^zLh_aZD!{q)Re50q!-Y5tD`cHd!7&ffgRL6sIhrDz%EQ<-4XTk3KnH7@=1fruPZ63~_M)t*?IM$;FN1oo}8}{iIc}y9FkgX|xkAe$TTwoI%&C_lOD?#k&xmuQ49kgGYj1$nBMC*E*wmvOR*{vrDQkAji02P(5r4{r$;r8ZS{qH{N6sI&lC;;r$qKEMMb%^Cwss{$S1#7o(J>31l?H+(^pgm2>ZuO)|O+XrM<#wp3*L&b>bbyamk@rHO_S+XIqX8jX)nCq*yw4Y;%N)+`<}H(=rlJoFMWARDx1-|h47&-%GLi>`{62%92aVlh{sB~4wl>^T3xeq*x?vRkJ2Ig4(98!+^WD;De+X+qJ7AbEBcoA@Sb`WdxR(=B!1@<NeYBZAD4(zCw6&d&*J4XqIb=yKM_(We12>LqfZ&RJQv)_tcBE_KhZedZq?BAtpbbUh&q{GogpyyY?lZb4Ck&sBncO+Qj)7DEV1T0p0{U^IK55t&822dyA2sq*hJMP}FE#kLqId9C0knbDpaf!rAx+DL7;+Lv7SkK>wJt@hq}uk!gOd(r7cfHGUuWQKSf4JoMlMc_#kr=$ie}hU2uXryYLyrg2A>R4-J(!f1i-;;liqM4Ssx-|1xDO`(uEkvKe5fK8j}>F`hZC1bN)Z~7)T&erfThc>pf%i<-^Dzies#4$MFl`3^crWEmN(V4!)_MwYUM(@oIrvv*rpPqD5*2?-y#Q12kL?WLv<XjmmfsyUm1SCS1f^TAiCK3G+6-ozb~fJ6W*QxsA#Am8l>Z8J5W1Cx!WYNzd_AAuF-VFY|9Oq{)HqV<he#A&%6uwJ}PK7f*0lDXB0F_OO}i>kpjm>uL3273FrJISMp&0xb0k2*akdOI>i?)@eGuS8^kMe)m`L@rS?N{n%P&8S26VE-1`>i^yV+x$`j>3nx87MF^bZFe|}=zwf{GG}kzVjJe#i=n1(kAYE8ITHGw1G!I7N@~yP!>C!%ve8xYFkW90_l#yf>buV;ub<;gn`9-F+8nAZ{Yeu?|h9w~`k{9albpk9D@UKj#u<j0JLW`8QpcR!W(DVNCr*|xL-D6dQs)JTz*{A58u7G)H{_%kx*y{ShZ%H4mn(-wehwpf1*1!6&vaUYh?TZ^LuNWR-UzV^s>!akQdnQf!^2`5FUa#}@M&?>YXT5aLRk#-R+6-|hA#{^LeR%+HIN2)N_a@`2Xg<r)bn%&j$Wlj<V3Z#7fs69O<06jv)fAT&OE<ivqDQt-pCE^YgpAIpvwF{Al8bi1!1*d@ajFIj8yqz4)r{t~a{uODmTzuD5H9=!ivZn2c`Yxx$8shbccic|EFaHh#%|q>#{r{vZn__B;0uzTn6VVa*WzY5K8EEr{+vl2e|~Ay8C(!bAuyMtp_y+KxK-X^LAIB5%)t*8t@Ke3ML>Pe(3D$;V`~hZS^_TF3}Fmxh|~!;HWCsxs)L=Dcpu?(PI5Lc87joTCNcTn(xe6BBTcnDF^|3pMZ80~8K!@>3cfY@;ae3=UNVfo<)Xp(rRZiY|IdPGnn~Zqg}u<ZGe>Ev*v51ZOBK4(W}?M5aCWcpws$97*Au#Kn<uj_8xJ4<>5S2=z>`T%5b3s2Nx9(wv100O$(gUw!1iRPdz_=`M1R{T%BeWSJ{V83LtW{^Tpg7c@<|s8ZaG31&2i?fpkTep8z1wvT-B*8jsgRC&pC0|sYBc>p@VpxY+B(w9t@@2gwurWf`q6fEPX)aEMYDScmtP9%qh3I{-`7B(Bws?vZMoTItS$gDSqM}J)46qbD3XX7+59M6NcazQ$Na-f~8#zVh6!05&=XM3TEc44&&A|&yb~^18(;F&7~QL3YX1AKy4qf2_Q7EG(3uRhtdr~8^xj2$3d1w17^Zp-(6ZKmF-BxNLg38$6jjD?a+o%op6m}O}xrn(m8KKG>I}U!H5QYR;x12t(B4Ltllw5#AeP&`Lm%GZ=$$2RtF|V`dC_2m(^PcFNoAIm2)oLBwX6W>oOb}boYnXPwf%IZX8%wC0RXqwmec>Y&ie>Zk=GbGG%KE%3Q0*U}vuKm<2E)?ME!H-3aCa|CfbL-6>*+e&2+rjttr>OwTWuXpNIR4u$6KeG;1m!3dXOAcYBfJEIxNMQQ5j0pd_}GCC5}96UI2a8%H49`}e<jm3<1vpm_~Bv&DiX>+XR^&G!_Pe;ED5TVMI{`j@om+|<O3taP8W?HYxw(K}rv*2`Ri@)JHW}VErPG0c8Ect4u;cLr-HIDN~aDA2F($ZR1{@DTK<u2+ql4CED<pPZ7u;fe97f(eM4phMc%jiDIF1Tp)`8CeZb8(&FGF3NIA|u*8p+{jg3NUROr)qph`qfFSE15hR4^7?7dYmq<@wj-qbdw}*Jwt#=icT_M<ZYJI2BcrzINOuK<b1#L?~sjs=)D;GEv2&~%)2x0TS=$K1#iFyyNWSXfSH|KQJ;-<Y8<n(yBiHcTtF!6v=W!pOD?XPTv|W5w2ty7Kk+JG(Sb&;0530J=+)JhH#wlsEx%lbt_&EYWB_{J*~8wxHnmkA=jfUB_M$*1xWphv2t98YuRB0?uXj8**!v=$acT21PV>5kK32Z9juesFChx~A6G%<)hVgOa)c8Cvz7T(v#h&i9@A1Ad7-h(Zj*Q|GlWS0vWMA(&^B1dnvr|<xG94bF@eTQrU>TD<Jodyh)c4=E$EcIfgq43-8TPC-Uj!F#fe8%{{rlH%hJrU&U<X@u)sAu<GGL1Ifo<l&*XDtKD_6Hq#zK~};f5*B9`HTL%?Y5?E6w%4;~n|Ejxs)|BEHY-V*btVuCK*cI!U||XD86_v0?W)zWleB2Mhqi9;gMu_#_7<AHZP1B#-gxSgCSuINE*i8u^%NnK=$a76$`&6{$Y%|I7wOuoiY;E)?(az11Y!eaw@rqPCz{NFNiYky@d+toiDqdI!aInp1CNl$5c2lu3%X<(~+MtRZTDsE#p}S_jfwQ&r|Ss+Ol&cPK_*U)_2i3E!XXdc@1sb7Q10SKrN_n-T<Wnu7K(nOR>hUq-Zho7Uz12qrYInWE<~(tqVahxb!%)w;}o(L(knK3-q(MW@d`@DR4P`AsRG-Y(TA^DEW7J8iZSiU)O^>;I1c*=0_%yH;%GFcTXus>_I-^8uK2M$0YW+mga=_`*@QsjY=}<t*%!#saHEJyhhx0_SXyW7a!_;W)LO&G2hpxCaA-lb-6vT#<L<FAvhQq7qLK0)RBpBb)H!7Dm&hBs|}!?u)$M-Q)XiSj&8rMS}KauEMD8Ho^D%U~jx6-8AL*R??$4@aE0X!4;9#Ke~FAI*n8SQId_|md@MFJbLoqP)h>@6aWAK2mk;8Api_0U6)u0000XW000>P003opaAk67ZDnqBb1!sZb6<8hFJo+JE^v93SWR!^I1;`4S8%i!DPucHPtT<rI%JY;XRz3wpfd~1AP59nqHS&@QA1L3j7I+ZzT$^1+0HIDhghbHRmE4YKIkM#uJ8ZFIxRY`xs|G&vM*RLwy(B~w-uw;c&oIP?TU%k>MiS(Y%S}JsMuo5u2sz!^CU?olciP-D~e@rdo7BB$%ZyUPphocZYGnsrdKd&MEw45quThajMvn0yRPLTs=3G4C}R3Wr&TG8*{y6%aKLg?=)SeG5%W^DOSy^~9_U!tZfP=^RAR}xnzyMJ%05jP{z`?dEN8WdFXoM~YgL(yDa|fFIw>PWWyVr&DRXbbhQhtk4di}iKe8ydhAXu+ws5B01}%VC%f7oW^q0t1TFL7os7gM0$EJG;)2L^G^`9=-+40L#3(*!q3#2}L{4ZjF(yCI`<gnGMHoFZU2EySqE|7)cO;?Ko6e(8FQEi<kwB_*dXDsz-{m3q}oi@3Qcpy~8Q1wnk)JCw2u>;Y&gxr%2UkNj1sueCEh-Pn?cjq%(%2zDKJ&w#)G@Ss?!A`c|M%1#U{jvhDrQObqPXkdU1FMzft5z8+OESDnz6Zbe>pqZgJAZC3&TlWz$*ZMoIfZ)4Zf5u2Zg0+SX8*i;_y<0I`Rxz-^6SGx@N|YG!~=3sErc05u;{Cmutg)=-U?IT1cd?d0hMEgrzn4oi&1c(K?~D2#1%kDVZ#Z%v*$io=Y-yQ$Q3P0r4cR;khE}F&P%O~VZp%2Ib)%wSk`Jo>UnP!MX_XjX@zEsT9waKIPNb#Ge=mce34><+=gbRUWC@N0{&D<)TR}Bq&%hybnkN`Z~<G%bc`dJzrCE@4fZ=hAS0W;DRvub9ECtE5p??Qsu_j$^1`Qx19T^DRgH(X6bE`?bR`$E25-o;S{R{UfYtTg)vw<ql?wM{sP<?mYhcciuvMsJNO;96g-|)p)9Ha*xrPFtSs=5jWhqBgaE4n|c-u(75(;Gn#*zz$zlcz)ogrgY$12f@miWzFq=wqw21uho7rWp=+2h!zf)Wa&<1=hDTtcZnbnX79qm_?GD?c9{2y|HR@_B)bIeho`FS9R$wWlGeeTk=&!+Yvsl`M(b3$aw-E@zSK?21C`==Yi64?LcQ_k=$})JPM}zlH8#kO`t7T&r=DaD8STunx^-qooyvU)u`}n-ND7UGmbJxCP-;tt?UI*W9cpz7;4%?T4CO>?|OwK@uQ{1-;GY;<ZD7kw7}l*tvtkB&Tie_Ld@A)q+>(3@H7?Nf<XR?~rQNJD_^o^_H63<CE73F1sW_Ld%vJ=l~w%$$V9-MVg$QpPijYCFUIDT?suShP=+HuP0f6js9ME-3usx*NZgR8uyA3QmIq&n8d=JJZ0>MoE@w>(d6mVh*6p&%<P*-=T>W_(`E7yOz%1Y0~uUCqV5ED-jcmTH8Rdnl)k_1)(9o7HiS^{BNWm@5B%mw$UY{|B6|5mkf_a0GFD+k!AW*}TPt2APeVgq)!f-LpaOt$!bo-7!!XNp<nD9wwD*$B8r?7I{lAp`aHz~;h!X@ufM|~H3H-vxK70|>qX-RjfzrZdZ_d&&T$r$;JTO%oxRQ+|&JJYa$aExzCOFo8tlv#4L?@iSQI&8A`ih+Rf#mW?17YZ70ph?5teC+8>2L4Q{;8vesCu*{WIRqEK&7m9rNn2@P8^5hDERycClZ?zMNB$UId_R>#j8!TY!20F5_Kd~5O@p{jy>KjUQs<jhmn*6V!-|E$h~>rQCk|j7{STo!?@TJ<y&KQPdq57lI?0j{=4oE8gMM`IR!9eOGe?^G08VuyW{-ygulb0J$BZ|;n+i%881fl`LUfLvGe#aejNeJ<HPv%0#z1}@Q!>G+?qV}gwYEO3fvwV-b)avkN45)DM&*7pYx6;hZMr;X-;O@=<X8U(IgdAOleF*rNs&bDXeC(yV4MmLi0=h6`en{2zM#eV0XMA&ad>UrxtPV-lWK?Qp=9Ic3u>fDvKf;?40weDtOqECbOA8H>j?~U%kYT;JEbumi>S+n{}wom<d?9RuXs3JWc#fnbxT3(ubQ7=mEo9`AiRVr%bW~gOfeT|I4a*mfeiXM?IX@FgMVVQI2UQL6!_>rM0NLJh=gVW~!~X?BG(5sYhU(^;dQ(S9HgR|MQ^2`vqsHF9i482RUe5jIwY!c2|Uac$0QThh$8+ZF^Bv5@qJ1eUTc4$<>|vKYzdb?d#pu!`;K}?M>o(c5;!pM0NWe-mc^lxcjzwqG^xE#0Ri*tUbSob0GSc|MB8}i@pFXWcvchmENM|Ekun`9iuF3dUW)K8=$qS`jY1B!;8;6Imj0r^1yKy{7(nTM5qpaXqfmKdSY{6*M7erH2u#pVSEchlgYjk(lyB(b%ZpzB0By;|E}rZjiap_-hoGQRQU%NayRH7*ZdPIvv*4fo!a{v4nAd`)j3sVcXXn9;TNaz9C`SBAEq%mPC(??IXQO6t;Z<PIn(!z_`E(5$rG1?Nd#yw{x|kjv5k<cH74R#*p1T9yNcjAh}(!^b`65!mIr1p>DG))z@1RKpD26{z|?{kHQ(pL;C@gbPPhdYIEyf<ei3OFhA;E@VLy88e=Kun9)<PMY>}m}qU<dP;}OzX<BHgpra?SW{dYD|{b|WiX-j5*B~J(acvIyhcd$xD@t}L3;K_GiJ>4%m9XBXbiJvYyjma`~vrW}Eoe523%A~v0<{#k9vhK}#P}M_2@{;H^h;31{yrEwtDY8XD8Bi1<xULOllm7rvO9KQH000080000X0D?=fTdEHL0Oc_N03iSX0A+Y^WpZh4Wo~qHFLYsZUv@VyWNBe%Zf|pRX=8IPaCyC3ZFAc;68^4Vf$GDJl%p6sy-!Nixui*2_nJ5pH|^bMG#rS8BsLVO5Tq?N^?$#80YH$V6z8UMo{24yi`~UyU!Db!;c)nUkymArWhxOnRmN&prA02%yi(<@%JGX9GEc-R-K%7>D;0cz*Q=~nd8|Z|%5`4oDvfnG7(CDOqLOA%>Aw}a2KKk9hIfI;)vYR#iz<=Jq)N9+RApUlL{SPU*0NIJ;QNit*J-{6`9O=>6y@RZve=7KeX7$^Y4}jqd5$zmahL0=RC4Rh<Ef}NsRrFMHAo7e2lWKZp;9oTY9lML1r4$T@X6#-Wk#6Vr=T-@-b(!u*{zClt9bxP3<p=%0*`;Xp{W8@RhLVeJaCn|2K@pO7#C%!;wo2K<MdW_mQ7@?@4y6CKZJwfa5xyO%3>>`XjKCf6-6T5?uxPkWt>3`O(H89XHsjJjUjT60?=-k$yg1{o2{%iJghqGa6;z!8@a=;fw%!f;w%PsL|$)q2O+^yJ0mc>P1T)~wM=ta9wIc%U@-Wn6FmajKU6;ZrL5I>K##-~EQkc=uWr)>g-izmep2%^N|UL86{Ck_{pKsIakM||4^vUqyG+ejSs|-H^!!;6J*qPWmBT!dWhoEhPw}?M6}{V)1-K4MB>rM1hTs878lzQ*M3R6*#Nb?!hNS{B;%s;Ty<>%9zC_ckn4MsU$VgnlYVRO4qBKvdC>m*%t;TF2veHV7+!y>oM?#v=vtNq<JS$T<@d>T{#^Rfq_(zuvS;;ILp?8F-UZDnRv>yv_bAiDv+DzPdPe1%AWvZ3<RYD-VD$AlA4eg49`zS~#7`KVZ9w<W~J6zmh!*hQThy(oFU&7x@Fj7mlZ;$<@cn0J+g9Cm%TzpxbTbY3dRxAunR0)pn$p^+HkwQJ?41w#^<_%QfT~d>Gjog4SI9$3#BR^@=pEnOVk`~)qLqI9<<Nh~KA@ctw${Jz82^I84?jYs{19Yl7C(pC%FlvRIp1OdacY|{g2x4SI4k|!^WneKFi^+FhZ1F@%4y+bD|3j699?{@HB#>y-Y*`c;&nQhaP|d9dzW0*5+3)CePE8JKusJ$3a3D8k=dS3x=ZMEvehK=gF;tL-`PEBpEPPUCEA*VY)+>%Z;z;h%`mdVx{@geJdkPZmS1zsATPSl}M3xGaRcz^Di+kfNybOQNS{~EV-YhE*fy!5xIgc_9V=6|Z<yJYriInk*kx8MHFI95Ze>&+8oXwP(Im|8)Ck8+giZ`lD48P3#NBk!)p#J1w8<)g!a>IbVlU+80s@><;sQZFf4{Ghp$(!dl|9bp#^5Xj60*puX9z7<l00^);3BZ*lI9q^I*N9uNcG}g9i#o4DPULTpp5AI;1E48~%nAjxP9WbUT>lB@fC&ZFQ^~~+(-<O=>Ok2bByJyaE59jGjHG~4rnyQ&Az1S`Q?k@fU(P7H5I4`?P-5Xasu-`}JgbcYhr7c(sPt3g3aB~EDzho+uBfvFCYY~ajtPySsf5;gHZt9Mm>G)CW<kUd^ZWg|0By$F3`k)H25CxzbGe8!$ma)T5I5@H&mb*cE|<8cV_Oq3yBK#Ii)XX84(BHHh6nL<atI`hFo=_eLtiQ~+IdpLTV$$7<FV7y>T^!~bq)x+m>|Z$D4wKEnQ5HUK<2~<-=<pQo0;g9FDCTdi)N>@8F@yb)$ons-i-Ha&}R)uQY9n9X?$;VD(Y%iS6D6<y}7zcL%q4$cMRi{Ho3GB5QzyP-9Q)Ml4ZL9Lj*trP%HPa1?xaX!@0MxEqX@55YL$N7Q2N%ohCSDHpv}9k!JLn0(GJ|QuZ^IM}k+#%-53SjnA0tSOnLP;}<^g2zuY5uo+?+LNuL81z?HYf%}R6;Vm1^THpEyG@ZcA*kRg=mr288cv@p_0A^_LW1{hiI;}FTrvE3iTI}qZ#Np{P3(-HJzrfH?t^O}|HsFR08;H-2Wa#f^45z{@{_sS&(P9Z!+|zH^Mo1<B06WY{E>8!(R6|O5A!0ZO^zGtvdVpMk)`p%Z<rCX25Cm=!6?q_S&jzchH^N&uFzpUi8Cv+@vNIrs?C{jKh@x2j4xO<!55&5br37Qbh#DLh3pT(6Bjny8$}KE^bPJu<D#hYycz;bV(z{d80)&n|Av7=-*64;M1IVufhh8!9YDll3!PyLZb#vok0ouJv6vjCW2XJUP3d7M!ZGeHUx1-*|2J8Bu7K#AG<pcDrWVu!bi4}2NmjGjBjA|B*Mkci8gKh>ot-#6B{7q{H#BuSZXaz-wE#F5EYdj110HOEIqwOvq5MoVfyR1x-#$Q(w;vYh44f4O*XZ^v`qn@Yb!4f(QRM!fg3T&Hw?~XZI!4Z`SfUkCFirbDLi+7KNgdii`jn0A1R^ElBdHww7n^y)-*fz>{cPqRljN78$(d8@-{r>v<Yp>8W{jLiM-?A--!sg@a_7~1}C$CY?gyrD&Ra|5l7#Dh@wAw%p^fnLsb?b*@eF5kj4_mD?$7~A=WTbm&%)Od9Xy6MCIyIL#Pg`kcVwb-Y(~q240w?850ZHszwIxY<;*pt!{eju_%2r@{q3ii4kQ_a02eQ7CyNLR09bqN=>i9Lr#a);$?nxLq3e6KGDlQWE3+u4mRUFH(SmaIrW87i~=IcXE>0jyUAM%fRahJD7(xfJ6$5XlA`kzCgJZ9#r8m@7tu1C|LDGD@d+#tD=b9g|FXtbV~gDUu)Y2QW9tQLZ-({DW<km1OsBVr$|Rjx{Q#Td`3)PorrZyx@JIfXP9tTFAD=S6eg#f>ix=d&!kWg>xUQtZfTBnJHSsW*%j(&ELd*Vpe~nIjHND!7{@!P{1J_*S)!`oa#BbL=A81?ubunn3HKE}O=I0C!>6MVYE&qD<{m=(0I7ke}P2YYq`7gi{l5Rv<g3GsMU60RZY#A-BtPT^F@BmTz{->~57^G_%*yr=8go!Db9Rv^HVE^y9$akKnyKe>1{b!7Xl*;%T3l*(rvoeyJ731QCZ>ai_|W=ld}%dzyegxYyM#vY(mXJ!S>N+B>G;W(D=M(qc(RpHWprcSV_K7CD|@F2;l7_Q`BQHS)&@F)!P_DjO!OcSzyWp6PFm5p9FZN|>$O(in*Oiodxf@)M31?MxRrH-N~oTS)ZA?~9Dws3;@n8fJ-Ln0edZ6*d0Q2(X&@B23fNOK&FEA_s_bg4(&}L>$k=)2z-l^jkEhmya>zMD(faVkR<n>}dqa2aM|CvDtYfg?vxpN3BH{?f!hoVfjE2PFrN|Y|+QoHLM++C>m=#g~fHTEij)|@RZUWpZX&{VmMlL(%kPl!marzE7obHN2|2d71VpksyU>CM}er+c88h1Zcct7NWzP;OIq(#If2G$3l(1cq;e24x!IsPIRq_)|5}-<_V)QNAKpLzc>=O<m$(Ad_4?hnPh7S#=j`C-@#g43nCIcD&MA@O#+??g2R>qGpXhp8(;m0?9g)Pv*)46p!GOiQHO`iwU7^9gx2m}t;-O)*A0JC%x5sug-|_wfy{3cHXJ2ShQ*20<H6MeJXy5``FfVhk42HNZV!5m{2^B)_Q@g>Q?GS)LvRy~ISXEoO=cNs41xM(G-qa9*ZD*59GxpQbSTm1@4Bo<EUNB&Q*m%DiWjiQ(pT-pG6}=ceF=ritKxx@7M7lAO@h5uwk*2}i?lw@fM?3MIxB$d29tq$I46`i^VhTEpjk5G3kd4P7?j!h`eWZE$%sw%*PsHrgNoJpVn3+V)v~dQ`KN_L7e@8KBpeF(JclqqX9!`^=M0|%Ra(9Th+;Qwh%R_|sO&s)Y0d6}oIi_QX?l1-(TpO@{Dt3b&=<%*B5L#v6QFQ4AXc3=R&~RkHNJ82#CjbLt-n_x9PzGPE(yXGZ1?F7v7sP6&P7^Rb(D|Q&&a%#h!h;r0V1)PRI#kRzv9xGlB_F%JGncMNlj?wMw=hf+BB~0^f$@eKNFAq0gAEd%LZ;^a18xh-GSvXLXzLk)gdbUz{0OO$PAA(tRYns~-JlBHIS716Zn7s&c#1c?et-1?9ui)^_-XPZet1E@U;i-qFueCunH%+tVOZNg+?idM?AS%l9-X;sTv#9IwA5TILmyZbb!i=e4@1+^E_-u~#}5d&&8frd(P56i7JR`5Khhl=F5PwlV|VF6<_E(9v!`W0zgW<4cLT3^b2(i+e9Pv_ySjIjU27Qm1&0UN(Seareb<zQXPqFP^J*_lGQK#<sCMK-a;iHV{Cw}gI0^}x%ZP#NYPi%kE$gW9x0?v#OzkWD^3r|r0Mp3$<W0o;j7(@gx9G@giE^3MCDYL-L$zpDh1H)sT^BHu4(#Y$(_+&EU7M08wGhe7``m-@jpW8O3#v62km}p3(URXM_53n*y@JIKJ<1_U?ka2Ps13yn5xoDmX!SwE*6mXhLB^6#yLT)Dv4QBVV=y0OSd47BOfyKG!SVe|_U4`wBvZ3Y5JVgV(QZ>pt#HXAkFQ_Cuy{f`?VLwoK{^X12Mw0Iskg|>QyhUTWbIeKzIy*3!78v6s!)}7Jr!HX)F9Sp9)@911=9tGxanIPCFiIZoi$+vf-WCyhC8z6$_N3j#B8I%ONeu`444yrU9^}_t3`;lPYEnQz>!4CxT>ihVd*pWv@smhwgps3B`cPSw^a7kgs11{C|gdAOgtW>dV8uMeHD7{Vq;rhP9y4bWu*@>n=6oHH3Oc#iy*XA42tcA8bD9q-qW>Ceo3HrK6$dRcSGC~x@y>X-b2g5TRh8~8?WYN3<ymxkmr1f&?4x4jT6C5s2t`Bpag)%I#<>sdiL+Jtns2gWApNt?`EC(U2>h{w`tzY>8!=y`9T2ztf7o~ZNbpLjmXYW!CYXZV{h*+THKD;qZ4(T#%g91<ZnJF_#FaTK*dLoMsrXdQM9*~6Y*q>fWEa_fw&waz?c5P|D%=_reFpjJI+nbfOhgWqaRR6@h4n^TdQ(3xeRa)-U|)+O8ucmEoub*_JPIn%_fN14WgV_BmjDN>v7Zq2ln_aXr6E@9zZ_U&GDT5$}hd7c)eO>Y2NxT1wDYZd$jtP`53R%CNW;+kl%T00E>amC`UOLa@pHLtV=xPp74K9fU;ARTYFjNU~k!>&IH9bMWJzjmHJY``tE-@K-gz#42XRL4A&d;UlU9|YNs)~*e%n5Gx@f`PIA;WFFfnmcRctnP)h>@6aWAK2mk;8Apkb;zr%<j005O`001BW003opaAk67ZDnqBb1!sZb6<8hFJ*RNY;|FDX>V>WaCy~TdvoK)k^di`0%58WQcF<}dtdS?ckA^!*^0fp<+W3ll6fo;7?Q9+42*{;uA-{@8u#_?liaVnXMn+j(mJWSs@o(YFpr*|o_^0@G8&CuRZ(09%S8FF|MKeX4=1nRzkYvydiH$ke|Z1ljUS|up9OKQB0t^+YjqqZRZ*(kFUp{di!u(2ndfm=f2ET|?+X9BxCmaQsXF;fkR(+Yr)a6uld6o9;w0Pv+AP~M<R~bElVYFhbiXZT<!yQ4ZOSq$E>2F?ak;6MvrunOhT-`$-~V&2D}X8Us@(W-;g=ib|Mc$d`**%xd2tFH(@^<gP^z`gV^#2=dijeAOMk0U0#o>L>I1HytCh-?w9mK(1H6{5doR>CD#!f&rNSKjNG0(S1*J;%epLYS+jk#-eD&@%Q7qMEAfR=c$V7qLyZ1l(ufG5K%@5w|_kVly0|5uATg`)90UJy?OH`>Y{J=Ma4U$4xgb7yD>Da5%qRKLzmnxe2C`PX=*LnqF1b9h(>Q?T#+p@q5O_&H3gk=>Zezwu2Ugtr!!9xCs@={-`H2#D&Cak~v@vr_)=h4Czf93zEZp*I|z5HY0e+Wu2LwX#kJiY;s6mhz)k|0Mr4_Ms9k&;#UurD{j_6wgd^%*<mV@=jr&BTInyXPR#hakKrnuCH~t_oe{SRJfG!PMG<5wRRtxDfbZ@a3MN<`sB_y1&zYP?dTMLWO?aU*}#!K0ngCG|@p+_?sZhRO;_GF=U|tNZ<mBuoPI%9$?N_I{pTX0sf|zCri-fWE;fkiMk1r3JgNLu5Ll>WbdtX?k_Rpe2+C!Q5;%6ds`NMl~x5WNRSsG{x--8UtzX;tZI6ksx>gaq3ISGIw`Nz{@>qxoD#<*5aM)=E(u21f)ut9!G7fz3NV2Rgf_>R=mFefw4$<YaIFfj++eYnY7^YV!0{$7Vp0h^%u`pT51~!hGy<>MY88hu5Dmhx%7bu!^6lBo8;wSux61X_Uo2KtS><Z6fXd+Ic6i*h2|-X58u^n(ky-^+QW8<5&8kYnQtPB{k>u4P48o1_Ou;57HjqR6`b(ix>6!&4RoISTuZ?Y#D``^hGn$YoeGLsjN^pvmfLQa>_h1M#nff2quN5f<&%#KnEh(_VPczRbzs1M*Km7pdzw}3+t-(gjTo3pB9QGO+6kh%G&D$T}d}9DEKK|v^`O{~BJNA718!b<tuAZNsh3c6KPtVSuM$z-r)75JE;#5UXUo2PQ>ipS@XVL0K_;MLNJEykKUWUt;=Px4l<+JnWBX80j=<n~|zx(_5ANxZ+d-3$?7tfbpJX@*rv*q)bXW@(IXHU<TFP{Z3@uHSbU!1FGwK{vb`XUO>Prr=9@Okk3<>}L>r{}A48p``URt20NhhVi-M@D-aYvJc4kFt`kCJ}|%vnw+|`v9R(IV32Cutimt#AFyqJ(6iSf#8>rngA-|*DB7hi?6_?d2YFMr>i7F35-fyR;2>Kpb;#y8YDWRo~9&R8;CKqa7tPjGG~MeOa#rSNS)8Th2b7UN#cUhC<6=vv5xa(>M!+v3Ms|=PV>)o3{4+GPw|ghp$TL-pZb~78F6k9L6Cp}v~O-&!m?4b<({DA(mE($4%YNo#AMUaAeI^p+9)0?)Y()do+^?c+Edw%aaz&<z!KJGOBmA7D<!|{3JuCUxKRmp+-f2Qsss65WAcfNki})lWmyvJ8JX*te^MGI$RjBW%KEo1wRAFc;W42ARlH8?iMPfAD=|8lkV=I(+LEIrok=8N(v&fJ2_;I>cIaH_Bx1~9EYrvUCyuhT+64(iieNhAK|_HtS;KvpfW{bQ6Z3*qJk?UEG>{c2tB6MogKcJj+;9ajU`#hvDkRwGj9c8q&{^p033w&TXGg1FtJJ=4<205saQwCM!T?a%Lh5Bq4Q^-_)QTSp74c7XgTVtV(h}yF5L{0EUzEy-cvVKwg4>u&lX$D;)x(18+>8rjsu^{kOATJ~Y*C{ZDX&{^b*3T@-Xa(yRWxRq#HryFDnONe-0D;}rC}bdO77tHLdCKwh%BaMp#pyLxR|L!4`oiqY43r1W6VF%(8HQcFEjLZ7wnk`T9XaVE22pjhu2j`ZS#Cj)T)@xR`8p-g$I@`Y3X8WB<K`DoJg@uo*&7KZ}!A@J7tQBOsPm7L+C@r0<r=R?O_U0cpdBmUY0W0LK%@n=@fPo53t>ZlCC!joDF4?vrv^hm7O4|fU}hCNXQKF%4%)K4>thD+9Al-%5<pl&m$6-C`!ygNxU*oD-y(w@NB7qt&}ob%i|JWU!sI4+=3p7?4)i0Xid|D%c4|i-)v<85f2io)mgh`{Q`cVM~@c4-|Qoh2Ic0#<BkNyg9g85=Ls%1WC@0TLz-m=v7)(Y_L)$mcL}o$98^54Y5B2UvizX?wIXhVryjEMheXjzC^9msq3nu5K-clwR9kS|S_avY`#}$@ScZ?+iW)&7<ybKu*P;W*k#uhYVU!{P3$>6dDcETpWh!J9q}Dm%1#JaaI#NLS*9r?q04md*X1|IH*1^l_(<eq`2jxN?WwynRCP7_(A?{x#0pYGm1LRvRlx0+j0hsmHx=7f0;+e}p*+vBgw}r(JgM-&7i&>q8JW;Y3DRCc^GDd=3*Q6iF<jCBDd?^+sSZA>!5(lAXy@bD{wi6b^Sk@z}z%GDy1A_>clv<^c=I&vz<K99(>8&*>Tv-*UN0P1>kx5ESPhv~TZMocgkzNz*ETfeGOAv|~DFD!UQw8`h%S{?opIDbeQVZzsh-O}vI~DU3z!?k^7I4XACa;+1RISN2<asRCSVuGf9Mn8u<T3o3+C>^MD4U*bRhCk|XRb+^4g{4^#p0@rpD;FS39)_oI+m%6SyTIs7AzI{L~MUx<WnBrRKZPx`5{TFitD`xt}B?V30uo_PfHjoQ;Yx{Ga`O~3(a%I;G{w{32zo88hoUu9Lfd6En%{UqZhFYv4ot2=0|%x1tP+BNkeSy<C;~8+QFO=|Aeelz&N+GYWkLK{;Db8Af6}{vBX9}Q!&f0AhlYkHT6HKWkoiEVhQ*z*@m<Q)|kD5GzvU4$O8|q88i`02gERAyLg=<V8fGFZI`)%JbUOGEd!PT5vHtSs(_?w6$WHJXcd=)H~#o|^NI@YP{6dqlP`}+wO+&0BXEg}r9z|YxL`Bj)lQOEP-KD-E}%)!Uk|KH_`w2DdIQR~d08iLK)uG9!m#<uDD#B)Umh{S);?lIOY^*gvXgEQjTCVsR{WagC%-UuL~Cz{1;<Emsl2etDE7Vfu2T@57=;>|7yJ-1VDl7RMHtgEAS(%sQ~bw6BnE(_EOxAmmsKuaQ?+Cl7>>Oy+JGzg(IdSCqJe)xuxCu}4x|qpTauu7SqxPNNlq)4oNB${CoGss(3V^g16p1%2YFypm@;xCO+C1S9z0HPvBnlNqybG=gms};0m)WqJOG5rnjH)nXKAgVY!uIk)Z+?59C;b2iHM#&mC~4Weg;!U9{dFk%OzqpKaiknuhcQ~T$RUcDdEIyHM#4yA_;CNU}0yYEaC3lup1S#$6A8*Z(???%_F4POM<M_?DuW6iuE_*da!~9`=_duheW0NMze-B1>CFd1lJH#@NxyWM|sfl!b%}ThH_Ej&5=K9-^>p+Fpnh*W?c!wL9LXgB>q6Z-ZmzK0&C)FwkTMM8^zf3llk!pGim^O1w?bNfbS0EAZlyM?-4QZ9I@k5|0?x~m=n|ULimT*-VI4}5ynLU#Rsv7qP;IlgP-ZqBfcQFUbqw%CPxf_aS<9V^Qzcz1_kgSF#c<5@STERt<wvMsYj#t?I73WF(q9@X`o1D6xF1hU!%oNjuHECmNKIF6E#@x8IMCCO{hvSCo}D_jiQgYlKLo)OLbdP&fz#SiN2dN(>YQrf3X1jm5YT${qzq$CY0^bv`Hdf)VUS^AO1V642FBh3*{#J4z*&ylZzUDyy~PU(Wa@Tf1DCXlk(c8c3R0xuhq5zb=hg=jOU{Hz>t<wGnLNbbVb=Mi-DUIZFOvczHGH~2|9D}te@LW4uk46Bt8Jcpv~QVT|;?ENDF0#I|+$d0XLc>sA4>6Pn%L6ssJGLoROEDjVIE4JevBWNqef!*t05&z?0)UqWOh^b9FjrG^mX5kkb1}8~d=%2wf7{KdBTtbKC+h9mkWVXST_Mf-${n;WTQ((b06MhJ({?RULJ9D_Vg|x6%z`4t12Err*<sINcgIvKj0}0J{~faHrSH`W(HA#=GcNxgN&BI(9gE^LD+5Ze`;=bZd+O?-n-(q`#PrIqBnIRdsKeT3z=@=vd_5jKz=Z7I$pmz-l?Rs6XMYqS4ABmZQxaqM%bXZ_7->{&(_tVI+d(%{&5(Y1h^Yh$Il4QgugmC`9TlL7xK#7K&qo9tFs~@4?f%Zjp7#dvqUILH5+Ug~oC3Nj1;2hd{M->3Y#uX@+REN?VO6$?D$;y!+9Gp&-155vOJQRb3s|0ofW8@awd#e|Mu0)+dTUnMYnT`%H^A`h`TxaLQ=jEbea^wC3sy03znb4||d9<#H*%QbjGdRO|?`J2F51N0w^@{_s4lI{}pxQ0AWXxnpeDzKShP;^oXXv*y*fF^y?O^jhVqf7e;z2r{Vhg=r4-c;Yr+-(g4ID?}T2bEnHPv~#Yah4q>i;)yMaO>qA7S>1PVSLQKLcJJaKu=dVN2VAOi@D*%-GyuqKn$4u3z0&k=+})CR;*Wk#M-w<UA$-bl+pbS$8+9AOS1!u&q)S9Qw#7$wy)M`q#+k2zaKmm#xlyVAiw3{ahNJ)8x6YpCiap-%zU_)8xp>U?ynJJMpY*i5IAw0bx~$uH&4&pVe3*cc96CQBhYRYX0>|d4fbqqPz5<g|1B)UAue(5x4bXk>ynqAa-bhNl@j2cGw*}Vq^4Zg;Up$-Cl4H<-|GFUmspvZ7rv8F*q~cQh$A9w63Q_fy{6)popUr0Tsehm#rtovHg`*dFbaJ$MPU)DcCM_4Q?c0tGlCy+z-|*iWGqzO-=%cCMNSZEUp%{h&LRU@O7)7TIKD?VB`~OB^DWgzyyjWxZ5Ci$68S$XBx{Cb^^n2o;&)W>&$6>|?2cmJ@ZivvKv8}F-z37RY%+JDc+g{0U2M3wwlS}`+RjiUiJqFxN?hwGUy7>2gC)_MD#k>BY^&I|4%HJO45CibXnHxD-t3U)0EO-ZZ#73hjY@-b2O{t%#=dH7DBJ?R?QS|Y*oW<1y{SyL`!<;iffR%9O?=B6Btrl~yPgKH@yEJ1jXk<Hw`6HY|{Rvs>zNqo;_ofprYZ8(^RSr?)U!sPkgk@Y4S`9du#?-d02&;$jFk$ROI?7kNxPvia=f4ra5HTO@AQW+`CQWu@72onP7U<pZT6_}Hnsh+L@lB)=d)IB<|H^Dn^EQ0p2X<c;!A5}+6u>v0!6uB&em{9nrVTxk=iD_>KkhX3DiGO~4V+fsDG=H1?uMjYXs>(|O2O_BI!-Q8pwy8TuyHB}0I}0cApv1l_$j<`I)_%fs_;Suq=XF|MN1Ifht)WZtS$n-iR=W18ydFazCNrbyc5K<9DVC+E`jVugSGk^n60?O2E9&fXCR52EOi4Qi0a%u<&zN&FIalFSKkTBf;ce@=U6>rdSgGQzPg2~mYiSBpELNE+g=JsFZo0asTGr6>G%<*U~+Xf?`nv;FZ87yv2T4Wy4L>m8c2X)#sLNDS&S@D6xv6W+F?;@AMblWC$1T?D0J*h5W|lBjgm!g@?7WR)ky4|Ieupd=WpX;ONSpW5PKOw_fBqWIM!gsCltx+uuCxMMaw3awP+kn)1{s3>U2Ch;%XZPwcJIUKGvb;6!IH;E~!o;f2V_p=$XV}Tqb)z4d}d}J@tEnKVP<7(yB1${^H!87=8N<o%tnr?=<KHgaPB2ay@)J3vKDZB_9_CebQ>XR5_ouD@Y~E2}QV~n}ALg(|{)ojK+6PqeA+VTm_IvZb^sk049`EvxI{0e;*fx9O9#)P)3J42~_P|+e9`q0oq~Ym=Eou_Yb`TzkC39OS^oWD7PegG~wNFoAK@!M3v&AJ~>Ffvi}c0hd6IqTWR~O#z)R5G~GoUp^rVSNZ$FplVGpgCuSd^wW~1HxSeRHv&8KQfxKMWM*Ly1>dB@Nvz&H>Nslcxo!z#v)~k2l41`w>L$<xLP8P$=6qB+~z}TJjxYa?l<YmWwVX|QUHJNSJpB-%f4x}F2Itgg(8!6nn*TGavP18o~FW9S)5UWrnalm{I#GEs+&<SNX8BP=X*Py6$VbP%-WqPlLfouW=_43mmx>=Bgq)G-Hpc*ig8sRb~jvi-|8L0~)T*ee@@z}hlB?X;FfV5b0K4>_pX~4YJmDARUoPv;1@MkhZ;GT{-_W?z6IjYLl@r%)fPXE&WelieoadPUCk0a0MZ24FkwtZ|TsnFVrX{@<XVL+DvUX2_pG4Bsb^Z+E=?P^2?qxn#j!%aoHTVi8f?~6erJzy|@N=d&O^U+{@M+5vWzaRButY!|xrsD?NI2z3-bL)q65V|6O^B##r?br0-*FoCU@_Fgk!fS>cqED?EtlW1*+}01%sV!8qAc`7oaiGJnQn<3+`ZISvV5q0>>d41w(6ly$TO1@ud+P4wF8;~+Fi{5k4)BPJspS_zc0LGV;`MlB!3U-rfotO09+Ja;@}RnP=3}zvP2FTr?L1&N|BtkeA*PBJA}u|wEndt){`frZYeBsCdeTvUU^TScFo@=mRI)YhwfsZf@PMQ6|3`)%XSjnh)Y788sIDzi5_!#gF<zf^#%Ggo(VfKM_316fR(y8X?XX3T=P~Z&1kJd|U6<v-W)|Mm8;lEkn2?_D;__R)+ttVhB@TZfGP$UIj^gsJ;IH+@Jp?9K7iXvQ`#N9WZ)z({%dRsl+XWm0B0Svas7Yq!db@DEwkURx4-<z*s8RIU!k%{vOdv8ChwxbsX?5w35NT2bH~Qz}l%u2TU68Mfp<sl^vQEpLY-h@F&d7dU{$*4C<>>yAMI8OailmN;{rZDSacp?(lRagH99hu7<x*63O|o8I0F-U>$QAcH0iV>XzPRH7?q{UBM;-sSX>XVQ!R9r|FYBjn^1!8WC*w(9JxUgKT0<hnE!#UtQ$UF$`K)vxB|7A7wVWSUAG&U6FP<4#w!f;q^*-OS`*&wS-TOQGA`kG$`f>KF)INXLSbv@6tRHK_seD+**<wkAQ=a+q5%p=4Vl}LFStrmAs%)=k+iaiDbN%YimAg9tefvG4)4Tz1Uvuib-@y7&Xh8%RXu;cZ_5J|b;c=ju8wXmUF+gcq*X8s9U!ckDGZOAj#evnC3~WKj$I+l=wH@=ghNZA5E6!I4Dh#UW&pWeb4a%`p!gr__=ivlj!~dOVwRQt88qaeuo^>*vUOziR(Jsl3%<WJ9>13!?j8@X(gtLZb7M!}FEeAHhr?b;eU)if(ScWnwN9o)<UsS2#hOn_bboWRamAO5J8X4O6)1Ibc(mR`DeY{?ex~iXCHIahdg6|ArNY?ZxvNfW~q-$FzIme~ab-58eU?Apdwu;jzNRn|r`gwWv^KSlR{O=c6fBc{S`tRd8zfO*RUXE%VI_cu-kjp-HXTxe`UA75ee1mn_1z2m813eB>tyYxXQ!-W;_D;)~BMNLetam~}MemJau^%gU&n3vG`>uarKDwUsA!4g}uhqVGGpzhH#~9kyhez)3lgfDnwr@QxCRn&W{Pa*82cv;TM@M&q9yH_6=TZl&Nlo$BPTa)qTIP|BMF%^9UD2oUh-&0pWb1E@+)p2gi)8Eui{kLMZzthr+7r+4B6Rf*31X7axwrK?r|S%Aj)BIYuk4DkbnFX=Y;~aP-2l;B9NOO&88TKJfm`P8Co?-%tG&fS41IV8HA~m&e9O`uW;Gdh+ul3up1#wTtE~_Cl^<!-+gz~Vt8G^9>Fx{45zsaZbi8qQBp;qy)TGeyC&+SI8R!~^`7loi4A=Bp038ePVE_736<1m<puPrgZ+9N%Q&9E+AQYbx>FUo51Z>MKbLKN>d>))tPdG+RC`!K3)U7Z{OED#PeN8kx(7huejmO}kZ3+(|3iFqCLAdTjb2-kcc^)u|1I|R>gmriVj&vC$eczzxZMgGk*L~)+hP{zaqajaWg%AQ>VFqmy7@q+Z6Lu16rX3<j4Gb7T*Dd%fq|Q_Z+@A;eiT<4oc&4ty;<;zQx=3}L78jRR)n`)mlgjnNth?2I^o@}L?G2gMzFN7w^iLm=V|V(k73-a<<S@NHTIpn7>OBcrkFQwr`4%RhZ(%ao3ai%b)iwxxGPP)~=z+&n?C2{cI3cXY@muexzoE;0np<&JabA?ibkP!N{}cM|E;y0fdw>vMt+DsVe2NZ@D6qLv6aFZRk0q&F3MOSd`!&rBM*VDZbvmEr^zC1$#-k}EIE;R`hc2nF?P9;bG0JF^>X9(aer%6+NjLXECYc(73nIzS2rSPWHQIwq4UxpxBMObZW{rm|o)Fh$jATkJUrvh58YIC|C9P^MF|{{A9tUY@b5gDR(LZfi`EDoEPOg)pj!PxRbYzWGm6n_+AC(!H?Ud3?sd-r0k`<STyJh$WBEF~S)!w#rNe$&_>LA}YnDWHQ4wbo;)S5c;i9I0eDgx<IJA<@ZvVixGo_cv}&|_k(-3!(37;1YXPnL{6`VzdFRh{?F4ir`*jri~t4U^w|Uce+&Pb?kHH;p7sy(_?i!XEdLyU^efLRXMgFSXAX@i507aDf|v#{?hI1-)Lf+;oW61#@Xcr`cPPNjxD6&iG^z!p8snxi~`1h?f6I^@M{(OER>2`{#G>e|Yoy)yFpzn_Tyt(1OzCqmsqDJ+d%`$nR=2@t<_k>Z5F+6QgT7rK15|Eb?$gE8d(KF0-j$(lrVAT@m%hZZi|$h>C#_i3ExGi0qiJ^?*tkn{QIf3~GOjzfS;_Ew6d5^BPwB&<AGi_Nmi8?Y8fDXGceZs_T~3p}iPDr^?=WMrZXHewo@_!a(bKK9oSS&rZD{k4kxPU6(VIp8H1*NsjDXG0mp*i79QVgMM|}L1-m~XvM7oZ}alOc^NLYy)08d+8*YE4p5n|KvIqN0e|IDr`|E_NvaXw#KAh%biX8j6fte=tGPeVob2+elrN`K?#F1&=R3{?YfiN>CiGQ^Jhi<5pt?>)>_rMq4&S^C89I@*;i{wQ1GZ0$7u^kYMTDQ%Ud6X`V2xCq(SG=4yr9Z3g4xEVTXP~h$oeVHj-j+u_&dD!M1f%KQ;>a7R&T+b`<;Y?g7V~118Gz~omSljbY%l{uK0}wmT9$8dF^Wy^nqJ_FjaP0MU9CrGTB{&iKUwRsh6uMFwcU$OU`OEK~cwqm`q5Vx|o_08J6#jgDsVM)=5kSD&rPX)ZdDI+&amgAJgfM@-6OpQD-<9QU9he7S)!Fyoa`0iI#(>X77%U7z}O0Sn?WQ=|U!aWnSmgn$Io_aa@DlJ)ve)qYQ0iVL_L_(#|6~7|Fh{HzOK?^}Ydc+!l9Vd^*rzwpMUf3#B6QSLhJ;0}b08(W#MRA=4&BII*B6W-@Y-?ZRAPyI5*nVuT<Q)ad{#edxY;%%KgzPciMkFFw=tmE2G*`X<_*8tP>#D0zRFMCbhU)Tm~-E-Tt)2$hLP$$^}`d@8j>GTZ00=K@3WKggxMnSvrj&_(zAzBLES4EA*6Syj~2CVCdZHlu5s{I}n~`tzHQ=9KR%*c=*9&f{C{!_w_@4|_Q<Akf{Ea=|PRHiV|DZh6Y0jS5~kJ}l1;Ti~-d$Cp-^D{hTN_EWlpNZL>&PEE`}z`u|Kx^}-E=ror)3R)+G^$D{Eue}`bRV2xU_=oD2aOATn^=aQag~SWv|C@hydTLL;p5nDx7^-_Pai{nyR)+JZO9Dd#bo!H;JDr|Qi~==|bC(gDW4B$tdtmIZ=~%6~x!Hso?xm~Nb0(2<f}MsL8aAQhiBo$-kyc%-D_u8}L-u$QH6r6K^3xioBiG~JRGa}Pz5V}oxFo~3m9@CR1IlW0z*GqCpP}(rqi<@iGUOEIVo0p>JLen)w0OQYPV(xbg=I}4F+r+qi1CmuZhP+E^XU#c40fZ85$2nnNjO}t2NK0seD`sJhV{22OGVdY))#xH=Dc;+-EU4nU)rrBny|g9!;sR>=icp9>IGd$b4%RACG<5VAIO_F=Qb6e>8E|6dZfvHvui|30_+J*U>CD>Bt*0-$1FB~+5M7G%-L{adU`lHbzqbO$-RdL9m@4J0}tnx9~-^SQ?fQOu%zL9_<|aP6nXu19Mm4bW)24Tq95H_eQ1;P!5!s?caNQImq)W0KQ*_Y`?X!Sv%8bub|#~}Qigt0{|xjIZ$Hn2ebM%28x6`@Z$Mi*EXn}TKh0*8GA+hq^12&NpZe$cUsUchb%{1?<DY#w@p{{Wjp}R0A3mqbLteEd@Jb|h&e3{bq}u4#-)ZriR2J2i$-qT3O7RA_adCNS4ukS6$@%8~bFWTK@5iKHuFjlXFBPx~2grw%9jzzFidapvc=s1RQ4xwErAq^y(>&x7ZpBa$TntDG>ZE=F0>|=l>eigj?SaP}-UBZa-RrBjGxNQ~0_9`UrS?1fqzoK>_`Ux%<;rMRZd0Y#@GWk1Y|TAN<;MGZr;u+6pf3(}Mfcbh^irRLZA>eS2TLN~T?gO5x({H?^B1oE1HZ{v7r=4`KTxgfIMx0;<;&Pw9LJXjj6uIKz}3KsJ~Z&8+hH=0|DhZAD8XZRP4JjEd~r6&7fpYg3>`18E-<S(A8+yh0H45lZ%!jJ)o1FqB(6jo?hZ9Z>sY_>5-(eI%cgEJD>X&f_D(3{W(nwrd?<2!8)XvZ!OmFRs^(6kVH~b33|rEblABl<>(?q^AABrhPW`L#88B`<LCPVW16LFd)yxm=EXwe_>qZYwj)O;9G=3I4`wOeF>OC$iJZ`ooj2DUC)lCxJsocctO<lW*TeUUbbik>8aaz>$To~r#?3eJko1x)!5mDUa;)HUO{^qF-=WIiV-QBh(_G`{4P}>35bksyaiz<!zb{huy75~44nCRz3;vxH~ek=E+_y`>RG-)L>dc#bY2ZK@Xe*sWS0|XQR000O8001EX{*FwaR|o(Agcbk*9smFUWq5F9a%pX4Zgg`mbYXK}b~i6&cywiQZeeULaCwbbOLOG55x(nJAk2jmbE%abCytysY!p>_tJb?)UON|AVL>D~f<+QE07@FG^xyq89()W(_UfP^fkvaBUw1P}lFP@wk)^bZSG>1E8&)gLR9$0{H%vTYt;55wgV1}{@=n&mSi^W%u{YoU=kf<;N~LAD%abHIIjOa3Sy9x39keJ4Cfi<VFz-5Lxs|FjCnsU8RGkI;M&87gdgw~4RAb_T{{cllSlNuLDn2RWFYLL!1y#K5BVHqk7M{fJdtoBMC;T?zJ#U5Sc`1U}mYYHkot3S~xwf+Ar5%sF2@986E1fZJRV&Nl7o)n2nOpwl*Z(ML*@!i#id!C;G>TV6G^sn>Wb9sOS?`Nz+CsK`Ct#+NO4QD{qTua~&}pyLFQT-CR?22j3|p1SScOBfbH>fs-)O_AL6^d8->Xh+&QBQr;G7HIG|b)#_*Bw~9TWGw8K8+~57OQ;32ouZog{z1fA^6Y0imocaY@O`TT$gFj+yMFO*goRR|o{QYt#)+aJ_@II(J1QcD&p(FG$Wl4SnD2=}LRcEd=l}=a_ZT3#^upa1kB%=HfRE9=m&0I^~>@YQXVII`MBrSKhW<-yxJLkw-nfh~D7Dc<+TKtFf(%jfEvQIT@_E7wN{)&CD6eXG?!Tows+DgqOG$oi*DZ^&m2D>q6bRSAS&mPh>wWz!Of)Q<k{gI;G%A4)3IB$Jx1{IN)OBmF)V#<`h}M4i#N<rPae#BCF)uWjEf)Q6Yk^Jz))}bt&jK>@QpQY=m|18<LB6`_vKAA0-pEpr@>*&>JQVLxO{;C0cqsWB=Z=iG<$HUcdgkyMML}F>r-r93s`3h)W}uDK}m2-0Y!eCpw{FfqM~Ah;>3hWduG$aLbCU-2-(5>%|P#AgI`3;q8C(WMu5EJIn0pPBM8mzFjS3c_PD}v(dyEOX8(*xdgPrP0981;^fR12GJ_2bWG|yvEMduX$9=CqCI#2J^#(f<Jybl7kmuC*~<u$T`drh$ijZ_RW-dRLAJ!!oK_10F9%~q6Ot}X3Ia11Ttzfrl7-v+&8@12Myy%-b<dR8r}S?#%8(Np3Z08b-0W#n|Kn-?$}R|WkV1%oX77Lj1>%->J5f<)U+h^gd(lWZo5~U4SI?ye9JI(n2@jflCSl$+J)|6~Oy20BaKM(f{L#qY#P*xBe|@=0C-S+1IS9+U!t6WM?ORhkz*B$1?(D0x*Z%~=(PJ-jnt3u1IXvAS)~bmYohrC73N9g6ZH?74EIFkxY&<eKJlISDY<Ml)t!8~_+RN>uou6uj)hGt*BVpr^ek%IwBeRBJCEXaZ1;L*?OZhjKU$(M=hwZ*_1#<g3|7O|^WevOw0;+AYOQub$`r>Z;)$4ic^jh3YHAFzp%ZDmJEy@k>ltNqOpDY2)+Ye#O>iq;*%D|_D{_|2Xd5Ws*)b0SzK}z+<*a|nK{+1d3&7z>?vP)+X;knjSqnl#_L|vkgyfdYiJ?Y@D$#=q%fnqgDkJgcx>fWdM18gk{l0ThThG_N92mdFJZiH%pFG966x%lqQ+aG`z$%R6nLl56M{J{GznQ7c|TT)TeA~%BT5>>HI{{I9%&*wjzFVd@@tNi+l4Ngw?XRL1c&TMh!`=8$Z`1ak~_m`8=QP4b7)Fx_#me&2#3gfEGhgdIsxtaQ<3YCSb92$vg0P0d}r3HFq0~C)YnU$6))@CNVf_B%0AOJ#}I$`sjRzshjZN^q%XvlR><Jp*O1?pLeGKvxtR;CzFN;SA0s^X!f%(5pL^(cJj3d&gvn+9y39T-!0I`Fj6jheV6lX<q&?q*f_Xw?PD+U`r>%GLx2Aa#Udj!8$hSNM~?;H&4E8C*A6AdU?c^K;)_=0#j7iaBkmdaD1)*mIx`{w6wh*tvWgo8{1yPHk=GLhJ_htqS5juR4^Y$*Tp`;h<Ltq8Lz^%et@>Mhpxp<4lqoJ5>9OLh4o_XnfMRnI@lLb?mO)aMD9j-(myeYPmzWIxe|?jol^b;#cx<0*W(FY#I;u-6-U`^uylpe}rJi8pF#tuo{cAp&eXbNd3Z*YjSq+@h7SW)Wx6r$qbE$GBlku2<BR9V^=-nI3;h7jd<xxYGZaofkT_U4HV%u_)0wxs>V}S2`wbot6}D4qiwL*L({=vdOVyWj*p!wU6MKe%i36kE4^!xdUsa|iR;#bR4?p>o^)L`$8NWn#Z1#63FN%_o<b`cC}UpC5T+RwOU%0|aDkd7sxLbY&+n7Xfu~l5^JSSAnmc+2&IgrlIZI7^JuQlvMUHV~+ftv@%>`b58X4THfym=aL<rytX@|Z)V^`OcIyvuq(N)AK<~Ajz8}~e!Z!(nLkJjx8s8FIhOP?nQFk;la3V4|;Pk!!9OO{BzHkb{dhIa`>cCCi{E)Xn2gN#XMr61U^^e)?OS<*ZBT-NMZ4JX`L*A9Q*8u8IIJYa;OxoRB8vLmOp<el9fO{|zIn;a%=Un&@wM~*7wp!ttA@}GHDLIvN=zy5r-+EWaivO0P4E_#0TL_Wd~IqCU`9j0$cK@&d!j>|7AZ;m!vGq^rPXfyOaZhc`yAbtGs@iL!fZV2oc@CU<p$f0u<!_TMDc?iQ*9-X_$bVE9^r4ikE;rVmE1I*2tEXwRivTTf!5CXX70K%4J?(=Hltqa;1a|cvDg`CLFBnR6K#v<R;P~$N+uj5DC)2S&#LXV%}zD1QpHDGq2LE56;N#lXiyC|P&_q_Pz-~!Xp6hWe^rIyzYdN3BfWVg-9?@&tv1QY-O00;m803iVR#)2p83;+OBDF6T+0001GcyMKMX>Db0baO9sVRK)0H!p2vbZBp6b1ras&06bo+cpyauD=3hKUC^bitM!08K=tBN#!JMY-bXuGd+)o1Cfw~GhZP=+j1`d`|SgK>*2dk=MM=5EOr-*-5++BbR6eHktq?Y2QxOwt%}96$aMn$C7Cd5C6-z8sN!*&sil4pSErxE-7i8tq$<;qip^v&_~a*<%7p|rnKDHdt3bS(pIv^O3!@?+lrX4Vkr!#2WJWA?Xkc}`az&aYzRL3?%SDn6av7x|j8BA^OMfkNoEAnzMQ%i(azE1xMJjzdJ)h&!C^bUHfmo>+g@9Prax(akB@4OGp*9<_NQyWB_0lxlh_ymZ;=~A(5U(`Jjnx9yml}vb=LX~z223#-TmY?#K~14^5ha0MY9Q9JNJHr>Q1@MgJHv%$%_K#rM38`4P?n}?5SjVw$-r@(!2rw}3C~*=XdBNfNn{N45~b#rCXhz@p#=5viYO1XZ`^vy1;djn14|J}v*zVylcHL7{Oa!ybMN)r`RhO5UA%F{IT!_I2LruqPn`?|{591c3<ked@`mgbv#YF7qX7+xb7j^^@F`WkT^yVUupvF)X?35U2vek?x`shlOeT{XnqqIONv)Hdz_BGEjmR)v%M-B-6KTXu98ab+yilewI-UGXqbnIjtzjna(Edb&Nu*Y-;a`|Xpu#r!F%92>u^PiimbEX_<{+~Gk1~m$gtRXhcix_UJiokn_s{w74zEgL10l!r{}vZXtbh#vtERJZd3)NbxCG}x-}Q8?jpq$>6)s0?d9wbJ91lk)I)LD+f6`&#Nd4=2dLzCAEiXU5d*jqh@Gp}(SK==j7HXblNj7v&MMOspn4vJPVmN9FLtG8MDYVHU#xsaoWbTZbN`)#O)^bNEm4{4ocyvIA)-3)xf5AEt%Lsym@B)yci1n93y$}VgwdS7vfI9v_*cN~@Y9COeG`7>?+R~a-v$#m{9E-0lYA1%6y;F|c9?g9j_6ChyZ5i_7B#UIIA63wu1wc|V>yAeHfuJK<{7d;}$81j_6$0@VO<9I#?X*kQj_Hc)7J_dFj&kXIj&I}SzPfk5HjMpU_RZ-ZL>XMwQ8@4CZ%$wT?YONtJ0#hu_>>;QY%nw$Y-;)FHRxg}bLbE(p-2QG{GXr)AM9^}7knT*!3RPRd;rXFIzDa#7J+-B^Cd>L8m@qqF&Tx1OR~G6vBRcck@j1)F=p2`Wjlya_Eh{jV+-S!@W)Y$#)3#q_B1N8L!0hUe3MOGL~Q~V=1^`1tbaXqk8ZYj1{qlV3c^cx153Al(^=riAq|L?i|9@&@VZikIEiCWi2`YlQ5&AzGj<O`cEiX{C^;OtPl(=w5lm6s`h^B@E?@mIe(RKbslofnhYViYFwi$hVZIa^6G`U06M*~RyB8q;?;y|;2pb;ur-#PtB0@>pGx8Haz<F&KlhYDzNy@H5%w6rxOl*DwAcQR4Y^v0vsS{@wV*Ij^i`3#d;os)>mPdC>%{k1bacfoDQ~s$C_R>fQu8(iFB@ZvfF_R*L+FwR>j${%KQ{{RO_^YQq=hI9Cc`hzQz|=DIIMvCAv^a#K;JVIpZ<rCJ^H)$vs0ZmAz?lFli~#P6A~OoADcI^#uZoOwdOSrVb593KP9ivlUnW=$Fm7rnh~PGN#bWJ><q906HtUdcGk4_nhRRUfGX)Q?4iTKa8GkNG*p}8h*VU?e>qBQgg+;7$g<9fF!fCiw_B$&t=7!`ZkJsCHP))Ie{-~z6c(|AR*s%ufC0P5G&5xSPEbY@bHAks<f)y-T29zJnpitSh@cG__<2@^Xa?oM^2a!MTUARZ&*|aiJHxb=u+q0u`j}<!aB|k3pdm4FV!3N0hT~HZr$ATLR2Mhz=Vgr#ov_EWtOUTeIelb<FRg#YTVA877coeI&*-X^D+m|PYBF_$Ly#G^ex@oi(R*L^xapYv0_p_E+l*AYd-a^9O;q(t-++~Ug28?sKO2$?0Y>TJ-1(bt#Dk~ci)LY@07W3(|`4LR`0FHTucRh5mD~ByxBhHLms4!>nL_k%?VKYg053xC_*c=^XQ=Ynauk>ndz(`@T%8N)nq(gji1yAIoZG95TkjR$+!Rze}@9=IG|KJoK*bJG;XhhyZvn6A6nn&FghR$N=Uy7;Kv{5^-(8lf7O9{?7ib{@<4n)`odsJECv@pNEc$n$+^rl<I^TY_9wodHnHs&^0Z8H9y8|mNLRW97a(QG;@|F@S8TOHCog$CCqXEpd4PZIC_ld@~YjU<P;Ur~F`xpA%EpEN>pK#tuetlatx+XI=r7uO4E{Iz?R_;OK%GTYo(!Qx&6r^0Xt5u0hUYGp<;zn$=rU$C=ZoxTw=gB75M_OWiz3u;b=8!&DpwM_ybJQiS(1$K4<vq`}wI`+dNz<wO^i;B=_Zc$*?3kx7o2?9+WNf4**1%o!F<7uPei4I0xkFyWYk0<3u`<~hzn8Y#U*s*WpT;}48w-)I<$9-eQYHVPYIoO!S+D9}hz0xl1x?}^j&Mj+hN?{;;t?=;MiTDa<1@Skuj7DWU;BgAdw1r-CO4Eu64U53<Vt|9ksVkZuFv61XaXN{?1F;KrTo$e{rgM)PdLYqD6CIl-V)vu!J-bO&Y}=*VJc_%>x!ncDJvOd%3GLnNK7nJ;`dh_+Zg$t5v}%&CWvXy%up@fzmg!6us5|YKbo^rnA$n;^htE$h=TF0sHySof1v&u~k!FActOb?Y$lIxXhz(<lKR}x@RqUnlkJH_2I9=@5Ek~0nLNe1+OJ^j0#2wNp4)<u$DHm(jb$!=4HOr$}T-As8z4&NvPR^qQ7$i{OxdcDdTRp+o8@bBvs(vQsA1YqKJQ@y8sljepfYFyS(&6TZ92ThMQu|uPhEN~lLdifROEN`_{;D*2n_PsFUwib7wbxYjBxR2<H{4eDR^;QZnd{2qF=-V)_F%Uj1@-@W#rd{^{9F5$>Y)b}LN|^2XX~+JNw6(_s}iDFyW77tkK1N6vM3rf=^5pFxbgr=R1Qu7X(SYo&*(*!nxvZ(LV1Dy4Oao&b2jO+cM4q_aFslRtD`<pmNm10$e=WA>f!SG8558`&yUL}omD_FnwX@0$bD?+M=9*N-t1mWH*2<SMz{NziSf}P2)d{^(G^`ao0!FS$inIAI(*tejl{F9NHW+52a{EOmo_2cAmlg)x-VOz3jdzsXf3)biuVO>uj>+~aM6Y-2Jji?fL@*~O(Y*0rsS+|d;||2lB7qn+Zdg)vuBiT$@{R>4g>(2jZ3!FZLXi~uENys?V@9gww~b^px+@kLAgH4wQ{Jb?TUsLG%~8TT%YO|#B0{8d$A5L2D_fS<7Mg4nZ!nSIn}t&gCx>1CQubx_#r?~+vCtsV>=B-(oqrBmag|lJ#?_Mj%P3jgtJ=q7Q9GRYy)<NmL0dZ@nfx8wSw$lFB|tNk!r3yH&@&$mEtqM%6^*vd^zYoT*<(F4oAjWxre2~u<G=I=AkWePCBKD+kCa{;<p5<dW{@{hxi+3Z$)!y*^epXvXu%}7lDHP2bclCt-nt0;Rxc5SY%*Byj$*7hFwhM7qDqt)VwEoSW{<7Q20QFg$vy{Siz#AqsFD<OG<$uuT@m5(Dx90K9<6|2NW=Gs}=-6WRyw-V{mP%S%f&HqLeIw-(o;c!m_!#wq=l{_5+v(sw(pW)9Hl{V1t4_WZ^GTa-$&+f}P40NLp-Y18`Y}kkRfgjRrek$?=w(FS)9hW&3IbBBHc6)V)_&AgJRL{j#J>@L$#_o#js{RvIZdxFAFkI0&GB_0rY)BH&NmO=`Y$?JI#HK+G!f%meq>;8DpXqXE`Wd*IHV54Cgdh5A+vJKJlgFZd<NsO%(o%(RAv8p>MP5Wj)62~XGfJiIkY8Z{C~>gnVltZV(QO=opzb8^}h*u1DAK9l8=^9kAwCSfd!b1%$Q>~FlXwj$kx37UXswS76W5-f-<3LOSs+2$Me+bjFqhG+D#*BCXX;;#JaVqE4$woh3KprTj`f1Tv`EgBjDSX(%H1qIUWGXN~W{(6N(2^81W>#bBjD4YR_2m*`}g>UGqt@@KB$e&-}!i_lp{OJl?l!|g^kf>v*!Lp+B0!Ls?W&nJ^wk`Cil$q`9Xj1!e=`{Kq-o@qFyZ2Y~k52V*+|0g0vHWHZe=O&L{cK#^bW);b?$~^xR>utYR%ZP9Rr}_qMZ4M%D1^pm<)=ZSq@)oln6_Npr7b3KQNx8Z>{LK(limwVYoy+I&uG7Y{^0T3s~#<>J)?j7ljS!+&W^>E$txe-@v^sDt=jl0w>|g|P)h>@6aWAK2mk;8ApoAXkg}Bq003<g0015U003opaAk67ZDnqBb1!sZb6<8hFK}{YaA9&~WG--drB`il+cpsX?q9*FA11A~+6~w+q%g1~>(XrL8U)*b!7v0`I@w$#Q6;G)Zjk@JBaeFXOWI)u64>Ou@wvO_$RG%=71Ue-gNM2hsW7a7HBYy%QeNG0&F%!;YsRaLiLz`AUlqVgQHd2d=>~Uo!wZ%JH;n=v2SG5I<Vuz-N%F?7B?&6lQW>;SNyCkhm7Yxe+J@^*Ay(b*swwL&L$_5;X09)<TK&tCub~w*Ql%U3%5a^irZS?0ID=GXqFT2~3V*9C(E!=La1APj+D6ohUw11vpqs*wPD~(pnQ+4J^A|3i%PJS^sr#KO!~hZ}Nh`IIDQGQKvIbd#QQL$trEP*ak*j|oHBOJ#F&8{3ExM2;3W1j;{?xbANi>;EGRV=e%UB{TAQX*1(UiHhq~>NbW&QC^_6{~>DFx5InKJLif;4|UiB6o!dto+&c`R$F!rpRZ6SP5{0!Y+F!0*}Iuf}Nm4wMOln~Sp`8XOlz_Rk#-jSRz%-MLxLi(<vo+xc(@49~dXi@@7!g@UECMovB6jO?^&z#xw1=5w*MrQ^|P%?y0n6D0&LI_k?L)M8z7%qUu$(|$Zy>XkLB4XY^}pMq5912?2viw7vI?qkJEIO!s#bxPmVBQpO(H4w2}DrSX!*)UO20I@Jo>M(j57+wTdZ!b>I|4u%f{W^OWEYZ=92TBA5^MZZ2xx8Ryqp_!@VOkVWnF3oo{n!X(8_qJ3=b&&ELonhFTv-#Iwtw+xM)B0d&K`**Qi-*wkZ|^#W46e%$elyS4>5d1b_#ctr`?HcjKMPMix`wbNak!;h0S(#V!lFvuqYnzzlAG|29+Z;uv4wE1+a47TBChk*-sw4ml+f@&#?|G!b3C1I3AMotIO*hT*5~e<Wm#}oylmFr?acm^WP37h?=>=hI4({76Vq{Cy2(SK~TCH&%_!qi}RMNW4+<ueg8vP)iKvxDZX{!$40KU1`w1e-oOKCc~DekTLBe_Xpp3HjvSQW`k>!>rVa;$G#!O8WdZHY6s!^3+1<CEU%aQDFVW*MDh+)vSL>11v&}bT;-fv-Hk9q8FrZGskk8!zUoK{U27_X2^L2Cj`_CWVCjooubeD&UyN_?-WDaD@^WYaE&_}VuaNWjd()C|bMxo7)WX@%8I@vE|PdFZ*qrB4@IWSoqiSHvbDjC3QO2r;KGXdERPJAFeB!SBSZVaZpTc>0=)3`gvF=$5>?mUj*3kLP$jrw2MOo(2J{WBeFw9^X3+pjwkV-N=Nfh!3+rm^|i3b0l^!QjOW8Q@i_Wc|jn4e}OpPse@TWIgPu+dg4mHlKIQ*szk^Qa$QnlJ3z^UPx}jozg^`yh3_;R-ZD_nfRO@l(n(o*pac4vLFRsZIAhBiUP$4qD3(heWI^V?N!ZIP;v{~5CPdpYz>=bPXaI4xgBy6r#4c2!Bb#n1A-_7@7q<~7?umAP0Vfp7cKvDMwT!?s8z~4T=dq2(M&hByQJo#Fra7vOSG!)?M&*DYejLW7@5lAYu(jS+>TK45vLR%o6lLW!T@h=eHwT$fqN0yv)H|XH14_18(a*jzx-Hw#)0ooqKAQluW-*z-QH$zpgn}MK!h=l&9m^y4slUThZtrr@Bu$tqm%y|z>f2QSHI)DOeR0k9Y$2{nv_CP;++8}>hKBRA6|)jjfLGSz;9!@QIy(U^V3aJ-6poz=Rf)q#P{0U3SUh4{NwnXg9mJ*_EBe`!kaCV==u%Ir0bCsrGRaO#@M#A4Dex)*(+Qj*H}{S`R8T_P64doIYR>mheLPv$0H5lCZ{iKoSi;ux9V{0*61nrG~n;15+55pb0JfDT-qz}(-i;qMlyxfXTt6YJir4^EWE;55x^HH;DsE*7Q<*<I_?1I!YVA<QL&_{JQ6&1Z*nTJd+3JG-kJu*?rqr&V2pI)aacVP3lg@2(zzOmbs#N5fAP1+kE}4MQFUh)!~(M%l1+cR@vnRba0jk7T)D8YfAKl`+Rj^iEoM3zR@?wZ4F!%SR~{&}ApH|>=jA}B$5S@Oa$!IE+fYL45!6j@!-WR`t83{(!!I_l*5NXWMybc`X<Owe_DR%XD?0Tqcaq<_B^PtAeE1qcudrc!BU6jc*0G!{`}*wRwb*u*87+670EY{HG?A!{k`BNTgS33%kKLZh5jq=5E{-(v@|m9f7f?$B1QY-O00;m803iSZ6}mS(4FCWDE&u==0001GcyMKMX>Db0baO9sVRK)0H!pH!aBp&SE^v9}8f$OkxbeGx1*?9Mx3=QsZi`-WDzMni^%l799&Q&c+Q1NKiMDl-M4d<_aTodTo#9K=!%ng&&;rc|M<R#Anc=*tAP7FCIjdy8%~K-UvZPJU#Dd6%R-&d2Op!V-xFmuVER{L02(2<w)4X9B$xFIpFhGlgU?K{FU^d$}yd+7oZDrfABmtP3Hxhs<E-4}q1}5c&K`AUy#<sLAWR|CLW`<I6XCHHJpGqqC8c7XLMZU2Je#Qw4RkdY(AXJd5Hd{(6lBNZAOBT@vxTdLen^VclJWYNTyju7#g<c!6J6g1g1G^%e5%{CKUBh-%vLvN7PA4UkO`eK{n%1I9(xMfTHOYo^DL}lno=tE%uXf2TJ1l1N*=z<fuqW{^7B(#9O(y1-3Oe5skRmaIdJO*>2Ewk$mmuZ9U2sWO3lazrcPro}sCdF#S+`OIk24J~K+GgI9Gv9Bk>Egqihyr^g|h>gN*!Q8^2(BRvsyb)RtN@URTe3dQ?Sb6)Z}Q?zb7Ayj5S2iJ5VhH0a{W?3drFc;4DiJJBTGCJ65qq9Y!JmvdTn6KJVFqq_i^Fgv^TqL|NCcQtlhx?)D&q1=+L`!BCP6gtRGgv4{PBYvJu|LuI-r1qh-_57A6BL2Chf#^k1xIKPi}Ooo9YDH)R#EIPQ!*;TZfcXHkG2H`zXswpi^L(!0kn?Mnqm}!T5{S;PvkkiW*G{5)um;9E2p~Ch<mVk9Vi44sr_Ol2iEnq$9c4#Cz_mnYe>kM#p3ON>7tWYc*gw?5}2D{qxwn^@3Q^w!?akV0^dTK__%=W;seo%_}fmduX+|aQTBc=Lr-a+XBwWZlV-WHt7u&Sey(aNsymej9T$)uROm4GfqQz}?!U>4-HuNy$PndsYApN0Cn*I%>KcduW(lf|zkoG8B&qsYy~fc?%&3_(hODtt_mW)wsVq0iq}qe3n{RE*w&`vPQ#WOI;A#3g+Qoo<h&k14l@GUY-hBIPxU1JpzaXDLUIRyS-5zUUA7{ffN(>HW`S%P2-j2)2q?rAiBMTD;LQHG_~V+Zy)4Si|rY0y0RqI0WN!Fxy&J)!&g|$Sc4HGIGD?1v7D5yE+{^#1%vq0xqy+XKLTGjw*uj((GwXR>d1_xh7`qz`@pdDk^pz29Zn6qWkbdd<c$LYJ_9CcEL?A?YtiTh066ogxd2gSTD3EtQ#&lWEEy$<abLDMSy{kmUY1bPv_SWt!q|gh_=UJx9gYQ{&wq2q8}FhSx3nwv5y0YAz6S3{`G-gz4Gz^e($T=W6!;ZhKO!6g>A0HAW*16sZgjz8Re3dA~gK3ZO)+*ykwh0HV5b8;{JlXS`cmD7x-|;P(i^*7VuD|EQTO<VNMwmz)ySF3K0;#5v;*bzNbaerhwPxP7NTz%t3%+qJ2<O@bnfSEq2nh1&d}1=UY3LV=_=;Qks$bT<*#B`SM3(4)q$&RQ<rgW=;$dxqg7W3k;y<MGlFs4t*J<b`?5$vVoKW(tthC00}{sV+0}~&=M^}(NoC=#R(vwnY6lv#1Lc)+{t^cvxCojAS`2dtl%{=rD#WDF~6Wh<hu&9YZMitHbG<gTMJ^A2eNCys4K}>WF_cldYI@&xqw4~GYiMUpCd{HT#@hNiNnFsb1zS%Ie7BFK_1Jln}#=GFtMbBM3rp71rfb1*)~*X{vhj{)#7{|1#_RsbG<ic2tHh#+OE?M4R7~?*n*s(Pz`QZj0`iLsl?`(@9~4$z(tTaCc%$ye);Ri5AXkZ{Vq@*SHTQTQ-d0(;blY{3MP{S;W4S$gACC|eWE%J?C#cIDTJuK0hlPpRVPk|6ZGeU5elsiT~=fPlyifcVeTMtXBfbp!{F_YZ{Gisqq9>_7`=iNlzWHq%$~WC^!&tu^_|w67OEnU43vuY9HDeksWa3#f<O>L<LspBEK-@OGFrQN6gb?P3}=API&ml5(Gfc#=U5P3Um)w(->Bbj)$eQk{o!2ws@WgZ><`}tomWL=0I${O-^LUxcGP_oeGllXkD-&q+BdFFEYCis4_rCQf?=*;59Q|z&&9C^^ze#S>dq9L-8OGZ?WJA0f8v}Xj4IvUz;#^Ea+6V%)Fl}S@5ahwy*M%@hEq!&=*Kc!qh+k-184;-o0fHj>C|2>Zy2y<7*s(f7rgp<P5dwGjz`|4{wSGR&!4eEQglk>sH~UH?@Z1gL0nTIJiNpG{v4C#MEF^fbGAIc=;Q3A5Q#}4lqWr<X5@R1Za1i9(`noo$}gYieL^t(kUViIgd(Bmq47_p6h0kspRgV<`Wk<M7%mVK3qrHvsAZsdpD>;=(%#4<BPaM!@XeT&6P%bO&tT?7yv#>drE@$qht9ZrKwz^5M%$)u>;n}5P1)s}f*sSof&P!Ad{UuD(1w^P%^MJ?UAFF1^&{b=J8%KB<7a(ysnW8wUjlW7Paq&EE-7{tB^67T11#?@yoxlibO4-v^kzE(V7`0tAgnRMynP&DN1<V!jkLI;0aNLW+zFL!T-ic{Xk|q;(K?8gZq?NQwB(S(*o@zm5T;<b(pkU8ai$S;RnQs68e(J3g1mp{3u5ftkmb?f)QT1oZ`Hb`8^MbfLbla>=Q3SdeuZtsb2xR(0!`xs=WCu<vUf19A?%J0P6_qEsgCOOV87TZg{udBM1YqyHRy-ZUPS2EKccGutKx8lcL^Ee+pSRgZ0a9k>mQRm!vDLR-K#D{my+D)73AV)y&5VlXkHp>9!>4a5L;5PYA5$eXX2sLpH9=dO7$HRX_Esrpd;o!L$pw{$}3ETvK-08^cINK-E*Qhur4B{t)K-Kn|vGDx(zZYfsOSqeq&#?6)wr3KCr|+8tcVPpSRL6^Jp{0&JAG?P7yIDzl+tI-KTS7fC(q`mJY5Hkq@YFML>!O6{KEd23}Y{T;nk<jTF0Mg7S%F*bB&iYvHIsFLsr^TsniH=5QuhcO@{&NUiPq$r{i%n@Fn^H-4g~25w+>eG?e{xyMa(k*1lQoc6M<v7<D1K89d(=b=x<Gb2Oy0L+X|0V;_kxdJsEoi9M!&1&uQXrLeG(?0e1MbAwqj0t?}UNvn7Pqm0USATj}Pssm~Z50gKj|~v2H#E{FbVlp}Tqx8?%6||%S>2b=-B9bQz{yvsc6CoU!Oo#Lk7`{?Tj@=-770+A60fyP`$|3QPVW`TIMW`cyPX<(P1Q(J?x$qwo>sXSy2Se?2VrZJ+6*($$y7oJSs1O=?LGDKO8?u^yuhAx92(_e54B<=?qm$-M&VGYH{|X#aX&_-h|gE6)o>1_9>b{d7vMJ0QR0nl=t!Hpsi?e#(NNz(i}-Vcu0hqU3BOgZexC^M#ZESXjDrUkp>5WW7~SaRkSHu))u*<Lr#jUyZA~xwh3ctJwQ2zO8@~(FmG&CImN?t{y&QFvi0VnHbiiU$eX5_X|MKSTKLV2)sIHZ7k$o4h_r-M<?QEFq4@q_5NvXU?>Tu=w&e<$L9{o#ak2zh>1ANtXl*r~Vgm9rKKAKC63Kw_Y$63;+Elw0C{%-bYM>~BPjr5#);w~e``EqZ8bI8fb>x<`KV{|_E*d0c9m(huG4DonGLhJ#rpuB_vzvh6!%l}ImdI@}6X@?dZfovERDky>L4MAY@=gS=ZiH^-L=|jO_FFzXC;K<~9U!ffPL?)bM?pby>xe{%0dPXxG>wiZUo6t52cGKW0+Vz9q7;rus(V{r2#m6=I>C*YQDnFsaFVo}a>9Y0-S_7sb9-olb^L)eqv`n?SX~?5`_cF_WuJxaiu&H+AUh79ERftnREr!(&Du#OSW{$*CWW=BHR@e*_iy_FmLi_F)Gd3x8AGKY&Jhj(L(N`conEy?XUon`JR-c4=#DwF^qX$#uWp2J-1h9UN7rg!W_M`WOBT-*tV4onbEE!kuERaY-%T2y(c`IDwC>k85_Oxf#%XaN1ldop{>GGgq1)WB&6FOD@csF$|WOrG<-7@vf=Z<mI$-`OImJG`@RS>yr@rDbrbXlV6$c%aLuag?q;%6xsTV$)cDjPV4lJ#!RuZU_kkv%K2B^(-22cZikY!84;E3jj4qqVJhRJXMIWHVI7#Ti-x);|%BL5R|dFdsq}4%~iGrWO?V`JP`xOfIns>XtZ-HT@u9rhkuQvwGixtaJ6oM%6(#<$X@+joLsx80o=G1nzFKe{tsPz8~5B1*31_`g=`%p4Vgh*oSdFpkv>(4KbbgFsuhU{<6#K<$%Xzw1o4@dGtwQ8}9u_)8|qZbpunWxQpIFUHEr35b7lhbJSR*Lc|xkHya1}uHucor_(4R^>5E;2CK&tn5Bfq%j#LDY-Th93WB+YP9%PNplQIe5<gOAP`(oo70-bFIB4Z|`KO@wU6Zzi*}nl$O9KQH000080000X0EU><LBb{g0Oy_n02=@R0A+Y^WpZh4Wo~qHFLYsZUv@Vyb98lNc`k5y#XW0p+enh%^(%TgIINV=vXjZ~+$p02oJ^7(TqZV>%;FY;LC_Lqb4`(2KJ0i5|M#m$ceA_Ml&EB8Z?M={WIw8_tLt6OAP7Dt+oDSAqP#!4jF&gbDimc>HJkV{OT_IDqN<zKy;v8eczyn>*p|sQF5@~aauMe%QRInucmC^iH2Qp<h`6bX4ZK{6by`;SQCT$5W}OvxBCVjkNWN^dbeYzosghC@w@F#1t3(uQQA793GELThu0}<fUZr`QiE2~aBu8a(n<jT+8$+k5I6o1~qAZhTohMZ#uHrhW0OC!Y=3)aN#kiEII9<itI=xLs=ieWnzc@bs3xQjg&@9cbL|i5!FKUtGxXzWhyaxyX#<I!tq@0d|AQ+9-Ww8-av~KFAOrl7nn{83nFc9=46jY;;d<N6T%Pg*{q*9OmSrxhZyQp}uZCqbx>7^R!9Db@M)pb*+nfg_yn}oY8i!6i1;%uthDp|)(R<F`!ErG1#Dgw~4*EBBcbR92i-F!R}@ZYI?{H836GNea<s!h6#5Kw-wjPoK-mvI(d$JI4ItFGhcfB9>)PO~H&O}f)9i+r74skuHOwY;WBqtS>kB0j-B0r;<))m2h|OycErvKqhnvP`zPs>zI|0fe85Z3ei7-DnB0s*7zX@&qVF#9|$%xLN5Yz5=2tMRk*Ix9}kO5^uIyGNoM`0eVDLH03ghfVcq4`hL7Ei+=(Zqp~P!IV1il=ClX)YkG_8t>E>;BmaciRw+^)P?rEw8wUmwf=~oN+DQq2=XG_AuWo+`<V31JZa#Lc08xN}F#&Mg4m+7%WyR$<crxAI2NR>ShG$XL<rs&VmPr;Pc}I0Y<4h(O0IacB)NG<*&*M!(VCzAHEQ@dB<4v(jvSYyeRRc(k<WGDR!T)b!geC0hM6Kd-dA+021_(8UKY^x_d<l=r<7Jj^)sMq^UN&hqxRcz|Vs|$nU!bjuj&omws*Yh-{j+W&Jhtx3<Q4z`Dmbo~FnivnFn5kyAFYy_VC`!eH&;knJ}}GVUk#8DQyy@O_!1a7aLW2iy{8!vtUQDR8poSUKw;a?ByOTjBr%r=C5huJro}P=UYd)CfcgY8?hD6<d%**n2aW0d<WU}L7C*_Dna<2)H=%eE3L=7P{<&<D5IE;GoV02_4o>lazQX@|jsN$PK3%{tNUB{U!o=I8US5lqf8M3_b<xy<NT6(h38Uo5@5Sq1e>&y4h~*FtI&2tZLCb?B;1NF!#j3c=vm#zaKz!*sMXCGC*-vlYN1x8$y#C}L@e+^;aKu}8&^+7vd!STu8!wAx5#ftaoS&V)iGKa~{!=Kn#TK|F@>SrOAl6(4$q1hU#Z+lk16eSuoC4Paxr)+!Jkgz|n;T$iW1J4?NA5=Q1v*B>4gKV{+azBBN~6pBnt1IAJx$kg8*T21G!%+zn=Fx!8OWNZb+W0(F295>_<|TNbXrJ?yRe%P7|56B25U6`lN22A#xa+v0!h>G508y13d*?P0v_SCJRP<PE;{2*I!|EDsErV7Cq%(|gcKQ5U_bSE0{=u+`dcFYG?z2rFW-82O+dw<LCtvxq~r*pVd+7zKNZ0gn*|g92-fC2or7}q#g8KV?<DMfWdhsV=tztu$nJ6hHR_GV3~WInPA2|2+vTjwBq4@`EPA?1tDC5)fW%p1PVx1~*a3URs4mg~Kf0(8&rY7dc=F_X8}p!90#G0M<dKrq#nw7F*9Qnyn-cYyJV{nC-6e9GhjGtb6QHYS$$!nJ&(@FP-77dDpuiG%)iz!xMDD0UvlOIrZ>Z{_xP-ysH}XUAK>e)ors7usJ{DPAUV*079&;ENMW?t-);ObLTL9rS82PcjEz`WV*S-#ZNkIn$?g_fz@@88=3)IE)YMqob@nA1nLiU^>dnpu$WDbIZux^Gbe3=02Sxp0*64qH$T{GJm+3*p1A}|xuby6X7v-$GaKJCGzWkyVMAHd$Bfq>;2z2TSc3V+KJm3&!d%?epNs3IWf<9vzaxVQ+FRtcj^#!79GgH$9qThd}h)?vd=>lXKFN<6Qq)=zP>sxc1&N+SrII{5;}%iJyn+af?dP9sKA9J}(I2bws*gSwuSpo^jqakK(juf@l2<1z(xDilqbA#>J3$j`?mbMDB??%{&*Ld~x61R$3V5GMR%9gyIAkkda-5j|NzdvgT~sOChsqkcY3>zT$|BvwPacF}H2QmMaqv0bd!U<;$Bid_^@RwnKrI=CR-J?m=vmW4r+>aI*{tPMp{sJH0D2l|J?ocRFpW<Eenl45oAAosrlqaL_5@UcgDX6tX#*n#6TR#=)$CX3m#=NFGWDkFzj*1W5ww1655Xpc2cI`4oH+B-uhWv-87_hr;>%e?(Fbf&K6^56DF+^k^AWdU4YL9t5XtGob4wdCiavB-{srWq|9ajB<WHu(w;Ag&#dHqyw6_yyp|ZZS6TB3c6$#L2d)EgA|cV85Fflx#F!j|>5Ebo<Oy|0VmIIs9@FD0>=+Q*p+Yn`#bejMVN4s`mZ+6u4+>oBI*Uom#v=d*heW23AA(iImw4t<jV=Q0K9zJ%i}=^K2$UjMgXuW+);oi>l?3?b^(zrrD>d76J90AfvJJX+`Kx%VL~5X!4sJIOa=1xF%h-0v^gPFHo>9v$VPvO%9uow@xuS`aH6fPjYoYMb5Ei>YBK2BrOFE+NzlSKB&Q%^#bQL$M{a13Npp>lnw)_2<Rfrv-_8%*{qu^yGPGZk=+8Pg~q6#9BUBMij|9GB?pYJ&?N}}LywbmW1>=5#n>M;Vo*&h2}Ly>vJ@ZWtnRD>*E_l_(DJW=S3lrLkI=2vODtoUN0SyVED-d<B)Tig8{AYr;Ych}Fzzul=~sk;pb8!BW-tpfdG7YUHBnocA%RrK!aa2Ay*>hyku>;}UKxt4Sdx9xPBz%k482`wz~0tU|C78bl4^ya==yegWY}g_S8a;4%E~_dze6BEup8Ejr`U~%941P@%)ddhh4#mC1BuhJPu)<v75%kCT1uY{a7Xu3Ca1CHE1)&4`hLU8R_ThyPIF>cEGMM?L`zl6QVy1*BlP4RNljbY8gYg@N(w)s2BQq_+EM4ZNu)p45H+Dko`en$HWrw!J7|DWCunKI-*HWnf_+T*{HyFfUhzsizzC0kp*2`Ol{}3Yc;E>d2?AlMMQdx{Ay>>-#Q#VY@mgV$BXq2rRtvSrUhX&_6Xcx4M=_|fOwuiStH{i%T1%|eFX_+?_4i1x;tGw5X48V>^$>J@J`CqV!F!<$hS6T1&&ct!*gM3W=y&q*Of+jS*bYqFbwbdz?1d0Nwz%=Q7xo^&_p}1Z(FM+d+@&rGQElKDvE8LM?=DHNu7NzZAdQ3+GbH+_;&qn9AagH4#)9OMh^aVyQ=va<d0iByqrjZfdpd{hplj|b_Dt+ubt0J-T(F<H-SjppGq-vJ&I9y#fX)^uf?yTZXf<V;H3SzcfoO56aV^czdvXR)pv^NZvD#%9?UK5L*7bQg<coBxmocP2o04;R2&L^Ade{o;1{9P7o5C?A=*S!~0Ax!THiWzBj~z|RJ%Dl?CBT%{rRHa7<Sr_^(cBk<nJ;?ue?G9Q1RY9Bc4{!R^MFiWWB~14WPgwPm7sA)#|h18!Y**ifVr5A9No#GHZ&{37RuJqR8-R>zfDUV@x%Mi{}cUm`uUW7Hz2w^9i*O@XZ6iUy0hvViWW@iRHR@Sl=b)|^bFumIc>JkF=?szZ*QZ&{rW08d;9kN&mZ2*1Jknn{PFa|r?+Pxe|huqlfT73&fcFzr@wwa`}xDC&!_L-&x3Ught_C;<DdQVW^RFYy*^rEU^y_9!_Q|QU;oY17SOxA?lz~HU!Q$=`}4c#r=LICeVPocr0gEli7K!(N)^y*?J;A6f%;7^tjEBX@gSo&ak-hP$GhjlJPNKcj8u-ssX=i@$Wm*_(1dBznAq015%iXlrA^2tN_15g&9;gz@6m6v+13#-mn-1ZQW~Ha;;stGo}+-W<nNaaSZAb{7zqqxute)tUX++BlG6Y!%ak1#FwIz-Ei^M2X?Kg@Urkb?1#Zl^cy=+lP|iANWNp@h7C`ewq2M2FTwczmWYgqAExTI6xh%&T&%934ESqAywvp;RpRw1#LSA3TY59g<6(yfIHq7TV#Ny;)!Ro8zOCY=A{GJylpQAF$eH=7(6!pKuJYFr~EKk6Ur+BZxwU9qvXP1agQ&C<H<@D%8o!4it&v@dFe3#hlu5gOTh}txn2(?4oX46Q-z(ydUF}hU7fv4H$^#AQc?O~lS2u<oYV9G(zg*HJ|J_@=5W>NsazD8*dhC?*2l58CcZ)n-f51FHj%T!sQo63Tj%kK6|v$rqf1T`yq7_sv`noOmiV?2=>O0{@y=sYy4jtscDE}c2uj&+QL4n654Z3;j-`gX=nyA_y>S(?)nX1}M(d$ozbL{dA9Hfi2q@aBp5>xn&#(h=uv^rXd%&Oq?KCLZm5lO?MwG@z{Tf`SqVLB+Z#fk$sO88#qqF%gN-WE+7lI>dj^ZA|zqKW}zZN689gGnzv%1aN?;^-Jw5!ptW)3M&q7DgL_xb1*5<Lr`#LRMN5f^a_!b(>McDs<S?sScbySh>-602DKr}58K~fBg8wqtNm1I2l|O7&>h{&0uBK3CD~i*7?_8m{mlK{6{X3}?=eG1T1|E*AD>1J_8x3Eplzq|CaJHBmG-bn6i4#$$Tcnr!_#H6il=E60SIY^qI&!leuIFR5Uxu`iQk?7O5T*0<Q&kJS#@1B*@~FAau{HC9LlTSFzwa0K@QLY3ltfxz{W+>s4QUL>NKgwClhmh2Cth+i#3X$ot%(k)e$$R@H~)yAe@b}4gfCvy9PdwDBPfx^P~9T1Q>_3ds`{<HIA0o(vJdX2>}rQdGhX+bpDb*1AV^_3Yy2`hYm2u+WyAa_`CGg6plE`NZbFSXbsk*WF@($ZxKjri5#35G^lL#u6Q}16@l2q|AeuA6#w}ZWbzq3NClj-4IKQHTrM9b00vEUV1=g}*i!;HPwtZPrTF|p<sJZxq@*UW1UMNfD)cKgKri4#4Cp{WHOAw%Hi)I|WS%pSZtL_5CD#&Xb_Gx?In1L>?oAsB8WQa#2)pl1iD<tDtpHSQ%z{QXl@(WMUCm8W_cd@+3)rqNlTOq&py&%`X((&?_6H|nA-GENM4JpoF7Y(3NJ%z<Y1=vpFo6$D3?L0&i52UDeOx)c<twKH)46KOznmr<Db!Yvv0ut7;18vl0NMRaE9+v>4bZuRRO!_QG*0Z`EXsJ}^tmsxBEnWq5B=O}Pf-WbhJ5olZ(aECB%)acqSo4OdNJv=mRdXPCGZ}MM?Q4fg6*TJ&|!?jAKg9=x^AEVlSWa=!S-*S9sSMMW{ELI)UWCygFo_i+a<K!2A)w+Z(L)7KRQa?-B3`v!CtaJU9w^k3_~pVPhjN{9T?7s;D>_3fR@l6Ph9%om~B6S;d{4_p`AcqpRO|QY#eE)SP|jhGF?Ft#p^mLBdUOKP=sxe{6MNSiP<jnjt2-$e^b0ADDHrJm?DdcFfhDiS6ztf1{Pt}UcAKG3$C=N#AQ}2k<<IA<ox^S{6%#B7o2aER#agT^zP)N{_8vOF2!uy%_gmnw^>}n9&*iu4qh*><NS&RY_&}Btgq9hN|M6J^h;wj;JG(|?&?^nO&JaY=*7tL-F1><N?aD-Q(7-dbU{@u1&j`~1DY<JhN;Ee7uKF%Ot-~$99&Tf2m&31WU5T|_ff0s^q`G4<L)p*M=AoF>xSg9<zz8?e&QHJ((BXY%8q0DHk?5i&<LOb`Hqb_I<}g25V?c7&ZA)?YRHBLka}P_q`T>N_UH)q8jmSvNC^i=^6;@Q+UofbSiw8@v{sTMg(5K0M{u_ueL6(keCenVH_81B6|*Mk<T%5tkYAAt%O_ZS<7HC&lo8a2;4(|m!%T7y|J@XqlnWRmnFdt0ff3~3(Q|ekb3W1z5sv{lhBbzMi8}E+0J=74uL9`ObvwUw`G&o!qmPKGPj%dm;>$ISus-R2{_L;54V7*(oT#<a(J`eRPAKxsa*ADP{1nHh+{KVXdiPq`aMqz=kQ#Ze#xh$!F)dK%9m+Hv21n&b8fk)e73ukEdYxa;uGA0Dn7zS%2s&uO#x0r*;sIip2t#+SgRS$Q-zx<DjFTSXp|C0GT)&s*M%ec{shZ5uPYiJJT#g(EjgimVyC1DH(+Dc#37tEmQ`%?PbMA~iA=WTAj+4%BFfCBYK`IimD3Kg1lsCs+Jv(}7)G>`P37H`w&vj0+>3Zw>6wGa@JOFgPg`JJ++QlBTkX@H6^l#0*45-J}rF1bOt!c4Ma+z}*S5oz`f#v!{<(QpviCH}kKE8Wp#)|iPzpDFuc@2sK81*dy84pc++4kuIrq--L>%tsXiO%mqAIwQp0)-DV`vKixgsT!*BL%5yE&-w?7TFpK%+yTs0uoT9Wz>?vdd7r;$sVA21oW@PoK0!aag_ktR#nvGL|!WjXL{Bqj?~&cl4H7Uh}1(hRa}<weTA~vnY+t}_;@opzr_%w!IflMjlsm7NDMm;w<_t!DZwJ-v^&cAf1t5s@wUQpf$OxcWTC*x^duA~R(^puM7Hmowdj^hVyt#|kNB((n3)+mMm`ZOpy#}mKDOzc#(tL9?N)mDtP^(ebc*v8a3;zQx2#J~>Tx&IA6Kiw+BVaj6Ykn)bB50DF82|O0mePc(R9Z-cf07GXrQ-Zj2m^{a)H$JWXB?j$ezx9BQQa#=qX~Ob37Eyx~{I|F6u4@jblmMc5|9Lx@I@kb=MWe3IuvZF!GB>%rqPId890SH{QWVkY4HBSjyH1(ulTcR@5U`xncUnLmiw3U`kQRn@K*@O{z7acs{u>>N@zSo5TT3s@1(Lo<U17xbol8#=eTPEGa3GIbPNc7~&u=R5D}^wjzdg$ijFjKuN8r-lLSM&%mGxmH#~5ts8~>&rfB1*J7ad7wZ6^p-yP}E!`62i`k0{om;%ZU!K}aPW)m?i2fJKE<#^|H|@&=1E8R87z0IlF@FC11t9zRi^*h)iBse5VbBYj;{T0tl*yEmUA+~8V;Jv6PxT;Zz{rpOtm4UkzIYT56e|dNI;2@P2QzWi^S-Lf;wFi>*&Pshvb*73x~i||&%yy!qqSL)R9l5$ikRk7X@P^u`bP+>7f%G?6v_KpNQaAtnuz0LKn%a0I9YA^DgmR7HVAX-^Jary=tTCkw^GhN-Nuz{0n?6Y2=O0!26)Dg{IjY-FkgS%W1SO0O}F_~FzM<g&T;oxj5=Y(6ecozr9II-bJOsF*Wg0{D3=cUM-D?EMaFs7>B?R<45Ua9BTVSo0KTca6KMlA4wL=lbs-!|mR>xW?@_%J<X9Di38m-9R8(<PQbu;MNvp)S9Mfa63l+!o|0-F}6^z`IoYfpIVib`d8<E6r><g!gbRC?Z0GF$<$Wy#8kh_WGOR{Xp2;fdL@n8(@tF!2+bw);j&VLzOpoXSqePxz5AaS%7HoKGC2T$zMv}$hGL!lEAFC^e=A%f6gLW<rZbt{h9_ux4G3MB8!FeNyzfHa?jg8wS6$N3g;DjF&slE1&wmf0bqWpFDQs7bGrn8JsbalHg%n72bct(-wPCqxZ#2clEuiHz(`LrbICW=S^=!}cou!B(owD5e;VNh6Wr!d=+j!mhtO#W)*l!7HB%+$J~8gp=&#Y0TMhvZYFm>lxD;(A!Mk%VCuP4^ibE0sGid5e!2{zcW(xTLeD1F_dO?ue_|PnJ>;lResj#<x)JDkQW~H^d25pk}WTKqIvZ19}?>&P9iHbt4<tJa!7?FW{s|LjRrK{KO^f5yM&1uuvGBE?x97XtQOBj4qadLxXaNn`QYJk;&nuOBSIW5qi6^HiI-L!1(<<m$3E0Pa$#{>8(`&k`-PX9qMb7bQ{}pliK^`Yn5U>-wQ`afp^*W>55`XDrJW~iCkQo>PrTV}O0^p+?yGxF%-ZuUiO`N(plv4W=@WV`$<}>^RXO;Y`I##574x}J5{voWGd>0Q4$lCR;KZXjaKdZ{FqKH_=OWUV;sz-^X$S<DU7fCNUn^tKaKEC=2QpP&4ka=|I;nhr(8TNd{KMHc1A-@-IZnFrabqr|?m93uPmnC(tK`C?AqQD^l@{z!D<8d8J1qQ@KJ(svp~QO6$d5*R74(g$N<m|ZNbA$L2=QB)R>|%{x9m)c>IDjmx~WjJ27)SX0#iEi(VNP=%yEX-4c3lsX=<6MPN?!FhkNi#qq0=rqTzr!Zx1!*)ft0&>Q38f?eb5vm-pfiIlIqAg*_KR3zw{sW(+CVI{jK$7lCSYKv##QtC3onW2Ojojp6ycS18zCu$kYLXypwr=$ERXF;F@2V2f)CdTx)p!`Q<;Zb{GWM+efHvwL3p?KDQF%8IUb9hCQ7pN+1B=&kVB?eY~yq_3mEOm%U_zU@Fz$fr-*A?ABvB@)Ud0&5bQO3fl5-CsK8f{Tg&J}hbk+esrVc=VZ^<>{{RE_PjxWjoB=HFUJZS?~2%9dX*<RAVSfRvhw|c8On9vn*enT!fR|V;XfQd)raTSWzp2)r?|TPZ`nNDT4<MG^~|zz-Vjtb{c>vg9fyc=2=DWv8Bp;?%zL|O7Zf03-<(jx{hg-<(1bRO=y|y;!d^`SX$d^=pFddmGzbtK!U<4+DUUy%iSBHuMeJky1G?C7dwYCBR*<~euSvYR0*T~Fd90Pyc`q5E!c;D`iX<x50sV<>}h$^f2I~Ymt=BgRV}xduI+NVgBW?|6fK@zjDDAM=^thSHu>0EBo{CAY@5<3iR15%l3SOG<td;;rHx{dFy`m>g8HD9ccsT0k|4xt!-y!IkN)G7jOD6l?wu<!uXnJ#VEw?^vbtpq_8JcW#f<77z7wq5_eS-t9UJXCUmcTy4x96EkEAwPx3NpJ-3T4PgQdWu_}V_f*EtsbhY0gn;Z)<WJMZSCke3wItW)y!Cej(hx4!UL7;T>HJ+(4n5SKLCqfU6W1S|PyxKO8h-Bu@5%eVOy=9Ys>UZ$O(R401os=!``sDeq#F+EJh*_{c|PfI=N48H#)?F?$0x=zU5gK7u*SpA_$buw(2#b7<qh)WDt9+jg$JgCQB7tDUdc%3qX<>}y_lff+y)@|7u9MtTp1Qg%TwOPTxgJGSu3wmzZR;_1x01*4ppfYV#KoQki476+E@>IQ`c-U@1(5v>!jH$Y~NpiXjDXxlK>B57s!3y&*t3rw8L9LMm4{Is$K#Pb5^8?unzrqmlx(9R|Xg`5;@Jfc+VCvs#Aa&S%@}(vX3>Tx5eGb%*{{TJtutTrzZ*bt3<V`L<MW?vV$<UqPTrwJaIR1CvH?jAO1#bZ7_j*Z`WkUHvoeZ3pQcfJBWJlejs*`v{pN#>k<ZJn<jI~)}zUnmioms8);B`FVyZ+l79_a6QK-i#n5F*a#+hEum9q#Qe_mn>8u(fZY+1nR*&KCGg@>QLOL{WM>0~e@^Ws$Yr)sq7T*%_kg?bWi4XlE2>7wo9N&zR2*@23R<c4f|SFw|g#aaWnC$v-BCAgvWJNObha0o~bUi?&U^yx*5~I$$J6jUbSG`o+Nm+iFNVuKoOgVdG6v)>!l(85Gn(z~J$YY5&+C_*GC0#oWISqG2fiqv6>m@r}G;g1>TLV8(rQ$TBHWII=<(v#*2j605bZO~CkxuMT_XK$w#m>8RaIrvA2Jyo0c2Cg1C^oX1}|q6^|d%8N13aAo5{r~MCu_;INA#t9X}6}MKO;ht0aXZ-Iorw<nF_}`N!4+pg;18F915AH-hu29^P6Od8mO<5YaJ2&bQoQD=)VXGrO9yI3T@9jpwds4IbGRXp&jCU{@EoA=-zRphn95jv<^X6JDIqX1#sFMNmbS|C^>}uy-4C;6Z%E`@Wx9bBLgV=wX%u9Xt?`msi?Qi=Q@P}5>Io_f_pbu3t0^GAfjuFHCJ3GX_{6=fvrj&yYja7_0<iEoiO@Oy(l=wlr;Gw?XCity}$uy#mZbi}KOgzAY#~?g>;2o)~zaz6CpDN{%%;i#kPex;>jeGJL_m1$LIG5MAQ<ZAw)B+rHf(QE<r{e7F7m-#22G!pbwXSezzc^aq#mP0^m?e}gI31Ekkya^vHgZXX5SsU_oi+8@`WMpo=x7c+0_1Gj6=?5rLUnzi%3zn+=GzK(&-!f}wmU7a>+<Y;Fs$n<dLZ)0bL6n?$KwZ;R`#eq=3&)<92jY$PZj8}hjPa{_l4DbtHRKPEp@BoTrBL2Ff03uN9{}Mvc;*SS{`u7D1Y*eqVopLVhGQD<i6X4owapeAc%;>Im6m*ptfzU{^&MQ`tsb*kX4Bl=k8H;)yxbF`&pQI4U4}3pnrVKU8WEC1k0u@=`()vVo&~3&2@@d;FyDmHC{*T&s4R%M%fnzy+J_hFIT6kG)XVHhyUuVOjQ-j=$~}5Mp5GSVcsJb^v6Y!3V{6W3-GMBz+z*sncBAM6kgk}b`@XmOUB%hRNKjIwGi|t13_?J7&o(#->JR(!bN6DpQs{1GA8`ki7LGtKbRAn3~KTco!9t79}oR$#RT&W_`{HB$9tUE6JKJwI)844Cta|XIc)wgwr*UcK<spETr|7PiYoEeT<@tUb1KNJBGV2$`c5NR5%IOF(FFv!nAyHXy1m+CyYo_#DdS|a=i(A7O=vFyYI1+&x^wHSO@z*McJ_IaLHtW}|J!eXn{6mv2j0rt{+zz_scgDjt}HU|9hbgZIxgdE@AMn>=NWU+R(nga2U}du@4Tqxl`5t_Q=2#av@1tB+qv&=S3OB@dTu-3!bIwrzGrQv0(yqK=uF9Lm}7xfos@v9z?1yU_Y=zox(u_)BlT?5Wm(2sc^`?lz{lVk1TX9t9G|fUC(^Q|vn%ShN#pjSZW*w?yV0p}u+~q05V`;nzxqwr8$QnZZujUG=i1_#x`)oi4SMyVtikQpaZ*0?u<tUryT-H81-eB&>_$evlI`0AYTma278qQ2J{z>N>id+tN42+PStd#D-4l9a<U*>yo_ZOJ$63ZBU=rkf=ngY9VSf%ZZ~LpD$JcEG-8aq-Dj9MSgFYa<E{hkAcYx&whd(De_)^rin|t-@{Lq_Oy$4|2hkJu3*dM)e|3h`u{2H;{)hteGZv^OZAO30H&b+g6k5$du-Dz2;>v&o3$yx|bksbe*<U((NynuA|k}g}1$uYp!Z7WNnC;6a@8RuRF10`X*#(gY_kH-OQhL1x(N06+|x3^!fD+fQ>%B{Y4y#4S219ses5c&ijG9|vs9^ZDpfapiAK6;(ndsjPRLGCR;QiA;(>Gr~hUkzXvqu&|#+KKiXhWtSYAh~suiR2bgLD$#vcTv&&^bB-;zr~wu@xy&+Ye*b|(MjJ^y;nEt;#R;;oIlrKP5Wuik`xAb!6ScK)%-j_gWrA8E&L?P+72Bcv$3i67>`GOGPfJCHa$c-`bjk}B9LEK>1CE^Cf*}|@T$Cd@B#6N*Nvf73b+JJE&&;Ii9Rj_zbC!IkLscGas{F?%bKOKV4EE8;G=zHPNUc~EGtW<;uC$W7T@DhiOaOY>z`a(QQReTM^r8Fa>8Ua4fgSNEu{yZ5$jBhgEuEg0B`6+Z*NZ=E^;wUA@CCwQBS$)At?Da_UEZdi@$q&y7*Cize^qrNnztJOUSv^=NjQ$Vq7Dx@aquxA;)c*tdeC$ca?Bu9(`t!b7u#OhGDvqLe*a7=yjiR)KbCImMQt0x;NM8eBiwbQtv*KiP^u=^t!ui>W#nl(yqJQ!Dy|ap3WV^R~GY4_IB&f8ZdA6plimLdjjmL7$jZIb@CeCcM-jMfBO1=f>6yn4=!P_8-f*YWA68FhyMoPKAwK~>FgI1I(heEnOyBmj`#zh+g*&}LjL;f!{;~u`#C!M@ctirf?X_QRe%0PW2imY-JNbJgR^t?@#lAevQDS6Z-j66B<g6~sy^s!4)*tfrgP71;h~#y;ZZN%s4v7}6%J;eHDz*yaJA<HRpgvML`n%J#4R@%sPm+xeJ9!sHu-_9>ZKi)J3`MXUHEUzi8nFP%`2;^mm643L3W)?e+b1hyjJzu^!pwwVT+$VD4NQ4IPIvzRhOL~M7e5~{_SWnGC%lCsL_p#^w31c-1z%Ydhqkbods1mTeMs2_$C(MFsXIWmSR3epC63HKJHG~+F$<G%9nv|kN#tHkic&v59X1xmlh7F3Eu56@o(6^amj(ANp6?-WS3I0qj^s;sV~UUe>@d{U$1bn&hqu{D6huFUsQBF?%&{``XltUZ#4G4)zI@9d0~q|K*SD@DK#U<t?>-k<y{B0p7mY0Jh1hFw#C_&EOTBEF~8Qr(4S(}(@};2!5(V|m2^L-44B{7(qIJcrFwm9>Hn*};d6T_KT|{=q^K;w?hed5)F@@o_bE?rW}MfvCO?GAs!Jrl?FoRBmjENzm=QMF9iQdxkYPj}Ba~Ax&2vVF9M<_1t*%z?+?YsNAT!n3xz4MgYy$kpy(y7C+5_)kMR8rL4{)gG^4mpjCyv_d@4=a&eD|V4E>AF$^E;nlJ+i(%>wXz7Stn&l^)?INXQ)KH!X(?<#QT-17lMmss$<S{Z~dO$=6^v$zsxyT44HM7_>$y;x9^VLVUrboV-JpVe3`|4qw>n(U@UJg@j6lY^~G-M&JC0L>sJxzN$CbtQa%*oF3pLNu#R&4uLj1e?}_!LS2;FE&%dB9s5z%OW6-)h7-QLi@47YMf!A?mQi2@>3><xWz}3O>*z{c+%(-FYgk3e8EnaKJA2CSD|22O;8FJn4{{c`-0|XQR000O8001EX`)c4$AQ1onkv{+cCjbBdWq5F9a%pX4Zgg`mbYXK}b~i6`baiBTUu0=vXKrtEbZKLAE^v9hTWyouwh{iWU%|3pNQa{DoJre}tIQ<vOWitg#%?;D(P%gj2}#UR1VfPLQ#}6f-CY1Az!%-wN#hSb5nL=5i^cAP#nSivcPCa99xF@HuqGGfQl_HfnOE^T<rTJgtj+QUNHTAq-|=i&)f|3!dAToIUZ&iuYO&)oUM#+88*u_Qsh1W^%Evo#=cS^qxvWH)A+7XS&ApmKYX+k*iJfnA3FEMW#>h)q@w8#Pf<Ij}M}DRrARfHQc*T)V<4JyEB@T4U>wKT5B)cbSk9jF;CGcu1n1Z(xZJEX13sy+JsQCe@U<6(hea(MqbC3?$Rjg(WFV31FM?tDqO*@x*{970r-W++&<Av}0i^aYcCof6%ZPV5~Nxb}2iMp}$DK$+hiUOvEEruxLd)5|BmZ!}^H~lO{xiAl9d#cW;qOwvveth-X^nG<=2d-H+OeS>;+Bor;!Nm3~ZMs1itVo_F$T+eeiU<@{iV{>;Bu6HXktdJr$ya|#_Ibg_j4gZ6oPX?E-YS~NZsHQAHfEE-%4d0bFiiFWdSr<*siY8xypgI`oaO9L3JHtcO(s})@b6VA8wR%Y{5B_>il|ksj;PNG*{f<KgEibcgH`WpUKK3miDV}Z?-b93J4|bS61O}tHqntqi%?6+Z&}eYTQA=oSzf>9cX<kHsfbRzIf|^ChI&kP=OoWG1&9b1h*qaa^mCHqI!E2(o-;DE9c$9#VzB@d@)Fo(NiN!8pVzX1Wds$MS-7GJ++9W9`N~^^4aFyRm!H~`qT8&VYz-~`4i)(|kY8)^ytK%lta@_zt;j()b-^m22i^e;b$9J8Va(8bWLnPHa=udZtf$mmxA0c|yxLm5RMVIJ+iN}IuIFHcK_dOz;i22UwB8#oH3u!0o%oNvfO%g4@1<!9y?pPfwy!%f4nD-Yim!~`PfXqf{Z5a-FXV?Ktiby`YmRyWXnCu1DpJu1QS>$dopEpDU+@t~D*53ACajEUZTXOmZ@==l%HH?jmyA6IHbLYSxoq_K_z&z^ptoJC(ZC$*0}s^ief{J4&F*J#O>)4*osCi)FXo5c_$fHsM*7=(1$gM7dQ8cuSlJ!zRaHZfqv3%j4`U{qbH#%ltU;(3@PwNPHg&J)ZhuSVX<*x-cP7a!myWKp;RjLY9DWySt4H3W$b&tDTXB6Af3?u_hcNMsH@wE}gwD(pC0{mT34gqT?RfFUI7;IDo1@3?sv=Jz_BKa&c@#w!W6&YVz{f~X!bTYbF~a9wVsttzV-1TBya{{=vz3rATSe+e&{gRdTrh>!yT}9e!2d)6JD>z*w_qzU*WlN@3M}40M^h(Wp|#RlF=r4(>m^3VTuV-3DApY{7*p2)g6t2Vlb#i^6A(7amUoRPY`+pHqqQz>Utk_M4dj73YNeK3rnPvVfjX|j5iyL_FZ9OVk49>6;Qn^&{b}9J!(kGxMm$F|BDV{E64o5df3WX&plH3{yJ?9Qh&rcFD+b=0)ipZ@8^!4kJQr{~!=K|e42Nn#7GJN8v5sXV<BC~@U@Hjrg<wqpgkfat#1;`Xpoaj+mmf+MaJ8j?k#w!j4pN&>(mrm2FfxC%L6cF6c49<OIWD^dC(5&afl21byW!_`!A`r3d3nQ6D=%2$Ux*A5H9!c^KTOE8)@WogG`Plole1g6wv`R!9QL$f;f#%Fy%_C}*5<>gg`7+;Q$j-fE3LU{az{*{b)IMXXp2M6urMQ<kFMprnwd~t%(5&X8E0rMWf83-W9WK((wTPa05uZ}I|j08vok^pIAMSXUZa=MN{6ja7KjuvJ_J62M_;ke;B1gK%9|+~K*17tK-fV<3&6p*M)KWG@#zMB2Z-AsDz5)sx8T>oC(0%fH}t4?O+U}1Vhy^0e%aNq??rvH@zI$lwGfTJ#gXBlRs)i@_4K%lUkSj{Cs)^hjS=fB!PIC{w1Bu_415v8*3M)wlpwA-%aVrQ!H!hDv@eHlHo6Ve%ic%A``$ijmyvr|4P(JZy(<BEqx+=I4lQb5qkNr*1*Y2nDf>dqjF1>cvW`yy;f=*~fG`}bS%OHNn1-X{->F48ce4}w*LvB|P{89LI;`Q$prUc)eej<xzyIdlzhIwy_!xQc^zEDf_7C5`diRc=K6c`Q;h~&pf2+a5gIT9H$*#>XSoQnF#wDPm1DbXMt`C6;LK@p%^?@vbkpdW^XJY9C=sPt4jWhp_z7XQ3MXdk9URhc<djT!By-Vo5{r#Y8YLfSXuf^E!sI0|$;4u4{t-QfZU6<x-k%3PCntBiW4+wlahcSeAm;_nA5<h*7pfCcLb({0|utbFLQdX%2C}2+Y(}$%w{DCCh$(NkIa|#~Ej|qUdu{dnUj$KFE4;B<s1~33Yb4C!3SsKXfhK#|qw4`7w7<uSy;iD%hM?iGgQ;%+)*PZT`aN%fC<xR4$88HWPg?((D0RbqEf$tmk-Ls<DvGfMIO)9zI2bP|bj?7Z^N7rh*=uc0*Nt4M?<|h{LK9~;<y=enD!(-+Vz)3Lx*T#{mcQGsoSQ%jAw=ci-`$2!p(ZVt~cx_TE=qMdQh6$aq)yk&USYg7LXw(h8-V(ee)b@8;YkfjdXLr9XZ%XlgxPkCr(^dtZ4yOjdHIFW!o6!LZI}FZGBc{^_DyyaQis8)Ay=Qh0k0YOl$|G;J8N<YFxU%27xNtk3C(+^54Cb0J;hu-J;vYUPF7a6{rcY^jj-<G=nU-+-+2^PUF*m4*(b;ATydGwuqpo9HdeC`t+$|WN>n50wGg7q0W0O=-Pb#$%O+*HOG9}J(?SJ>?yEm4G{5kW-rMzWz4nYbRv%^B}aso0L8KpW>Fv*i1qk!jh(9F0)kG6qU4?IC;16q~U268LCJwm(_>bMU85DlVPCAg1i<fYlPOA8S!Z=VeG=N^?Xfy_I^MgWfdDrjJIa#ytaK4P3MAqxX{ejo@q%*Mx<i%e*MfEc;#-8@@=1lDfIjR0LUgpKMPjB_XC(A+-MG)Ir8nPCUjHrK-hp-I!ie~%tbgshpv?ZXV~p>{!Z?!5v55Ip}%Uf=Sp9|e;~=3H?9X`OE8(A+eR?!&9KQlc!*PrbGr&=AmsBQwk4cs)0kWw?ofxQV$ZOB8yA@#!>sPY$57SvAkKnkOBNCdby6+h4)J5f6kV@NxOFCYa<mjXt6&&mRq9=p?^zvK4cf1Ho6b;@G>XtYUabZvukTge)BlJ><Hb9{-n?6?vl$-*^n3*M*DPK$A(Y9Zc*1ZbO%{^&Sy)>H$dj3F^2%xK9ednFS~qeo?H&r73MDrh^?j{`d6G%Lcb)Nds%EBL@2Ss5@PRxR#B_e98e{ob?}M<g>I>#y?9uxRtm-cLE|%-x{-oGp=RYUQC?mQrNSDsBMC0!MpR_`SMl)t>Sgh8o1IC?#m~iAkuFz3+)hw$(QNN4HrHRdaTFy>+`A^W(pC~&^;WS>Gnz`YYi{3;SiriArkyO%W_&ZHOI8)IJ*=>^SpTK68nNvG&u`{AJ?CrrEu)2XQa+mpW@Cf4^6Y}q0VP{AjRV%)dTnvmjsOPBr;WrKog@8Fcd$;KM}&X-$AL8hpbE0$<5XFGd$UJ-Nf(X!|<%7#hg6S&iCvDA|M8UEYIZnAzto&W0{}Y?-b}~z{V^sr97q}`(`z>qW>3nXgT<d7rdb?3+6g5tRP^tyIiiXrUmMb3O!5J^c#4BVn=x_LNI#rt9wz^X4HQ>N_0)8t}arLo#=~pH=}f6-0&hx%#h)s<ASbc`)_BJH}-3vs=jIed0Bx9-<=<#LQOiS#EH@WXf^i2Xr9TKrT6Ir_H0MqZ;(v0n9qtnNWd)~Ha+rrb3U@qC9v`w+_OvFlgVP%@qusnmDfThMSjDF8T@g-`Se2ER@r0**UYQCFIcPFg83@~rx!m}&3TP^9IQ!Fy>R{)JMG(|SX=Z(F-aeOLRN@4Mg1Z~p7zhiry4~}<ZP^rRn=0)Dl#J}G%vBF!?a8@>M9@CZqZ{fGn}p@jTV;x@ceB^G^d(5h6*u{-`0Sn#3_-SXD+su75EuyzjSF}rftlt$I@#fkX3_?>Cp8kA=sr`HyY_}N4K}VRx@>LZB+mVeSO1Yh<@E*OS*QV%7Ma<^p%tv*%Y#ZU$%=_1ONGzA4cAw?b;8!T18WSF`c~Be9gU=A=Xu<CMT@Q#=#_$K3&?CRrmH{*JLc8Eq{Fb{25FGuh)%)Xx*?raviuUhizCY6j}unR*E}P`B&+rx?_ZKVCSQ&`w7WL()r#O)7eInb}{frB)NOFb8!~Yq{VfIBe_=7i3YdlRL-AJ`8%$(TPEI5W^l1iGGLmWWI=4&K)4;;ksf%7fmfx>9jW^_gD(7zr>)bY5=EX4IcxO*k+#gZ5ee0XgeoURjT)06<?y~AHLi*R<n(nirC&eeGC^%e<ULMP+n&i~`oO#0#T44H3QF92hvAL0uHO--3Q#o8PuCwgj={25tD2}Qrl0I%e<-r-NVQ8bE@3ckOtn$4lk{gGHvjIORcWevQ{b>_nTAY9rA+SN;DF5?c4zZ`h$m)e7obFLk6{Uhkycwszon3uy|C`Y2GxIR)gcmt|Ay@<>PBvy60<FNzIr&#a4{E`VKCUV?t4`>M@@hBegd$B#VW?jF7-}S_XhZ(4}M3ZY6Z}^zIpxH!%F}egAX_LM$fM`H}wSveX`Sl>KpZq4>N@>UeA1Y2Ca(b2+Y-nj$OOzi<^yPZh4ss26nJp={DzfQZ<ps9)uCw8Vob#8&Jz@4`Z5DdG^_rTI6)?m}OuN6&zy3wzlz}-SA{K%D%WBN7)xoOq8Ym_6H&sq;lE=7fD${+Qauk(Q>2mv0!iIj<LZuRs4g$P(6YF=IyJO5OV!D-~My?5B%{h{r}?S@+Uv+FNyMOoH%hr166^YHvzf4Us_J7zqz<U1uU2#<KgDoJ%Yq-DWN<6l?Q{91w;u~8*#>O^VGT;b=DHM!@4)f5!~ANjkw`>GJ>tTS^cdSLzb+PM?n#F>Z7MiTEKE4fJJUt$%j~gR%sOSVFb6Wi>GTlnr^H(yNwppH~p26x&FXJZFA==-OX^|)07b0G9cBmSs8w=`OrK*4p@LMM#W8j&NV(MDDfiCP-YT$3o>3CX(A6`TZNK?DX*{Ns|#4Y4`82ImHp6SL%Ze;r1K|K6wO?5f0$1@s?Dk^*g%15tKvsei%YqZejC5TD*}w2xOv|tDmS2h?~1ner}=!GyEo@dg0AtKoy5{TOcu01S98umB94Kx;q~mbvkt*PQsMQs28(LW!}jgNE7vn#xV~<Df^vg-)LTk8qz#6^#f0u?4p67!2=PCW`LA3TE@Hq4@5Q}&_AtO}!&)kl{ycPR9a2^8{A&IzIRl!o%SFUEstmQj+R8y{(LK0qbqR|N6=c3<W<_zvdVf^1-yJ9)b?2y|W<g=eROg8bsAtPpKl}&hFGyXi@G|kfl!%>9>h9=i{~4HUL9w^F1oULpt3*>=UwrfW-E-ty;%N|&mR(kCf91T#FQb?G%jgp;^F5c%6wg;L6S^o#o03bJh&nmI^i=(G_Z@Y|R%aL=vmO5Ns$<P0S2+5}i)<;{1`pGDHdS9X@F0Fy^fLNPJ#s!DAAD+#<<#bDaQqzbWSl8}*mY=G4X19<4m`26;PwaOmONS({qe*@;rZnQl+DaV>Hh;zO9KQH000080000X0E{A@m&pwP01YYt02=@R0A+Y^WpZh4Wo~qHFLYsZUv@Vyc4=jIb1rastr~4}+qU_;eg&p|2$fQjlXlmgQtjrXPU@YT<Xop+&zIvtBqVX62$di$yIJ<z_q+fgC`xwLOqW;^_`Ko4^9~J%!?W9Mk*9erE?+$tH@Vu0RF|f%nzYV!DP)<67Z>Mm#Y*Xos;fJ()Rnk8{Yl(>I~fdKz8b$1i=wHFsMkuoe0Omn@{L?6LQM5$t4&@jep)Hh6t#Hs`r=oSmnP2?5>u;6H)S>%ya@T0>PA(Fk^intEYeDE;jPrwMi#sTVwh%O>b!1{fh>jEZ0kF*k><l>aG_Q*y%T3oL?w6Qg+fN-ZKYFX3}7NqUamwAh$72@N}1N}9zg|lCDVE`7!HSn!LrgDktEBeZYq@|u*0pcYG7IFTB5kd!ep|RX(5dPfgF&@^0bcIM+5s{BkMI|)puKxh1XBZyMco#o6YtPNQiRFpl{DF9Namoc`z9K-BXSL<abrhuc}6k2DBu8h7$$FF7wrFAm9aBH6ZXzfU3n@3~On|H{Xtzt7d%jWJn-R3m(R+ElFj0BMpO@e6^8Dh5Mh0WuavauqTs~Z<vT|jLBu0z=@WrtW^v1_2h&=7P`t*<+)U9S*Wyra$?0ubpagBR;J3sfQ^m~{!wMBGnI{-ysZDq@OY#Nd{xy}7j!BftyB^$sb)B`l7yfEFi+ck;)y8ulc%zPQ%B%TE#ct7h-)C7*GUo?RV+uWOMK-^fh!YlI$#oD+~Q+jYq1y>WR;+_-SAUO17DTQjS@f0qETm6rK@Q8uKZByUD-Fv;V9&e{7~>oZk8ybA_*tK>5Rx#_7=r=;^ZL-LhwyvYOw%^B_8D6r7la@Po3YW&l0I;60KmpN8-Er!}9g9oRk!GdU6pfwz~G10<(*~5%Nb2j6n6_qxz}YqD!jG+O(O8mTc@k;|E8wBUjWRPg3@cS^R0=klo8Ev1izX(lu&_GkRc4ou~L3zF5n01v^Cgz;7c<AZcJrzcqOU1_5hZ3AUn&JB(0;+NcsWLMg>Yovz&dvk6?DWIy8W4;{yw5@b+eUL!opQyB3B@!%l5pVvCUP`&Trm4c{W+BnV+A?hN+A}?jd8jtYL48ruJ%w$!`yI7dD+(P848(?+~{9~}U`u#|Z{}zBT8%Q!Bf1aTUOG{S#aQh1@8&YMhWF|CdDK9{5*j>E_J6P)?!^C0qvL;Y7!E}Wbi8#bG;3N%wAqiRKpl({4kcg;h5;CH%;m-FpyBz+A0(}gyKZ#63va>Fqu5I?`kS9Je%y5ry1{sg|f_Mw9i^Rs08W(btmyy-s^+_zAydO<uQAGC%2Eh_tgml0`0E49j2iIr|$V16;dNO??ZV<~@iQL(eRMMbW5duTOp|^MQm$ER*owkoQ%cdw2+9taReVlPVp3vX8O^`P2@ycU<JD3vV2XbOOAkzuB%$!n5fXN0t#Cgr|9$j^uXGoNM+;mP;P__ZDMGvku`4HDo{Sq+<Dud$*0CmCf9zS9B!7oR`egAyV!JNQB#4CFU0%HxNMTje{aWUz6mlU5cs|XJ@vfx}qr8iFpf#MA){iDYX)jvbf$@4dt-=AG3muD|8&R$$47~Ol&2SGbvE(z=bzP&hqae8&epr72Py7|iGm%+nzgI5*fIE$BA-rzPj@-~9pM8}k>yomf+JZ43X7y*N31y(R8VI9%uu-aNu5M@ZwqjtN|Q=n&=ekFim_j>?IO#eBjx4qS0_g8sKSKzL5tC4Gdf6p)-?`D_5@|i`nJA%D~$IQuHoR6oK0u3Z+<K}+nZZO~^y{LbGW!#AZDOcq|g)aUrUh7f;Fuqxb$E<S=&K8F<y!Uy-H%;Qr<@u}Wo9F)+|1f?2!;~5`2#MHtLFO-1^$1#>0``~PjSHQAfFtD~Jz)p<C9l`w$N9-j=*7Rlj4@G?+ZQtZ5DP~V(!DIe61Pn~@kmct9E(UGhG7s9DOoB+BIj-p!>Bh>pmp{HB;*2_sFix0frE29xP&b%fP{}!atmGrdkCtIn_OD_ge9SKoH$MWv1mCAO~O?>tQp#%b*pr_LJ#3%1SakiXGS_zRUl=5*im35^RQYcdfWUK;JXQ&UCZ}6@GvYuT7!%&HYe!o+0*x<;CK>#7z<P<x88~8W)V3~#@(-d8rT0pcJI-+IaKUZyzV}4oBf1JvH1Z8pgU^dWeF=hL|CpmjmzJ{@J_kd5(`w-$#F*$clgR%*A8F%%pZ#lR3|<C_7Pn)og;-_qtb`x@I}!4F+Nf&x1aXZ>6Ptu!Yg4Y400QW3&=oWVhoZL#gXBx7d!WCD2Si|^7BKOJ>xirjWOZ?{@!j6V1KefydMLOhVo-!9AdN&IWsCBL9bK|9;w10zEEj=dHRd<U0W})G(1DIBLS${pW%sMed^m3$Igah_mze@VguDHM@?PjEAaXPxNKFCm&zV3gu+VhI3|gRCt-zLsDif+wOm<i=qfap5`qIOd|v2VWkwKvPu=*;s1>$G+;hqTBbSY0AiZ940lSv1z(S{~O{nOu6|{!Xv2er}yR~YAH`mi&8j$T|k9X?Q0Vna#2#p*?Cu^0j*4UEKw_Tpq>yZyY{^TF5B0FKa4brxK;5l3iw;wN$)F%hr!&-LI&XkO*Yp|opugq|87>g3BVrm~^>}h3F)N{-}wt2D~+wkYnCo5G)i0tt??Kw7Orj})@T8=T*CM_5xI3OeIx(cBVP<0z>_v5Bqc3Aemdu&$`?55m%chEx=6u+&cPNAZJuKOjq>LX)k&tvGap$h^ln~D@i2Gtm6(d|^uc1NeLzqgZTqW5Et#%jb>1&#}GV1vEIH<PdbGI`3of>dCi7}Qz9+!Ec?VAh<Z5?tYK9@8K(Pl$O0MI|^d&WBde8)P|nEY4TVR4<oc8RXG4_b_(!ZW^wa2AxBPUB1nWLho?15~m{1Y!3~H!Va5|lrm65s;M8p3&?z1RT8xV@>wNzSP8cC^-S9xvCq>EU@h207z|v?(&Bsd=6Dbo$RWT5gAia}>03|}tbb`hdF6N=t?s@wrc|p1p!cE&5BaD8pZB5wS3x|I0HdV|*2o2g3oDq*24pHg<Qh16Iq)+{;Yt9qFtAtdFWe%XJBnA<(R~4iMxqQobj`>mF96)+`T2`?@V||s576=!0>mg#&VrtgHuC@h!tKqSgFuA<Hb(J`w5FvI8pX$<^Xo)qtA|It{dwrw?Ut>QGCP_2$44Eak7nZar)Lnbj2{nsv)&)L9Gi?H9eXy&;(AZ)_nrL{@aGK9-+Dju(T#BL8K09O#e!`MBtqlz@yYTh7!-h^x>D>DK!!9DjfnMQ_hibD-L!H)selNJFtyLXUvNE0<E;x^J2wejr1e$Y9z#b_hutXlJ~MY8T3I_D-F|VTI=eVdx93~0&rUDTe!AN00WgXtP{4)${8@e%MBzW`e8l?;J&-q!S1qK8hp$<@%lV?YO&DGvtIsLZY^5WBHQyvByN*v;L7<Hb78$?mU`&HtOgNsq&+@nzn6}$IeV?cOnJ#%L&u?XBIJ5q)D*a5@?2h+HmCl+}LAe4s+}v$2s^lpm;Eli}AO;1^7D{nj2j~p72ek)*f-0%l#6i(Q*R|eED^vyc1H#?y1+=*AAkkP9vC~bFF#`eTfdePKofc{dj%9Q^C+<rI>huXpeCKot<l7YHme*bFmTH&mVtQNKK_+7dDGxKHTVKZk%~AL2jCU~?QAAY5I3OXWQ}GnCJ2M)GMCehM6zekqpxAx3WkWxCu!95Lh>8=-!|iEwV4M(OTs#DSOs>fm$_W*bguL;CkW_duXdA%}V-TrM%sZk5kHwY&HiCL-&wp&s-|IXu0{hIKN7tV@`dxdfF)<Kd31<+<$$sm`_&vFiA3{ZZ6XuAIMj_l`2rlKI?ZMw+bng%Rjs5Jvk0^qXhrf`dt{b7(%W+!d+cCEcV^=reZ!`itU;eFkfYd1lI~NX=^$5miz~H&tN^*O5dxw`2_In6IA~T62HD+f!*uD5-app5y07F_0K>XEq0OQ&X(n?|{z5q*|LwHSM+gdu-y&F$Hxlwqzhi7V!%U$H+tim>3bY{AU`n~7>>He)vW9`tQGYJUd0#oKR8lA)f|BYPp4EmA@UAm&jL5HO`$L^x5azzUaHF>2<Rmoac$8Mv~Zoz8D$bf#s1z$kK6-|U{AqA5Wg-+>cFl}ZE`oRoL7`rnXJh_X7t2Oj+R;z}3)(VGFL<|_jR5Z{ib?SarK;QradFZ@$<0r`%^*JPze9;g-u;MA-xX3>!dj*ah-JJ(Wo$4y%+_#0^o&DXY{jSlE)sA18I}lJ-zQKcjdfy?_G|*-FdZ*juFN?kAZLhO*%W=z*FXjmek{K2+-ADF5sVQAG+&*qLk!wQv*LSo*EGJBYO7ozd%Uhz=8-Fnc2WR;~wU2do4;Z*WQF@Tc0`H%w2E$?cj#G7e5y^z1hPU+?4So5!`K_w?`3PSFe+5wKoq=dCbJS**-+(hRT<o}otk-Tzj%bh{Fe*fd%s`x;h&5C!cFJPV|1eWI5_o&b@RV~7BkVlFE<j*KUL&8GO;N!vh}_I&GK-B<L#))D_EFkU=bqp_q!6b=r+cpd1yD-^1QY-O00;m803iT><wZK70002d0000S0001TWpQ<Ba%F90Zgg`mV{dF>Vl829Vr6nJba-@ajX?^+Fcd`hJw?EE5~PTrbfLwi6udx5lOG%S%@0XR&FxKHyBUVzG4H^JkR5r2jI4Fq)pC2@A1{_PMTjg!vJT<OoHr1lhR+pc2CP~!vwr9Ymhx(fXi`wRbX_aX;?&GJhjfwh?ee@%LenDQ6OdB1$sfrVGJr`&Ku`&iG_-jnWhD4<w~rf4Bfj~5c>Fu^P+IQE16RI?ZN}ycP)h>@6aWAK2mk;8Apo+Ezm!V>008j;000{R0044jadl~OWo>0{baO9bZ){;=En;DFWiE7hbS+T3Qo}G1?DZ9zp^mWpNWw5G7${-J1r(IBe2F!YCB-@?n6Kw#Q{LY0-rhd&H|VX=2t98X9D?VSkOc>Svs3URczoucVKM`^iCdpgJ4f6EQBS0D3qOU4zqVhGW|pNC3@7aqVIxdQsajNxC>e$ec>B9Tg^(E<=JVP5^GO@n4Re?AGUM79aFD%Ms1}kXeF6lZ!X~dp!x9ApZy;JQ8&MR@&tqJalA_#;nzh!saHlaV3Kk;qZ@aiYnS&8th-1y-VuA@6f+P1bW6V3eC{>9kI*#yzoeOI~d8Bm=2@95t-EPU8aMT!GKxey6R<fvluUErTR4k?$ty5|*Bt>bL@0XAFJW*~o!F7g&E4RvIy!K-bxId6{)aIgdQM9hJ?}9{D>L@Cr`Yt!EX>RgAHT8mX{Qu?92)e-T1Atq5BM$omP)h>@6aWAK2mk;8Api=OHlV%$00001001EX0044jadl~OWo>0{baO9bZ){;=EpTOWV`Xr3X>V>Wba-@i%uQ>=AQXk+y?>=t?^UC5Kxmm&p#z2Px-))6A<@v7#r*qby6LLB^Pa_XEdARp_g?&N?MMZcK#srfx#zZ7%PCOgq?dKI7N~hdEVXU>BNauGdb(SyP5{YV`cv+G+b@irJ0BDEZDnnX)fIMqfBT^4{o*iyMlD3pi0F+6lnJ<0$|N%w966$lLj31lvum}~`766Rvs;_W5<g#Wzjo4~Trgh7Nhgk{4Z*+wQ7Xa50mqakH3Uw608mQ<1QY-O00;m803iV7Ql)QT0001O0000N0001VVRK({b#8QNZDlWCUukY>bYEXCaCr^M!3~2j5Cp)#t5{tD7I{Z$A`Msu7IVU!k<TRUC-WT<dGq!>S3+%Lr!B-iwCYg$6M9~)54D1^tK-Vckqcq`u%ouTI?lGT;!7?2@F*f7r1WpsW+~wXnc^c(5`Iui0|XQR000O8001EXJNEu}77qXbhcy5I7ytkObYXK}a&>NWX>DaMVRCe7W?^G=b1rasy<2UM+_n+^zQ2OimrBW=&I#HgSa*vWt_!qjoFGoq50`_im8e_#w2~rHcV1ubzxNrw=v{kjJE(!gD~TM=4Cm#UA?<nIn?s!EYxeSoe_xbEeq<G|rI3mj31?L;4nm1LE?HLAtmai+rcEMtIcNMmMv4P3lnhs^Z#l1++H)2sG@j)=z2P-03;puv?|v^?UM6wQQXy|c_7h(A7dn6}XjaWG7jn<#s^IrnE-C9&GEp#bI5aAzWs`kV+)AP#q?@*nB`ZpL;yDAKs;rYe=2T^^!j<QFt5sH)2Np$HqnesWk^RWxq9|3YL|Mqy%Dziu6(@NtCD^d9u)sbSJNNibmWBI$h}GWxE~Qzg!aEFii~P_VqWGc8M9$l1XwXbj$4V48ZqAG1xMttQ6}?-nRw>U|l*C0@2q@9Ni}QwWXi&f|{z89mRt$ec*5^JHGHM^qoB$H1(3N6BibARw$~Rxu!~hQr9r~l+vh9FiEK`vvOUkV{r9=YvDAgQFFPK#IlQXTvy~!9B7e{}@@-2sYKrqmDi6=4=L%f@+*N%nc>?er!s;<l0_g<DoCT^NqZ#s5QuSXPMO^sA+$B8c9NiP^n@PJ{P{yqq7XUoEx(PPU<NKeua9+#!{Ra52o8#b?kslu*iR~jsGy-YmI%UE^$O?C={$V34G{#aE;O+V9HtEbmxabX6tqB-o~t1Qvp!bA$67ugY{dqKc!chwjUyb~lix#NuBQ1P1W$`XM`6v{C>Re>JAn%l_C`*$ImikyX~EE(*ob&m(KXe(+MMMZpo=Rbl`<4>(}Bl735Pk}?jjpc;KglTiAq~9_*_Q+t}ev;rTBCFc|R72wKJ+ESLS=Msvduwmayp7kJ#vv2pQ!Eo9wttUv*_(<mUEV{AqW@tk48^Lagt74~l>7MWKmN&|(?k#=Fe9M)UZb*$Z@eG~_xyb-Zn#vwTh+{uD4_Wi^ft5-`>zfFYZIy?DL#M`>@s4>AJ`Y{t7l)kzlOXS+88zB&KTCcP<u^RFBF7j#S7o7cOFb!VoH(byfYti6urDo9*Wm~9v^mTys;z0n#ZZ%F&MxaJ#Q2{3`1d4Ap~+WIOuE9exIr(!Ui8i677zP%NfN`NZl~~RjYvBnx$zhG|<V}z71hFaI^4mn~K^u57xNgg$N)~d8;4Il$6V)78ML_t#eLU$Mq4EUH$<bNx%K7=9zfE&AjkIFGF7P7{V~I%(l0%A*<?Rrz5>3r3|xFZ$^Jf?|xxp=#Slk`<fyV=^~i>oaGG+dH9JIlgA?Q-55wxlzz~S!U&+0JVvZ(#a^>cl?Z9@qvIgIV2&14t5J6IM&^KFO_7V@wl|`04n)HmfZc!XP1EfkIcK-~meACE5HV+clzqv(UkY#Pt5ZJm8Aj^zUe56W#PlE992O%+H${p7dx#5>!NxD4T@9QxWm4su75olR581$CKjc)7tn;pfFmWCd{#ul-)vyuM^TKzfD8|x)Q75YbqUAj~Ht|1^y$y>Rw6GU7aupaeS>wm1AjG8~f>ziUVBHR*yd^;JfW)leT0o$6Qz1iP)g8}E-~zx)2wMWz*2BD`C3Wk*u0FhslRa-2`I+D@GGa<Z3&37=5vOU*0jyF=u@w?(UB|#76)d0q^Ue3K*@3H=9E60WWU4)Pr73>fA{L?+V*oY-ZcnO*Mg$<cFAnhyw_ysq&F=(pAmI;y*7~<eBFNzhuXO5^Tqy!SYj(p+EKlBZCjiFp4EqQ+DhY!Sdeyv|7_fWCp}LgvttN<wf#v8WiAxZK;;jUW_cW<NCV%_Z<g-d7w>dW{?d{vpX=`Pp7=md;C?tw}$@7eCwp4v9?l~z92<f|vp!7SKem*CLw#$=U+NCx6n|jM2ca8Eerc6*|$L6J-ceVNY8V{NGV4fetqWaKNLDeXKAVvoY98W!%pf_gYp!Ai7S{QNN4RK~QGYDGMux+dVvKV73udPz7*`oTYR{e<TMUh)0iafGj%he1v?}N_%H1b4v2e0pV+JoY-c9<ezzAbxr93*^7U26wF-I5N550?gyql&^z)+Ggz!bf^Tzr9N>LHqd2FYWV?=)6oi3K(YeoL-$FD9T{10{KK8XX=i2KOt;Uq5RSh{VJ+oCegQ0J7EapGfsI9hu<6bwb6%s7uwv36mf(E?Jz#98?uZUY<J{}#JdKono@^p$+M`)+8Wcp;cDm?;JqCqq}Dwd-zfX$X=FHD>)&l?+_sYk<0Wm^miZI;hm*H`5Z}JWHa|QDlc^NQIzB>z_kKV$S{2xLvojIzrrgjU+F7C)@n4&GVa!kkM$^{T5xZ!ft_)1&X>8zDU{UO!mnGgT&0zNJj3;&CS!Y_ECKb&2nmpkz!z4Y+miK~HW!2=dF88!gQQ2>6(gc9;hCg>%3Xoq8a-sJG$NkW(BmTfZ_efqy_6RtGNuC53huGUNPsQ@WY|X{pSKfL!479vK+kl8q$LNfKfio#&G}zhckm|^KyE%{3;CVHS(AOruP#<d7WEmgr!3d2RbxeTQ+5mtp2sJ48zFAFGZOfYlHGe{x^C_i{7uua#Dsm=;&gT?6*1=_b$3H}bD-Gha70jA_@kP5SQ@1)%3Y}LDol_FcmPH>|8l5bUCMA-CHIxWukd<zVT~VX4hh5n4+2NG)5Xot?v+=1f=eUVzxTgi5Fc9o?UdE{{+NKCsn@L<Jd(=r&6}E9kH=Br%^zGoW!-lFf1ym5Gd{q05$wELwS*KnXwF36L?Kp5z=;#fd55#S#n#NQjITOA%vxMeCe`ev5=`jKm9fUl@Dna3@+Y^u8tazg}jR;pd@LvCROi{i$gCC2d#@V6`29yV#4VXk8CS`T>Pp4ypX!RuR=W=#5*)k2LtsNN5Wc1^lg*$-d6xb%3>mHh;QgNuE8CX9a|LVP~FyR1rH&+QvwO$zDkn#MeD@dF8jq0%lAcP%Y`uqgH+v592E}LBaOP@it1^634#GGH~wo^T7QW)=@bdg1Bewqpv@SICqe!oJ<24ZiL1mQeuaz_vNRflAN6n);6A5ki^q=waz5we`e)FDB%fv{K?>?+$Gp~J3?MlRa3H4q1|t7CXV0eTvJ(g~f~$BH%7q&Nsga0umpZpx)YiE7Cd(LYeVUp5jGx{>xsjk@kh-lT-MGfXDxio&+SCtBCEbW$OcxK16#7kS<~=EaW3wVp0`p7v^*rmAuQdb^`O`ID+iv3E^T@cKF4fJ?naD~l_16;@<)RA^b$r+AmtbGRhalU!X&Y*NCz2FBA}0_1g+G$ms6pyB1XH_2wyIcVp&_Cfn1SQz**C`Hd~!`h(iaBmj|>yR~VElg**9-T{doj{k&bHhW*End$(E8V4e3f!9W+!=7|=wjf2>xE$s<W41=!>b5qPN;>{j1j$L*$$>)5h?y&`5L6cP`7P<q~Pep{)~bo^M2%6BX!ZL8-@rKbU<C}*dzNU_%Pg#$5Yh^2TB-n)o&&{0oSbrBcpNgY|aEMXW9aCZ#XY9y<j?tyc$ETExWoNjixe8s!b1-O(*rbGiH(tG5u*M6lTY26xD0PnHTg;W5f~mvlV7f0Vk2%@C0Kp%^fR!xs2%V_XwDbyn6-&XBh&PKEu+8TyYfr)5H##%)3Ektn|j8#Wo}veVxZEJ50Wve3^jLR{rvS2rr59Jn(?~C%3XWU_53S=;+>g3sX#AdlC9{@W-?kW?Bujlx;E$z)ZXHEaAr`Fw_b$#ekOC1spivDKqJ_qw3J$f<C~&ayJK(UU`!L+Rz0URN&wc)tldWW3b!^s?F?SP}L-+adp1+^9fSpKT6XzW%DUOUsl0L@0#-tnY%|PgOzQOJx{7wHFDk#u1#0BU=Ln-ONZ;6B=8<TL0SU);Ki(VhLo+3*eGY}t6|{N14aU`*)uxY`sx!{Bt&|&w=<sNu(D{-B?cS8=Qvsvv8B;#(rQ%dK3uuYxu&KlZNW!KwK!7mdfWrCJ=ASd8oNiR2J)>5+rz27MOstYY<=Q)oUG|=&Hifdn*_6UOHVlzNJCTrl5&c_)5o$;Nq+{)=+MSXa}wFpgUj{S98eTc;lLhD5_&wLg|1?n`>e}}832)a$ktqQ!P5&Gb@4!_sUiI@D(J3LxDyN7A0GSC<TYDPJ~lH-&ToAq&dG5KAt|9t8Yf3Bwb9ORrJc^YUFZ%?HK@<67ls-CyBhi4waK;%b6gx*n03|0p6gRU;yf$u@4&M)>F3R>SKmgjUjOZ+j?JYGL-_x>7cEdUdY$Vl4iW04foP!7Bsk7=jwWr*9-f-HxDeR%FbJCj^G!ps<=kC5NavWRyK=t?uVY4ZYePekQRwa%Gx24_6?Kr0B(r@o(@nW;2+Syb0NHJ7Jtt#7oh-cXsWVdEUh4Vn)nV#ow6_GG>t07wyCYa#Gmvd5OqI8yV*0|w2wMdGv-X8re0Yp;BGoz)gYNTZ=haS`_VZ}A-6=D@*Lbk+XClCphV6)tobs%^=Dp}_-DPmw3fl309)HdfIJdrsr0;|GaOct<8P6EEG91d@j3M@CbSXaNXwUr|jj+4BvAa0>>Vo=a4o$WcEsPw^wV4@nhlzSL%o)gL&N}8AGdg0lHQDvlBL<mosmzYgy(D<zKG@9Nx8LkC;S&jckjULej5GKsbB~72rR?*P$~Zhr8=qZgh7tzWu%}Ll#B;5kK}Ml~MT7i7)Y6qXEJkNB^Ufr%PB|aNxpb`mwW?tSwK|RjiITD@RM+YO=A2v$9~a7FT|`pQogSkRI909m4dgTQ03r3QaXl$;5t(i;+gm};*q^tCB>fmM^3?sNe-!#qp+)XhDFq!ccpm{Ny5+8nWY?`mb!Yv+)xJ!78j*nSJaSgj-37?yd42M=)2e>zZiIRLQ<h#vqPvdf=!Cn~+k*#Y8m9VWfLc3slha9u)Njl`5E@?_)%`N_uZgzAbk~;e_#C8DGuU=CpUh4q1+fSYe{BR|1NvtT`aYK7$B^so(BM+7UCc7wswx<<-ZnHRhD={q{{v7<0|XQR000O8001EXs>wksRv7>Q)>Z%j6#xJLbYXK}a&>NWX>DaMVRLh3baO6ndCgqwlH0hI{;#J%m_L$7qIkb3(NuXIPu5lAjLY^+s?yUXg8<lKT1_e>)$N{6|J?`JN8Bgba{!Pcb!&O-l-;hfBoa6P&h<OMAq~TLljXVKV^WGuMU#~vY$@3Fy2$dH@Zv6C6_i)xE~}TtrY70Vx+v=`zam+lic;iEkTqr3^h#9bVDMVhl+&7$J6e%jQD$i-IH6ZG%d48yOF@b>%~(cP<fh<a1*5KuDyxe!6V<d7D?uwDME)+at7Sb%i}Hrn6``dd72ISsP!k{5D<Dy??#HAm2wl~p%xNvi`yZx1tmwz-oBQ`g$(Do_YoIDS=S4lBjAdNpHC?Ul+m#7!lJ_tz%IfS^kiWk9nUtHn&TfDYG$)%%;H)cLWWB6#_Ny!x=D;wF!5}S*8?soWn|f1<#X>WO&>V)NbynonV4$xpX|-HsN&B;|*CtzKFrDVoKdT~dz896Eje|n3hVUEsIVNvlus6W8kNC6cQHthUwcOO%s{I8MX};IuW}RlMwk58z`MxPvFpw$AvM6`1t!Y{9wkyTIHlk`N|CTAja?<tvTIz?o^*jeLy#Su0ZjH$+5Eg#Z%rhw{Qwq$l+V=XwOPtA=(7L$E*aDd|Mu`-8#^`E+>>LdKNPgI?P={`_3b_rO=L9FBbhz5wRAUr9=-Mq#=?2trx&l?HkYQSVL2Y0E^p}@EF5bO)`Qja!lZOF;KLcibOO2WSDKDY2E;0@mja9Z;(xT2N5X6R)Tif&}VPcHC&DFfHz2`+XcS4IX$4R-*JZ>k6&r{op1IMwffH59BuGj64E2*IKMSBVV9(upphB>_vI1nrHG`lj9VI5j+Q;;9FLj9{|)9G?^bp@OOh6|Gw(_^aw%dCpkt(cy>jOi28sFu|61Dr`}+YT2|<OtWcLz~)`pIY2agP6x65K(AZ)U#|aPOaGUQZMnXkh|2g!zdPB>>Q+?{f4}-?ZEEeeH?!ZE3ZTj%8GWp_(r&T{*1N^7M|}~ekf=_BjBm!@X&RF#HE}kz;lO&G+<FmX_AIvNPWj<aq4hCaMLhOB7Ta&j`unIUEkN}rx!93Of2}bCq*j1IA)T4RZjn&xovq4dYST+1};lo+q2@3$1$goO<g|<xbM+8q@39v=(|OQ<-30BGfJHR7#={66FiUOFbG`F_oFY%dgmKvz4HmIXO>7}Pb6+i{V<{)3tT@4Q%5iwBo<|9Ks^sE!DfyhQbsMFMp2sL6)r+YoM64{dyW$Zj&FZ4*5i82Ea&3mSNAQSI<JnlM<K8pe0C@pcn@$kejIQsjX4W#EAjw%xn3-iD7D>$M~P3<C}2+D1r|6O*FM2=n0WA6ie9Y+qRmBJT#LL~+|cz`XY?mbY0Kl#jw9DiQzxJ-0`K7p76~2$3!RACuFr)7&ILS*MN>acJj;qg(DTGjEg|A0Ji%kn{VWd4WdNnpl$!H3IeW%bwj7Ox?*}yHp67sC1_=-qP7?XxKazyUQR>66o*UB0a%`FcTO%RDG_fNablp42QE<|Z`$d@gtssxHC)Dx8sb$-q&jK#|Ftq|Zj_ov!LR&;Ww>-;XZUn=paS(ATXaow&JlAI2Vqxf6=?MmhUVN0hm+Gs&VzdzaFpe+Q!1HVhsn~jQo7IyQEw97`@-DWty@~4v#{fMOHu$&Lb{O>;2e%i|z;{^~3y2RQvLeQ4!npl)0sWK9Yc=8A^_-9FeLJxM<529D25!teYPlh}BMu<nwvsS(lSJ4&4I|f%SqKrshdcl@kunI>)+qynfb&@~7&M?PAor_^&$O`r0E0hea$9b5$9E&JCZOp!ocUhBArP=Mc6`X6B5-zYlth9-^tBT&4jAVyxW71#od7WDDah&C!50NQ-v*b@nN`P`1(pk{DJ+2L!sX22UYbC}Md%JVA9B(JxGGs53OjNVJ7Hepx(*<EJF-q;9LRd@7#v<?pX<{S;KNam_Kt#+Nnvebnn-^Ai%t?+iDQl|jKaj_fFK}8^kP4B60lhAI8g-i=RUQ30L@nDd|gL&Y*nq`&t1m({ws)LYJuL+IF7(urHpbArxOCg1rzhV6fl>Y*Z>M>Y_T9rEXxJa2O;Pmv*PHK|MLC#b4*GpZADX21B5Kf>T5FetjTQI>bhYRKoD>}E>erdU_UXy04HJzAkiQJv>H2h4B0T4o8zZ8a1)?x02dtqC#P_ijV{o#{K)$@g2R{N@$ZWX7NtQT91&CI3hE}{5rM@KbA6yGAWS5luo*)L#$YN=BDe+63;1n2l#5eN%k%tiA}#xJEdE_-SrULAgU-4^KtpbG50W%sD~nT}x=9+vfdz0n3S-}K0Pw^y=Lz*=5eAMM3gMpIT=Tz7TK2UW`}9n@D8{pCU~ykWspWxcCQjmtSfnBKzyTyqn54cF@Hh=D-=axu0l*1D3Mwoh(+E!G47TmY!8eeU{RcREzDw{TZX-OhQW3zH&;!SdthA#zv0O{UZjgd%#v#C_lv~seV+;IsV1uLZ0I#25upc|0!`<5gkeOM3tiL8lPfnuaSVO@ef`vX4kl}ccL9^JhKzCDDxNhu)5f>2Iq7?Um!5#R%ojS440Bl=sL{FK$<J!Izor?tTi&e3U1h>#`v#LwFy00@<O=wk#x|-ZZrfXWi3(Iwu4?!EOFi4U(34AXB4srlWwul*r>@JLil{zfK{XL%tz9T3Yb3zmFc!0uLdUC_h4r15#p4Ne@tX^&s5dY1zv(BknXM2=9Z4P>;YUNYp*lkoz?uv57&GkV+ch&NV_K(LTb5kThWl_<a<MK&ED*(<hPXN3G%)=~;$6$=Y3tT7i?I=xwzufsuaRrc@09nU{e8qLa@;MJvo4Y&(j|mx|@7OMnz`4edGttPlB2Zb*J>iBtV9qIVd0;e=;{;EOtE6|P64|B_CHN}<i?x*AizO{`u;za(X?`_*E&2BD`RnPq+?n3VVu5P9W%(I$J=OTAP%!~r)B@bYQJH!c<kfy?`?2k^nA14&Lr`dl{+2@{huKjga52oGp$qI|9#|6YEhaREh?ahKn)OTee)3-v*Ezz29kHm)dc6U^$xQm&rUYbuCCpsZGTT<*zM6jSs-{oOsuQ5T9RK#uZ`?QpcjP=R?Z?63^-urx>c^K0(8R^tm%qGv_v)v=k@+Xs?BLB$Z(jDgf?YxENn$$`K7}lZI$(Sdm6E`TUFN$1pmisX5@RqJKyE}9m{TpllToZq8KQt^q^ipinS3v6L1uEgO#>LBR~jNajAWB6CBt@=O?xw|7BqnXxv9l))ZQ*iSu)YKTGO(wx_4=QKg5e9%gKXbjEP~&|4ieNKoBO)y*>psz-YFe2F)r#exa+4khLDe)c85a6_chWgfwam#)cn%C?Ab>J;txU8aN8xz*!ly_UD%wqZIX~%q5?P&vw_M1%#bJKul!odWgk3GhL%2KCWf4(v0wo)fXVNF;>D{O40Q%k$;oF6}f0+DQSTyk(!BBR|}k$ByP&Jeg~{>DytOWQ}eCM`|ZNBJS`d`SOV0MuPo1+%W@-A)^-i`pk^s3m)>nCa#nC~$#Y{<r<2GSZM*kWsi9|HiF#->i#A4NK5tpjYm5at+YZt$Wn-K#Omu-)PIS1zlBGpn^*c3J+m0PA>qrvG$m`-G6Tsu)Pw#Y*(N;k^J=cIFx{828(sFMw#iEge;t04_X`jlAD0P6JAYiE!^6E47q5NHQ8%cvm@I&7QHp~cC;2pSm!~LTj*JW`9ptqW}C07@yX_sT!=JHS%SRNMNso~~CBQIoWv8iVyg@xkv*itv-lJ6IcvL#G0DT)=`{DH12Ah+X~48wRUstxEYZCN(f1c*ieV<}-(<s}RAWGJeZvQ{w7a=MZhY7QjD_dmWPSw(UIM8J$>y&^Zsf4_hK2Cq|wf)$(;IZ#4hv;sW8S{7J)h(%v%_7O|<DpOK>mBPdTj3BY>Dlgy$c?%dsjOFNP>X{xNsh48Cf`o$;tYrjKAqDhbL;S=dLP_uiGDTIORkC!DwuNY|Wy`7EEs-Wlc`B7xGNCaTQWYCWWt6OOi~CRH!-p>0=kLpn`0#<qj0aXtQdI?qMR6wZUrjr?hb6;REP8`cB{^W(HKAz@3x)^}koFp-F3NJV#-bu6i&Ds1P{~Hgh#Xu3w8Pv8TbYe;wVIdMbl$5LG~tK*GK4R!%T`TI#D9&~)|b%MU^e`O=QduH0;@M+X%)HIfG{yiC4z{X1OTlR%hs#^{PxGn*d4e6-odbcm}~fB$n_$bVaMvbP3O9)YN_v`I?4IV_}6L}#z5N%O$+|n_+vJ+jmIaWp3xLMc8YDWx-tVXpkBYz%Rp_$QNe()^O!)`x~i8riQ%xpPh;&^#w74YV*(D7`*0Ed9~0LZb>;zFk#wXv0SBa8XxLE-6IDto->^)L3u_Y7NqYJ4@SsM0e0((YM0S1ce#Cv1L&J=#`1;sd#8y1o1Cc)y8+BrccCj}vdGXK0I?2nonv&EH&<$BzjL@iWUq~u1@?2cWpsr_v4jVBP27LU!y^2bW*~-l|&&r|tQAsPrNS{FnDz4?vk&Z8VpgHJ=!rSowxPx*VDM|UvlzbYLiXV^4-C?YeyrY277_32lQy4?~+Iq7F506fGj1l3oC*Ol?Y^i8i&;(ty3K?|{MNZ}3iMDthzhHH`sM%D@>0pdIL9aXaVSU39P6@vBW(28)P=U4W35cgNNlZ_MMwvjsrxno4`D$m1Xcug`$<dRM41C#eMQ;*LXS%gSHA9a)Bevx<A7C4aVeF``UYEI9D}E!<(OwH!g<M@{8T(`!9vq>?vnz~7b2;2nd{n<X{L_5z1FfNAGmb2oSClQGxz?!B9(EcwlWn@lX?4cL&LEa;PvDKZx79%%lUt;#?r(FEJ`+x0&rUUWe{G+FcAwl;l+K&V9a~yWFd9#851wVTfOXc|?5W&TE&F(W*#)r&TD9PHwufAgZugA_+ZYBcfs758GweFSHe-+yM!(*H3d+Lv^OS9d(nIzQv7^eZ6%$Fc*QJLVZ<V_BSX(E$jW`+a6FjV1qbCRbBYEC|hbkO5z_0@Yl@p{ZWX;`D<XuE2J-3E}N^e!1+RMFC<Dpf^M^iVOJfQ4MEx_TZNgHL7T$vgyYG&&`TAB^8d?LxiW^HJaJr=EL$)CNR>Pp+WlNc%hByX!Fch#qtHA)YN6ng1lckq)r;=!!%ykx60vJ5s|twBw!G+sQ64VvZcgtYIa^mj&S+HUeymS1b%)b&f<<?foviUMu<jLeL@lnX=D+)JujutsZq-bU?&U{W#EFy|EWi!`l7-I4IliVlcUS-90BgT}#%8^8v&*aCbohz4S+EF>3#H&1DGq5T02(L@hD!GmA{|CXR0qO97dX!e}o&j9`>&#wSLp?`k={M|&JZ1WAvAeYmp6w^cAT6TXfO?&cE>mJgO&va$={edozhoZMy!r_>$r+;C{G|;$&b)<$&q~;H*-{ZvC>!}}x+U7v|g85c`4kh~2wCDPBA6ZD~vqJ%XN(iKXkDI)Fu0JI#g1mKI<kb-zf-1B@i8Rejb>(1V6uE9cCyo^yCW(JKe_u%5?o!w5QyVrfRI7>NCql^4QHnC%kd!*v$I;f3@k%yi_$&WzG?D*L@&AAS`fLP)^`>}~(D4Z&V3l*~7yww)P+l~z%3`w~+TixU8cr!yKV9iP-*$R=QsdvL{RtHV@mWehxj7m)7%V=m+2`;{dWTmkRy#tsjxVG2Pu3Aq$CU8c`f)$DO_y5lB=>XllpfUQ;}OM?xZzgjT8HN(t*UXFDxIK-v2_H>e;RiuQjGJ`jn)7@DdFgOAC<7&mD%z1<KWuAo8xZ4Q(ZLqbhW9L!{f~Wyrk8A&W25U#8<^|bht-jV|q8@DL?R|Y<5JJAd?+(yMex4@gL^W68p_ZpQd{!d277tgv}Z}NR<i*`u%l)#tJn<<tN(dNw;L2>2glsXQ{QL-JpyE9)QwK<gd$34&l5rZ}}*5RcEUeZi<w0PnMhH?`4r+sm(*p$OH76t&~#S6u0=Ccu_A!d57$jE{EI|n-wS5LagOR&b@s0x2{M{vz2)$0PkXoW9aP|#yEiDZnCO^ar@gaPy3#~G!Cc938r#npr+)B#@>PcJ8IiQ`EojA(ocELO>lY$`<x{q4s3S%_tx6nfd&53Jb~X?1sOhcQuBu0v+<+S+L7^e@^u2!k9Qy7?mX#kqF--evvjvWslR|N!X}tZu-#>674E*%9GDJw$Gzv)6@8gm=nX$+0#20MxSkyxvX-Co>l~C|qWz76pJ2`RkTaqmzXz{*EBEmmI4#pMeI~I_?=8p{?-%%9gp#7EkQyAR&PmZWw5hdkRG@O3Y@gVts5mgDNjJVDHu-aVsIx~^Cau*9tQ9EcL+9Bu-EmAJ?cz`R%8t>bVBiQy<Z;4-9`Ui6NHrr5Ae=||*vj-#`NR>C5C3nk>vJ`E-VM2B3F9t)%hqaGTCyd0)%~+H4$RJSnYbK1XuHAVff6Ruym&w!haEgtqaPR20-BRB+tI=M*)w<Bq$Q{uS=Pd`EPeI`mR?`r94^}vu$O8%Ad#!hikA9tNTL$B0nn~Ozfs-app?gYzXFu^#oHI!Et$<xOJ}{!K6U5PXt&L|+tp=J;7YnZb={ms3GJIVz#~I2*BnqAy6XFhO<gTcJPO}(2Pj=!4qAAoUms!qBU}G~+gn=V(nF}ju>S5MW;5!#i9eC<u(#2KXh=33k?(s_8$1;hjOS`>G)htz1@zAEn`L%fqYMx&OSaeRqLw>5TQFJ)k?%EYVm#k+qCuB(j|age&VP6Jt*!LfsSS~lquB{FOVDw!9%E|kptc+74b}D|2nNsLRtqrLl6?O?v4dXQ>nxYcgD_#fTfF+=_4^@q-*Y^&9SR?h!>sYP!`qjC?fm-7tGAD{!SBXpX)74?pZGc%aGYo|P2ILMEO0hGPCV{ou)X>BI^ZHuV6$@(#0K9*gD;=aI;&M>e<J;HaSW4k-WOcIO^pvZ?N}z6wn}ga6{;><^TiG0rv0KWhG^=ebJx}|TvUv%!Dsc{?djzq%4?iPFL~Z$wQ96KC2e0%@n~yu-_5t=_sdB1q*T-IJHCE!Znm&x2LKq{H>0anaR=+$$ZQ+|mp)oNklnQAZmNQT;ZW@^DtYy*Yr2f<pMW+uCJlF<c4vd*b_Brnx;<*<+d%nH6(x)^+#hbVV@{(zztyY=W!tWKcF^KLGch_H&_=DqXGX8xWHie+;(!@_;t_6pQU}TpuCzuZi?F6BX?YJO*}O^#=vDZAcmRcqH2XL=OdS9x2BvuByf>Y}anH23H{3?c!wwSDb}sifx^r&#H2`U@H}&CNRUJxYXOxZXkmb-`9vtm=I+AFsH%|fNUf*NCBKhaAL#3Gl+1(mByd2s1<Yq;?QPH$Lx>3=|)LJ37#@9bh#v`TG0nX!oQDw@+N$<S3l#Xx%nrrC3ZP{zsBBRDBbSI?KuC|Rv$oL!>XwE_<D#I4P?c-ncmFK|TXP;O&wk`qqo>>akFLZ1nahpN3E@my?L8o2`SN*A*b5>*xDnRhnVDja3Fwa|v1CU-dC1Fdk-5mY!p*aos@Ij_kvI0tjCPhvbL3G{@CQl3@PtyP&d}A&q5>uoZBkB+pIcx%PNdYKhi4^!6m`EmHfY(&f>LKoy8B@tJ<c+{hy#Z4Sd?rQbOTcS<=^f;QSYZW@0T@w$YpDz9^r8R>kY8T+E3Gt6m$+XBa?r#zEC=hUAk_CUyA(gPme{!Selv;M)y*a_OS~@I$?NT*A%bW`t?H`W=Eh6vXHy6)Be_O@-Pfo!<w{LWZuL~<ugt@zyv0-M^?*u46ylI%+Ui^s$(hbv8Ov@dE}l_6wk1Bj+tOCr+j~pdt_1Z28g>?bfh{fv2N1iIn|g8=>sP0W=2X(S*gisV*{Kp~lwz)Ae*pX8+*EyHt_i7U1}=AsWE#x1mpI^^i|z9RmpkF_aG~oi=?huqbh#fWwo90{a~K~gx^etqt|;khivz?ETQ!XkW*unKc(Hl&Wxq+Ukh1$@d?Xl4i7j+-*zj;ZhV|&pBlwLdlwGw=rhWNbSeIS45BA4ZTC4O4mGRj#BrwK-G@!>hVf9!$L6cSsEP4Al96b)Yr8QeJpxQ&NOxu3C$e{7DShm)$p;52mY91kz)s5;0G*Dk2wg1esRPAEJd>ew}&LQn==OvEArqKvm$*DpU`T4kaGOy>Yhn_Tp>8Bps`S${7(@4ip*;TIjSnzCn%nsM-H^=-t0d8A;C##;7rzaJ~(}0$-Eqhc9c=6`v|NEc+m9HF?(Ln9J{4RGdk192B?bm0Wy^2;UowAO8ZP`iQj!u<*?5M8h`}X+1eQc!}W5)wE{;#Xlm!p1ep$cE6Rb^Y5UNlO0X=Z@2D$R?>!T$qLO9KQH000080000X074`FOjiW}0I&`K02lxO0CZt<UvhPBbZKp6FJW|aWo~q7Z*DGddBs;tZ`?KzzUx;^)r;1Z729nuHdMfU1PJN`0qo=w1O!*&uGp1GnWXG&gZ%dnhkAO~KIBpzY-xtWd42QPIF8>a>$ob!X{`%h!|k8%uYs4mb;7_#8*r6?Rw^Y*xHDW?q&31=_$B-KSGcKp!?P&5^M{Q7M-)zKm4eZZn>~nTEy_}=EpSyrEw;Sa!`~mlb!{!ciiRsE3k%!`Fv4+(TC(V!hTm`Aynl6j{r1(i)@!tiJ13OC8f1f_4Sw+OX`#`j+(;CJKX_50nfIMQ-L+6#SHXGIS%Dhr8pIcAoi}DD_Vx;tR;Tv9^SX<wQ#;$6;60WGPq(!!q{Cp8(#Zz_4Y#{2isLwnHbytV*rs!x5sZ1Ow85iF>0p?RqMqjVt-!CS6dPc+Y&pXSK5|ng5#VD*y-De<l4U6r!)!QV18H=ddENOttwdn7+Df9!($2};I)>h{2VQq##?YN>JIBWRZ2p?pwr661Sm8rAC!U<_(8w(svbrR~;Epy$bwJ3J$~K*%?E7?Ccui+0PHqO946`L_o7Dsy`oN4Z`W&O!w)LI`2r=}OA&d}Gq7@vydx(Pe&7o6(aP{%b3avJBo3)rwgBkT2Tnj(sa6TRx7{NLJ;Ni}wz-5l3TD}!#nXTbDlB?la=4{EEr|k)6k+~)Bz|SRnOy80`C2*ju=xxniQKfLtus~8^kT0FdY{gqaCqo_OFu6{#xo5u8lZDi3Jd!;~@t7=?keKQkRtb)dhj0JRO4-ar2`CsNk({S+5g6ewM&{6v-S-<ZM*90rBJo3`9{yLO2u64-+I8kjWC<(ZEqzY_tU&RE_v!3&Nd2eNi_dgdp}h&x-$jP>1&PI&1dt+`T*ybQKLwvw6MWJ>b;oKtnNFmmwscV>tD)8p9l=gZ&P)XyG=+XtaF-+p9+IwA3K-7(VhZSx%I8JTQIy`6!q{Le)}%)PIT4O{d&i;-vG#_}fIRlHAa(=+Tcf+yE|E4#jq!g`?nk(*Fi^I_rdo8)2Tdrxu5m<aRqtWF$JT;_4rgF}8pdS`ooaF5;Ss4e!U$XpuwxWEp>Qx0DtKqH!{<P=Vo1*xC#Q2Jp=ky&wa?+EI1Oi4Q>KQ55>gSWmoIhDNhQs*05SLtC;K3PGi1|~@wW4qZtHKE4Mp!tW^wfqSZ3P!@-4e!)^4W)C%pRM;^el|olREh9DL7kC^66QP)R@ApBo#c(JdOv+Wag#o5<3#5SBdrOJ_}B^eenPZ5InuGDNVi`eQw5=wb3-5iXujNFDlIP=Qqua?=+?zwTHu3nS}cl+e&m2UJ~xbtZ+A33->W)_O!(j~d*%ZXYF@2pnq7JwR{5&d{%~fZPhL!Cj84t>5w<h1^z-mIB=VaJi-3kE2x>fD*`z%x`l&B71|Vv0&Hx8q@qo^u~T`i@(h_1un6sKl;4>WsEk<-5~yeOp>K#17)vpnZ$M4Z$__cZma8Yn=N$H%6eR7y=sH?C=IT&@xkCOc5C0g^WJtWI!12^wVaAus=+?K@yL!>p$P>k`Wu!rG_U4+v=iMOZqJBfFQbF{LA}JweExl2)x4g)>4OEM$j}NFn^e477AOjj0vTfkM#wUTZ;Cg5x5l+Crd*F<i<cKnDY?U4`$$HHXA^%=wrl;8iTIfuBbD&=B`dyN4L^cHW&`++<Z79-gCHK+Ed6vG**z_KMsV!T`B*MTj!qXZ<3K+}vbO>YrSzdn7QaW*>4w$Q1P!%;?KM=x;sBxRB^RF{IRKH49kd5Dx__IuoPqQoP)h>@6aWAK2mk;8Apn1<l#i<o001{9000;O004Ahb6;|GZggpFWiMiHZ*+5Xa$#^TaCyyIZFAeW5&q6!f$~0xOeom0lQid)opE!yHq%Vv=8~J~csv}4ge06Pf+0xRitGR0UA&N>Bs=N!W?IFLNxUo;`}!=PFbqHPx{*bfi+Rb3>gMp4bEOEcHlmT$npf>BQt=IMNW<%rK__`FOE#Y<QpuLjrMxEBys3CegW%JO53QYov7&`>tQ8whF1GDTRzz0imMmmL6mPqlu&N-1P(XT1$$44AGK#Y%Uj?}=IN`TE?^+~O5k}UmVoP2)Qm1rw>~l(PM7x4vwWul>^+^I5GH3ZUuL=sx2}RgOib7eAr?6Jbn)XPnYuU6Um(`*aIk2n=xqap<RxL%f1X79<u?7}s5Qbq8EE>5cS+?lfuHjimEFD&X5Ma}?QbFL_nq|!z#ohaFN>=V!DnqGe?Wz=WN9F=v-4M0vT2c0Iy`uV-t+MP;*pOH-+=W}Zm1~h_xMD(bR>?}_tjtzStr8&2PM*KW7NX=qaPiMifBW!0`}NJoU*3LrPf`-1i0SM_7<~TKZ$73k>B&hLe0=kd&u>4z`3c(Y0s^0~ZCIs1y=&g6@Qj4d=;?EM5+=4+f-b~Y+Xs}rzN%Z-TWp@ulNa=vYrYU=DQ|T9F?~jl`{8AKjjgBjM>^~2^V@b_%DJWfk{;7qsum~R2SHHq1<9JOigLLITPhOr<jM67s7A$Snn2x%s*S>@BqaD7(%+=0qBW}{)ixkJGcAr2vM4*XN<TFnkM+z3thuQ?%}`vY6skQqJzw$sT2J?Qa2$4<sFAX@JQ){Y=%A^!1A4O(`6>##O=wmZ3vZa2h97WS1y+dj2EO#tne5uSYop6yT0+9<)XoU65`wZc8>GO!sH3>23(MJo6g*e7QG9uJaz)aVTp~{i8LZ`M;Kmcvu&QCd0zU-bPYLdu91);0g&@+x!mJMUn7OD}vxOmUFpPq>(2Ne97{K%FP|9yX`YMXge7+Bf2|wB_MlJXmi>DSf>@Bnmgj!$_KbE|OXWI&v0qmsd-5`Sd@M=VOkRvbE_p!0pf*<gC>hX7QOgLSK5A^8W9Pa9O_yEl*L~LAh3N)O_OtoD-JjDs>V80XI!OjB$-5+>J#Gvj6?H(*EGIkO_X84|U{~`7Cpv}C08(1@Kp3yf<K+^cwRjpX_H%%j(DEwUU+nVPs<fv{#dOPoZDjbKDfm0hGU`H6QGf_a&6YVxyL#j-$`8S}Np${cGg=IBwa$aL5vR(ip!NJ;1Af?`gdEKEQFX7MqF4w=ZvU5D^S?f-LcX0xf2xM@aWLZVGO?AeKd^SukGOM8b(s_5ozjUJEn6uSUj%jv^xs|rJt@Rb)b7unsbDA7wbq#4fdW7UKR}&ry^={Cl0>fA{%or_GQ!^Dn#dI_CG#u6%a=@r@j`T*f2se<-38bmfT-~z~+P+L;`f)sf<vA~T1BtqU0a}|QOFP@MFx}(t8OGcpexMaw^D{rKN7x)|$c0a+b}hkse|Np8fX<HK%`lffH>?x|YkB73uE+={F`Sbpd&Zak%plTP6I3}Q6SNkDP#zjeEM5TwMAy7!unJ2wu(5-*7!%@2Ga(u@K%0j;5_0kO9rOS`Fusg{05b081YBgB%UrfnmMU;QnBwBJl&nw@hE#}Co6Ff%8~}#uSef2k^X=I_Yt(vc1l>lu@PNW~bngSE6i57J=%lVAA(3fHTubbR;`)|uCaQ(Rz>}uYX)-&G6TjC@(88cu<|bt}WKEt<;>%;0JJHrSN%ltu7#`hzahVDD^8|D>BAs^yqe5kD!$gTOKLSQ3f4%rT0^T-$<KE_7OJ0Be>6`%C?y7zoo$L^H!44OIwPz<DmwsPBzNDrVbX-)?EP0l^OioXdf^S65(+LimBt^TexgEFwWwx^y@y?1uEkq?+9+}swqKmG|(LPy86+mKMWQA-qQxUZpYg>%3pyDqH{sSrGtW%&$og88cJ4R46m$zKmrD2AGaCv!oOcVGy!Jn(dzRmFG>T1k65Ky^n_9)OF6Z&$0tiL1U1XR|FR%;b$J`zJLq0eIq&cT7hKOYCaGS#-qSD-k6;lRCb1gB^OTdvRutj~2NxK_DsMXu1j=a_aBTZ225q6T$|^}6TD8|&K{(0oZx>9b={?*l^ugS)jO7U0n;x|^7oGCA{%KAGrkOwRUfIWh7%+c%$NQbPd<B^&I@c-?&*>*~#90|hKSf{%m45Das&VK5#;k#L&tfD;q$#%<n4T?Ms6XPg5xWl&6bATMeV3FVOV*1Z}YdoW{~MoG{Md`1us=$jx?ii!h#zMNt(G|?Yy6DwUm_M4b*Fvl2y9LNLjW#!TUsNikE!OGCd$}Pzo4rv9d_fmo@0*r==n>X4p04Mpu29FGkc^b?~L`!>x?(yMZiH0M9mur~bfxW^mVbikQ=!mwDT!!ABFkN340u;|0%(%9C(^W<$uf1B(Q%-8&7F0$WVja>S#7Pv3(c@01!9o`10kQ8weje!4^MPmRl)8AbaA^T}=5%&Ey~FwU)4TDhK;*~+@SUAp-5b#wdj}c=JyU`mS!`rYE9WM?kA@+%vdmL7yS`qt<NIPwYu6AwP{6g#c3dxg2OQb8!Cmp}l%{6#P@>)9U=n!&;<e6n7v@dF<HUo?3Hk7geu&4Mnh>{Fw}fvz%eK2gt5`tZ<i#uC%H~!)fNEEq6g)4%01Iae^guB*98M$$$|9&a(&3Q7FPtBgyDm#FyA4zd8<4aqeXZJ)T}V7={>o5S-C`kb5%J~S>|_@9RY>m^CWc`x66{yO<IoFC1mCD*AE;kFmdBbg<`Rr+PMu29RKusIcV-K_h3CSD_GxN>{X>Ic`CL|5>472avfu}-Zxl#-R7F+r+iBO7d)%$swpM3HM}@qpFwRrhs?LzISsvZ2%A*6UUU_-qf3Jsd+Wan}@^=N+cc`rVfqa09^877%{qF6>M3Is&S-v&L)F3#Uaulhrml@`U4oE0U{?dsO@{u`w_K2ac+Jk6KLKhu`&E<w`xXf8W<O0GSpyY;^46_{m4;8k6#Dni1wYgr!&w`OY1BCdn28snN$69o%uH-Qd9zIUmKT6tF<Rgbej~om=q|}ELYv)vQs<$>Zdlm=h?;F8)8p6&Az?jBxVr0z1phJjKze7%0G@O6s--Ro6P_-wEX4+14^{}RYIOQ3%cfdaowuyDv^WBbb+8hrBzkA$qJtGk4x7F&n*TSPBbNCtt8vaM<ahP>~0D;<MI|7$%J?Bj??|ncw;KvWODc(#0)wn8Dn-ONtFCBOjg2!86C<m6diM`8$1dIN;hES{zMSA?$=f(ju{{M!;ebD-VA83(25)mKU++kMoz5UsHD6od@D^itpsvmWE5lwiNp^CB_KF`YKB!1;;Ml)z!EuPXp(&v+Y2+-1pE|(KKXmVu-%_ax-<Bmu<!t@L#M}0M-#V|UH0=2(p;FzQj=$IN5z#m<dyaFo$cn;FR3rGciqRqy|XFneI7Gwp1$4|lD+2~_M$bVdBNJT9Jm*M+&pZ=cxbpGi)yuzphA!7vY`UZFKi|kQ-jD_YuB*Aezg-wGBqT^(sU`rh9^mwWp)?IatS0LsgYuS7_#nmz&==bxU;=~)-6qUX`2^lCC?{-~Og<S!mC6s^=hL@J!npI+f8FIKN8Zz-vR)F$1LUzcUSK_+KAaPb?7)PTx)=<=S^ficvqj&c`T5$AqZDL>OlDCn$g^clTKugXu<VUXn#=i4&Bfs*BynlD$<kR2W>APpF1Hf9w@PA}`JE;s$BQ>TF8;)>%sZm9>!@D^n2I1e`$5ga@J%arDthkr6qvl_w>54&Hi82SxF<uJnpFIB|wJhvh1%5*;yP|QLCi5+RA;3xomjro`wNMI5Bn<lsFs>1ni2|me%F)j~Ne$A{YRB>_MWhyuh1A;@$m#Mbjt|;|o@OmOCVxtMk~@xt_MW*r@F+aj!A9a04^UCCF~aaD8z6a3O0itE?kdPnnRek`(avOaDl<HcZalsIeYWreRS7h9d%88Tz?KeU-B_AnTedrch?Vv+A|a51xIK3=iq60%I4XGnFIofNw26g|ZG-mRinc=5*B+Xgdn>E|PSQV+j~r4Jf(C#+{c2!nBO64S>F>s1B51UCLZXn11;QRWy7d}xmqv2vFwUERLmN~N+}I5FUEa-&g51K*tru;UrJP6M6Z#e4hPBN5okN%h@3|l`?%Z(MFI4QtvC(~#^My0VNbfuU8asi9`&^tkGZINWK7HWoqHyB49|1+%X0touHW#b(w`v;Ls{4iwJ#UsBhQ$lr6BXPX!6Hu2vO?xr7JD-(D~gQSDG>lNMED_oLqiJKY8v@hq+r>wKI_Uh)y<C*LM4R>v3RWT4+sxT2z*RvFrA~|mPGr}Fe1}t0{a-y1GR8L-&_r`?E%(M`<nQpqjc_u^_jjTNkbi2vlg~t=i#?j@Kn-F*ME?Jiuy2#ZxmM3@YG8#yXXBrUK`DEs7*KO$Jm<s_{$T9;(_yNJ<(Kc`q2`_#}RZ!<OmFB8D5}fnLeA!GPJZT3(qY4K(h<}4Nyx11QY-O00;m803iT|slQyi2LJ%k7XSbi0001VVRK({b#8QNZDlWGZ*FF3XD)DgomgFy+qMpU_pe}eA8L1Ot?i~sW|N(1n<kw;+-9d~`;zHsXo<4fl}MGOyxHXDzxRODmu+pld;PL200cqc00bCC(d!RYAyet*%nBoUAwTj>A@JdxR9nk(ZJ65$mK*(1D7Jsb#D@x!rBIF+tTNi^R2Oj+MUx4HN|q#f?P?>Egvqkf#xbsxcHBv=>}1l=GVXX<aBGEaNq<|df{Kc}T_HE^`fGenCX??6YvS(zX|U|-cT<bm#FN<1LJ8wHe5F+`x0e%!U&%itO57$+-w0(dnN*HlvDNdX=Q>eVv<Q}8vb@mTGvkJ-tE3ty-vmYfW4~%8z@tB&nd!vxvMQi%qjdomndn)S5vjB=?~++(z=`r|^)zaDx~yEs_(|B*$c;b{q=iHHR>VOt)oP!VIuo62QSd!a_N%}*YMWFV!M*DQ{PIO9+)if;D@3*+;b5)E5xGiRmq;aDl1!~A@>%d4{bbIF5+8bzy<{r}h#<ghi4QO0y~SR_PyKRl5tml5KY3A$*T!fwjgFL+wRH?<g&ESxy?7BE-_fOF)8%~iR~*ew?CRVNxMhtUeMrfg<r!-Zxb%zOkwT;aGvc3G8eu!jhH#15A5hfofAxkJ_@>0!8u<9GiC6nOHQmFAA-v>852OAFp7rvq?NJbH9p#nyuqY+AoGc#mS%*XRRg1w#`v?`m2!J-LQ%X2G#+&Qa)0q9hGMVSXpiuOHX9^y%j5Ebckqyen85i|n_NI;k`O&8Tp=@1Ig@Gl%{rWQt8@#}IY+Z^z=zh10u-k>H)xuUdPA*f*E5S>9V*X=_)jL;bcf&dbTv=%y{FWH4Ln-i#LZ`gwX_V0yr~)K;;kaGwpDk{+xe3Nt2c!G?Qs~=2CtFb6J5CD!2TnL33Ztt5<M-gtSscfIL24QzLZ0sUsCqqrIcKXmTL-aiSAH)~N->0op2?*svMGuWuM3ytJayXKT^%oKc9+|dC!XGy;$^&i;<<Oa&<VVe4|(fQr*UO*MerM0?FBa6rMtw+k7E3IbhbHrGH1_&Dcq6MN#Ta4C|&n>EfR=<iRLV_S;d<K6dy!dyJ7L?wQE8Euec=C80aqaRysR!eaJ$)qibyaWWe%(?O4yd+s_DRWY1U!w<=!Q9af3GFn2BQ+lGny=Msv0w`jS3x>~MR!EeAK4#BH2k@b-X$^`O@yWW7+avA)O{f#RCdPJ~R?$E$g!!R5OCob@hO2t6{*Npv#>02O%@d^cHk8F0gn(>X*MePLTQ%c>+mM#`3wjmEB-zr>pGCkb$*>@qrhzY?7T~I9&o;zU@Tf*h1;Xp<3z>}tIK!1;fGrF+j+}@gd1ahf((Xsp$)1V2Kft8e8$z0ebu}|BPPSAs6P({W>)1szyvDmf{_lco23bPl$Yc=8(UT)wr$mkmGxA!RtY{=~o8=|I%2_-n;K+4GB3Dc+|KC?>%%0sJb(w<XxKd!m)MblryXf-|^6!lO-=Z7?2XVM)LJxorBDS82QOl@E@LiSiWrVmhZ%<}j+Cblsj&_AEKC3D=0Hwx+0ULc=($9x`ZyTWzS`7GQgsCYj9^bf%!I=5}q>S;R|QZo-AqDF?4#9!F{tM(@ATc;39VsvhJbcgW6e}8=a+r{<U>({J?tXFyUb*t~|7wpyBS8rh}l`nKP=ra{nQQ)@V8&RMVaggDg9f%vD(p|~T&Dn0JJ=N9esrtJ$4wHgE;pmj`=+nND#=3<;HO#`nnjN4Lt;dbXVs;%U>5ePhFl~OKpR=OkuuC+J9frtY<QI-gkO`6UJt3l82!+ga)1Eq;(fI<sqn*PNwc{uOg}`mfsNU2{{!<Izz8rz5vtiHykR_-l(_{LKhMjYJ5A}E~Q!aC4MmIV5=!A}gdASo@Hx4*$GG=Un3_ENp&SuuTuE*^Zvc7Ba$}s#x)mVOY%}Sr(zFVQ){d<AWaMG5Yx1?9rW;y9<Umkp#drc!TTG9X)AkcS#e#AVPKD$QnK@;dNYR?Oqu}x9?LX!z6!XV>o^R{l1ZPYghT_pka@gwH%I~r?&3^)LzR67v)gtNzu>(%lF-DOf8WZXk@?YrKI$IXv-)$;MyUoRIwQDwAHzJP)NZp-{D4m}%%x-n?Bnqqm*)>R+x!`6`K>vKUJIyGFpeTQkOG?cK#H;K*6^F_p+_MZ<z*2uCB_ADp!=v9q3Q?p|4M_x7#T^?rkLGusDuMZ>u;DJyRjQfNJ69HAWQJKTx;6QBl(A4tA%o%NU=M^_PyGO>c<Qi%1{=YYzF-52EOtqqj*d4IyFL~QO0w`SetXY3L=C{oyx9H$gcidLu$hA=i124C|U|9dXX9Xa33Mqkt%CxX^wx<&g6#H))rAZ<%J6_)*K)~~2bWhW$r=w<n`h4JW^xg{R21cj7*!*O8KMJ{5{(h*Krr!Ru6BcOLAF}j#zv(yM^b)<n^t5&(?)-CL)MRkDkwo}sa0&ZX^yDIqj!ps>Gzz_8mnSd^_Ckit^ppOyC(Vunj+nvpCNB2}8G}j|;!6+Sr?>$t?twth<1sC<Q+g1e`Zci;JWCuvI-SjClYaqFO9KQH000080000X0CuwjYe*OX00vM1022TJ0CZt<UvhPBbZKp6FJxhKVJ>ia%{*&!+cvh}^(#<z?v%{RlI<jISE=rf6T4gAX%bJ9c4k+rp(zrwSW~2y58J9%|NWi^KmsIXyWPDrHy>gNI5+?f4$d2L9OuU@yp@r7_wn6FQIu7*6=Al@^0E*?8i{OE##tI9LOzr-#}5mchO2du-->mR#!Fe0#b_|N%H<}=0|XXvA>Lj6e3Yfhb|fxS0Uc!=R!NW#MVg6?%tJ|N#X5^*g79%#ltEg?K?zGKv%!YYNfFeOh%6UTcAq9$5Q$}+$YLbk>4gGh!rNHLWd`3EaaNU^sw@WbAqE7vop>GGND)U8<|?;1b1usuPGMva=2-!_l0^Kol6fqPaUo>{n{ymzFj(f<TKN95Dyv-jK499w+<-LAO4@cY7^t>ii!5dMFiXQK&tbL^4=nV!&#PPp(N&fu7Y{P5$}Hy*o1k1J@j`=L!B>qomkd&FH*tESA<ok6P+Y><1dBut#rt4`y=uK9r2H}n%X(gSAmHDbYQD(xEFV%+P-g2m^l|N-hJ;(fAkES^3=)486f15kR>A4(uYEj_;lR_g7KLo4y!v=`;s12;%jKubj~|BhA^LYf<Yg?Qp(ui-^nt;0Sd#x+ycrAz$d3NovriZP&p-Vrrovg3<))a7$HmK0unv9$p1dzcVYVLAk>bZNi-IzMnMYZE<1mJ^4<9~$KKl$4e^Q8S#D5ab*f>J-vk%LO)4L!!JwEwn1gq*n`Sx$0FFpa%)7Pi|pO3%sU%xsTd?`LImvI=wRr&N%6mS@cJi;p=vc<15EWawm#pmE=B+kOJLZBbcfP%=1K~iLJ%3+pAz>mPtz>C{ach>jkuaP3lSBpgue*Nn7_}i}|`D*$4G<dm;qTn==ub07T=nT#;zCZi<$It$|k3W3;>9hao^6#|Topa_)#LMHMaNfd)(^vTE9W~*nbNF=f`glKR=JelRynlP~6Amt`O(L0#mX1lf-EEvkdw*AKlDOOx(U5l>28G*89dOfy>kw%jm<!13cr^rKV1O_n0Z||f%Z5-E0O<{go8ahZBQ)|Z$YYQl6IhRCl03K{-NfapT2uu{n=CEy3L>|hW$}3)++^t(5sh~ibMDn*oiR&}F)|hX=Wnu&^ml5ieQRf`nU|h}oXOU>a2SI^4?hMGaC?DZxZgc6cqEqsMF3wg7w`Q9<ZmwiMVH<aNB=Dn5L>e-4$B#I4DqVZ=M$PW&+ZE#z-&(6?qRDmiqLo?SZnHnn9d^LzNu4{%cE}{PXvWn!Qv9hIP~h%YQ2zvJ^?X^TaeHI;-r)=Ll4D?*Nnh_@e<^CNdrc(vUuaR07zN@)#J31R!5m{yO6LM*bCBOgo2@P5wT~1$p=&!;+G()Bnd6t3^ev23*tiB0?u7Jmw1umNc_*Ij~{*n(I$3Csr_Wfo8JR<NOS?pzMaN}W<nPh3o;DO#Cz`C@3jWQeXc9`umHuDDseFBE@jXnPB8+ZF4G9-=1x#rRi09xVkMu&V26)}eC?=-c#7C5PD|cIhq3-RE1~c;k5O?CFPK+{S9V|D8l{U<8aZ<q#F;l3SxtkA$YCIL161V92gd2R7O`4q)gTzcJf%%oTtPqR0<>}$wpk1;nHO8bw0NP94;(VE8iA>{e|cWiJW-`FSRS}ic>~hIU<6dJX|R?J9k9j6rl-h}OE5i>WF3^@%FUgBM7xvI{Xa&(Zf-y>X}sOtrlqSpXOp8>b5DFRHE8gAcD{6ePUXV}cwL#9s2kMw^&D^u^v5r(E5L-FH8am^xv&2rPfU!p4O_j4!G8vfgBpbA8!Ig~u2;2pT0@kr0fQV=5V)r{2;l$#`rGQekE3#hOXB0?uk#krO2#*<vVl_H+h9djfko|Cc|u#*Stic@@xAO#d7`*lNfc0IU{S-(C?>UmiIQVJ6fZ!mfNeYxBuI)7q(mcONNXV+v&NU=2e1|e8Z|lEt-)OeC`BYuq=94sLtDUg`02x6(O^vE8qSDK3S>kS*gSS*uCxHd64qvjilW)mt=vwNV7-U}<HRSeyI~u%Zbz2yIxR$Nq+hWKLgW?)IN<26eeBJ#0EO`M;m;umBpia^-vcnuVBh8(>sx69^bJ%Xo;vMC6(^BTrcUkFF;NT~Z=&4`RvV3$Q5>0PS!pRBYJH(R1<f#MFCU+NeR6zy!abmLld9keEJg!J*K5J}n>@P%HU^<RLB$0yAF@<YFQ}rpgk4l<4)RpryjIQ&`HFrK|0dh}psD^vXe?~#25xyjU~ZqwBFtkD)8z^sRy>vw9DSoyYgNj84_@Q<xs<<wI2@C`iyF81HN$1S9lsl&k8`<(W?04;jFTbxr(hP)9z;<sK)Q%*DbB&WMts-T#z|aXv(`W(3vfUuO##h_<@y>B!soWi!OK)MYCgbmeCKgso(EguYRj8_Y;vuOH8>)m_fTom<~LQo$qMQ5l{^z6sQMca?!@XQ!wikH1%5%=<E4fpTyIMDzJF!C5TGlqP8#F~3JT;V@bOm}OeF%5%6k=`zz))qDaR&~ttc}Qf;C;%fGF~Kd7pD8vH-TplVHQsDklr<m@2(Z!6hBSH~<h~62xol0s5y31qJ{#e*-I!d4!+65$H7_I64`wHv;!QOYWr0+KYEPCEjp*Ju%SNCRgIZhzDf?8#a5C4YdRUmoB%UIz~T^d^mF7$aC8xd4qJ@$g^_0k?tMZ%^psWGU)_)Ezl^w6DP`tNyzr#b&u$4p|)Zuc~<a^9S(L4EmUonn6dy}A1}88cqA_4yV|0Gi$IJo=z?ndMw!?r;=8GNfAK5foWZu{k1gPgE({N<&1=A!N?@|Bd6VoJ4eO6?`j9mkdPLAmO<nJ%XZmI;b$56y^VvVEI0rT0H9`#8ZmJa+9%yq#yn20jgmiL^=x!gMVUT_&j@hENY*anlqpQEDC$DKoQbQkEZKbL$G?$z7CfoEiem3Qdn2>7-gBkXMAxS<b8ncH&0edB|(`9^Pp?^aAf<9e=NYy6Uk&0s;-JMuKeQMbuR>NleYqT323BATQ1BQ+pdYOxiu|_b7X@V%)1T+Td6Z5sw#}(<uHe5>?YRFczF{)I8&{u*G>7<0;8x{Vg2n`WgL>Vy~*a4u9V=&-2q^gks{dBz7Brb(X4E!J;lQ;yV4U-E{y-5sbVEQe3lXV-(KnGVqG?&pEy&@9O6jT%|nM6lfRjL&*-_*g~Y1_tQj$$5*;97=dmLnL6^|5ej!^g2fM(bPn>|#I~#Mbn4UdbV=DL=cVui7iWfwp|>qw7qPU(Yc{jwEI@B1iGk9x)=6g(+={;Etq)5gW0GPz0%z4`tU!_e}Z{qR|aV8%NW}F<d&J76#}VASP9%4wxXh>1lVbW13nShB};Y&7duaH=^p%^^R@ZTEQzOE81|&jI<Z-VWz3=%tydG>!KZ>>Ny}pz;3nSQ5)KV$Q>6f)ksU2Vvd3(L{CDaZu|VXAtFT~UbVOfs|0Qv1;@whxWE`PILO><I>*Wxsx5Y3GY45GLna1;)4hc)>o}1qmz4|#zm7kVFYBq;YI0~0%Ii7HZDT+!!cgZ<MXlc=YauAw5U+AE5sKQun6@ymn>l5>lDwp8-x7P!LF;pn*^7sx1`pY-m*|0JRW`_(GsiQ>ZZ#owR_lh}8IqJ-OUIcO+5wBI6)d_;i;A)ca63U3#y2teL^_JbcDTfd<!>@o7ljp=+6R$?!~FGTW)ovCbC#9aa>?g(Jg_fR&$`XHTrp!X^w1PJW(wCtBmI+_PC+)8O7~A~0KM{=Q~O0u7yrb7Bh5fCMy!I&#lEAyIzBo-Vq?(pW)nk|&kYB$6{D=hi&n@K{ELdNSm7WHCP?U})waEug70|dutI~q49vhThVU&Bv3v1?5t$LMVcW#e+5{SuP_U3J`@@yE{2Hz~jviUKbhNM`b(J&%*h~tN&>PY|8j+-^Xz)iXSVt0e!r~NfNGq15EiJrggt;B-(soow`W^`T6*_$8TQMT0nAWemMk`Kgl}KbZ)!;-at(yKteE`M2oinu+wIz$6DYjJ@t{-Zmqx%RuX$2q-#w66t&5;1RAp*mKBZW&r4|Z|QF0}xo+cujzc8)o^0h_dVLU|2`P@0<A>o7X;(39%FId^R;GIc2jy=`c$6dl3nMl+%Xf9eQTD}VBg5LhI^MuV+lf&sXW49Q0WNbgoMyrq(aTrMS;pP`J5$a(QbEQ2_qx&jQ;Xlq-?GJqLNFr7z-;}>?wLWv|JEvaeXp+GYnX5Bs%F0GnUmL3Z^jQ=(R2ReUYNLGnT&d6WlIbw1w=9Jc?FYf^zDo5mWWS3$a4Q(2}wAVV#fd*W#fGt<0GU95}uBHUS<T42`a^aU56Y4oxj!QZhT9PcnV1rH+CkYf+7y)?8IInjJ`zK1-vvm@woh0f(6O~%v$V%8&!NDZdN5{{zOC4o<jybx)#P`}WE2i#Xq<ret<&U}ygvc}KfKeop`Z(9fmUN&ZtdriLI0s+(?z-N824y>U)|0JK{CN-zL$5kV^Bh>-1|14`4lZ%HP6RQ(>x7Tji^rQBMBmS$Rg;H{;<ZfKAKx|WXeXbE8)uq7Nt;uoL@RKcwT6$W=f*@)xqSEICoVrS;!#6;$zE@kpQVS+iSL@^omwU=gFuv;g7MzIw(oiH<JPxL0_R*^VwSs9N5D=i%3dI)yF+7z5qo>4nFqI>TV|7!<N3a^pez$$M{2E5i!ZR)rLjX=(F?{qv`r*#)Y{Bv^VT=~G?&duZ7xI-W9!saTM#lKmcWo0=j})%VI;cNLsxkgRUrnY*Bc-cn$(!f8%J?)lV*T2G^73^swmU<?&@c;Y3c%NHB>Vpu@fgRdl`rOaL=YSSBFk{-JVWihBDrudNcMoLQ&w8EA`+yBgUsplR+)N8gLzfq3fwI_Jn0%Q+IJz7466j!#_7Bqp_&4OuDJSF+uTM(+5{!Ln$&5m`N#%WsMyj%+nM9HK`y)ng39=2k6o&DSINrlvNmX&9^%FfM`258>%-{j~x6KJ6$R=v_h=TeqbiFaRp6L3ECRUZaQ3fLN-oQW<vI)VzRk~)tq&1`EX5-wM(<?L(gg<G56YZ%$n@4wVFZW!I>V1aqdI4R<Fk`Aub&v8Hwd4HNPE_-v(Zkq%3Pxg)XOzaYyd|j})*n{S7p*7(&y{mU;gdSz{x>&&J%rq<P=yx;mDbvgIpS5cgSrD?yj3C4bA^K${yl`$AJ@lVgp>qh7|9Dz?sa-Kv$rP0ZM2!D(tN?XI~|g{4%Q`t26v&oaRK|MFM|LpW}QBftLhF$EQM6HL35Ww=5#X%yeZQH2TJHyGd|d?(-j^{qk^<l&0aiG}iSG%3Kv5ua!^;`+Ld9$jD84iT&tuAy>8u!x=Hj5q_tRZ;3<TPwxRhlhD~K0V<sK}KS2Ng;|VT!BzjX?dM7_v^^y^|c2un+?U2fT~nknM;ao0}UvjE=r7ogI=Pt@P8xEOlk6GW+*J_!aLMi=SH+^GEcju+P0qQtFRuniWk|R*N8Yo@Mw{>jQ}s+w7=}y23$w4r+tOeHk7^`Q~hB$=%y7^Avy}{CO2fP`N%XQELW6VTspP2#9B_YjOBWwk#Hj^1oaV^x?5zu@5U5cF*J2T{YI1OY{$d6I_&8qbAXb4fEF#ACEjEi+Hv0p;Q8ALJx#<O`{_Vx%NX-48_Jm9NL?|}l!?Ryq{^Bk={k5;+KumKgNRLYP?UR*8YqI$HoVBU>H5C4WkcbBIA!+eST;7}DK{Rs)TWjFK0=@IP;kJIxv7|TdC8v2v;#V8Zm2(eZZVu9^+eJ6Kn0t2adzG60QStEpw_;p+HKfFLJU4ly7!T1ZAzMZrfbW@dS)Zj#)TZc7yNhJXv+C2R7Yx?s8ZbC2$NB$(Pc5`Jx6Cu<5C$MJLMuBn9AGtWmJ_T)>L);(c>Cu{{UUlI^WxfQq^^&yX=&Bd*CDPm)~|m3b!-*_c~*Q(-b6rPZbAcdM$eWk+k^tDt*Vkw0Vq5Z2j(}xDL<*O%B&{(tTDbUk>9GGpB3A=-h4(j82-)jcZ7Ps8L*X_RWm?C<}jT)C6|B4dZ_LctML{woiKtb4=rXYJSnBuXfsx;@0Y`RfuIepPIxoBS&GUdo_kfs;;mip6YINj*v}h3d-zsLP<p^)T4l&tfVoS{h&vO(wi`T@2Yi<t|fYS!kHOwpL6`ZzM?H3#k@yUwAOY}+NEoCy<Zy;+g_eNAqt&y{R~u((v@oqS>8*yQYWO!U!DR9n}7xQtmJ86oV7P$hH||xO44Hn6LE~P`}t5yH<N6}>|cuSL4ekn*zo~}M-{=BZ^VsEr7DJ@Qh4p-fOU+<el!|b;qC7GQFk3YXgW^2ClC~xT$66H7}oGG6u$lXOM#ET0dk3QL&Y0)aj4gQ8di%_wy)_RT*5E@OT0d&?1Ru#;x~+9GFtCIJX_;}Yx9db0Ins{+KzDT$2$X+8OY{S3j<}*A4vJNw#EnAdWYgSK}N02c=VAuN82&bnx&NlQ<Sn0AF!1>C=88fkAF<v%?1ZdGpl|6M26Me=}aLwP-SJt?$AoGcVPxMxk=T|s4A*{X6R3BC{@aps}$T@Y{9?Q8dTYyCR2br&>YB%7xqGi;>C;JeeMTOn$p&J*k<Z(aL_icm(#U%A4v(+qr9GBW}4s-;J(lQZIp$MKC1cW`u5ahX}Xjy<@A81tlH0h`K6RmH#RY(<Yh|rS^`Di)($;^n-knp<*725kk_{&Se76qgAG~;K_c!~uuTIYy*6U#jig@}*aSJ5&DoNQsFENDc2RLE{nFSl#zuxwljyVsR+#nGAs)04I{8}HstzKz(q&3vi&5j&y#{(6qiPeOoNgwp%O139pBaghxxL^g-&#*PH5#z-rDH$yb`B*T3=yf$%QU1NE;6D^*%FuduP=|qMkq-LD~;78u)J2*oOp4j6PK6@Gy~ZKo7LMPeKouV)pZF7c-K_0{0<k%K&y7(*9S-|pwH?J-4%SsXf~D{u7iBbQ)H!FWZ5kS`@sTG^dS2n%u+CzW2_iau5d0F;2K2=-(uU1;0sk(sL_5^D8v#gtvC7!Ua#nDJ6?@oX~nLZ8hxzar#gKcYZvLa<(?suYKhY+9Mct9J@=tqMAoKvUB=Y{G!J^xBfpz-H=ay)ibtLkA5#3*8zI{FzxV8YUSN5p2TI~PWK|hRyW3C|d(&akZ_m|Ol@20fQ~ySc@;U}O4r4z^e=*eDLha@Yb$n-+$eSSCI=bc(xO5pm)V<E5nT!uvz77&BsHk6&onZuqe(XU#%VDjLmn5so+9Ky)R59`x=M|<7@I_hkP#cYQxCUHR#ZjLsDdP=XOTb;wQM?gRJ(4CQY<jeQ?=})2GEj10Y{G5I@tV=O(d)P*Sja@Zk<*TX8r8}Yn7;wh6nqwzf+G7KgH*OhNUVD3sXc(}w*q($^;f+~Oh_7B&2*jGfce%MiZ-bgav`ExbG6Eg&^9)TSk>D&RJFDdWWA6|O*%Z|QT~PgRij#0>-d~@ym@oY1E}Zx?tt-}to^XQ^R%oF!2DeTq)H2w_eY{lot^59s<J4u$Lp7Dw8SwT+M!@I-%c*TBk5<}R}bqpso<;Q>gXypyAID8jhv=ZnPBiYbBcd!t8rbCr(y)08x&eX<Fipzc$llhCrN6+V{c9j_8J9I<Q`_mHmMF>F`eO80?!-t{qo0Ce#DdS!>xDZt@Rv&u0iQIg2KlKoeyrGvvJ@FT+gp1s82mGkNjK$mnyqEhh!tJ3+Ojf-{yN{F7<_Y2*n+hqVD!>hvz^YmRbLNPeug%F`oxMQl-?YK7Yht+x5_)y>Dv;=JFqVzh^zc%QMs#K0c1isJ?@eb9UV`z0<3G12(Shk_la;(D#!Lpn)zpTLgL*|0l;AbsQCa){atLP(bRp5wWO~>ey~w7RIkuY|Ukv&rd};TRM$vxXX|#gw0<;=@!2_?q4R9@nom;mGAw<<zGti^iQRzWV{jG8(Fo+wwO>7kEics*q{2f6pjrO(TtD=7&ZZhyKoU-QPWrRFFL;$gpX5DFySw_(Db95JgYYDiRXDezo3B+nFjw0P)h>@6aWAK2mk;8ApjE=mONMt005mO000*N004Ahb6;|GZggpFWiMo9V{c?>Zf7oVd9_+wliRixe%G%+`H+$cMejP(Oh&OXO%vPkG_gIlC!JQK;Xov`VwNHp0^Exe=ePG<01}k6>rFe^hh2&Q&i(QoXrm~4yOW(N#io-@u@mY>6}_!=lfq+LS4Cxok$ZS*E`)4KQ8lG%75r+fXjM1wRH3_4ym|RNo6TNoJf*s756iL^+ui^vvDMwaucfGr*z{Fxfr!<@?%?xYmwl}=@l&O)jQ~kRD~n5od}dehZoZd4>rMa*)unJmHKNuRmF2xaAGQK5-6&C3#&*?)q({+2r$B^iFE3O!d$&`<s9F`44qO5q;J)S4>PmID0@(?@1t!w+F(8X9MWb%O3Rbn*EQ+GpY}@I*$n$M)`%dLKY;JXD!90z&64{&C%)Kk6l|?O$QO3X8Nwce~jsN(w(T#t$U^u33Z(Fpud;hJhYq_aaDt?qLE)4Xo?uwn0C~G^tt?J;Mxdoqgx~bl)*=$y-tzdWMMyWE!Ukmg_%IfA-nTox#J6*!#0yGAb|I&kP=vgd96eVK*H6B^a1pKt}w$`$Q71Sat`+aL-TEU-HL}ceoa*pigSLacR=ZSfeBHh?CPXg|a@j4x-sm7qSr75av`ApVEjcZ1=vXfv7vy7t@I7Evm;pK#WcM4q9Ama>8m2YmXGO_bSW_I%N-@b^QHkoP)U8*=qva-5R#>UBN@#PwbZmJ6~z$ao<F+LYho`^3#2eXKC$8R?Kd*JnKc#eVdd(|x8b-hYv^h$iEkkzY}f`)Cnmp6H%u5$`fL;eB#o_~2pt5)qHv?T`KLTo`?TFl&{Zu9o^>q{o|n|O(-vZOz>Sn8be8^~?6u3-t~qa~#cO9B1>L2M7ZRunE-maSoD{5VZq9F)UOFER6mkhHqc^$<bdfAi+YS1(`u=jnG5Z7?VupEU}6-V|!!b-;1t3q~f{A)xcT)yC#kQ`tO+sH(TgVsKBj#iZY6Q=5t>;v6c9fNT|K@PW3F?ZgwX+IXBIM^;9OpJd&ur(LJJIC{}s$-44Yq^|Z*B7qta$fD$k`CwtHpUiCZBC<UD7|XZ50ymH~Q2GSuT-oYMh4FBTw>LlOS~cR==!d7zzxnpxm?D8EK72%g$twOt--t%;p?Weo)PE$0(plgPo-`=*AOuw04t!iezsaHr&AYS4CCW41UCXX~3-wj8u-pa~TmiFp;8)$bU^I3i^yX(M_PkRy7j{Q+`7t(}k8hOeYggie1gN3Gtmqd|TCd0Og+^mAe|ZN4`r5_5f^u9#To<}Q0+g68a*@xel~&2ie6RMpyX7jya^n)W=)BN9)ZgLl(6K4ST$$fijReL}>!HlCrBZNGb%F1p>h>-62YBTw&$JM^41!+w^hOr8zNP*egxXiac70P&uY-bRi=8smDWNwt5Eb7)6W}e_Z8FYO7baj4qq-~D+(-=Hc?o?^IhgPWjAd2#h);6M&w{WbLeTIb7+!z=m{k<4A}G?^wzr{-ftCatz)|zdNBA4sIFfqZ>5be#zqGfakaYnFg|r9DnySREe?0&gP?pyab+~S$z%D3B=8Z*vxqE}B45A*iz=dj1BsiY2Cvuk?zDtkk^-j0{{DJoTi;R$lE3c1NTpeU1Kw)J1Sy6+(;smRJbMROQmQaIT4G;>Ns6Rsy7$^R$YenV~s5(2wK7NR>dx<%XwT0RW(HlC;$GBrPBn|<Iu@13L8E#NzPI(mnj-OI7=ajFSZF0zHT5J8=u-yKI`r;uU)RG?_ojP@|K0zJ*b9I{ZwLwa1?vr9VsB{=k@#Ihvqjag;p4>&e#devU-Ob|loh;f@EF2W$SFT*;#ZGIZVxP?}WdP5mJbfrvl0Y^zt236_V%zt}l1w(Yaarw`^K*Ev>o`CH+|r(5kV8v)t)Y6JXl>E6uo6Mz*dnTT!@zX~*c57v6wDTiMpIVMK@>NMprs|Du!qi@7$B3~1)vizw#Q7aFjZt@*4F5)#hu<p;fd2`nb7%PEdma7>JDSzgzi_=0%=&;C@6Vgr>?R*EkY$_s8mRbiX#g-Q#b{o6<-F^!$c0=0#hWA*857<ai_0SaUR6Va(GHsXY0vnkW0}1VGDwx_f<Q!Q5bk*KCDMvKP0K@5UA#RniD95T}S60t=tr}!CAH`Q}5P&YaNyjKnt(p^a%@YzFX1k74CAA5I3a7I1zFK;9@+!c0iw3Pd9mY5&@+HE{JOQ?WgGpXo2XRhw~_%(HwY;mY@qhXA>&I49@X1j*nR+6#RlWfCa#l2p-+gPb2ui)nSK!uo3ZQL1XuKuqqzy>F3uCB9UbNbqE%*Boi4~UV&>=%rn4hk)nZMrf9oLce%m62S+pU!is%wEG5^z$9GH$KBN55<`HB9RQsy|Rl)C>c+~(b0FW!|IpfY$^bO2MC8);|Gl-Byp;`+-c!C}vkq7lh%+7W~%<k;};V_NEVPPadvTxLpo&BT$+(sj2P|Wt&9v}PJh(Z!SM6cewcpiCqI8yHs;OR}v_be-Ni%A_5w)~o!Bn+rEpk_;`5y=x-FBr}igl8ZuW8#;%d;$l<`#@@#-CO1J9TWni)7^qeP!~P`Dqw54LK%kx<1$tUS{t;>=%a~~<UT#$@-T*>=`N5AxSfNTOuv&?oCZ*P#uV^Mmg=f1)KnXON6TX=f-Pe)<eKo@FUEXg$$t{S7EO=HAMbaDYnFsm#7ESStTP)cTeWqfWCX?$(iD9uqs0MFe3>DDw|O|EVCEPn1}hfaI!)hO*<n~L>7FYCgKT-20oafnhyQY{HvxpAoL>6Mn)zZdDffXvI|Z`Z7^8xMDuJGn_{v(~$`E3s)492xB?sMV;BUeU6a>AC<Mq{{gkiZNek&Ac&A!1$^bN_lKGJqzc9&@nM$OYE7IW`{Fu>R$WRGnhS~8~W7-qzSOO#}sO$JmP+yafOG~z9p0n8Q`!Acmq;cXoP4zmjcd#^feygVU6ys}ECgpy$9Hy@`!BAue(hI({mG*vxXi%+?9_-nF(jip{afZ)ju+5`F0jlyshb|Uv357_cRiXH*M35+o>+^L(l=>^g+L6z_~78pSHWH9luszpWsg)UTlNBB&JnU5qrmHykD3hPq*dUP9wS+n3|aU=E$zoxkHZW0R-^NY_Q`~BPX5uGjJLfDrM38|WXPg&y1$K=e1@p5`n1P<h7@c=^aopeLWr}WliQR0xg5|Vs49#JQ^I*bHk#|LVbds=a~ULW7`Btan2J?Qe@!Ru)LJaL{Q?;606ihb6mrwf8RG!hx=4`4BhnKco_O~W3a#-A-k{JUq6P^iaYQPwS93r`#{0zE39YYZ*O#}!Yci}hq&oSejf)PRS;^L0F3q8pB&qU3dTIck2YVYtMj^rG%NN$<rWW543L<MlA*(`Sbl(}Uu_9cq53fsqI{fkPgPVU?`p*N=$u40&iWNb34F4{A0^^@$k#6=wQ$Gr1sH%uXgD^~34*MmK2oSVCFF?*{n06aaIX)zrw7V;i|T&J_VeEW)ZxkYBgsKsw0(8!HeBJ&KDXwD2VktqJBjUEmI?0Z4KA2P@to)R_)2$H*obLfzGn<eu_$j8l&0BToxU6r&VhtoZQyUPPj5*_yk+9t2_7PlAhfK*}i95V~2>_Hi<}mstWYztI(lQ`_9-LO7%bw+nE7wcyk1@qU0`_<}r&l#Wc3;5$hdTjPrZ92R_g$>vB;VQLH<2c{vwbb)$%Ti%JzxfGqHVX5>GcjP*JoByCY3@}5X%AAzA$+=feFEYdticJM*YAcvJ_@ZFENf4i=qX0`E<FoP0g>Q9@U&#9}^dT22{I`Ot!2cW|QN#|Y>5i_M8XU5qr)`X_Y>&Z<UfXs`iQtCa57Mre`hhVRB-8t*$A?TGQ%n|E=kzf0rwk<19<%;a2NBai=<c*7V(Zyajb@mK3Wbr75Z3ymC~x4E$rk_1<k*zIo@{ji0yNkTJPMp3z$yBJS=-8DhhKSM?Knx6^n4%~oC3nBgHF`n|6ft3<U{pfyva@FB;b6oHsDHj{u6QJ^~J%h?F1)I=gaAw34RHZ;?8j4vUko|b3`f>#WnonO?MBZ2ZQO+y-a$CK_1djeNT21lJi2?N&WW%J~C$i1yD-^1QY-O00;m803iU2hm8Xv2><}58~^|r0001VVRK({b#8QNZDlWTWpZO>aCB*JZZ2?n%~@@a8@CbuzQ2OleCR6GZO-R5A5;j4oz_7BySdtFi$XXM)QXar+g)zR-Ln+K|Gm$U`~G6rXg^ed&nI$bI5V7iW`-Ar;m>Euv{5IOi<G^5{o_4T`IF2`ZLUn@2llU@UjOvX8@*SleD~rFD^wv<l}q+sm1kyTWyL;~ER&@;h*HEs@V83ZQt6zP@=~r@l1h<R#hM*tDHDkOhQu7?QB-MZ-4X5aN0Fv`kvuXhGm)1nv4O#FDrfRiq$*Km%zkXOOT}KMmgzjbf+2-)=ySr!_D`%l%PY&}lQay}`1$&^_{WRi*&q>xQ?4*ND<JM(nne1gL?mYkjf{i~4ziH=k(W##Ng6v=*70fRagBp841?eZA&m25RaQoF&Qw-tQ^KZPmx5&4nuc7Hq{3RruZh~FAgEXJDl6b~%krXTi-jrGQ6y!{WzXX7kKghmZX*bS?>ZswXV03q(!#%+fZ>w~`?!JBI6nwu#MLdT+?FCwq@C>S^<`$&Cfdv_Hy6KQ2bGk8Tf^ltM`0t}%W>tkppTL8RL(xM3-k#|zi>l$c#P@ySlIGf$Y`&1x>^ruTeTa8cHvW8&ZLIJeQBs+&N0=Ze113P@dh_c{@W+HK6>lFb5{Ysob{n~2VEz30Tv~xU9%^VR<h?9@2++0PPX+VW6yy(8mQ?Rcix8&>)E}|XA1=0Jqw$)A4a>4qTT8W0Xd1n3R#@VP3s5FRjx|TU0|$PA<8qX{+iI?NhNX{Ce=Y~ZbRUP&W{QSh{fCZuIH*c6956pdAyjUs%Y0-3YA9W_NYxJO`Ak7L3M959AdRuyRWabX>jXBZr@9z^3zZdvdM__N}$xpA!z)kHtAtVnK`iJR%C6~ii~IaAX82NWlM8ivv(r9H<FS@n<2l%&GS;&xCLU~U}tLqaJmJ;Ki8W<=Zzx+0(lq3<k47JE}n!+39O@@GhJv2WhJYx>MzD<GYm184LzJRRZghOez^ZT6Qu)3d4W}M8DLc=%6bg`9{c!S2;;EO1NA|_&yhI?IFZmFh*;V<9Gtgmzq!owk+Jq)x~M>gG&|j5%lQ?1i_#>Wa$R}Rm0`cu*X&CV)u@}gM3Hk-nZc<}-*~7-R~5JO?vFXo<$Hwo?CxqoT5Voy%@SGB#48=&kKfoq3;c}mL|r}%@6II{5*!@JV3z6xu|?Ixa=MP~&!&eli#od4#lb__OL_BYjb^lRamo*btXm(m4qTso`#FV&O4BDUb>tm6TBoFnR!gQ{b*f=so6r$*G^(|vz3^eRniKsMqxPv#Im~2^^Nn*=-Aa^aWsw^w;Ou+fS$QC4A~703>EW^}0HmlbJ*n@iqDZflP!zz<2@DdG$3iQ#B;rXcWRPHIbwJ#+qcXO<DLkA|sx?^_PtEj~852_S=3ePbO!UL6>N&xYQ?kGlEm$}C_lqjGuS9Blk+i#g7`vSCa61EqyYLAW0fM8>!fo&94o;g}7$`Z=W$AwX*?q_zSt_zggwQ>-&_O-xtNZ77YXGam32H{!S)@loAGDc-d0qDdh^#A|%EU>!3fG~z2E74bZU3rs?78YYTwE%5T|02v!yvRF!n!&gxpQK#OBsc4e|Q<LI%CdKoz7^+->U<<_ARl;XeA4|+uS{LNA)0X<ewoPoMg${Z~o|Ex}g9~&1FaiGHyVL4juuHK!~OcbLrY^kUoq%!<rv0Tl@QZfreYuf<%HeB)AU$8|}lc-{%J<PPqmiCgJF4Kdo>#jqrL3;eHB%7A}3p9P%WRD{qFP1*)$2HLYs5G9twgg=UD(brl)(xAASif9*M77?h3iu4Yne(fVgI+ZDzh)1r|H4!zgym<I?DR_EJL!U@+dqQYS?)vfP%wwaa{q{D9#z#2t^cwbS9O=f~2!|j^==R8hb$MXl!Elv;v>f6r{PW$O$+%I0q@B*44bK6Vi-?Lq)_rIVG2_L3e?P>zLFkZ7p_5b5LAeR$<w>)*ZiE4zSsN0U?k3reJw$hgestN<C&LV;%C%qobF4xO7PKqj8#cm>whMM?sWXwcoeuBx_#HnVE`BFP%YbHDtq19|???NPP0h<oor*!xLp}X2^J9;^|JfPJy?h)aFpWW#X3lhnn9Ctu1w3qi`5+INAx?uobQKB?XZUKFuFei*sAX~e*SWP65PS_@mrn%$cF+17c!u7(Yu9mhkaiI}qX}f_$o@aK6hDzzuo}tiPl3kssW1ra`I-mnL!tBj#81~BPFxSxWxaPE%!7;*!3TSlmj9_3sL_v?6uz<nbOB}+*m&8}U$Uqbpzh<4-K1_70Y`bN*{}X`DHTQ1>|If(mGrGAKkIoY!e>Y!_zfBwIpwdBJgeI!z_~wm;F00G)Un`NSQjM40<EbBYRDP6(z|ytF<Jp{Y7Isn2`e(VR;=(>6?~$Er#v6bpMH_sc%^L{Y;@{49pc@Qf87@wdFs`ezsKy=rd{zxCk3~UmrjgT#27vy9+?20%O~dEXB7z~{?a@1NKACV(@))%TPP@WgNb^N)IWmYai_)h(Ylg$wd^EyMPLA3u;?9jlevQOM+3oEr&ZNkr)oMwGEA&nPbx34&PsTFau}V|NHR5PaMK1QamvU~xxNbtj1<tyO{Py@&H?AZp@Mip0TMr(m<TInl<aMM!QTL{=LGye{P_k2cv+q^v-g}kr2>pYppTy97;@VI0X0I!HfWE>LDc}j=>)-hcY5cSm1OXXUv>xKrjg$&qn0s$(zb-s7)@J(uQnim8j|Gz_x|VkGQs@0MdG8J_EXoAU9bmudbi8?b)~jKm8<FYpd8Bc$ILdJUrvVxQy1x-bQ^jKLb~s=4g$-W5=0(l%4Tg`ox#M#)hkFC^Pe0_7`rW7R2}<bU{`0&3zU!S@f@Wan^&gn{496EuVqpQuK~T|M44blCWO6eTC2{>45D~#@g66b3--3lM{%1Owu>XiXx8Hl4o@G~AHn*+;yUEi{aV^h4zNO0d(Qka5rWKk=-~DrXszD8^qqNSWCip{7GasKVJMb#(WYY{^OZyo78&FFF1QY-O00;m803iUiKS&oE4gdh%H2?q_0001VVRK({b#8QNZDlWTWpZ<Ab97~HV`VOIdCgjFZ{x-h{;prKVLwE=Nc3IW1_ciaa7~*wXqp;Lu0UfL1Vt__#uTY?N&9?;|9fY4_LW@9^0fu(4|gItJ3Bis&&=#@6h+^+c~-Ogw!5vG8&*~|7wo?3_w0^$Rax<3&35h36ue;fUDd<azGX+=iAwam>6z%;F1z7z6h(_g*|i6jrsdEN9ZyqM9gc0+v#e>_KI^Nt5sQU-_lsy7{ks)3@|gAey4vYsm++`b#D3_jT0iw!hmx8{emJ_%A1it!u5s4&Rhi|z(2SQ027m9=>pyl~+pXoRAC4ct$?`p?r@wPC)cyCoxZ&!1*0+Z$Pf>Pyk!MZYRG`6hpNT!a6Z`DT-+q;%1nb3Wu~-zmWIIq{&C??5Gr{|%XooK6Y1g*>+R>(bu}UNxQ0QgHk6FiA+i+H+oi;@!Zduml`|6G}(K6CB`_tun)^Rii*eAPdtAh1a^T7VlAzjC_LP!Hi@lH(5wo>V?c&jzN8Vf0T75vD-HJbbZMn1|{_}63yp168&8L|ucePyW2wv%sI)v#rxG_*NnO;y5@VpMjtW>IwjHxnCj%UpUBeIrg!^4~h!@fu|D?Xt9T5^=|CU;wI7>ov2InMAY#Da$Bt^LFznc|OJ9bEH7{al=tnW93~KzG{-3S4SkI5iy2UDu%MGJ}e_w_Nvq*`~xr)%N2x7p&6K$iBVbAlr8W@7IEEzb(fkJZJqY~L%&>E4FmtBdC=oym~R<X&2i{)z1yv0O^+Bwu>(Z)krTNRSwLhjSMXnopUg-{T2cl$Y9sl`_XIRsb~W`3&mpW#*9+||MFpz`Uw<Hv(VimEOGMsw0A#5HIo5e1M<^2D3y{E*6CucZffM3OIhQZ2kz*BiH+8#PMz5lkujf*ZjVnQ6u$m}VekPv)OWyMiY!5R%M#k^ukD7iRZ`A*sejJ;d=xM?eR!)3Ti7}{8pyz?hX-Ic{5C2HD67@Bd#y@ey%B*|%gI*<irH};jy;Y^cqg1L%MU)}KNYdeDW5~`&NFKLf=B3LBmUkb?8P(flVivLns!IT>^I|D&I&wTmz&I<4<>aHq#C7l+3`RX}Mc*m=$2!c@zV)DT73Ki?8vrTp`1aeZ7J+H3C5j5%6d}U}PggCLG;fa&skZ1ybrhOw#brxe71j`RSg?+8jtJly)~6TdEF>ZxKtNpFF9-!m7zZ3O1u=;e%WDU+ot&ysY%QhKP;G&OQmIOGI+vz|+2aJYri`h67ci!3BpxW{NiY%Mrz_IPwZUdhgkhZRL|YF%gfasUjN&*#|1%Vju6tn}u*)jJ{X%0Iy>CQTa;C&L*<41f3d&GK+2v43wfBvL{>aHYGU69g?+9jaXWW|k{-gYcw4of}TBm%YNiCW4YC;nqZUH8jSjIHH*#6WFFquqfX?rUlSBta2pql{$1cC!U?0AQU(2))fNiV&$sR3RKP9(8kj(92FUG2p*IZP6(tbPD~73WF>U?2ZfQt>@1M5R4UQcLd&QQ?#-3?Vw&`VP1uZx^UsqZlH8&H!}2#DmqMqmV9R@_=*Q!3@@EWK>0=$(HJcEET>0O%X6a_8&<geM@@=ltvw_MB0S}mVTu&fSr(>FuLE7-?c*zC^pp(^(8Te8?tP7zbrK?8nR};VXxSm*I(-YeA!w7Vjt@)=Qd?({9cU!uc?oI<KPCKX&Sx(QCP-Geh)Ry2WZ=V{_YYwp9Ug7D_C2yJ>X@>3xU5oerS7WF0$rkfC_1Yjv<q(Or@r0P<@mDYnvttT_9E3OrhgiOuLH!Rs=eW34Bbe+fm^aX)%SWWTs^=!h%INl~yatBNozPXlM@}M5yI&nLndf*q=a%37wC_#>?jOvw{^>Y3MTe(Q}&UI5f9?3`&pIfe&e)?Lr?jkOPcdq>u@6e(bxp0n!`jZd2aej7B^q(-=~Hb^LR6{1)R!x#?_?7FY}P4UY6lxEkX%RGi?s<Ko;yS)J)Ze{Kh%10SDj0Ga^KI<gvtlav>r7#IZ5TjjS<6s(o>7nlG1_4wOUkni>6El4q=6JxmlPrYD?2MLIp2_vvp!$maDKzAjhhGcF?NN0lohe~;7@p6%$D&iOmmpZ_`<hNECQf#%*t4Rgy!^s%vP8Y^utno_UR|a!_XdA9@FF@(T^(<14+7&ECrn^3-j&>^$Z==Kg;I|PN8!A-%4~Yks&YZ4{a#KT!7~G6-NGE0M%gPnEOmVr`TE%^93Kle&EmuaR#Bx%S7)844bsFu@rmGE=ZzYuo?nW9bGvHMu?9q|)Ny!y1Q6|0+P^w?P)N&(L^)aAUZ}(9?SI`{xRsW#;a+cC>iQq+GlM>(hK04YHG?S^Cqu?L>Jw2QVLXgDXNCJvNZ3cu3D}xHxq3(s+vRV0FP8_D2d7{aVVlOveWf@a7Ih17}0r1t|mU6IPA*fGRkzR>hS9BfngV|F4J3E;bq#GJz3wEf)A?x$KTh}XFyBp-OHZ%0_Z3i`*V?E7l?Fg;NbXytV7~)_{;**M^t&+rA*6XAl63_Nd!D&gRrKQ*Okx^-9G~e?>7Rfk_pflPi&&i?Sz7?ChH+Jg_aB}R8$j6!z)PihAZ<T=;qh4*suy<Kqq1RahlR(UDtt*Rm?Y-QJwL7d*WGa`X0JMbYI#4G&nj)BlIFRIe8zoTHD#Fh~qr)i#X%$ypyPx6Q<V|Wd1aThhxh{>pweJIFvoE}(uEoKch=Q($h7olsw}$WmdAg$K?a=fumt63F4ZHzu2SX7i#HJQX7Mf4{sFlV#k_ozJoX4FWYkRDj^7e47dCy~eXdxX(o>ah(tIeBh^c0@ois3*SQPb8LBux9|1(UOVCFD+j(Q-jME2se0@A=e?2|}EmO%qKCo5a_8N=D(8%tW0tK&vs43)h>OgiOa_ht;sjc653eZq+Y|mO-+3U=1|vit$^P4~IeROwt)1l$Is9Evp2ptA=NCw~KZdY@7Z++MDwbbX1_uWk&_67AVgISLIm_iB6acw-*>U#0q<p8Q>i?!j4}MX>2l7m{5Dhh}t>y4fw|E(H8}x2>`9rW-sgkfL?bnqlcz`Q55fZ-8MIPdJmZg;-|wjuVY6d6Em~WB2xKwB$|N`_BD+<ErhRvOM7oXl}Zv^vc1}x>LJ|ennCwg*Nq@}iT~v-EW8<@UXE=doy*9MoC~)J_0}h#lK@y<PFA*IZ&p{YJv1|1W>M{&to*Dt_8G)_2IX5>a+wYl{=H>`pnCwM9d$l@u7VI-w&yK=<yN;aE?G>DaCK;&EU#gxnp<vhuu@O4d88>nSI?hcC$E23JZ&Dyo9H)jy-MDEb^YYPige{L5}ADux0|-~Yi0x*)iiN-#A7LIn-R3A<1yqv3TSgO<`yh=%GH=$Rtm|)LB}v5i-bhxZ<KikE5<&%WlppP9TZ4_-#ATAbY|_lW=_{Jwf<s}!wiC1H`do@OoIEpJep&W%lZn62UZ;pL!a&7*AbHj9?FB?zzBJ(h%T-`TGtm>=)zG<JIn7FRv-{cD!Ck>;D>C5FC+_g%O510AOHMrqjy1Y0|w0lI*Qp}X>Q5w&=$PD_{%?7yZeRb3WpjF<DlH?Dkz;l@*-w`!^|uXJL2Z}QC%89QWbkete3h_*|lUTG$vJ_rc1%=(m#froKe17v$7tbT)>$hwr{?2`$8BV%h6ytty=dO(5H0nWNlAe*$L3U<WNQq^$LRV<QI72>sJH*K(8zql?EgB;rc)~6C>J5p<s)JQGEcW+>leq;($dUcUN)D&5ut{i|MhvH``3N=QBN{#(&?4;dqoc7u*YV<jgcvG_hGz0KTv3N&6rk=Re^)cMn%mIKG7tY@e*R;$p~AA$}b1gYI$Q+Pu=~<hqf_RSS_-RGZ_TVsNKTQcW*8j8cZj8%^?{=U0HH&2%kA37neHAdqHO68vCwgyw=d<S^M;cEzY+l@z!(yk+7l-REwPjV93`Sv-GgJZX(boI=OGGmWyloK&~VpLEdcR4D4q2$L--<(Ymue5kMYG-r?VKlNDOn%Te5F?+;+(ikr}%6IG$z9@2PMEC;9S#!{(F$q7Io1);L87)0`vOj^N(?|T1Ls17b<_=K*YgYtJ0QHb84lwzNlm77RGj|NkP0hJ`1)iJJ^e?|aaOxs~Vo5{8bDzFHFghr+=q0bh$vAd*z-&Gz+u@rD;+xxrW^;RlM6&jEa0p+fFW9xav`9nFr*PmIz)=7n%2y`Yff{YH!%I(H_8{<BuFvo?i+d<Gqme<foDN;REW6=p7}2@8Wt`@csR305a7ClBruQx-seM0}bj7~jPO5<bXZdwG(I=JqT7g7_^s8j-6z-&<+&j8@3q^}Mr3iQrUGtUq4bAOam%qNaVT-kyo8NRL)X(^-X-RETn_lxi>q;yj)pi7SbGKx@GGTg0)=GA63cWugg$8vetFhS(`s`&{9U1HS<9_GLtSaN33$r72t>*bWdvVsU(yf8v-_adf_bB@G-JF>UPMXG?o||bb$6v7@LAkZrp#c_dE&o>H7siU-eiK{>33B_axdC9S`gn0hd*fRT6x316_laGUX7HI?YM)mg2F($@n^~UR+jE;6_TJypLWPNbC?4YI#0_Z|8FXer52p)jllxqL>*-*tgP@O-=~OF_xTc>WO5I%`dJK@dIWRGD`-B$D-0hf=E!WO!!p&iv*F5V0$o~ycO9KQH000080000X0NRBSIjjQ!0M`is02lxO0CZt<UvhPBbZKp6FK}{iW@&6`Zf7oVd8Jibi{myFe)q2sezIv|mi8?W=(01V56g6*-S#02MwX8gHJ0Q`a(3Kh{(FxuapWXJ3k?iooy&Lb=j3_*2km=exd1|COFn&h_(%j$GineZts!5zJrH;udd|2dW|HinmE?a@%atUJ8f8m02UF#Fo@Jd@g4A_4S_G(T!o{GpMQEw4*DzTYY2jbbn@WJCEw%K_8G33vrN#Me#%9opfiY)_^EcRHN?LezFlSqlBB-Q$&{>wX&?SCD(ZY!{*gh!<<t8KeJEJPi4q2dd;Z?D*r4=_dJyG7%riY^RiHazM8{&Y7(!^LDsbys>)z-M&6}f}R%V^5|669OIO7!^0s~B7aF66dA*ge<{X*shiBB_9lv%4a<LEVG)bmtn`dcbM`-PH_>Xh9c&Z}oI>b>h&s{?&1boszKL!}l28&7Je_#hLQY3jYd6T9FUchmw4E&jqp$4#v8HJaM$)TCv7#@D7YE-mTg|*%1?F(B!wb;5F7IQVEx&m_5wBTE4m>oVdXzU63`E#&siTvlKsqi+xF@j6n~YT1csK6vLDfo_XJ^uPJ5h(NI5G&X!iV5DTQ<9v6J<lBQQpN*gAon6gUs3n(>!WT*8o*~1)R+LUS8(`4@Ahb!dm%vLAN0H&;nQkNcpR|dnqF|VG3=##QPx%Kg=wbK7?+qL#>gK7vRLxIbhsMrD6u~wrUMq4b9cln;%d_{6pTn|(maF3rKKR)CyE8^0*{Y{V9qQ<tM^)>;0c>Mk8%jeHCoSSmZhd4Gd+~SvCeLSnwoliJ9&_0KkXP0*)xF2GxIVKs|%SirHxFKX5qL$OO5!?m4UjH?Mp4R(e+y?AA7BO&Lvs1fe=CqRP02K3pxw(P8<m!g?Ztj+%<>Ix-=7ord2fIgtNjU#A<8wlHFP;{*m$?w!*5I;*SGO~8{`XKMJ3EMnl~tsBJ)VSZy5Mo=)lmU`!(C_bRu&$-yizn39dJh5Wd*<8WH)y^G~Yz6%5T(LApt_^sct3<<|1tI4EJ^5-BDt6pbf{2PpGnsI)eVG<_l8^E|tD&hqtTBXu$I?6W8;&bvE;DuL9-Vc}1Bnuqd+-j~`3HxF(!n{JdW1wZFcbSdwVXmnPk+em?5Z7cN=;BN4z+xg;^Qfy)FNK?vL_n|YE`AG#mD$HQgC*w0Qh=qi@kga!%Bin@bPJMlyt1*EJS+}j61_3<u(l+`Q{RNu0H08mQ<1QY-O00;m803iV38_>6N4FCY*EC2u&0001VVRK({b#8QNZDlWVb#8QNZDlTSdA%BIZ``=?`~C`^`k_*?QEXq^4_hp7uH#&T%L^ALEwEh>w8Y(2t=Eav+D?%F-WigTNQtuDD^SDOU6I4#aNaXi5Ck7ePV$PSJSA`6vM=W(D}`ib!k2{KWkP1<o)lHebCU91R%SA*N<mne!gQTyNhV>%jyD{ZkZYFQ@G=!~5Cn_GuBi$_>8_P+!zm?MQCE$GwPhul!n0VIv7{;`zms`(-Hqrk7>HY$<r-Jb<N-!HL?7X?#~`aFIhZ|TCM7R%hYnP5N&7)+$TYwyPpTA;X<%Ph%`I!vPe3DIEEY-5gdlHoc4U9k>sAW_e{jU}l+}_qgf&$Q43P(J@vO~d28?4lsT*FiMx9tO4saDr@#?xNdAv~Q&T|B;idqU_D8YG|9LRwk0qJiopvif;mxq|VW8wxR&~U=?{Kb*wS^8H}R^*VSDK8bWnx%N6mvtqSz(~V0U~mJJXwH*N;Ogp@r!o2E3lVr&a?SIq+za5PI!gi&yotNhYC1s+K&Ge&l!k)myQNl-C3&_az$!_*O=({3Gbz?rP0J(AGyKCfvDT=yUg}QyZ~&xl*rEtkQydIIRV(XOT6=2N02V!?_t1k>9leLlcJovW_#B-2$7Z!8FHYfOwe7)r;)gTGc|i}HrJ-IG>1EEU8nzxt#V2~h?<okn<)ViINw!VtC~K(qTPh6tb91tV|5zMY%{MR3wvlGpPs&>2-f$_>7DhcgK~HvBo*Q;*wWL`JsI!9Yxv*F}nA%#~U<^7fXrA5h&|a9z`<kyc#C28Wk<D_AH7?VvAnP^x(dG0X$cGwt02|1PqLu7A=SbAQc_MlGV#gUe7~n`NIJwQ_LD^3M8lh|`9wB>^BEg{CWY4RD%jW*263t6RBA7)3^0_z0;M&3;1p!gG2$d7b(LKcA#n5-61#4IG=H2<vm-OnxFPHD>oA<9S|G99M!pZk&kg^Fa4FN2>Y`-LdY)g<#L5sYmOuL$3NnW#D*!I*b1k2#~Ur<Xfo2F{Q352ra3zi%#rWr|-wBiDs2zbi2*8UXmm5S45Yv2SpKTzd@A^X6`Pd130k*2yu-QR3oc2x!w=!79cIlQk~9X_aNgHa_qcmVEDV?v#ousVv%s{G2EDrADnw$8csL|_jSoFIrEZCthm&%=oPuqLM~FB6}}YQ^r?j&e5O$um4KzhjBa?+Mt7KB?hu4Sd#I@UVBvV&ya;3ZuUO2Osu_fJ06pd2tO<-tk7De#-XYys?_5nxj+g0%69it|9#SPpd5(wx4H4sCF<i+pHAf+h|aOC2;`s!Jd;-aE?*5oJ9*xV|}uGBDbQk0YX#_zrpE@k388b4TS+zN<8)VIZb$?!Mz??n(|4Y0!0gMehBX`ba*FiiNPD>WRihL%}nu3^xhX7P;wI@;iIFbCNKcxdr-Tj`gB12t~~VXkuVHfoo%Pi9Y**uOZeJ=>ZgVO5FGs3YD#NQO&8BJt|g&p_){|&lv?Lo9Fi0Wpb&}}xN<>3taM1q{KB#wkhTB7ug)($B9q}sNAX#sbIDKy<L+yuU^*qJ=7C}B(l0$)K!Ug)$akPd>BL&G)`0*&MH37X%3Zp1I_aQkmdMd;G)_#^dAqdCBxFxZK3ICh`okn2U93=P+M{`g%TMD;deb&oVq<F0i8Ft?9XXkaR=cxW!wm}k<l{$|L=GkjwLrP&E~z$QkCD0b?mcWBx25>4<@_rTPyT03H2nWPs@7%Ch^FVl!|JFH{(n}=#ZN=|8JnMw<ba(w3slIQ?^$wB1uGzlXhKu&m;{tIRV7vN!&SPJ+Vu`pO`jn~U;<e+**?P*)}VpV5Ah#k4I?6{YVi0FAWR*N+2)dz{D{S6R|~76y3aTvKPv?+FL)zBjwROxrmB|v{PX92{9SgJmD>J#O4j4~SktyJq0t=3D&4BK0qWMO@(yhVMC2t2I0%tSCIWoWsN(v5h+m@S2_`B|Wgp`x#Up`Sa!$Jx#?{9F@gme$tk^UMY0odniZy8-$%C}ooKLsa<9cHxA;gc!p*2>0ktXJMK%s^Y_16IB=b)}^0np|s6dD67?=zmKTAC<}+?tw2*$K{VO8nY_?G+}1I@P&TS1)ajaLbn|1J58)9)?tmqafde3sG4Uj?VX^&xW6~4FC=DU>hJt{Rtx%qNh%IPxIRa8Bwc5?TonK_8GjY?1GzK5kT54pw0<+&+8i;Vbv}hke1+ULX);YT!3;E>^Y30u~XNpRDt@?A_Q>PIGwLB;ONzYE)pv>g5`%Q?bQdBK1_0zEy{!NI64!3f}~B^>Lhqt+h^jmIv;i3%Iaz&S*xlxeG`b8EsE7V;uOtQC(R&oJ%*Cjg)XJ16RHLGx@s<1D_H*aoiAoQRM@+Fy}EpT{`u_{ee;ffx;p>$N|&L_Ihf1$uO5cLO7`RV)y3b(V9~<vnqawZ$}W&ZoCylFW;r_Ta8x+!TE)1EnQA4tf-u#zP=p>B0GAxg3PvtIzdFZZTp1G#XKH?Jvh0@-yr8;)JnZ7*=TWf|E!^r5b#Vv!C4W4F%t1mn4o1q8OeobnWcx!fUeb4z;fQ4!UxT|p6s)-cD?kN%02&lWZ4s+JCzZP&&mLa{Tg-#&#!Tl4v~I;JfM8!0`z27kDx=v1N^b&4)i9prDzpiTAz$0Y2xnXr^r+YX_4loEc0x9x_Dve0o({HA43%<~A3?;?)Q7`D$DStSjj}J7R?M<?V9RUOW16AeA7hmp_nOv!MS?S@d9zMDMyw=9?F3!v)#+bcFqC9cx3u5|1#0%b^ig{=(8qfHEeKZT`TI2g6l@2MwuFEp41*TqzSM+SKEGmh{rXm4f%HMe3XH<`fbPP8XB|0FN7gY*yLL^unROmm^gu3?VQQveM+*2bHQ@CcNx?d18joILCZ1(8;~>3G(NFM<U(vw;ECOiI;JQS{+Yb?9Z>OiwG8b)2ecpjpkZpZoH1@K37Co%KDs!1&W^t;krbb&q8WWnRX$nQ+(cS5>bhnW18^9cD{*Ox<h}15_N9o;6if7R_o9)eKe6y*ZZ*UU6Fw1b$5?$&WM?6jvC--)$hV|U%jVkIs!R#?F`|jG<lUX+(*a0#GK@r{DFVPPjm0;J}PtoIPFwShAlqXDj)DI;959biSGtZdIZlKT<%)T&HXO<i)4=BhX3H+q5FstU1ywABeO#`Q4*Pgx7QLSL=*~wpezA78b%F(92494oV?t;(dO<CRQWj+1f@)h~k<Nt1@6&5j|M;?|>t^(~_nb_kRN4?7#{I<l%a|i6XS7>&Ma0FB?b2k0qwm@2aA37<;(X{Fz@Cgiip&Dx{h)t>u=I$Q+s|?^vY5SAWU>gcMqE-?DRf(Zfxm06HqbvoxL3S-?Ylmwsh;KQn0+${e>KZ^@Yz}G}cobg9s_w`WyzUebe}gPRsj<!Ps(F0%j}NLn-9w%^=*(GM5<{F>5{!qVIHnzV-1pWTO+)H_7~_h0C5)J$u<f@5xXskpc~${*54U~9p8WNIdgFuTM7?GJikEh!KzC6nNaqvP)hoID@JSI`Lj;b=2YJAo!%l0v3oydU;q{RYny@y_<ZKCn*Tkg^LraUiw|=56AT{>f*-TEX4wx`?w~YIDCwuIcE)FJ9SnFy}+EN|LiYnP1sF8^(3X>|N6ILEsrxydURo(TAzXfSUb#?=psmaQ5EaCV%2Thu)?k2t~jaC@7&VKuDnP#awBff@-x-ZcWp`p!VQ9$rBLs;Z0U!YEBI=F-Mw)RD9&n86pv}=IwR?y}eRhMTLtL(n6X(E*gd~2^V%8V=&Rzha=W4f$tXtqSY96`M>1TG7^Fyl*#s<x;z9&u!ydU@EdxyfqkOVMK}0khgoMwOeucZd*mj8!WdT{zm;+8e(%D}uL+8frU1YpL+lB-8g@=Q@nmHQJ!mrg>?NT&v9TnDN`Sx?A%#6a|N5&AQz!*c~nTZO5pyKUs}T@G{sq1*KV^#wT`%v!?ZQ=*HUBVqOpg?;({qa&mcsUbk<TU!I!YJ!UvLFTezK%Ax#T4|dBMyv45?Y#nzk`$ux6U-<nkED)cJE4*kjFz<{lyw_d+toTW7Key;ruV+~L4<b+b9Ul5*+owNfO1B=@M0NcF?P07}JDlb={wbzRgNjzI22{8z;$79;(5BoQQxv8+5}wQc(H)HU_SVk$C#~M$e#X)}d<_nB&&gG|(g~Le&#Nf<Fa;}HFkRHt5I-W{>m}x7tB2m5&^@aj8D;7XEFVI&WE9?<-uU^mnM)J4(==-^;l<-GRosBi?~xw7gY)mL{qcJ01)Nq)Wd!VL)A+31`aJ`zEMI-j1KcQp8$ZCv-?)$dB@f+I+5$v@K?K(lTW^$1x63E68`Nn=hyCiM#S^zRCU82$3z|5F0J#Y2TYX^bGw_r6bj>AOpTvKj?mrE>=Kj%r<L}usz40hr=>Ar=lJ$4nWO{_b$xN=wX53x>D_Z;qP)h>@6aWAK2mk;8AprjRaCa2~002A$000pH004Alb98erV{dL|bY*jNE^v8mQ_F6oFc7@+6&5G6Vo)-fWTlmmHj0!Jxkb5JJ-C_i;s>(LJof9?#!Mj5`hakCRk^#}&hz}4I0aG?iy%qUz{fLDhNC2%u|%Uk8{2O>i%5j_c(CAw(WE_M+q){y^DGl;FxHdd>e2bz&Kuhvv(6foe#wk$oh4LCI6*ZXRY}l?5e~@aZ49ySRdACv5qdGY6NfTwnizgH_60@<vU$t0td-y#c^CTnAza9kYF)h<K1wWpWh5+)T`cPimQ#naE6E@(DBT>6YcNJ4C-oeu6oRH)sG4r>OS0J(#V55hqxHu~oiqzI5PQTS{=VB07sUX+je@BoRSTZdq%CK&q9l{SBC+U4t7nDt(Dp$SZOYm02f3>r7hRV9P8U&i;u55o#iFOa(z(p!r9h7$nM36C=R-QMof;exT(z5irgAbZ?3ojPvEJR6pYTy|jymBF+Xau~Vij@$9#iybw?y@**sPRrMeFMA*C2S0I?ft9B5+rK)Ao2}Cy*n~;gwp!g$>4uNsMZboC|#jl*gxqv^S1<(}&{)uUFOJ8ef?Xz!&^3;p?VVO0m%)6N-gfvYw%iQ}hbb;hN@09mZTSB?kr^hR~xgG>dMO{&of5hnyytqQX!+Emn&8C7q_h51F1P-_=|A4>_bUSn~(lFcE<qUP~i1f~C8XgqtLL0j{1>yNRb+Odi=kP)h>@6aWAK2mk;8Apm;x>}LK2002@H000*N004Alb98erbY*jNUtw}|X=Y(#baO6ndA(R$Z`(E$e)q2sxQ9xGqcuU-0tNC=^a2#vx}oUyVgv*w9oeiYQWYsDZs7mEbEu0Y*-p~5o1ZM|T)%Ur^2!uJDcnlQB#=dEDi{2PkF)l_gQCo(g6_H2(mAkhHp_|A3CpWmJ6S+1DkpQ1I@`$!X950Rx5w|Q%2acI^s)T*Mx-lfo<4!CmHQpCCA9BTp^cWQP<$oqYI^p?8a<yyv)L?F!rETJXEe)(oq;huz7ZCpSDrtEJh0&YE<+0{Zx885k1@GvDp75N%id*_&>p^(6{OBo+b|llRmM2vAi%=0oLGwj(8f-+15v#-*ZOYlajHP*x(r90*73dfG6RI$ZFvP>&?w7Ak;r9jYW$}&j%2w+2!L;ds=*%Bhp0j2-hHiw3i~&Qz=yXiVXuP4`8>F|ng>kagLjqPuxJ;}?`9=xBt$+E5|WS+(;mue=~hF-J_%{j@_)zypEU#f4^shEr3Ug`rV^1sRk<?4g-i=gR^F%O^R|4Dl5_N?hnnrC0a8AFF(|I0(Ac^FACJ@&r2+>Tmqk%Kkw}HXn?|JzD(2EHD@+E8z^h;*tu*>f5Q-ZK*OrbqFFmNjr7L<wU!+iM*9%)iBS#iu2}tq$0?7+-E0YoZ`{%@N0qE)Fo~`i!IfwNa_Gv!vjf^HNF&2lZvJmyoQ7sUdq-bALfByLg+sy;(gsbgkOA#H*$@fB82wS;5b|gBwM{O!fhggx~*Fk2MF&o4l1A^rX*>`z2Fza(wL)Zy9Y1ct*^&Co2l6dk36ONE^?20PlG_wM|7tr`>!zg+LQ=|QooFM-5LtGw!Wf1%le8?!I4uJ~FI#F01L>B*;4=3t!G&Ux+`4F?wwowW^Ld~gExqbSy%a&nX8kL=GUL3Xtqt%v|G##;-I3clz<IqIQH-J8HGR)N2L5!7pNe!c*36^m@ENger!SAipK?e~-1jc2<j0QGAZ4H=~Q_NbnFQ%@vP_Zd7rc5L(3WN<-*y9w8VQbFf>k6k(m^`_~Uh}mEonkLKpR-bIu|H$)8E9rwJ8TMd>7I;o6PsxE2uF_;E=d~Ht#>}Ur;e1wlI_*SOJ$O9Zz*JF6w9-xaUyZh92S;W*@5k@%@k}6H|~VhdmA3;`n0wt%Y*S24^pck=Q%oHxzrZ76U?`q+qEo9%)<XM=S(G+QqurC;ObJ&XA*aa*LYi*kX7V_ry@K`B~l3Y;%Nq&Oy&~jUi9!K<Wc=$0>(HRUGr*#U};g48MbJS8*#eEylFPM5(%N&0Qck4-nlYZgdhdKc{N%aZ^vr{d+@9eehrv1sYmU;ssE5ZGU4coq2Cmm7Z{j3mV^@{JZ{fBG{+e-@<_w=*`^q{G#?vg!((}IdZ_nYhjPAaiMXQ~HS5qZ9>OsCY1w->iU&wuS4J<{DAA2uK}GR93F#QtwkP>`o@D8X7SFC0{p7gvDL+WJ6erG-vgbd8W8{5n)hLmAOQ+SY%Wrsj!^-9waNhG8?ck+M*Q&W6tN}{PaXS~-0>re)@iH;Cvb|aBx+u3ae|6cvhMN@2F8JUnyoaw2Yk?X(U5xR1<V8pnfnL!F5K4_&qqiEN?k?+N5DH<}c0fJ4ba1;87r(veTT$PE@F15rs0I(0U<$e|wO5*b@I+OKttDwA`x)Nsy$D^#o+Ml>p5f5NHJWay&#v%K=I}OmnHZxcw^Qal#2v&o0=_|IY#1W{1r5v-oVzg?F|K3HuGyGK1)xL%ii9pOBmGX#azru_=^77x{qCZlBxbyV@7Hds3drU`mqiF&qi6}uouj8j&nYW$!ncLNy$Kweg~Q8Pv_M~V_VovMG{sK!qVt5J(ztSz>x2D>T`%05i95B2WRw=TQR5i-p)NN;{C1c{XaVNHVFK)qMP&CQ-zyGmup)<1D&I2yy#~+DcYPw8gg)^^hY3U<VJNWeYLsj-s$s%1LDe@b;zuLar_O99^MG^uoW*%?c^NRy3n4Y<tl{fFxX~##?|%VMO9KQH000080000X0OP=9pvwpV0G1yB02BZK0CZ(@baO9sWpi|2VRLh3baO6ndDU6nbKAHPf7f5Z!SsPlB}&TVoT=wD(`(}Un7LjviQ9*&Dh47!2{B3V0LZfA@&DdkfTZ{-I~m_Ym4{g5?qWZGy9<e|kji*_ruV7XP7)<EZ*NQyYOZd%@`QW;H>UZ|*Vi9D^4||!o1a-8r(8|F>%HPE7WoceezGi+lI63Oxsz8ggTgCWAK+_6WO!C2BIS<IRMIL^C%z&+<iZ#@-4R#c!c0at6$2a<)xy6BC%Y8RG*yS(h>V9!8IiEa=!)-$^1+o-63k*oW+I{|6I#q#^Ao2(@!<3Kf4sl?m45ng_3qQuKCna-E!XZrERT6ASxmu2A`v`xj|;^MrZ{y%-80P+PWhctpb!NkrjuYYnM5hjct0TBHJGja+5!0W9Rw)2wCsPilo>aB8M~u-;sMt`i+A|Kk5XMy;s?Hkk<1Oxjkof)B<6;zOyoiv5zV4Kk8TPnax+WBojEA}pPYEFyyarilAv5(Xb`d9oZi%O8>iH78$3-BlZIflxXd+wI0YM)RHnv~{&aQiYn~>Zw~bSIOcl#_91cgqa1VLLl@52@_~awLA;B204p~v~JofDwAqIKcl^gWAWs*P?;6T6k3P0duzau2nMJhm_6$#eMP4#UC-xix-SV+=UWRQ%fntO}htG-N;c(sk*YUPo+*yU33nUzg02c_4B_D`_FH297xt_zuK?!R7K;4vkW)$H|RF{p~889wwR*@bf0$@n4fTq4d~n5UzJ3EAF5fq4(^+{f8N%5%R=<G`DF%g1@^(c_4)VDhB4wNk(Gd}sD#>S?3=&UsFcU_{(Wn)XLXj);fhQ3=B<x$DxYkWjr!O687bTx+)DHr1!zE2egOpsEU})JY2w%Kot$Y7xFqmapH23;4GLOBcbks`8*E!}Wt&hHv0>ZdNDp?mwD9M&4pOiW$#jzPeTiJ|rpBAi?<US6jdpI3gj~_Acl1TC>ZC?m>zo<Uf+hIL;c*)O&#XF{rJ??GFY!LPwYOVNoifJ_Zf>43O-X5v$?73wbu=*kPq&>K<&Yx-E2!^5J`-cx0ryZyaOUl$2P(*sJ}voG@n@?@;SK1LAUsz;Ca=yKrn(_5#4(#bO7*)Kt6j)_KjtDCI0a6m3oo?N#JC;9T8WBL@8{rbhuGIT*@vY4+g9oSUOmH?%~??qLqw8fMQulF68-;a23NChIluHo{nTn@fKNl(hUEG_ZZ5&0`(4s$AH4DZcO(0zk`;Z5qXm5?rsAr^k(Skq6s*YZ+Dh3jTj>vV1rzDRcGvfu%kw5MoJCi`70rNlW%PXWuPT=dv?AEk$uU4AFbqX@O}+u{iuw`h=BZL!>5wQ`Nw5=Lf->IA@W~wD0zo@4AT(&GV4K&<KcMVufwX0J5&>(=B8^h1t^KV<Z)pdFi4~`U7}+&^qjp2SbSG*OBga`ws4-y``>)EzY2RC1zFqG1w`3)mBljmzSH7;%jn;I56ih_us@q4RpHkU;v|m7});Ta7L9Z>&xY0IWSl?wt?ZP;{1ta=!Zvxhw(Axpi*+#s=>-p<0DIzQJ3Cu0k&wf_&Xr{P3HWZKqy(DdZK8n&*StNL*)~wGlV|pE$^eyv(V^kQ0WXhJu&5d6IzW*Xd%{mw%ovC0EKGBtss42Ip|=;QAvOWyT}1(Kr^&+%wd}fz2}-HB4;VZso*(%(F$vq<_QLzo*$XeU|MUV?rv#MC&1IKXFTv_(8&Tf)r&U+1kVikNU#SR#E%R%7Z+fDCvZP+%;_UO`WH(N{CN(lrc-{$vH~W!NM`dQWl$2(O8~0z;BMr>YnHj1i!jJkXy_im8NmaVz^-4O5iuiV{$jJyVEkOXdPRBib1-Q`Y6GaZq%wyDf8e<<JYYbP8Nc>kxJ%`>*&alG%Tf`~UXdUG!OPx&3+o2kLewk^%~&oHd>*l!ZaD-JEscBAbHO7BTD!H7^8N;}yx(&Oq=cj*hL;5PK@ka-&QXJbngiNfp&?h0m|=;%SqxBNvS}b$e{I`CMr@-f9-N$rook@euj!QIknv5O@++HQ?`dbiI}tp#A;Ml?hOvl@-@H+@5ryd#)*$7U+csQOc>zIh-zWO5!cBzI7Aut8!Dy&g*!qLc=n1C9PsnjZ<AQpXQ0ccOA_lh!a}OhLc~<jT(6}44nGK3GE>cIdsHC4!pcJ>Ri=c756-_pccY}g{ljawQzELMFpknO~5FfHaH;i;X13BGt22L2m0eHWN!lJRZBfL=z@JmA*R{S#%4B#+R(O%s0E)?TUaR|aX-MGB?4*xBRAF;8Q+%P8`NdO-8v^=PYx0A>{bC`=}5|+dMY`cP<9~mTkxx|2&hsVzC=vij?YnF2!j|s4zK+`(+-tDQNHlpPL*z=16CUwTMEjV$t9@TsEoP82Y-0I`tc(l>`16s__86Av`1uqk<Fr0H8v4W#_x2)MDM~08a^f7J!{t<NGr5Bg8p?J2_N`0G->6FWTmWuqwR*PMA0fk|+&9)Sg$<<1~`;C+eN_weM5qTm(euMWHc;b^%;ZAKJHP{HF7EapC7`zo-$mQ$B+qZAM@4xq!7iVLO{J4vA6V36S_4th7o}0eEHJmS)ttnpQLZ)urjJ|B%_y6G*VaK?m)y6b1G{dBY4Oip+B8ARIk6i5b2CtbjW^5`}>(t<F3QjL@2nxarl}0BI!2!V2RlnP1EbIC-XhFAnAQ=q%Slk)YkV^Ph%It{t*iJXp07HaFmxA0Zwz)jFUVh{|erKVtaV_PM|E?Fazpz=ddANA<_$P99L;VTHWKgXVf-}<wZ}-pf;51V*;1suUSM(Z_!0kL5mLAgi$)jo*YsXkmjGHsYZ^m5ips7E=x+xt~3r0OHl_n_E9Steqk>cJ+$TND{hiZb(x;UAL#H03dh*JAA8l@OcO3Po+PQPU{d$0dLP)h>@6aWAK2mk;8Apm7;S04KZ005#E000vJ004Alb98erbY*jNUtx7*X>=}dd9_$ukK48qe)q3n*%vvG6??PYHo?(B({^{8q9+Z~^r6Qv1X`kPW+hQ2soiyh{P#1%i!Qb|Nf$T<_9ANd&2TQ?3{5G0GbgvK6a_O}6SL!l<)&pNk&O|R_?s7`VP;3zmWf&$qS)Q65%;_#Yo>WAYOWJX>147|vLb-O@{(x{SfXmAf(^VWS<WZ^@x7L{lSSB4tb^Pecyuq!zUjCB4u4DjnpIpk2orFcJ=kl)laVUlc^mkB!<DFbZFFLo&ThX=SX&4a5bZ<8>LP1IDNRgrSv5@YOxLW@J885x1NN%61~2d=m-R+$1Hv`0xnf4h`bWDSliy8S?A^ohX8$+r{<UPc?5$gyOeWS{<ejMZ@8k`y!Q<-1gurJlb!KoW69q}h!Y=ah;+b7+STS6jk0oJ+{_Cm!H^d0n3%(&)mb0?VvIru$iHYI&X6lsUM{(1t`q1-{sGHV+2AwW(TQW3dHSNLI%;dYQ;JGZgR{)!0^5n_Q9aCF<tZ7LJP2NjUM-^)#ZB$H@yi1HMg*MT2Iwc#aaEXW-u~y)=?_B~U8*vX9e1e*7JmF}XM-?b9;bc{lVV-%NArk2*2vKK`-!%$GG>yq?S@YOtXq06eQSvOCCW>oW-tuUgVD><^Ef*_`G#%<n8c2u6fXb3p>w=NLTCVo7RrP}Os@-z;XcIayyw*}h%SBASi^=S4K99+HOfFW_Ued)j9%$aW^aPis9_Ko&fUy;;p}aHAc@Ztnysh#M0@OgM!bvcNma7<k>^pS5s0)6N&R4A7^5|?3LdylZ(*y!V6g{(Mo`dH)Thi{oX&-)2o3N(gu;&96729&iCHlhYiwU((foJ*?*SQjF4g@8T8bC6kAA6)=dO+)5;OE2W$@<J$4ampu?EBx|zPg6DPAq;v=7(QqKX*dp37IDs4~YJHdHwRoKeH1%o;|SR>U#FaSn~O?<j3PBT5-dWh4F0E7YBN<5StChG(`q~nzQ<rX(HF}d2Yz9;CGrHpndF~oSQubrbDjl{S1_=p;!SwQ&O4~ia6?uuLs_C%$XjNXk=uQ-K6t@w5w+uZJnMSu3PTW>A-`Ic{Eh5PNTDUfW!=%2D-8}%V8UkSVTGCDmoP7_dIW{b+kdbVpLb2t$bm}-Xrlr^$fVFV(JFT0VvccuqvXOA*WI(u6L>hh$~iV4&?dp4sS1R2?i6Yi-u9yhNOr4vaVTY4UlHO&$^bL)!^e>Zu{MisY=#P4C10)_<Gjt`7$QX(wKPtlQDxnba+UQ9&VgAeh%nU#v^<(M*hF}bP63o_OMAW7L!6|?l@^UmEl!`uc0{7MKUL1;}1-r+&EO*ywCn|3fLB&9u5)FXZ+H8mOLfYZl(?qXC*f!0q~M)14e1OA}Mt3245C*C<kpj%6B}!SyCTZ*Mcideb|O<ZQonA_Ml->%j_GCnR7z5$5Y($M9#C`g><!~ofAO|a4_cYBTxp8cZXbfS0BK1mDnTUb|{OTEWCe~^m2~XaJj(WSLbv!L@Ie51vDEYIJQ;j($adurR7j$Iq$S|-s;<6feE@196KV34%H93qPM{zEc<-`zE#k9_aR_kvO~8G+UO&l@304Z7xZb`XD}q?(9*9`Yut-bk$g&<MLDt$>*6HFcmt`WLFXJJkD}~nBVXPLjKBt%OA8zEQf`IOU6M_Dn)AuE;WmM?Tq@Nzb`LZ((46f+VYcQw7$CCExv$MnupH3g4c|9t`lA_6DzOzcj8xt%n^pdRVaW=Xe9M5EyNPKjpc<B876I3f#spW(D1!bB>R0`o?N1M69<)TRRSxoSVA7G9K|0Ax7;mF~nCd3J*iBlfpVw`Lb<2-+`0Qi#K5{c4{|g38Cl}k>SPn+I89qEA<j5Tdc=-96T%D7e-$D+z+TKkeBiyE$tjj&I7m|d$f?SbzbvWw`*{(Q&))5K0lWkeRbH(sJQ5&wnF8~M;_d^W16F}8?8<rrsg>xrIh{2Cb>#J2a0t3^!3)9KTZnl`?6z_fqh(zyL!<VxKjNJuZf79-beLO;%9$ulkM4PyKOqbS)hJnz)G}h%c4YD*okwXr-5DMZX)gDiE)VZvP^^hN16cZ4k$ls?fm6C9UvxJb+*Rmh0h?`D`Z)`)t{HOi2v+$rX<M#_mDOsVTs6Rz8J;B!$?{0WyhHEd}F8^Qc6)35&GuX*4Gt}Bi9`t|3-`i@T4+tOhDccWEcbLG{(5oyvKZr4uMk*isunXkO1v0pk-$U7GppXGqml6l)FpmO=)sSK3W>8q+y|nxtoucQ*67m-$??x#7s4?y!?SRi<vo+#T^X<V%fBGuIS|jNLx12>x&PEfe=AgIu)X{>?pFNyU1BVEy$xEA!KBO*3ts_soJDkJ<$aoSPq)YhnEeq2Qa(>n|Enkh64I`fzA&AQl;l4ip)?~SWIX))OlV|hui|@YuX4SE)liL7In8n5P<7(`=&*Ggu{{pzL$qy30H&t+_xX;TLv!n&TD{}Sb*%|T2gbP+{h;3)sZM+T8=)xY4<#wWolC8}+=aJdfo5cmJ19R`NRe{$EvfleXNiG+Ozl9xq#Ti-W#a<yMfiyCgWlDd3`Sw4*y?*uI%b)xW<pj~cw1I`>98*0MK&!=Cl)~)80f!|253F#b5Loa(P)h>@6aWAK2mk;8Apov<n#C^!006`Z000*N004Alb98erbY*jNUt(`>baQlaVQ?;Rd3{w)Z{s!)z57?NJW;8l#L4a+0{9TL-AjuiD1yBh20=?Bi#0{6B;|Zz|9gj|q{xnUeMl&BW;pZSn_+EDOFH2jCAUPjow1%g;OAmkyBD_@*WHT;=NENw@xpQ2YcE?Uw#Inpt?0%8oi(*oz-0w!;a_rc5K30U0|$MO*60=w7K=p%H3>22+mjcsr1Mhmdg&T)yzF{j_Lbm5SKOK2mcWOLm~T5CipcI31b-G2BCA*j<2<Iqv%-Qi>HsV&I$`k&w_a|dIYvh7)1(>2eVWH$Ir%D-gDjfweqg?vnLpZouolh%LK5!yWfgo+cGnbPZH%THvRV;(TP&}O<rg})+660bY}pJ(H^rx7d1RtMvuI}Fzg}7hKsvg~g85A}e3HHy7%Z%i4jlU?R1bexYwUnq_<gL)e}rMioG5)SRAgszjVf?#h$`!#qT&G7Nu#;cwOQZYYyuqYHYdyL`Ryq|x0fBGj}4FtN`-7y<Ts-PDpY&0odx_$q$3Xzx`-J~a9fB9T%A=M3wgjZl)V>Q1v!ZnAMup@krYwx#SRcdgx(76YGYfhU|`@=ar?Qrq31jYq6*h3=^6c%X6x%C+GgNkB|(c7*Q(<ZI|_P-U#6~=)DV?kVXdVPM}y`oD{If<&7<sd(i**o7hIRkVuU^QqH*&t*YsYh`Hoh8AHP4ewXj~+NUs}Bg*UA%`M=KS93zRF&tGD5op(Po6=JcW*7F`+P>A!OOBj8yYQ!`7M5hG8*xUx`O!O8zK(PP*>+w(i`~BlR3YK1Jqv1IQ^fXH>&u}UnBC8L_PM9!DR}2m)UX8<K*d$j3`Q3w-ubAsq*XzA6US&tK0@;hB=tp9Hh}8>^EpTa`G%7mWOKSP-d#sWmGNjkyAfyV~98@_$5+ps+9kM_b93bhB&5Vxn!>rnUB`q7~-RgZRH6qMZjQRFGabYacqD9;_VKmkiAdCx+%|d5oSx{KUA&^kd(Rd31zDGDF+oZRmG~G)?XM%AIl9QwC$L(y2kTPA52iQq->9L=9TPQ(x0l2p*8g*p@t)O>^lW><PtINZZ=I`gB9q=^#D>xr#7N1#+jp)#9eGaP4%s8crmZ^+%k|3XT(+Pvq4MjK%vo}?e%2K8j>O>8P_6kd(L)lZy%;M(BFH2n%1`txdl@(5Tb|LJ}<>caG{}fhps>4G;?&E+%9O7HSrqfvAH^6x$QO;A=r0t!LT{ApJM1y}ac27i!$Br0{C)o7wG8xi3=}zIX8v_#vZ-qUhjHnQeDe>&5P6bYX5r{#I&}-T%!p8b;b2Rp6vxz03!o(hMPhFO9KQjOV>BA6svJzjMZRmTprDY@Z4yvJ*&pg2Yq`?<q#Tgz-SkbT6yn@E*@GwD2kKX6gOc(zHP)h>@6aWAK2mk;8AplLo8ew1v006fg000sI004Alb98erbY*jNUu0o)VJ>iay;<Fl+cpw^_g^9K;UE=?cGt}%xu8ao?6yUKLmL#mC!ayk5@j<hiz-QF;~M_oZ-&$tW!by8P3wnUiR282^YNP*O6)4BOz=s|a+WK(3o2%|g;)o|O?-scS@Uf8uF6Hh+xNN@#&B(Bj?yqqRb3jf;|Wtn<SaG1<@2r?_(3VD;(!^s6Dh@Qy_e#YRmKc!xrDO-e(PEk8BJv>>5i2m2N85^X9~$O%D)({N>(_AZlaPEr((x8@HcA~SBh6mamNUEal?uOj{)gtu0+1?c5+;RMKqhu(t>Fn+yU)xC`8z5B*E8Prg`+%^2m4|Xm0*fg_;+6Gl)NYO;E2=F|`jCosNM+Gk7NyPmNUjF!ITilm;l!LZQzmDcO#XR<p8{h8ZDC4Mc}yH=9cp=&BGV4nQ@5D1$I0Mln&&uWw$HI8dzI@bD%MZf>F|4np#o75>yUPR?O=a5DTkRl#;kZy0}vlM(z8k&(9`1_#sUf~$CX6)K408DViw4*mMK*EjR*n0%@><P7P9$jlbV-kh1=a<SPOSn%p}!SQ4Sb=U8Xx&sw@t5&n|`g>7kR?8%1m9+;bl890pR;FB&kq)i<U<iQGh*K!4$T?v^24;of<?2*Mk-IGyi=f}}?d0ikpQXogHEF!gh>g8a{wB7X*JdMlBkzKgnS>mjvZi!2tt<TTLi0<a0%yLFO7KR~Mj&uU(zpjMA;;C%Iqun+f$zqTEE?+R^3vvTd)l1o^6K?kenAhwP>h3b<6ypegaWXSO*^_Tu}+nIKoPFbuRNZed4|<71%V-Ykh=<qmbJp@fM<w!bm${JSlEJa2e2k5Tn1SsM2@kwcXYCDjE>7Of9n;<!3R_={;=3~An!mob8xx$@E;UUZsGq+_<x7*$61#F3}@<2K9rY}K#~@mm30Mh-%;9(XUXE=n{VzPnA+$81I10P%E`PK&MtefLwXSOL=7ES&^{SVxD;lC%j9jk7!e=+SKr#E&h{e+PvsU$0BY5~=q=zcVs$1Ar~qn*#oV82_12F@L(@QJFJyYpQGvLTrkT*6q$mx25N1mqXI-x$k5g7+nUNN?#uJY1skp8S(|WM8ohM39x>&m8c@5Fp57SD?&%DHZusAfXk66aPfJ3Ijyg+fuad40h>@(|<2!3c26N)mI%MNt_;UXK6xB&7!Ok!v{fx%?%<Ybiya7P_R$59QTHCN_G2B_>G2skgn{a3+1+~X}g+SJ%QH^X;>;rHI~#8C!2{-$ak+NIVCK>6LGBb}u9;rE&qVZlpBruYtASd<%B4%&Vbya1I?68<dBhtgFvpuPkOI@$05f#dE(akqW*G~A(Q48RHI=qt*w)pQr1WVHnRSVhb0)%kM(n<e?Z=4y}okvk5MEY>_QTcKSkkdX6=v($DK7#6AwurPXb*i;i&U3g!SXp##kgw*AI3DAW#U*{}kE;hBY@s7u^d_V&fd{-5$x@a2DeOi{TlrsA<RGf>W(^#5gEZzK>Sh7wyEy@@#GzYYB9H~@&hmb+T0DS_qF#Hpg?mT-6Kp?Zl>t17MqM_P;WkPcudVSQWR2*}A({r;;!ld(oTSv)*f`F$6I;W5vj{hymSC!-SSaCAX3sG|FIxy=%Xr`=&3{Ulr6$Krn5{tIj{;@Tb{fcjdHqZ^7=+up4t{yjb;rUF{tj)f1vgyUA*sL@>28Z&OXAoMpcXc7Tun_An0sU;ifO_$>EO|VQm)6;=TbZ!c#{ktW()h`sqqe~dqzF?9iZZGk8V725dPVcH?_P$rLX=qGvtAWl26%@F2P+xI9w5A}8}S_zl;yf?+n?1j?oNA&T@QS;V~L2|GxUxz$+h*n3#thx9qI}l4-E05jW$=N{vS%2$g`~6ht?8Ujz$SU3|nhi)P{#HuiMfY&ew`PI${E~SPjt|MQ1^+yFI~t1C15E=e(jj01i<>_`#@lQ3<ZCp^9)WJC4Sw5+3hhtP+a8SRY>f_(!!RyEpRWON3U&i;Yv05g`-Y{&M?EQ>4D6*X{3S{_UlE1T-KLf6$PHx#9uCHuw8;%L}&VSSFJ_`0vEcLj1h5&Ht{mWpeZQ*-H8N(N<H?)#by9`@2E!2GO*zUIbN$ZOs-0G6^JcR=iM<B;<}fQVg~k5KutdQgD8Oz`rvLmnaC*EJ1y%3lZCH0*H<2pKf`P$=VFj_D{Nff6LJsnmL2^#&hN3lw(a$ub)IcX}f3C2T!mv+pW1GV{FQm{DXt*ciCoGC8YL+rdzy|q-(x~9^N5T(}{KkTIku%9cmz=th!s$nSUWOC<@w^ccLsHgSy}}M#;D|m}s6y6r<_@2qx+S(iR9X?Ux9BZF^c6ovDSOS#Gp2@Yznj7TCRZf3gIajczapDYPdZX2fWA;c^aoq@{N&h8HvQsh}EgA7A@C!(X4JHs5m{jMz^JB5jP%L}9b&PH1i0CPksKV--GQIGo%oJVW&9j~C(c(8qqWy+wfID(-VOXpW~c<E|Qw{sJ*huxTh)l3W35YG}kB6V3149@l)j<>@_uM`{4(X(o2O)Lo;oWqQj`R1Zb3e<J&t>f?uhw_m0{;5M^CQ_3qG#5t>wi4IBMiweIJh2*0s3am?Y0A3r|NGb|&s9xumeNys=ko@}7d+@aIO&q*}WJJ*MqC~UL-1ka%Er6Pv$g@we$Iuo%vv~LkW%#+ov(Hni{r(;SaT2I+o|wEHYPOAxl?q<;md*e9$|m(!FaLIDom9}jo}i${WoIG}sQsixsr^zzDc&bh+I*?$J}lxWG!Fj*P)h>@6aWAK2mk;8Apo4PZvOTH0024%000>P004Alb98erbY*jNUu0=vXKrtEbZKLAE^v8`RZWlDFc7`xS6DeYQWTYCwNRuzRqCmSs#k<86AxK;6FapX78dc}JGK)r-48WFNj#Z(Z~SJ)$gbDM5$V7<ty`OAGFsHo^+&=jQGK>Ex+DFu*0l~!Wc2B{efBQ?1M`b4y27CcBRf#eR*qY?UsSN??ZBPXY6|}L#HING2U%<NVVHG;I5Hu5qqKFhzEB6c5;Ji169~qY(k{R=sjTCw1{*;=kQtOi+zR1q=+h*AUM#=B4z0_wOu&vfur9?6G*UH8TBa2+r^ACQ!QfE0g8<8FqhEv#wn05B{i0aQx6oQztTKXEZ$QX8us^;broqKus;^$bXiMp`B+H^CTa>&g-}yer?;p8vWB8G-*Rzt~?;d}flKiuJD9J;F*x(X_cCF@aes_z68?|(v*iLzGbgwPS9yge;$U3jN%S$pn^!7K@F5jGESP|}5K`kn<P5`AqjQbc4jW$OXEVBigwp)Gh3^8}y^L#>0Vjn(_z56V7@!z~?l{VDbJH18v`W9)KAVs|~m%Ay8O}NJ<_L_Xw&esmPHd2{tT9VUjv6@OSTjHzGO@1zKMvWF}MXbfSCu@rH->vg2FwzP*<hbMbB~3`=n}Y1L37?2mQ~n$Bm{yg3!8S|UhN6lOMY`{iy#+;2@6x=-axOfLj~NX`I<w&<>UDc;P!L>oxFu_YqJAtBgK2>^a*wn81kCad!ih4?sy~Wq2Lfwyowr3CHZ%-Uxn*(Vxt^`Cf$@eikXan~n1}E6>*S5|7hg6}h-+HtdpS%iz<dQzKg|}EaE7Zm))Qw@jNy2`-Hw#kr<K;x#^|9hV!c>N!qTgGIq<7s+Fn5`!}riqZ3LLd{Chr<#7t8Ok?@pT`3k<by*7S6VlHTjFXs^D$t^VgWFP+m6rrO!WwQJ727fGtzXW(CJY%d9@+NS|mM*d3L!~a!W1o#6m2k%%d=vRoGd<~i2jL-KMSm8wX7Fd=EgfUaOSy~eFHlPZ1QY-O00;m803iTyR?ATH2LJ$R82|tn0001VWpi|MFLY&dbYEq5VQh6_bZKvHE^v9RS#59QHWL2sU%~Re$T+QP=i1Hg0tT9<0eZM=QtWLD91KHXB-&;piz-Pa>s{o(-wa<Q%9itjUJy7IH8bSAJu|~lN-u9NZ^&AFHm%~Cu(~92y%vRF75Vk%*DLa8E7YA%iKt87aQLVVP>>C;xnf4hniR6+8KrbG5xYhzL;ldR?jD-G;o3|lYbAGte{Q%EJCLC>!*qUsp7DEDwN8-dNG1e+F1JjmfAi0xkag--WFhV9m1B|0t+UNW@eMOPFIa<u@*Ouy6xzS8#HP+0+3=jz`hhFI9@KQN8pTUdAh&zlz&^api>lRztNd0<qm5#Xd#eknxMfTy)5&D=gVQqec1+-X!xwZX%eLZlIw|>@R8E&<`oW4&94PC7n#qVL%gTeoiP<%I!^}1f;=D9bYs2aSJh&iV?F#--v11#q=?Aj>N<|5OzdL(OiT{2+dt9YGZ2=bCNF^M7w~jxaVV)9NvmJ-86#NKYrjO|;;k$^kl$#S|Z_gsK0u2!p#%E)7EsYJ5(ZivZxz{i4G}*hQ`SIpY)4zL);e2+)@b>KZ)Q|S_Z{z~}yyl97Yu;ekg9f}Lx1!h=6(>T28#ls$e8b6FDsuVr#Xmoi+t!d@e*4cSa((m33ljCZLL%8JqIts<J_N77>Fq<p*&r?NAwxHuK+4LUs4;#rE6u0LUx=00`or#rUcMkMa4gV3WRP2~Mac;ZHp%D<tRIQ9VMEXir3QNtv*kodxy$G&r*t_)z6GU2-sF&CXS4ZvhTuh~*@F@QD29JF39e*iyK8jPkx!?`vVvJ(CMr;SNdXRyFfK#j%zK09rBF$yS&s+7N_w$Gqo2=K9Y!qtU(1CUgP4LKlUwWMkn|gf4?x1LVj5wx%(bXDZN-%M;&u%cA{pY2UxrBX>jAewG%xNFrySFo1F-v7OWNQ<NmpdCAn!&g3RcTn6mX^pmK^;oW(d0m<To;ZkgC)Pz1)hbgv28c%oJNp3mt+^XLP!pTWP%|1ELKPF@8iKOHbLb)eu8nuIKXh4_+8;O$nt1&Uz2;Jp$yrwlborp!}G9(t*1-sjV$w&~81mD~U~|3p{@cibx1_<sUjcLH!u2n$Z+cf|rZgzyVg!lDfl%(|EgnH$rGs%MZx@VK?#b*R0Zq1oII&7S+Q0Sb%*Y+>2Ru_WcNNhqVFE0`({{(qP#>2%5pBK9@$w&SH_&Ta_ZL1o_t28#CY>XM#N<#sC~gyip+SM4_Y)Cy@)CgNU3trE`eRuDwC*KMI>l)|=;|CI;UKafh*@x7dUoOS0H|F&bT=0Ryg+%~-O;6Q)l5Y7mwk&(i^MNTx?4VNBRJkm$I3wDfm(kbY$f5PlZBgYO!%?}9sLfNAIuFyR`rr&n~#s;Vupz@obR9_kr*`gM8zX$!po{DN0FYmHd96)-@L4N&zO8(Wd9>8TtHmVM4(aFT22R8Ek6;Si@DMB<nn0f30>R3cmQ2LUzIcfN`V0Mq1mFcZI03fg?KLkwR~>k{hi`DE*#ELQ`NZtJ^RKGeCZl8DMXp|QqAm>t#FH4|0uz_cr~e(RCb{)_MJS);=pI{N+l_aPU9S6`;~Er5O3>yeA=pJEdncMQog?n>GkN>k~25oigVUBZyDW9lx$X40DN3wPy^x;ZJuX{`X9^i{8K^AdV3Q*5;qcR_1a!e(C5X&RAz3M7bzr^)`ybp`=v8$#AJoGEST_5hIs!t0yS<+cVKV`}u>=wdCuIapueAlHS<!E6ZU0r<V+ywQELRD@Qkf$fbOVRJVvJ2QGKl3>P(_nJ?QQ}buy_@<Z~9cvWImR!`}ZBat>>ITpWdW%w8cq)Z<7R+tm*ll7%Q1`=Ufn5?{N$=FJv|_is!nPXil7=wA1#nR6{D;oG7h2pNwCWKzcz3{zu~JLt{Fib3=KOhz{OEWXUH(vkWi`OueqK+!>M6O0=9blFL0JiSr17bu1LRCX8`NOtgoiqMSjTVHHC7R9Dq&Feze9PQGNBVGidL~=?~u?hLL?NR3#Mzd3X4!?221PtumFys{vNy14w}1W#0O9JB8Yey9#y>FnC$?EkM39ifK!|#2Tprjy$1FGq3yS4HX2Neu;ZUCh{as&?kxI1afe8c$*=|=9JP}>C?x(~sJjn>3^WX|Imd=5To<{%-(HXTf^^gk+yZ)3qUI?yRCOszSJ|!6+|GC#fHJX(T+VDXovnI91pW-hN?w--Rv0{u)gs>(#?2glOw(sMeH-YZ{1j(wz3{D5^sk-ynVp|Fvu&+kb80|LnAE^8r)~q25jUp51^{gIV*dTi?=?RkB&U<Rq$RHm%R5f!NuXHgL8eR1H}f&hLgbZva1_znR&2IUv4sXG#Hk;9>_E`dcx^pWI-0k~;&A!@Wtf*CLMhg^Su-&tyyghY4{zqH!8u;D{ZB2<><F5#*&arXW%$0FtyZTjGJqLahv_8mFUO->1e`C7fZ4=?jB`@lylUC10yH>;oGzOC^zh$!pNh+8uJ1w%2e7u4NJ6h>_>A7Pm#wSgS`;Fy3n(zdGDxt|g+iX^uzMWO7=aiBGO#KGY8v<QqQXhLpTT`ZFQco^@ms*CkqF~Gb{ZGI{CM?|T;BYC_0ir9g#GPUgEiXUL!cGyKLTLP{~u6G0|XQR000O8001EXH}4#VuLb}BLK6T0761SMbY*jNb1!sdb97&BZ**m1Z*OZZaCya8ZExE)5dQ98L2w@`53<%2>o5cmFgJ0RpotUMPKUuW2wI|TH4-V1l#?vTf8X&%vYbR|+J0C*xDxM$$Ghhqo|+Y<kQz~?dzpLW%axD?ZQjc}%|N=6qCi@RTt&+&%TtnwoGu9$nk|L6Mqb;>+RC*8Nm-t4nY^ntKMbcMa&&Qiemd**dMR5Wvk$Qjm?V-Zk#CtI>rBJTifBPr*)2007e%H6U6iDxdK02T@N0Gl&&j5L*hBE=xA>w2x9>I{qnu~UT6hVsZDmz0ODPhj)XwDV=PhW)d)7b-p;2tz1_VOQ86o}1A0y~TC6$b8H9uHb^`FpCCL72I(voszon^}AfTc(=P^X&ZcM<x;Sc&uogvo{_*Jve@N`a2xNHUsxZ+~Wb0BJ&Y!%Am@hbZGoUZojd!xMO5MyQ1Z+a*;{4$hoQQR&FKX=O4}OjE68U<Kw6cV`0*mt4~|BPMIUKsanHb6PA@YBLlfCp|<^OkGJvylLRp`_8Iwe^UrI<J24mGFulM{ep0_aaP{(C4NQl>P|&euD3WubL>F|&v-DX!$tUHG(2`|kdd-7YYLpS!sstr;hEY8?r}@=ib0G9aZxU4s@hh70-6%dZgtRi`Vb!o*9uIyS#IJ?Sf0<Z5|vCCUO=s-T`~o<d(5$URUpb%UfKK-eODU&I7T=TDZ{NWIW3riESvc5AM6slWl2!S5biYkURtumpfGMVMP!jeTw-?d&hgsiPO|L)FsoRT$OY8;CXHRy&Eeyjte6Wdg1ly>Paai@8RDs<nB&Q4IvP%nPCP%jgVkYqw&=i*BLK-j#?gJN{t$mb@gcyNszE8UEwtCbC3>(uXrUFG=BPn@*ygF0jx*e8xrdg5Apiw}`-isGhQ53Mn+c-5A9X}PmcPGmi5}Xl*s!QO2BA=L3j`BEX}E^+6tqYI#Rn1mlC&!GEU^F#d|GOlTKor?V)Qfm^(~;>oyMm!ew^A`(Sx_)z___ABHQ?BmgvCl;T3BstYs<WJE%>47uao06U<#-8MtRWWw&92dd91Q$+m4|5AIK73EXu;)HK4b4R%TnXgabO4XnP~K18Ne27K`fTUr|!>EPZ5@__q$E_}!lElbAJwjZbWMF6Xs&cXk19GSPko9tm7CpkQ!vz%@rw{S9ZRA;-VW`PcDPjUf$re08e<NL5CbjM>U@L;mRZ#rx)|DK&(j8BHslj&%59OJiLnwW8A5W)C1lY$p;li?0cf5(u0Oa;@6tI5%bTn>*u4gVZ<8DCCLKM!Xk^5JTHd}g#~Coi2@R-BX@XxG4<8~5LQT@xPn^;&(lX54};B{e$|Y8oL2swXfBZ3HJb8T?aee+9ZZprgT-uWhj9UW1vtzhIil|Mh<G%o6OP0Z8p$48V*Y?mxLeUP<WnsXJ^8@t&QIM|(EysquNUOMvspZ$K+8aEIP>W^m~AR=K$8)Pv5LI=wkm@*qU-7G0;j8k3Kw<KY>ZjLwIr<Kxl96hfhoG*_&{Xm~ZdAk&kp+403+W2fHSlTWpeE<TSY5IC7#oWaA`wA7`G?p^d2p!|oM*CP)x3vIQjbSiFmbBzR@BQuOit??H?qiV^yNUNO9U@1gXSj*YC4y*V0TJZ2sUJL(Z5B>?<Zmmnf_8GkH0toP0h;_~)EAk9&pQBWokve`HVX(?8wXv6E#S2k!9lN3Q(Ct?9IktMZ-*yRnp|j#Srxk|+WozEyOR}5+<J8_B&8L}WMPR1V!k)`C*WRf@w|!@iejiEHMfwh&_(4+eBUv3lOCGwswRyNftgQ!p1=_ljC0m)R#U&QlnfTfSPY~`=-MyO`ZXX-<>g1V>K2FqpaIn}V^=U?oT4+W*=hO}nU*||43xM52&yLa#H;Y|2L<ua(;1L*}l1JR4nir!>Gews<OTQ(;MT=evD$Q4)t-H+rg_eX7$~&AJ{t8_ri~g>tD+-aMq=e7ZJX>$HuOn2~m(lhoh+$vz<!|(XCLFGx6e~o2$KF1J8ArVzE&dDb)GWW)R_);w+rfh<O{M*+vcu0aPWwIo)eAEp96hjRtvPp(U7WCyNt!8!CN7U7DTNGhlj!<e3AqYjzfqHws4DGr-gw)CRqXQ&316+?3(Pd4-95IWREV0{DmMN@TY=b?VMiWjX!8G+6xmo+&<-|)k7?k=_=T8vd3X-kPHWJZe{1^%b3Z{EUr!p*RMBSvF+eq{wE^D$0Z>Z=1QY-O00;m803iTuMdViU1ONc%4gdfj0001VWpi|MFLY&dbYE_7bY)_1Z);y<Wo&77Wpa5gaCxOy-*4MC5PsKR!SPAT6nQDyZU|st?UrGE2s&)t9%dK>Ezvd`Nfb!RNgCvT-<?RxqT?jZ>KB{j-FH9u-5o8PPH9V4+=#cY6X{l4Znm{tg{4&C*Jq>JWUW<0I&M*f_=7)S8H7x~>a;3_F=6M6?AEdtNmzHcRv4Q&B&$?yidI>%QtFCTqLw?MZv%3g5cs@*_typc<>Te$hfn#i!8%@E@r_^zY~0?s1+c7CO(U&4&|pnz!Hnv4DS~RAWJ!|z?3^jq^2YXBB$Zf`ST4=Z9g0$x%!g1@HrQn>D*#*7I}j>5u0?Cj;%w=SmO^%@F(}RmEi5;u4!gii(m_tHgJ>acq_HO0!;g~&43sZw#VeBrX)Sogthljh27v?(^uOU(Vv2&FQneP{1FXB2H{zV6-Y;Bb4sC_GNHellde|ni^)4*vkZ8KRk5ILMq_qdPkp^4&sO&T9E`=^9{(1!Gaz5lLA6AL;8Wq6jJ*kRvE6OXTdfW9j9dQ<P1IRg{TI^2YDvZFz?-%cWrOTX@*VWuXW5VX2bno!9!dNZ4)H`%6dEUVz<Vqc)TX=-W!$}5eA=kXR71aSFnKZzED|91UV1g_eZ!6~6YuL54TL_!=og+fBXUNEDVw;X(aV9>5@nG)?rZdtS1i3K+YtwC!`P>KL?g4WSwsX=6ZNN9UUVsV4kljIq=FxVgp`7VqMI0^2boBL@IUtkOL&~#k>aa(Bz%h)#13oyL>ZidNdMj?gplp!5GpJU!`*CJ$U)>hnEzJtJ9wc}g*@w;*W1ya_1TUHjbV5x>CinII9Id~O+WUOP^o}VWb|5j6e(LJrW$uk0RdT(?@6%Ik7pJPNZ=difa)Hu^mTS?bHRLNzCp1HDsY*aDH?>-&^rV1!LbEJGl(RwIa=j5qtiyZIeQV)5XV8(>a)Y-xkuR>bw8FJeDxqa*-FHxx1iD|j<?RN3)~e*SFBj^3RW;URpz@?KM=W^I?6bIGA0UnWbrEq;tzf!9M?lYk(S&wdf_F$S7s;dSooZ%Lq~U}B){et>6y0+{Opu1qtG?+>I*H`4I=`*b@s2(4lSTJ<=ua1*f9e!w%U`{Dn|fkQhhbK1#Z4tQ5J+4@V{;jA?l|L?p6Ag37^rsG{?}aAhONXJGgj8B6-;T?0QqZXV6b%mtL3vt`n@i<(3;645WmG%KfUo@hccLxl%7$1t9Y5k4@>eL@DVmeZ3LmOXhyytPtVSsK8`H^+$%Jf1`_%YuX}N!wbD3^Y`LAg(P8?AMq~FVaiYd}cXBeu8aaO+9KX*ah2!(;Q4u%0;`5Q3`|XG0;HaGm75C51hU+VQIYRe@F8##^I8X{ZsyDu%MKKem<lR7Oh;{~u2HJpM{t3h2@&YGL9ydPFi&U5R>K%D_z&|ZMi_#iqHHRm-;q~zT!2brdY{d}KGp~-E1kYbaMmJaBOse8NX6D03Uu+e(9QId!gcJWxr+a$&e;38Svil9Ev%`WsTmvP!37p(h%Wt?=hW8nK&IfDGu4r-SAoCqw0exYI)E!?$p)&49;5{A3E#Dl@$z3pVe%Nyco6{yUAuV+>^nsFhFVdT+#l_(M-4(EKYJ2TU3~R4?|ALzRyES|*URtp}Yx&o|94Euz8w&0}P)h>@6aWAK2mk;8ApqG=_yJo6003nc000>P004Alb98erbY*jNUvOn|b7^yQWo~0-E^v9RSW9o?HW0r1R}k!B<%J`6z1toF_z?5~6fG7x<kTP_Xo<ENOQcFtPP~EtdxtMsl4ZwkHy1}D=fQ90n;B}UWQi(j)&*Z6URF{WbPtcC@PD;4Oq=*5>ayA)s*$KhODk!pCaM;Omn@~q@FmTRj(YCGi{F%z>Ue*}l;+wnk+CR|j1c_Y#|87X`t}RW)@<V5EAT`rMhcd%nDXzcdQot_?z~fs=7ft(u@ZF9f@nc2y_RM&8jnV!oGp<(6wzy{a+Z_KnjpGXTfSwQEWys2Y)e8#P854)a3qpRrRvIXDH2mw#3DAn^&T|dqB)9dXo_&IKpl2_mRxDOe0;K`EwL>>;Ymy31TBjid|_LrcJtZzpewCvle}SG(iOwE=+l0J-f&6v1|CF}vK>plOi*ri6`L=lEXHobk`~2+W_Hu>w9ssSTH~yzQ3|FLtMkivg1kmAOtoeZxq^uV0%4Ma_vxyTiv)j64OMtN9;5jjeR49IWm4pq;4$mq5iBZ2cY1<KW@t_g1wCG2x?){WIv&RDj|Ui@5Z4|Fw-JtM^MZ?2>Sc&CMc6+zEfR0`(PQ#78HU+gbcqFS3HEvhki*vWMRCK(!g4KH;DwSKCa~XN)?8f#)#m}IUO23J3a5&44eC3j3@O~$=^;4VclZ+d7db&m&X-H3z?>yDkt))JWb0aN`b^<D?Br$#r+LFV<Cgg@PHxpR7}mNpA0Wsv$tI2NaZ6XKAT3mW#FLCVBEvJjLm4$G-k%rqWMCFCE9BQ%j^VX7)YN(&7RLiMDG8-%!0mCntmW?=eRk2&L$*hyX(r36V20(VnbGX^2h)k8Y6lE>*On0S9SgJl)d_g!>@P}+)rr%$Gbvhi*%$ej4B#G^7|wR{F@4sM595|}kCGv;@%KDc)cIaiTMRt#&AA7BqXJ?<Bs2)`aL7IAYHMyS@2(?ZY*c65c3Un=#iq=k(dPGDq|UdgKu!z|37;i%kA+V9GzmioIjcRFt48}3DbRDoTN?JH73%i<;xxe7z!Q>77*TA@74s37g`HA#T^QYu_5UlWz~+E|t7QDg${O?P%F^hls6oz+7FRR5HJf-%tr<irzTyJFbrhc*o=ZENg2|)_pqG98@hNqarLPI&$-uhp2FVzHycQy~-*r`c;R2X#E5qpyb6(TQq`eaT0N;B+SWLTUHUcm~pGBu5>Rz_sic`DG-xCwPr}a^^<MLxn4E*HW{QO-S?jSKP?7I$<i&|&vIL8gTb(&sK<KB0}+Dz=%(8}4Jby^aP)SB5*Ia`Bh91=CL6;jrQz)VV(ys#^!H1>z)N0sW5%i*FHIj9K+Cr#|42Y!9zb4S)HW4#NRe~DWg(sGk?6=s_x2w4=o%A1l3zO*IYA3qEQ#*P&q-r_EN3qA4G)QaK#ha*)ktojBTG+%}qKCI1Rr19<B{-hERIpMEgpVIWXS~ZH}?BW)0qKju`Q|~5Z+MtVkC2VTpTcc_g-HaFk*xd*P8y5|}UE)ox2Hd&xcY~Y^(Xy)pHwgk%h(31fT4Vnh!G3+~qu3vYPwF~Cl)XY)_r~eVakIX}!?al_imJ~*vSF;!mTm=MV&!Utu@_7JjShXs2rQGj{Ui#rxtO5QFgP<TbM%FX8d>s;SZF5pDm?q?XICqFB(@)<5~))rwj8bk?v{$b{`K%D`StGM4kC@G0w%UsfAx*spD%rA4yhiF51vmeXiA0{x=6qAsuh;j*ye{jz2Q}Y?-i@4a=6{d7Xfj$=F#7O+@p{Z_ESPF=$7(=E{d4qKcELo4Z5OSz^_>|1m~}5f!s%@95pT>(7uKrv!()3q;~=y4bM;^<)*GuKM%=@AMxdI>BsHR>2+;*;gL*&U<ZSyoWRLISkE7T)nQPus^@4(s)t<oh+$1ZH>-9wdY2$RaW(=A4q4EeH5%KHR!*Px6gtdZaDQ<r15NYow58j2VkOS%e%~I;T4@__>erGfEcIrh;|<dx52k^6(@YkH^yOctu+Xmrm10{iYdh;&&ZV{%q}IUW?UVlwq;`kgJlX<ckJ_kY$nU4aI2H<)m41Q79_s;1{J4%ZH#+rI<`Fw$!CU)d7wm6}pTay36I=0)NAXOfe*sWS0|XQR000O8001EX*slW(oCW{@x)lHb761SMbY*jNb1!sdb97&Da%FH~a%E&LaCxm*>u=jO5dZGKg3vHP>S!&`bxmPl?Vj1PHpm`djDVme%I2ciMB1@i^nc&+B~iBJxNC#N3CX+T-S3`E5Cng7_J{<_St5B=qPk&%H7h2NTrf1FaKcN5=Bz51Y&M7{O(g_^#iB}4QkAl)auEkXFdAh|RiKngn&ean5W$POY9wk{ozsMk>~cZHBIh$hE;n^Y_y_#X*&A9gQ7Z<=Q7(%bt`?}Q4cG1UjpMm4=)4tOZzL1atz^|C3wupNOOaJg!J78A?{BVeHTP{(B}|B_u_mx}%^F^?Qi@noK~~RWV{gkEju3p!mOM|1!vO2O1#OC|<nLHY9C-G`h#<2K;VB+PgI9qEYTB^0f0yOX7?sP$iL|6{7jBfvqJPAdfrqYJf&W{(fP<1kNdf%<va~8-DIPG&*_<XDQqBH>Kp~3B0t-@_Whp^uoU)`!c{%U=o>War1Y_wgF({I-TF0CvzioH!m{{g=m&g8g3^zGl(L1v@8jW<+p<BLYx!t_dP(W<{C76z9n#d*1wdXZL#_^!?OHfu)4)L$pg^D#5Nyy=C0$c-Y2qwMTW3y{0QkEe?c*!LpA%s8{89mgOTE^NZq;^pUvUWo;Y*{oL1k~hxAQoUYs6|*hXTcRZKl^|YBxFr2!GiJmLh2KYf=8apg&}LDMV*e$HmxH@S}Wn{3WzM5((cDuPNigJcoxMvnd(Zg@D!snkU09zBwn2HwGuf};*FIFBta;Pn$%P-tndv+YcRU3V_v2l><z<<7Z^Q1jUs)nIe|bU600bnO+z;wB)}=4$~=VO8a+ej=u32V`3*)JSX{aV9DRj0ky^s`+2v(~(fBlqsZb>+1QAPK%5Tk$=2NK%*E(&{*C^1c#C16jI?ZB1S1i=_nAC&>G3LBuP#jxhFq&0W-qB27T%BJ6V3$`HFYbZi&t}O+9?(jnbpzH60IUMirLu;yLlvWE7_p+3o5@c!SL=-C`3zujYn6DD6ASu|bkLv^W~|GMye<T`Od~meV4>QyfOR=M*-789Ck6Lb(>GPg?y>JseRCL|U~~b<!A6!>RGyzjd#FwZ2;K*_OgSVV_}!l$u;*!4NG82<5k`LLOyRUwMLUBmdv~!0w_CMHbbDK^EfjoSP$;oLH93h-E<7u2-PPVxA3eoQs5k{$UAN=LB_w3dLktvDb#JH<VbDT@(bUF?3dH+J$C|D#4J&~@C7=h%crIB(sx0gJ8F51babrO*EZq$@@b>6R;M+r22X<?|FUo|s6X->_`@@6AxE{p%jW!H^JPjOl+A1Kh^GIko0J?M%*we8y(lV8m0RXuA-QF@s)06wHz7M4PhJY}do^F$pVeuVNdxfjl9&)|Le;_`m4tp=>rp|n8I&O`{o{D&%y7zuamJq`b3T86s;BUK2JF*}LS@8Dtuj5|=W0jf|-K-X~f1y2xJ&E{$K&yEtG5TbY&ctn*2`yJJi5y|2@42TeZJp5*kB^d0c2)Jf@&&r1kGdMDfo#TL_JwhtCrkJN+|;~)Ql4dM9@cXfRiaZ<@u8lpX~`({(#oSQ!*o)PdAWiyvVtdp%S8m$qypIm%u&J$2NEM*3!g#SJ;Xj3Wp&U`Jy;^!56uL$u5+Hih^1xfQf2F=OQEhxfi1k*SF3D=QbiVAdis2*UiG`?U_Wjv_HhMCYd%np^x4?h4V7Bi#MW|R9gW??%MucM9{3kqJcjC>J{+4ZU!X;?VRPtTR-hMjJ3`wK9xQsj3ynvIFZ4qA<LjMY=eqi)A;NuB@UBgM_UuX3;gr<!yClS|;?v;u&8zQ!b0x%tO3LTzVKH&f;8-ju46JG(rNYd)nh?h9YNnca7)9}dtqq?YQ5yQ_KhHqg(93OT3WMtm6};Lq3ga4S%6b3M;XU6%J%{uxfcJ$#bZk9q95`ON@YZPJT~N1lkHxYGBQ%+yZl%f9TQ!lUVcS<$3;wp}?fWa-eWivrceiehA~S;iU!V7a$@jg}I(o9?A64UbE@;`iAiI0@=7+bxlO9EMsK_3n!jOHRLwlRu@bI<&MS^E{cw~}#&(lNL9@fyDD7EgcfA)<!29rH(y~bpN%o?J5rH(klh3!?39Bx=jVtbuypSykO$jNzkU{(XYH!Q2=ku&lUEkj@NRL6L-b{{U;wjzH@5)Fs$PBn2FOaoI!g8Q~|n<^1~&ZPRdY#K_T{{T=+0|XQR000O8001EX5csr-@dW??j1vF=6aWAKbY*jNb1!sdb97&FWpHnDbS`jtrC4om+cpsX?q5Ohlbxxa?9Odcz{83zTMKL*&;kQC!w_hRwz*2AMpALJBL98IH;J-kw`ubsk;FUR@t)_7cflCH&l%;KZCQ>KEorWq;E3=7X;v}5Lz3<!RSLeU1TkLFn!+!xk=UY26tqO5(RHKM6k|LZv8onQqxxP`rAMQ!6cwT$Ybsd<N@}W!%I=;|CADH++8NHajFXZn4Te-xRYbD?Xo3!sX_`3&x1ZTVHsEwcr@7!;whJ4**G+N%%6=M+Mg`rXoYdw-<BICNC=zs)pj_NhNp^IZPHz&lC8eUvS(+f@pgAVIz#FSTBxA*LmY|aIo!*=8lNTcdKa%Q3a`ef*m>(R(@rzb7=~n-3sJuVBswg!pY4pjt3m?7T-AXNm7P%;44ZnW*?)N|5zWMw1XB<iFD2MZzqhtmkBdADKmoyd<n}1Bm!<nrm_ZSJ;*u*mSLnNSOLx=W2PfkQ%y?*)TosnT&kL@D*5v7x32^OrN?}t3fiKaUNu?!%BczBG(%qLo8UYt>5Sc--h;~<T$(4uFaa4s~+A89ROSR4_ugOG(;bFXgw!4uB#K2v*OqoFeJ*2G!0{=!39k{ZGaIhGXya?IQ01uNij+)^wzR?rfyTKj>__;&W>HIAI`7UsA|@N^y+o(_sXJtsaJ6944~9M<Q4eeIo9O{u}x<4hj=V{}$BElPOcxwSX3>O?6D+3R*$VYdn7#2PK(_{QvdNbYn?rvdv~@y&ha(gz&-B*kGB(rRtUa6nrj?O(>bEdvY+uODKYz>RLdm5K8@a-asOzK^l&QzKgwC>o+e9K;6+(<rJkvQeUJpzJ~6%pnV`4FQ>>ENv>H^F5Fz)Pe!d98={8#0c~2n-0_75gv`@33A`G-E4A!2_@Z;{NCVtbrl`e9JE&<f4lwl<*UE2gYv-i-rr19Nf-or{3j_JdMl-nQ2$L-mp))GTK1p#y>iBr5bo7%y*_`47-dp}Z+8HkJ_v^=++!u&20SmD0&w4$5=EfA8I<Wq|FtA?M`czDr2^6$0&X!*Ap(7*d(b}XrB`%0j&Z|*Xa$~3PyO~OG`|SmDtK^#S&Qq4S(8<K;dxkH2^r4v@dX6F#Q5`@p7n@J?ZzEPdoXd4K|f)TGB;2rWOkgq0A9)Rd-wqGKxxRJl;lv-TOR{Xn$hC3>G+wnGkcuNt=6MQ-V?v1`0FbDIVW33hCCgOe)4GtP2GnnM3X>qH0|9O-<b>qW2+}7T@p02zqh3zdW<>YCRqNB@h!w^(gDRdXtScbTx~YFD;Gf8P=JL(9?`t9$FjCOGGA5<+BPF92t#=^nE%e)H(EO-|N4@#h{hX#OOyZ$2eLAiDQaR5_loJGGg85&10X;>+Cy`++d~_Fsr~5MU7N*r|8{o;3fy)@*8pVF7ws%X&1t7GJJs-V#y9bSawioDJ+x2e7SHS*PWX<FuO|^(-56o+iFTj}*G{iAdO5hI7il(pF3aWX&{~}6-7LON5A1cp@jB+#&+MGwv}fnEeY!e@y=Y{1AhL2wPoJIB8i1g4GUAUYi^w<EAt@+%uHcc4WTsJu1(<F?+#fs;XejinPxw6k=P-X*KOQQk&|AUjqoKN}_r*{}F?3^V1|cG4=BI~TRFH*CK?rKXq;mSMvooV0RQ#%jCA~kr>{rLPeLJSIc1)e86PpywB*hqtY3S?P=63k9ur+^R?473<XNbPqyj5EqYlTt4PU_3vT;HBJ=??o{=P!H}LMtsvZ4@ZcYPs;ULpyr7omTivtO4FtIygoc5G=uMU2r(g?Y}^RMCiABBD0~rg7>E5_w%eDNe0E+6Q0VaL-N6q5N;J3<5|CV!q7J6k^w?@6vy)ow47Ad(9GBr?9U**z<J{C!NJn@IBw`*e3yl)b4=4=WLv_t5Vv1W$lMY5xUmbInyfqZAx<Q|o3jjUvHu!&bZckuPtQqr90>Kqk|`F6aBpkQ|4>T<1QY-O00;m803iU-aub1+2LJ#)7XSbi0001VWpi|MFLY&dbYF6HZggpFWiD`ewOH$K+cpsY?!SW2zDS*-*>2LV9t7;wbQ_9w*b5j8fuJSYCM-*uNaZ#3f8QNRNu=#K*{}sgQjvGZd;adoAP9cs>b7oo7weXDA}Yl<EmNYd$gNOYqPCp0oRy>zRmBVP{yqEstIyt-Y|n_;HznWmN@@Be2!hFE-PU{3Ftshk3JK~~k)PmmGO-`cz2Z{!U#f2NtzlA3(zdIV*z?5x?FGPbH|%Sc?|4<jq~sfx-&4u(`1Y8YDFr&A1(scMMTbSYGIEaj3ZB=6s5U)YI3e)!Ew1tp4f4uZFKI$fS1qgZEw_tnu!I-1Y57{*(X!qMWp`R2Zwo3pFXBmL&gFLvZw2~JCaA;B>txT>wk~9U`nr?67emY*UENaw4rof&NwE^8Q1^7t<Zd#V6nqWAYdR&v6;t^(oka_My7q9gXH~~atwZ>EWYtn#TiJbZj;z~oxtzu1^O#)Br_-2R#^hC&MM=vW&XkcGlUWp7Ca}LSPQo9%{ZjK?>gC8Ph7mWODNDKrSzXH|1#Vh0+^MC|q1mGdTDM{&D$o{Oddu1(+=I7EYmO+u9gVo-r6g%@@!VL{Gs1>u8FkZKV$U{QriM8tV@9A0Q)5zHfnufD@eo+RPz*xOAj;{lZO4sKiHXW`0e+fg6Jxv1DLB|oRjatBFsD)pwZehSG9=b1p<|YzAG~yyLm!avO6Q}oFSJr3C;I4rNbH13$y-J9vX(q_HA?zK>s|OHx3AwL{Dy=0C-G_nxRRzAltgY>!<Vy+d`f17d_utga4HxI=?Q5L6-8F}1ON&#1&htdEpDLnc;UFkE`<l_e3ivy`5KZq_O$TS>!mrWozfUIeAK80(s((|;GebVVxB!B2TI~ggMm<#?G9L52Ibv~2Ob8Zz?57I-Ugs|@OBD+g156Ei^e+I@QNFRrr@KtF1j3xDOXtFJ~o;Jgv=kro~t6`6azI_LJsDHjs#70Bls<{{uO-q`TKXl!V3GSH+N9A8@+43#oEyT-GU}9hT_z+mcfUwf4KNA0Jrsuy&%)%a-cFolddVim!ZjZnir|h9h1DEtZcR{ohMWMwc?7U)8xu1jtB@xIG7v)*=+vO^O7dj{cpsO5%cPO0?@RhZM6xbcpy0N{jjdRY65|sFr*X;m{W?tD(muwhY@B0_5!GcnDC0NN?xSLJ{v^ZQ=`R!b}r20uV~l@)ZlPL&U<i7Y#wo6QOpc~!9z`NtEjE98XWySRX22(PLCIxbe0@0^L>M=0G=Fg?%5r!_$}3_Ak*30-}V98-+S@GB+a-QEvK=MJBut)bt$9@`>rUvQh}2Wn*8XcV}ys1#A{;)4AW~?O77%`=USb%PVTV%s$k?{;r9LTq3`^TlyD?HngfslGd;qRGn|R+8a+=y^f>tu@O2dBjW*%2qY0pks`I)`gZFRWef7;BfuqL~^D7&_`5n(Yl#~X5kAjtf%~2OLjdv4$1i*^K#9j`mnYrVF5kebubU{T`zz|YlMbIro3;K+mAS6Vm=%Ajk8lO8UJOW<;wgyKQE(Tq}^7Xrm-~81#AJ4#LaD5p>&zuYx?424O!&N`<YD-MQKr_<=+8e})Baho7{{w->?N%TNBWux06$5<cHCQ7m@*tX!#bo7R5Ykl)$M(udUmR5b<I@C6Um|bTpV5CYD9?1xFF?)K{?~zW4C2xS^EY<R!<az~v`%^fl?pY^UtYb6%-B|&)CU>oOJQpL0M!P!iuB29oecBv2_QKc1tU5(1={Ns8xyUy;ItCadzlSE!K%13hC$&wk!b7Na}eevW_*bLOCxk)g~Qa7`80p+7UbpxmDXqn77gJ~1Ry)i6(4;8)>d!L2PP*r{-OpQt73Mb*JA)Qg3##3L{yl4*!sa$BObv3=iycF^wdOiB=nG~Xc(1$cF@92D=IWFRnQ6g)ooYp?Cpj_F1F7Gs;A*Y%X`j^LKDI$%A2Rh=9f4&U+S@0v$+zqG0zYYlh;}bV#|S`lNa*7mj>uhn0vdbVMPIR5H@zp2RqL(^&cCFf(`4xwQiJf5-r0}n{Vl25{}V`wYuY))*v>DL`M_e<M27$!OW?{=Q#j97Y40O@Z?4CTrdV5v@HAOKMH`s8lEYyO+Pm!wrXou<P66$<AEGTEJ<wsk4`~B6s7~AD20i)Tt|M-!KFTi(txMO+iQ(<v0I4;a-MuFx;AeWzu--8;<0{0t}#5e3xW3d#Ixsx0Lt?Ii#|#823spAX1oN$8u+?1FKeDW`)=bN=p(xuFq^JZybI}H^Zvh>z2ow*=e+kE3B@0n3TH<fQGB#?4CwfC<u65kF#1|E!7$Ais{=3W|5-S^j7V66Q)Ot*N+6>vxQ#{si@!(5UYbWV)9YWI^2iALJ>oz2c5>9#$9Ju8u3zU0b>DFN+y(Y-oZt=Pu-Sd!ZQXO>n_lG1$B$5aJawgcSQ@!b!#;Nr8LyT41{EJ(0k7O!2n;^+?726M<s7sAC9-5L@d4ON{f_fz$N2-^&HMYI7GZ3~-`V7EP)h>@6aWAK2mk;8ApmC+{f7Pz003Gz000#L004Alb98erbY*jNUvpz=Wo~0-E^v9Z8~bnD#`$;u6$F6+WJ+^-Sa$49!<sBfGjwgyJ%CLRh%<STZlOqqq#V0h`nT_U+>u9$lBIStU|T%iefPazcXD0#ud$L@tyU^_RQ4cBC9>LCsmFR(%IbGcmOI-#k*O1JWc;DX6}-($=iLu~cU0w+^5agGG6`MRy|`GF`PNB9E#g#Em8=}KE%LI4Sw$*hd7&S-qTXDXhitbko*Yp*S;6anc>k?g_kJtZ&8p(5mQ`(@>b#6MdPn(K$Wm>AiYlx{6+PU9i4yB9uWA)HNO%_x{JhVqT4b?&^Pmz5{{tEoNfJE(jFpl}v=V70qf)L_p7Cqo28<V_gb$I*R_U%?`tK@2B%(B5tGeQ`Qf~7H*+Nu$Q}dR%5mjO<n^%$SAXI%4MU;wrnO0GrrB6IsO94_SqZJU32yIFMn8MD0TYfL@RjO(|b(aD54RpZEKgg@CtT%bm<J0|4rAZX$SzYF-CTq3h-5NSKqTJ@0`U!Bk2TW3xN&QA;s#YS6G;AK^hv@z(QVEX$BX*TyyrZqCJ{-gILCH@&EJY~=tgC@5NmPm~0YNYrd>g~t8pMF>cw`ORRiNO}YL}*wLiOSosa%WrDa!BvE#sP2Xavg~xL4(6w3a#GUp_^#04pJeUR)${<s_=AOLf1ik>Si8&%e`>1|!Ox&-8#l?)N{wf9Kw@_J+000J67OjKZ718HE%4GhQzJ5WskTU@d#~=gV(~){Sw=6#rZgt$X|K^6x{-Xs0Xs(_8jMQ-!t`b)^{9;y#rTa2eID&f^?dn4s^dm27A>+BEO$fakNwz%hXQ7CaiN!~t#At6+&)zjWqvXVFsfX5?CQz3x7ZJ5Lk)=}qU|o9?@5=jp2RG_#-HcJ_PQf%&!r^KA#_TLW`x({9^=hm`qYwJS)i!KJ=M_j6?W7^WNkBa_^3-@N<!^*`OiO8^U>t!HsLCdRU2oFtDS8~1z`58w`J86cP8e}_hqrk)1bLH3)kfBeh$-@X6Wn?JcHZRFefXnDD4t0Ppj5#VakUBopk{Up_TQ&&-z*KCXtye`4cjefF)f_e_xtxW=34G<Eic_qC@s+Jq{v^VTCeB%0k2sqisxMze{Z1gkw`x5N1oh#}*osZ?^H4pu-&>EeD)-@qS-eEdAj+P}GkQhinfh-D{iIn^~`sE6|NFrejJw-&(9$aV|?^1!B-xJFT&hjPsN#fgRc9y%eI6l1cki#62qa6`bZfAQwlS0KPaEcs$W1ylHizrL+6cXK@zZflBd4cxROH}>^)4yPvms&_pGfD*Q4HLF{bZKddmH^<972y{A@OHOtF~TwNNMrC3=_q2~xf~iF!Di-j?&Bnm9WXo&XV#7%k!IlU%y|J!aK0io<7a^U0zn{-ml`i#VoVlRLiED9BU*O_A3r+3bv}Yq+!~9F>zx1xa*C2)HA>#+I-r57s<L&+wiGp1#2W++QbDWh9pmG9i^<aY{ds<10opO?cu2jc^9PImO9ytm@zJ_Hw5~sL(hG7aKZJ!S1*Rb2B0X1cq!Mty3yM!1i74hQ{njpJEux=fnMVpT$x^Kqh=5`^hi))MPLQG`C#M;2M7EZZ?H#>lv(tSjZh1gnI{)=LylS@qA=F|%0>)MOss@epF2~0lAns&Y3fqshl-D-YqKiR-?|Mtf9`Oa4BM%0~l?2)|EZNlaU0v+z9(m0UJuPS{JCz~^e~hM%dRk8SgD46u?o!53gOrsClqB<1lxu@<1f{`Kw8AXy7Y9eE{zQ5_1Pw}?tTb>4T@-b2Zg92ZqI|5O;F71rj`0Y7LNLY4<`eu`E(0g2p9(qW>8mss_4xXPh|+>N6W+l%W22@&90$i5S;cuNdkU%fGUy;}QPSFwcEA+P+GIM}7rCmm%Fr_7stzkNLuY@C4xO}@=j52-jk_tHF2riFjy_0PKq4nNsQ`Ue6>1~uaSp{Hr4QiJ5?P*~L`=i!6w0Pc2sgL4kWY+f5aNG@9zf`WpTL=h<10uNE^orwczlZBj7x6a5(5o^dl}ee<CX&b0^JkS8G;nHV3uF{;k4)2zUpE-LDd>U3qnvY-#9n+Vr?=IE(djMe0|8B^kcE#uPf}y8e`-TaNBT`pZ$(c=b3)5adCjI;bLSlUUraaXPs<5jCH$k&&$(Fko#B8WM3fWZP#P~IZ;MyvE6Flk0z4riWH(zEvbnEzQ9BTvMrTGt71Ai!|xx{ugM|RIlp2V4i|4ITo8p6E*fmCFwhVT7<Ri?1He#}Hj2&S&~S^?1}j#g(gBuQleiTHZ@pfkaLf}9k=uauZGE?dyDvsfRa0Ko8&yKJ1s*ZR3QM=JFqK=4&XDiP?4$VGu<w|2Xo+M0zn}Ks*otFSK2!)y2lT&%GtA>t^LSv-p`~Zl(H&VC31P#w;a;Sh!CEe+%}X|l5iUw(3~3oSnhJ8qb22e>OfX?zxfuX?1$b{PsxdoU*M>Be%w3K6C9iQi*%)j)ceRVr^Q;ME2=mqoD6g>srd~1U7k1V*l(86g^LEc>6;M%yvpo$W-LZ!`b7yEe8MfHMdj%an=vi*iFu)|yh6BI6Bg269+AyQ=_Lwb<_vo<!Fg9GdC0f%kp?LAaY}-p*FiKPqI6g?PXJI;<7>HRmHQlWmfdwp8Enw3SvY4%@?{d4DXz{!4CUrS(c@G<(AxgWv{PJuXe27*9xIyWVtCo*4W-}GLL}9!|=2Vf047)nV9ryVzOGNo(EdJ232LUt39&x@cz*$69CW>m4*VTnSpeier_~P@yrWquCc<x5M9Iy{os}Ux>{+Fc|81Ond6Ut>D$|qHC^jTe43ROYg^ffUU81tqq^O7LMo4IRq!_f_JS2KfsM~_yI$P6^1-R>hVk7Tljkchrj*9{L9B|3ZDZv1PH#499`Qshf02_U6hjHwjk$_$IKj?(NMU^VJ88Yqnp&mK3$Q3&x0Tt3)6cG$;F*T@jCkMfwravM%ZSJy9ZW{0Yd<A@mWSisJ}intDQsc+Ryl`wQZzrM|IJ1}srl|zR79IX_v>gOZO1XTn6bN1`1Y}K<}24i<fXwN(Fk{`W}BnH&;A0pjJQcX+w4&wr~-q`NV>7#Pe#d3Wg>1U&8u}o9cGwi9VEKxD+=v@JJS1)}Q`p9>FLn-uRHUm^ElpW6AsjPmf8@lRI5)>G&Y7u|%76g~Nvyf$^o_+*<5s#{xkGXM6$DHKm&HK^FU!o@a_5mfGvxn$_s3Gm~b^#sy&)py2y>dIowp#CWaefb|tV<L<ct1#>$nwB<R@MP)WDZ?Di#z795E#o;QABDO6M(wsso%^29lx8RL??IC>m{uCTe;&iGM$Nc&U)>Q0$T6)Yi-z3R7|Xm6<b=mD@#&RYZf1$b#e(Lu{&wtc)5#9rPf%%gd=QnUyZa95A>xGT*R?RqqF!K39)Nj9ykYthJe^Dti_gLBQb{;Hd_xY-eZ+XC^VjUw!ltpbbQ?_TC}&u(m$LS$!!6aNsOkNmsm^s2Qd`N4%PIMR$%?8r~~>(KmtpWgK9{oM9r!p{X9{to>x>tW>6x-j15Kh<eeh^j_$TnJYy-!q`&Zaf{?9eY@+Gv?{kzFuYL_9o;BBTczu$IwkgX|UTFJ|sC79X`THWVviTEt+`qUopj5EEwt+c1PJeL2VLqP=pKdSswn&af+$I^k2W-lx|Dah(k#*9*gFMv%&az6(6ONyoRqz*&%)IGM;LN75#+yw8XL^I5rjx+AzQRw}5J+dYzI~9^UIpSCKurUN8wXLGsv@Gu89g9<m4|!T$$edltXk#eR+h~@rq|!T|AE5q4+i{2_u>&)*#4YW{<z2dvkUog$wdexz>v64Cs6RjWnM&j!iMfe&>P6kKB=VM@H9&U_yQICG5#C^^<tMoEt9RitBF2JLsVc^KWk`k(rY1ok!v8;!;E@%4O~TB7UN|=ST8WUMgefL+D&-;;q;&nU_Viq-r>c7+1+H+NfY#?ElUImUlhZtP~eqCn0@fFS4$dS8FP4PGv78_fK9QWdFj!bYOa@Z$`x_;Ud`RCfCSmK^+q>O3n-ce=OM+u+%-@`SAqwABx*818o+j-1Z@Q^Frx#TGwy>J9CtxXY0%)=Ev9>XZu{dXFY2c5O{q45r;r^^Cw}in6Nzcz^1kXPKK~H~1bZklmhIIkD^t){0#Oso2RxpI6*^JHn}e7w0QO~`>|&kkH`O@Z&X9smgGZED-=G=?&dXcOLr2TjL+z>Qm{Qe_a+-Nij){74hoCzON~FTTp&N}JtK6_<-uUe%_x}qZ=CFmYV4zG8q<w?rh++t!yIY)hF1j|2QGMY6@7_Z60AawBKGdUV(_Sg;^B1IWJy0XuO|j;icK8a~2{jjYj28XNzn)u+v1^&+Yxw7lNz7CBN>n!J(p%P`KgTiPIe^0z;&9c$p{1;MHo30H;Dq<F!tp*@02d<@?_(cHuItGGAaot101upJ2nP?R2^1EGKX(9qu`ebk9jAEwwsPx`8b${;y@B%?d3?vTe%VhqQVDgT-j@_@OwPd!fsYUA8Z;fnAVW^@Ouf6$o<&#VXIfV3?#ko;lkR|;Sx`b`HKaQEqr1-t!T`x7Z3k!e9ZV-)w60IzwyzK=a9rgsbC+l2Se0C3XOS?pGps(p3^Q70HFwer!zJKQ|Bkw6vBFR<Q(0_8R=Z#PUm&=4M_X=fL2t!|h1*;K<w&-cnYkmt?bib#0+6owK<1wR+ra`)Z;0icON@N4KknRLG>U<F(Qz-sW(W~(Vi%VBnH<VFd9(SBfi2HNUqA2`2FoQn21Kfv!)&M{l~nV^(zm|G0PK&*9)2ZIeAn$6X!z9VJV%eUq->H8eu1MkRQy>0j?EV>93yeF&}2!rd@)4Hx%>M2*WWu^1?6J4c1Y*l2C(0~GOxcKg7@vJr!5s8F}I%qHO@HrTp&@al`N4z{LPZ~4!vpT@84cB7lhhIbknuJq*X`GrMll#G0k-P*Idcr@HY1Sv*GBbe9%L>IW3q#F?H~~<r$E@*F0`e?{*{fB^u`7Ggtav?Op6Mf@lFPIP|{k-VdbRzUXhdXo2IA)Zq7oxa|-)W<06z3<?M5X1@+&nuQDZuMu|O+$CN0vAbtxp=lvFbA##aIhNGiS+s5GS5RU5RsM1k9SCwu7?ZqecmF9b6F#DWEPkEAR`%DLz`mY4qR|(?CwMrczX5aG+{o^nEuZv#siv{d#}ors<67oxhC);-^GDA*347_hKs0=FGEb@ZnHYqSa!65QD9}mT07e?LRV+;cpX({hp7bm#v8^N76WuF|#a4-P83{Q57f?$B1QY-O00;m803iSmPO>Z94FCW&EC2u#0001VWpi|MFLY&dbYF9Hb!2%iaCx;_ZExea5&rI9L8vGorHW!F`*uNHbi23LF1o#2>>i2&hC$F0?eHRrI#NpF3-aGPLsAkY%Smp#&4<J$XNJS!ygWlj6qQsp`9;fewpFqq6|1juu_3}ve1flbqTW|rx5HoHZ_a;Y1=kfz`E2IK%ciLIglSS%mh01pk1fxKg6()qSnX@BYbRN;Qg5Xyxat;o_wmCg!|+MTlxr>3%n>t9Ra4fY;7KaWt=P3Jf9EAvtQN9-XU1dlrEapl{c^}$!D>R+yi9qw*><S@vHI;DORxAOTgoe3#p7}#pSf=G`jE%{Z5cl1?1p`|W0siTfmrxf);dAPo7ag(-8*yR7uXLN0+#t(lkIr@nWKW)Q2_<7uVmI4)PAL#Jr&t(HcNA+HThXy^HTiA)#Z%950{$OI289EfjlOg!MBQK-8Zr%!Jp6cUxJvN$7B(c`OJvQ_?A$bvOK3W)I8tDq~>>ZOjwptUGY?~+zQdlpH;&ns{#Iqtsy1LB|*ldQ>7;{WU#j-S)H%DQJ!l)NpnH|L~>q+w^C&h`6IcQ6L2?zUx_HqYKg9LQHN14P4T)><=E3hQC3Y28`sNlwOFIFhdVNwkkvYZb%8SKO_K3cX1q0sA&!Z@7S$6ramoa5Ny_RZfCK!iji6*J?qG_~P`dG%ur4Z!Q5=&3*2<j}7b{CYD8f#@u2siJ&Pb3cal?Zs?g(-TXGxWm^>R#mQ}~3tE((|9fD8+EM@xR&?(v1krb|}oD_K*1!_|Jd@MbtiW6zCuoqH@BPzH2i6;&J#4-CWgaxtG<ifCr-h7sBBI<!WxQyHrnObm7nLxGuOWhrZ;I?aRvGD8g{I+aS4JK7WIDUPJLhTmbu6o#ZdO{<W+H7;rW482ffKq9t+EA+*1aefh#7cXD=0fBiCm`pxQFy5I`Y#$C>y{r;QqnRuc@B{{%rmEb9z3?azWzBbx;f8$p1GYVX`7$QqcVJ{#{vwJ=hWUO8QXAlVGhm5kho6iIJ2P#A`+ntzq$+p8a5sr&H#|h9UIdyZVO7P;Y^dV>fx>~vf=kbG@dO1fL!U_pM5s$rLYPm`{Mvv|0!{Ee*QpX4PU<V6-A;lD29r#X;fCes^Tq2C&BMrmz3+et``udIG{{-q+Z}_6R`_ZCU|a{jqR*h=*8K_rM#2u<bilfJ9g_g}hiU;u@Zhuf?hHK7GymgO?{u9Ya~kextJ)#SW+U%Vl?zi?K>3$bf;8V@(KDJOwIg55yFb7QW3SiiV@J@%Vcgo21B$`K#Y41g@1RbEWerCu0&;^Wb<F@yK>4-Do*=9WqU2VIno}$@A&w*&pd{@Hi0m3EGblnIae|(H)eb)U2-_1oazc0_m9t@6WBwl&YIpcN1GUtVpUXQiwH)B6K4X}o*W0CoIH6ZM4$<W><7i-5-Um>S6c$;^Gm|6UJj5hb5{k8*zv5zdg>n7{EO;xj`pOXX(cQ0zI69<sD{@Y0l)zqO4jC{?T!{cIaagw!Kq%KkZ@>g=$}p^ipFIcYOEk=l3baSo;of?R))s#M7@aCMiPn2mCoo3Qsjzu9rR^jJ^MK3g)BvJ%BmVLY=DXhVL!HlH+SB_$Nu1XnQ8c;k@j^VM&Xd#<aE3L(hNa+G7)AjHM^XTQ1AfQTrVXPC2{IhsusXe>T^yeeER-UO2Z@Kh#|UJd8A_0MXbC&x!OXr%!2tE3M8PyV6qI#{2&qF`@gqd|GLHsitMg0DotKOG8aU7tV>CL9nQ<F*2s2P>JUP1hQI>onzk{9@>iCHrb2-jzbIIxp9}yn1&677z5g(Q}8%=!2(mmZm2ZKiq=k@~wR3A@Mi}D6eP8yW3GUauBgtCJCAvMov&Nk2{JV~!vK`X^Gk=DQ@WP==LTL<(H!=ce$?>net^L|?g)+07N>DU7(qiAtwOtBy=Jx7~)j$x@ywb0P2phudFp*S`;qTJI0<^^PG_@2r<=c$p2d<6$=ozb*v5Y|Kf)uFcCZz0QTzOOKtoJSvPebZdDZ7<?=P7!TpZYL%eSrl#ro6)ve24wCx${_-#$P_>PWB_{bevU!?eu2M#coDdkQ9);?GL3o8EWE^(l74expbIEPr><WMtP*j%a@_YhuG~R?&Or|Ltxk4?E%7UV(yj##B(X_F2a!z;{V5;=q;03sMCrloKoYol=*2g#w+i0<<L!sfj(QIGBkN{ZS~qZZgUn{g&SySiu;d1#+AVmQ`mu$r+??U<%${l(Ei+i4ql1N9`wRVol6PUJR{<M_;+)aG5AFHM2{e$;Qn1UUK(rAVl(+DNsT~~C$;tICe)5}hCZOGaYxV={y;lnGMDVUXlOER<jjjn`A_5%*u2;ZeK#6e6dH$Vf4!{ZeZ1kb_7Rgnd7;^{*Ad!O)b?WtHs^CCEznT9zk=C7h6B*$-B(ktu86yqvgS?Ontqy(D-3v9;=((^v>pF{8P>oZxiz9;wA$?~$G$^WPb_pU_ujXsSRCq6i81i5TzQkvRgYS|fZZH@#7X|AT07QmU0Q(`Arc&2KAN@TnkE`rUXoDTph9=Y$W-rI~iIZM)UTG@Imcb;}EdvPJZJpi-EjD?NHNGyhb}as_Cxd9%LZaxdAT2KVh+tYxvrOE{R?Q|C8j^u!o7ro<9*g$C7X8^ZynT<e0LVTVI3M)A&ilPS8PofK#KeUuy7mTYzz)5JE#Xk6!Nyn;b_y8w!Jrkg%7ePT3C5RbaksC?V@hn)g%i8tm}pRCO9H`%hl{}t)8JrrqnD1zN*vlUw147b(As5+Gm|&Y+6y`zjBeKaaQE*Pk6bBs&D`<m2D#6V?Z-m;jjLIY7RmXPyyNj^P~c&KwrtM+AUU!bt-q%&n7YPh4`4*CphAVgR+J2mqlSYw_}9XTb=1dlCfKfo6agJpB`Thal0$6c;ZTHy7|u4v5ku2op(2Cx&{lT}Uw0*UmuZI}3aW?GVD?1R8(vSFE}}sdfA~>^HC^3e7q>CVS+U6&NhLPj2BhltS|rciooQE=ar?#+p@|A8%Md*+2B-1{ydHuH+nPJByYF7jHbUFm#;#u&l0&mEtG(?4qvI`<y#};*0kOLjG?xL>ZPnuP4P=qa#o}07^>`y?nkc_%B4edNGnmlfBElqbq<{Esv3@uJ`B1FA=dI<z+XoOVJIjN0u>LRYxm^;}Ci1{E@DHGL>ueFenY3T*O9`^J8u<QX9mGmG0LA76O_-So?A)bI1*S{?{`Ax5_uqFd6M=z$z5Vh}9B|fyk-z=)pUx<j<inRQIO0r%K=Fnt!Adw^8)X8YLd$NgMAW>{q32&FAEKLJ&T@wJFB*2a+=Sb}+wAGRTjSyM<gA5^=7KB_Joj@T4CBE|otPSIpMsxIHKTx^H|+}-PHX>}>qwi?*%UBycyV}SPvPj{J6xNR$g+d@UMtxr+X0i)tess=xUxy$-H8mJ{<iIzAx(Fv2Tk+AD&q#-tg%{~QqohIjLVCvhD!f!yM^8*z?@sI3>Y$<gXtO?{cSsjZxC@4{%s0=FBY)`&rXdNcc-weyuod?-tFK2de0bAsiwjP$pgr@0?>pjsm78z9GVhyBiLj4ghd1YM4qyN-|To$dt=_=snC?!s|yB58j!uL?3IOE!ib}TVu%7W2u%~LEd-g^+SG&pQ1JwY(rc&6P`z6v%N<2`1>c}jQ4Po+jt3kZ{FbIytlS}<_{F5KaR)G=&A%BOn=!Jd!!gK*Pl$HuWl+hA2hmrH9bU~3*Ldu?*=}?hd@#U(FoI3hJ_-EyOV3_e<6e;|LVkm;z+qE?Cvae`-)W@@-JIlQu-PtN{SVd&&cGaJO=%K?*T$21G}xX6anQigDrY5nZOLnVV4z#*SiYhGESM;59&(SrKH9fUzkT27XpX$|9Ce{6fkiRhhfy@@I&krD6`Y>hg>d7w3Y}BHaCHT)hwT_{7D4ssN&{OA)7aSj{^EQ-Ns)qSz*Z&H7nWPni;Gv&w1`^`o?l~mxD!QFK!Xpk4BIxy9-5k(SDMbhlA^4;JF~~y1hBRODR{Okdj)M!qu(d6rk0TPpiS83@|MaC-)sMQrc;e>_!hghj3e&FWMJ+g^zjy<U476ZxXWDQ1<n~9V0p}Ux{HHxH}Q3CgF6(kQP(_NqLAnbKGtNS_E<dJ`b}FD@9NDM`QXBB0vimv62sN!sy~pQ%6w1iD^AP|;Qpq}nA&4k3I^M30Pm5XVcK5KlHLLE|ENF}sXSqje>gPc8c<R$GEzaKf>)Jz9j6V}fkdJShlIoxq)xzBM4Auadt?XEmrOy@C;oBZ<iz<&IQiINO_PXh`YmLO-oHGboD*AT6$E=WYy-26)`466W*0DI&}~o3G5~imKU8Ak^$6i&E#a)8UvSdfk*EIQz_m$$nVn1Y&*<ZpuzOs%f0#Yts=eC<vFC%wJ>2^ruu)v1;}4-x9|I@!a_Fr-+;i1zl(HQ%br;5PVC~Udj+>D#rJ6e|n<Tqy7`hb-(+_%oFy?fDIQB%B>f_O%b(7JkDJZHK4?}}jVF_;(Kb<G9ADdnz^T($zCf2N#gl1zjJ}~{rJ%<17_?YENNqH`Ic+X3_OQ++OYmL<#XsF0~_Fqs-0|XQR000O8001EX#WzWpl@|a2UOfN+9RL6TbZ>8Lb1!0bX>4R)V{dF>Vqb1=bY)_1Z)+}adCgnva@)p|{;#K)1OHgkp+L#@iL-RLwY0?8s_3>Pm2+|wFAM@h5)lX>7(g<UP1X6e53svWm?zn<duG4^#n?VosoGT~mN=K5o}RvU)APJHGLvOkNl~q&C}okWq{_?9;XI7)WGu40lJh*jdnHs}mk}*lhe@VHl8Nh0waPQG&f_|j%J)3);9ybaYY_yCx~fYV1R_}%d0B}t1E8=<@=P5Z*s*e1gr$=9GoThSs_e%fRi0^ZMOdxUWNu+y<D*@q>UmM-kyOf#RPsahP=*DLSpbm&*9e!uWN;wx!|;52sUVhVaxcq`1>NCA^z(W&9S0{@7Z+!<UNdNc5tJax*XyL(PL@)Ea5B(zvIpz=?tx5|bWr#|z~JB@mJ2bjlQh<cxZCOBgSt$I-m0n!HR$)3NwuoyJ{I*SVYCX<w;R=;h0DWH79URKaxJs!aP;PU<YT3$p*+>UcF*tXi{m}Ew`;Gt^~`6DllM;iX=w0kcI4%wP4iH-w>UJ@w;pVX6DCTE@4~c}<Fd@lj(2I^0L&SSqD<~d3JjMgF|L<XiBuv^77JPWURM)Mrl}G`ar^FoKNC&YVR;wl4_QZVb`Is$epnPTi??^DpI=&^=*OeQLA5EQHxQnM>Gd8jT+1qq!z#qOUw+*)uhGD)l?6YRbRUbKuc|UBI$iOF@P61m;n(I<cYDJqkEPk&!z+tb5(D7yn-E;x1M{SkULwx|Nu<@|4jBwlho6a1vkx4z3wzK)wgKFEOogi43|wIL=Zh^CN#<bD6CUsD?%CnV50NY?@fxU%`A=xF-+|6htTHOYNRlS~$kJww4<5{_07rGYA6aiHQr6(mwfy*6V*1_r?jOLO#1Q*fQ+|+9O~ip%0zWLh%rp539%j(rCGvqFFY<Ej{Yp=-nM{!8+|TEET@`huu<Q2S$Hc(A@5kA^5n?i9H1I^sMpJSBQvCM!{}CtGH&780_KJ;|=!?<u(UFiJ3Rxzk3YCU(Q>1wq!<Hth(ea|n#Yvuqb6?!xDo>vhu;2dkzr>`@M3|;epNd6Ns;WmOI1>u*8&zThq_t9g%13>5B;LGkZ}Y|b>$lK&LockW+<R}fsurgBkIo|<2k=@yS9QFhi8t3bzPP|srm9bm(T||%rRu|w)G6SQX(B^Uu7L{IQsA~KQ5EhhPN6V0xJ91EvJ_!d=1KtvrO3?liP{ZlD4~49rH^BU%<hvi&zN-Z1f;P-Q7NH?W0B?&uDE%{M>mALlVv7T0otZ=K@LL|MwPEA{~E%x1hMig-L&aXggUE4l!6l#16(8@a7Izj(<BmPMA8IIDqx3`1ynx6BlL|yg|kCM)mA5V9^JY==H;}|^ATj>%QaoWXJ|8=D&wRI=c%N1sigh-g&dEHkaNJ#%P@;pubPL0sl(GMMaXMxh!#uZ!NuCOAYA1K5XKl#A0miXVTQ2wjiJ?=R{^1vKh;SoW1sOP*%m4b3$@BCnuaz9gzz3GfR*{-#RJ>}K57*zu#9<x5%_S8TSa1#<_`lg$2;E9yIdcMu*UtFfuA28CTl`ot@As{L(g6`V_*T$5`zp8_l;$gf<s~fe99^X|M8Y6&mcByO~cM}Tv5gfif@Vi@uq9nRAml>O&_k04o~fC;8x8mXr6%R%krOybOpEqnC4)I5`KWl7y4boGEKne8tGErLt$wY!&*sSe3N8hY95p20St8gynp?ofBk1O3|~SK2gG8&A-ftv%6u`hGQ-g3b-a`nlbZxNU($mcIe^`=($7~*vKotsD^f}CuPRVb={rs>v55TUf%Q#Yl16k(BKxuq%b0Nv!Ma(TuW=o~WLWuJ$&#KQZeUtRG($uO1xxHF8bZPrv`>nk>M&&~>S?vXx<=my?9#AtSSIlAwUpIhek+BfVvKzSwZdhV;|7tHFo<ITk4%!m!l_1^(WRLoVGy8=%80HULdx+VAb{b`Z)#ADRq2CFmMdo6dsZ%DP8?T3*!9a0-mHLFbE-P2@VwyQQvOKlRX>x(cbzEkAgN7&!Qm1wnBG~T#5J`1MF}~S3rNP2g&ouvp;Gl4QpW&cc8rHfzSd%Eg~lW{KnbQp$%>v+L3c+$X(GXR;(9!po=s=t%ULkFx|)$1F=`jg8#=m|;Z4TZSHaEX96r-8*6IIKsjVS-&*x{C)`O-cjzeu%O~%vlXmaw_S)~<?U9gMLP<VafRuG^clk(_aY*>l(VPFrZD`G@_+4HND(fQ*|Rj3Z{UmiZ><(*!~R*#mj-D_F>CxXUcpS2B?_zV1ne^r$8C%ja2rzK=Idc#miSM5TfF|5`~t8aw9Z?#k3Y9IZV)-ipTVeyIi55FO`uv=luhhB{S8cfI2>Dkp~aB+5dGaFBV!ZEBn4lPo|$t=ft@rv;diCakPl1X06hRYF<WlLi7w#Cr*8jnv2=<QKY9QVYtcjnpt?dEJU4oCumua4kL48^RhwJ8|g%&vmz+nd?x)n6{nghPs2t+d)VVrX1YVO&rVE^tF_bd4&5BBkfjC*NFxS1!Lfn_P@$xVL+-nVU_n&ZoQMr)Q%#msiu-*~z1+Z_X}>QN|aev&+-*1k)UHH2gXFZX8@+U5|sa3%qe_`{@W=H)aNl@~Fi<PsHfr^?1^!omORPE*aW4WQEn32;(rsa2~T{0sTQb0F1*2jsH3uPYBJ6(dF4U<0&0U&TxB@5wtAdOka#=Z?E(tzT0tYp9r}wstu;Jkxw#&sX*-b#jA$Xv`U4!tkc-~lkg3G#?x7)xHWX?#t8omGCe)}8<Xpn7^f#*^u$;1=*%_VyC#D{nx_YzFX5|I@TZg(C>a~pC~^SZkovqc#$z-iRY3gydUW#b_)=fXlhE3whoDQvMd)cBJs(YHc(>^pFF(I}b2e+e-SqnWY!(oIWAxe4voDX2o*nbd$<=q`2~l!(b&fwSPw@WNw8&?{LWo0_3dU`&^0cmunaRj>92PkoNnnh-+>=o#*LjxwjKz5QSD>4eKTE6IO~|p0j2}f~12=kY$yGaaWnXQX=gU<35k&_!W{kC<8Gx-l93XbQX?RbfPezb_8;7ifnNlqQwfcdB4-3%W32^y1pJhA%a|d;{rpT?+US&7e8H`r4%*=G#Xf->$C*of#aF!I`QNWOAZLirx?bd~*_^R<?dm_%+CX=`9My`lxr&l)<dRVZL(cI1~)P31$Cy2W4<{z>)G?Q)g;cV$$o&D*v?lx{TSr1}In^fB$IX(f&p~xY11d;COL=#43YX?1+nw9h$!gRrFOMcew)jHge0|;qnsUwzc-hq6IsN^IHgkQT8?4j#STc+*PvbI5BX*t&yQ=7+S+_WR7*lKFeoltE>Lh~H1J|P{J(TdR|9mq+`MUpn60fJU=4~@@50T(BI>0#0^qFq|i@E+!V4#l)u+KI(<Ai^Q14oTUX(7kN@Xv0kZqkL!{#7I^S@lYH5L#gGSlRNI&6f*`_b(N$LBP)B1>6uP-`|#g!nI>6lKK8_gVS%1_JG(eP__TZ<Y(-Ct+m4H5uH(epLyMzN`R~q-leN3tU*E+^*&z=I0gh%_FdZg?{Ej~wdXe%bhhhi(Bj?L5G>B0yw=Ir4+1Q@go!-gtb}3)m*>O30yWZtT6Sxz(bc+{mzJ>iG$pNYk_Bp7%*V;%~9cWO3DU(S-G}x~Kc5d2hsMd2R-fLWI4IA=}NXT3lpxb9XB8QH%lpn3=-RuDku~p>^vVMdtfR}xGH7C((MtlgPEUOMYz}C*Li77^iB_G=H0jrymBMNrpzPr#dzD|D=2ig?5t@oJZPiY2%jKmdak$cf0Ow7WzNpu$vp~Asm)_8iN4a|o^DJw;2bli1#zP2rmWh$HlZygx#Tm_(ifytdgV)-Pyhf@^ea)DMi#=IGOd)-z}19&YveA`++-dZ|SA=7V7`&wK9=jT7RJ|yWyMQc)ND+ya};YkQhV|LxD&rbIJbL`U@+#qdki-bcCCm*LRl0p5se|+R0f5|#u$GI`4W(cxZaIRkX#|UsL`gEU=XbCL-XADx{jK@5=IUO;y<7XIrSV;+6MR5&rZ=-&}ORVoxxeVbu&dM-Tlm>=ui5LF!Kl{%p$d91^mCXt1d<2_8%fN>wJ4iANwqaFY$+gh98Y{%F;vz;abH<ocRw79ogP2KJSDZk`X3fgOu!znE>W5@FtT{i+4*gV<(HuHg_d$%#tq0;Gc05$;ggoNILtc2eEb1ZgaBnTwdAaec90pb|Ao7IOFjDuuEMLhmhA$1B8-)eeUA3_{YM;s-TII7QVFTwstda;)<s97Anf4|ISwY}{0npWG>x?STGdJ+4UrHR9v37+tAMp%cBkkyERmH%?)^{{*Bos<i!N>*0ilRuE0S{#GIOlH;WmG&9!+VUl3HqC;pO^5U=5j_sc-YO59-V?YK1G9$u*?<Ow_zDS;4BireW*W14lM4q)dZbG`*)6JF?ZyPmJqg$a{paaiJ10K`ulMVPoo`=K6}<GxY!e`6K?x%<imG?j}C<fb$5I8PJixrJ+FIvc>J!tj8}BU*W%?7ejZuniB`<UE^~Ud?SM83NK?yr^ZFG<T1;|nPC4mOPy$Z7lMcc9M6HyvUv=SopGP(#h9TX@k)7)BIAlRL2Bt@Jt1WvC4IB1#3P~4K{t-MnjvLpvT&?|N@<hF^pQwN|eCHE{rQC13w8Vh{9zA4@Yx6N&P9mKe!rg@Cr#!#v8!QwvU#bf<RQ0hG=cj=OpGO)ub3^T_JJuHmS745?ux1qsl$4HZLRc)c1sU#{?+^;T$0<H6QQp?SYDnB9ZAkjpb11tlp*|Zw-q>t&2Rj<o&qWA>xK*4)RR<oMxmHhHe?NPBbqS7nOBsrxcZ7|zt8d4bXMcmU4)1X?IzNZkFufRh3rbgE{^I)FWJQMFXt{h`QoTH%eH+mAhvdk$87WH(uiF#fU!7m+1lyb7^lYLRy3YJIt=gq+bqeC78_H68U2CpfVI4Z<UvGP4P=c`DU?=hb(KuIbX18ZiYKQVG-hfkR+nae@slhPiDSKQBUgb1kvR)XPlcy1=<YzcN11B@(00}F?i7L(9sgx0b$Eha5E=177ZB4W|&6f$4jQ6FQ9O&!tLtsM8V4Y-jMTqRl-2%Q*Z9zy%c<@L?J3zspfm^l0>YdbRuUW(?<z@lrM|U7O$!5LI$(I^9ob1z+I~h|KmUgB<A}a0g$)3_fk_`yx-SBKp63zi)>qtSBr;@6c0JT3S*B79Sf8^{X9R%I(4UClcsEt4;G6S1f^#+g9sqQi^9U2bYlUjAz2EIdJYi9u`t^-c)djst)d5!C>Jz8(j_Sv_fwj~~K@C$Pcf+`RI%!n5`{_3db!T&614Kxi#d>YA%pm6@a^9K7nB&_%$1Tc7bi{ki8nCCbTR7fR1gEEMp-3O#mu_jH!`!Kja#yXul!q}-)QJKt5V8^ubJzcB$)e--Il*XqM0ld~2uHD@}XCVXfI6Jh$kLz`zIxY3m6T0nY_{=CDPNRZb*?M;PU$=K?$5&?U#_GCDp77dZK4{u7e6j@hFa0PruCzYe>v0^s#kAYQ9G~U&_!oK45Am=G?V;9(_`k;kPR8UD*64H4-D|8gw#>r6Z-=k~O5d*fU5kKa8n4|;341HCd#&vQcvOY*{-{9n+LYmP&RuRx!loumj*70bYGpgnsrnCK(n_Wqas2YPzyJ5~SJe8FFErAeu2-T+Yvs0;n$|=1=HNv#munyA^8A{DLKbWFN@u|+HQtWAIfcsk4^xo6qDGSEN7SE{WDPtlT1XCqd+gg$DgNZ@<Vr7lc69vGl$+fo<v^rVan^Nb3SSrenY$8np*chpT(xqtG)JKpJ8KM)yESOoCL;ROg5dO3I8~W`wJP&^xk40(2tUo8mWd_Rah%GzFRo>2nlC1oZ}5tDl3GIG{+m`HPU|#9cY25)^EqA=L8mW9rgP^<P4~P2TiXmax0Xb-Fn+bIm@mR4ttl4Z?huiN<x*$1ZHd}+ACUu|tk<<}nu%6*cBdcIX%R5lVOpvvo}kjWPNsiWK+YT=aJ~Sz4>{}W1N=g_ufa3?sF!c!)2;YF5mV~=VV>hRRS-GGpL<Le3#Ne64X{W)R7?!*v;Tk6BcHE4b`%ItdwZvvh<@KB$5+mjNw0S<8oi=>=iK%-+x+4`eF|-2<=9Dti|IoGd8MQ<Jpl(fP2HMeYST}{m3H4c#NgE559|j2;H7ELC&22f4?abe-`%fI|0H7DF7*8w^_%Q21EIBCw6hyit;_TjG#eNz=5!_WOcr<cNRBvc%U4ev|8e<CUA;41Y1GJ4+N$jN_|53#dw$!uqshe;g<q$-KxUiZAX!x&!NGoItBpyvpq4G0;?x=ZbeY4QS*^Kc$M!c;bDu9x-j2>Dx?QR<K~P4O#1=}UQ}?2QtqHG4izRBsDLPureNxqi3}cheBwA=|<~KKlA&1J&YvZ^XJ7=J%UZYKWO)nQ`keAhpi&)$)M0nQl$;%2C!I@RQM9c;4ZS<{fTTHSjtz!v)jNHY7ILv9)Qk?29!i3w#%<;NJz{O0{Mj+{4RIQ_uOHB@$cuh1Fly1(Vji%We2Xcws@KIlzANP?oZtNOB0C0`Ier3>T>v(hJ@*Wgm70bw&j#w=<|44O^a;aift<z7I{mwM9N}TF({%pM{#bF4!(JhTiGhS0M(eYc!Y4Dt6qtop4muWtSrh_KBJ$ycRH~5G;d;&=z{q^3X0=S0hb7kcV;k~!IO#4MzdWX%?eTu_*Q2X+3jmA!}tE4)+)s{ziynp|mVEphyMu^zmr3_;o3h2iEA2O_Sky=7XbVA%gihLU77um9fky-i)jNJF=kXJ{v;CzVL-*rx>cYtpL?XqdV15?qwLf@?L!5u)oeQcRO&{sjY-i3L-FchENf_OUXxF>SAKXWQH_SgK^zYxJZ&;JolZ3koJ?m)}~#Fjq9GAv~0p7b?S^y@xP(?xiYt^u6vTxsSoWM-6|(no*dRi8fP*EF}$rR0XYo<8+<Ze{^c1-!#2bld6itvtXSBn#*awjWzLX)T{@OOd9*obEzeq-^}^{-Y`;)Xk~Z=<u517TErRZG#pOL~@wwG|sZ~`G>_k6Vo2Y!H#L|<+;C*f_mR-3Sxlod!{X&dbjAy5W43M&WzjJW12`9wl0M6SN2*xc#dlXe~q}JX#feE_CniYT;+u?*z%i}ck1&_vU_TEWWq-{E2|5eAcom2;3iS<<TqBsbT+r$?kT`i*J`f&@&q^KKTv1Vos@;`OKQ^9T%}-hM|xwA&bGvT)NR#XtBX-?_L9Hh;tljSUEKVh^YxP!7kj#jNXR^z)7dJ`=3HE+@2l84>Ar8nfZofRQ=o_yc-_8L7Ls1E3uxzyFAdB9E=$heo@wtV@80w|(BN}#7XUY-v^LI0)8PJPpueM%6q{^LK)7Qr)2Tdh`_2L5{OM(LU5dn-7kh>&d778o$zRRa5Q%537UwU4me7gz3!(m~T-KD$x#pQp<B;UpFbLv23WBb)(+}e~2+fubr+m1l;-hPHg%tNRZTLg4Q6c?3^e{ag<x+8ij6w1*0&3d@jN#|JS-=GyAnH@<A-xB$8WpN;kKZ}pzigL-*;zN`+pVW^>+yFiDOY=c$j9PChfekl5J-^sc04-uSi}Fj>B#=obG{cvu^eHz9cc=~Mxoh~AAf2}OhfaXQQNc7?u{Al`9^Aok1_YNMnLBlnM8diq_1z;%vuZcnVn0!>Bi<(PTtYpp-DL&_nNv7V(`R#QPF-65_8Zt&`v|It`>)1d5)gfR*5r*2j;8ABtAjF;a3n4LkEG0z%+Ls{4Y>T0|XQR000O8001EXI)=L%j|l((RvZ8T8UO$QbZ>8Lb1!ahbY)_1Z);y<Wo&77Wpa5gaCxN}ZByem`g?wb>QDAw;s(mX!Zka?cG;Gh?k-bkKOC3yD7H*gY%7)I0Ok1a_dI&B<s`7%9>au4di1=%iYSUMXnsp?7|BIlbFEoHR+6u&W~3HHTQNmwQxM6N7LpOYWHk|MCh;37`8~q7i;CxDK^04*C>o7QDQc2sWvg4svW)P0C8P#CjnGtc(WueL?ct@v1HW2QwXFEUFS)R&R#fA*$8!No52D(|N{XB*<##ldC9jxY{>fKnIeK?`d6xa~`Qyj)D>5U|;r>B+bntq*w<u`Iju%J8A)UUa?C87WVsC$Ou)m+PH%H$cMWf5lzy9*WS$1)H_0uKXj7J21qSGJGK3!d=brFs2Mj>+b?bYdJ_S+GV`T6u+aHgrsWZP(7Grz^|S4_gX(aLX0_D9Pl8|-P&d%qB(!bLI~jS5zRUX1B7R8nT`7fT`Z*cmf9vG=0rJ=09qyx~fNrL?4tvL=!FlCRmAGz@J@iDJ0`bF+eLx~LecyfPdayBYH$N~8k1KuZ{kiQQ#NRBP~vSzwnmU$Xo*6Rlpg+A0r!uA>{SqcKXoo~)=;pc%(D`3H&q^Ju?6MRVA`D`r+M&F=N=O18}4Nm8XHU&WTXljScFcX#mXWKs%QQ#}I*g0+`rOkRZ!5O8BzFrT0^jaCENk56Xjj_aj&#Z;_V6XQ3r_<hDj`tAYDdj2_f)RTlzMU-e|&Q3ZWfg4h@dI4tPjXO<mn6`T=PI^%M;iZF4xhlBaJoa4j#_2A&2T|GIMK*U`mXs?-exp^(&ZHDFjxJo9u&j}~RoalCz^-IBZAB$PY>J4cdPt*Ww8;#r?AlRoXwBwmJ1Z#IS<+k^ZY)HSN=A#=dm5ychGbm?Af4Fb?5*b`<yk;WE*r#)Wx6w^bsks;mP>jgy9AT|i?2Q)x1E&s9(dNB_3RZer67oA-Hr2$?ETq?pHHvO-uDm{L}E2Nw%&z~Q9ugdjfuaZcc99~q@bEw50Q|(JJC|ESq)yY=_XAnK$L*#bJ&SHR4l_JHH-I--%Q73&-_m9Z<1_bN<Gt=eps;(Zy?(CEif*V4XrY&wS-$!u{WI_9vqPGzbAW#1NcDr^iFb2hcZSaR`4yg4S6({y!Ni;s!OVtbpQ2XjQ5jt$?gk&18AL1jXGz}co=VMbors%J$6JQ1qr66(TU0N6IVGW)~hD#J(sx+D3&_sXaFCSqX1^$jsP`Shj2y&0Qizj!5yOvxMKbW`z{M*i#M;~)E49v7Ocrd!5qqhnMFiZ&Uxg}KDp>}%|Yh4b+Hz^zhEOw!UAIfYMz1`1P+QZx)NC{tG-Ho5}GZ9xFztR!O(Q|Poi{zr5{tsm@HU{O$cw4rd7q88v^VAGG0PebL3PyGWf2RT#>SEa`YVn-*d6zP@5JHrgdpNtsB-bNr8V*HfdLtpal`-0LGS=N;C}dsI~-hNH!(skn?G)mjYZ9oOIn~Lnz~r2Nj@!8y61%(YSspHQ3JXu@r*<sBe&jxvZ0$CJ6*Z^;9(P8TMV76EpQnGk=HORIuYas$dJcs$liI5>2=k%I3i5`mO*}vTy|^(Svp0nFYIDJLYPSmj(hY_Q8~pZ)VTM69GHX=AqBBlBH!^RZzHdXhouHIw_}<<N4#k^a;{#i2jy=2SpK7TQJEdFy<hc!YO9aco3%J&4{#VAjqlPW%0a$jKK@CMpp+ag?9<hOEgF{eZ094b70qlxSS_mVP6V()Ln<drHT1V{BD$S-$j|2Rc%dbu8eX$-@Ag@w}SpG8QXOaVeg`_B;YU9_1yXd<`Z!TYRl03OkoPu%_<vaj-Wa2P9Z0VJJWtv&~SO!6uL{T67rVpxltzEzn;#~?vI1Iv7=${j<y^3?r28?-(w6K`F_(i^nT-MOO*f4Oqn+<hBL=A^Tt!4DO0NA0YQg#xCbi1l+&q&tBfw7eQLEYdm(6)rjda+U0$fkEzr&#$hRFRc5$XPixVW|w=*p|=pLe{o{XMvxPrFKM1yIXoE!I^<JxBM4R@4c7g6!9@Vodz&vCFOmT=wqt1Y4P1hdf4W3byw@z_`k6AJF2k{q($#)wrxKH9=M?9{#(sd$alwBaSV)>fE0aUg=@j<-zlKX)>R!SRN?B74*QgIBNi{~{w!5W?<r#tH?<vDg-Df<>DcniYo-4_Vuun0zAMVC`hrv=StUf#Qu}un?t}+4+Z$SMk;uW59>~=i9fNO=sdA{1N)z4&5(jKR$iUT=Z+BXxZ0mKOWgJXZI8oc%>O~kjrFK@Quvxu0Egqom*WGG&KJ^lbxMHpREX6Bi)c92<J{lfeFLp5T!VdCtw!{$H8P9O!AP9hvw{t_#8VkJHj_&;_*gkIvICerd@UwYVO+7{xzW==S$Hx8;=+R>OEzuCn%>Oj-RH$?RviGt7cAD=2^^67{OzP*N&l{+Wd`S^sF{IHcAH8z%S5--tflpu}!&?*NZ8;;^J-B6YCn&*;iWOIbEDY@M%0@Sp5df4NtaJG~O_g)0Jm3P#2lOK>39XwiN@N%NKwSI=2Vkh~rbInL7y$-#NRE9R@!q-7tfUUb0u<qTsUy)6|&ntBgi--O}oBzpzC=ot?hlcFLFD)j-XT=r^`qeX*O|&`}rVn@IzP_B9tR7Jxw}z_#DL0pVy8b%TO~-;pFlg~5u@DNb)H?qD&QW++?OON4HZbr`PO@xvy}F%%@|AtvU`hs8#1{>s?+|Kj8282MtH3}a<?@$72m*TvC`^-f^V$il8bZ&ipe=^NT<+W^_y7`#KlG@J?V>fJ1$=o)7ty&U@-71+Yi0N2J@yuQU}PrFc77sU8VkJMj-@&v;ug(#m{6xdg0$dI9A-XhU5Y5&j=?YM0pq_PDzpItfby%OEejPP-HV{FJd>G~%;-vtDD*gxL|$sZN5p}$@y?s+O_e1+Y_BaE{r#5`U3>2~avK*Ud?js`jgj=wl{hePL6h}4RS|Nk(7R<!j>#qMkjNDJU%wm&w{kQrdJCP+|?|8B6h+t(zQ7&m0u4*T(tuDgb>OhxxU7~=umgxPHYIwTbZ8?hVXCm3q|bUYaC=g8^?fE%B$PxcPo1m2~@o&}64%bE2f^FDMRM=1;{K~vK^hTLrBUbt;CnDww^`>_&#cFp^nuM|e#&r`8)sex&Bf*QLdI}PK8O?NW-A5cpH0u%!j000080000X03#7l^<e`5044?i00{s900000000000HlEc0000%M@3FlQ!Z^}P)h*<6ay3h000O8001EXH-w{Wf(ift4io?Y6aWAK0000000000q=AS7003lfV{<Q5K~rB=H(x_fOhG~}ZDdeO0Rj{Q6aWAK2mk;8AprW+yJ1HF002$^000*N0000000000005+cJPrT=Wq5F9a%pX4Zgg`mUtei%X>?y-E^v8JO928D0~7!N00;m803iU4>Ph&n0002u0000U00000000000001_fw&F;0A+Y^WpZh4Wo~qHFLYsZUv@VyUtei%X>?y-E^v8JO928D0~7!N00;m803iUA@eCa45&!_HNB{sE00000000000001_fuaup0A+Y^WpZh4Wo~qHFLYsZUv@VyVRd9_bS`jtP)h*<6ay3h000O8001EX3@BZfSO@?B3l#tW82|tP0000000000q=CdD003opaAk67ZDnqBb1!sZb6<8hFJo+JE^v8JO928D0~7!N00;m803iT^OR!t24*&q=F#rG|00000000000001_fm1600A+Y^WpZh4Wo~qHFLYsZUv@VyWNBe%Zf|pRX=8IPaCuNm0Rj{Q6aWAK2mk;8Apkb;zr%<j005O`001BW0000000000005+cJUajYWq5F9a%pX4Zgg`mbYXK}b~i6&c42IFVRUJ4ZZ2?nP)h*<6ay3h000O8001EX{*FwaR|o(Agcbk*9smFU0000000000q=5om003opaAk67ZDnqBb1!sZb6<8hFJ*XiWpZv|Y%XwlP)h*<6ay3h000O8001EX`No1L><j<^R4D)e9RL6T0000000000q=A)X003opaAk67ZDnqBb1!sZb6<8hFKuOXXm4b5E^v8JO928D0~7!N00;m803iULwve)w1^@tU5&!@m00000000000001_fxL160A+Y^WpZh4Wo~qHFLYsZUv@VyaB^jEVRB_;E^v8JO928D0~7!N00;m803iSZ6}mS(4FCWDE&u==00000000000001_fsJ_p0A+Y^WpZh4Wo~qHFLYsZUv@Vya%FIDa&#_mc~DCM0u%!j000080000X0EU><LBb{g0Oy_n02=@R00000000000HlEfhX4R&cyMKMX>Db0baO9sVRK)0H!pK^b!2%iaCuNm0Rj{Q6aWAK2mk;8AprYo;7%YB005Cc001Wd0000000000005+c{;dE2Wq5F9a%pX4Zgg`mbYXK}b~i6`baiBTUu0=vXKrtEbZKLAE^v8JO928D0~7!N00;m803iU3BA%DY4FCWQDgXc*00000000000001_fn&b_0A+Y^WpZh4Wo~qHFLYsZUv@Vyc4=jIb1rasP)h*<6ay3h000O8001EXf8|9wp#T5?(EtDd8~^|S0000000000q=9A40044jadl~OWo>0{baO9bZ){;=En#h9WpXZbcyv%p0Rj{Q6aWAK2mk;8Apo+Ezm!V>008j;000{R0000000000005+cK+XUFa%FLKX>w(4Wo~qHFJo_PVPY*}VRL0Jba-@7O928D0~7!N00;m803iShmo}ij000000RR9Y00000000000001_fyB=M0CHt<b!l>CZDnqBb1!3WY++(8aAk61WpH$9Z*DGhcyv%p0Rj{Q6aWAK2mk;8Apqr4rEg#W003<O000*N0000000000005+c!O#EzbYXK}a&>NWX>DaMUtei%X>?y-E^v8JO928D0~7!N00;m803iT7_WpMk4*&p%H2?q@00000000000001_fmYD~0CZt<UvhPBbZKp6FJW?YX=Y(#baO6nc~DCM0u%!j000080000X0IJDBD^?i*0M=Fj02KfL00000000000HlGU-~a%0VRK({b#8QNZDlWEb8}^Mb1rasP)h*<6ay3h000O8001EXLL>f6R|Nn7unqtK7ytkO0000000000q=75?004Ahb6;|GZggpFWiMfLbY*UIX>V>WaCuNm0Rj{Q6aWAK2mk;8Apn1<l#i<o001{9000;O0000000000005+cxc>kEbYXK}a&>NWX>DaMVsCGBb98cHa4v9pP)h*<6ay3h000O8001EXhN-_?y9WRO(H8&!6#xJL0000000000q=A_Y0RVJib6;|GZggpFWiMlIZf0p`E^v8JO928D0~7!N00;m803iT&vjb~L7ytkUPyhfE00000000000001_frk|V0CZt<UvhPBbZKp6FJxhKVJ>iaP)h*<6ay3h000O8001EX6Bd>{SPTFFoh1MO7XSbN0000000000q=5h~0RVJib6;|GZggpFWiMo9V{c?>Zf7oVc~DCM0u%!j000080000X0E>r>10e|j0Hz!O02u%P00000000000HlG9IspK5VRK({b#8QNZDlWTWpZO>aCB*JZZ2?nP)h*<6ay3h000O8001EXwLeG~8V&#e-8BFJ8UO$Q0000000000q=Dr_0RVJib6;|GZggpFWiN1La&u{ObY*U1WiD`eP)h*<6ay3h000O8001EX+JzB0tOEc5*9ias7ytkO0000000000q=7e50RVJib6;|GZggpFWiN1YZ)Rz1X>MmOaCuNm0Rj{Q6aWAK2mk;8ApqeU(6@690081F000&M0000000000005+c8CC%RbYXK}a&>NWX>DaMa&>NWX>DaLaCuNm0Rj{Q6aWAK2mk;8AprjRaCa2~002A$000pH0000000000005+cz+(XbbY*jNb1!3WZf104b963nc~DCM0u%!j000080000X0DAQ7X8r{L08$hH02crN00000000000HlEgW&r?nWpi|MFLY&dbYEd|bZKT`V{~&aaCuNm0Rj{Q6aWAK2mk;8ApqmRWT49k005RB000yK0000000000005+cH*5g_bY*jNb1!sdb97%}b8}^Mb1rasP)h*<6ay3h000O8001EXWo%a-`v(93q80!E6951J0000000000q=7Vb0RVJmb98erbY*jNUtx7*X>=}dc~DCM0u%!j000080000X0IqqO#V-T^0K^FZ02crN00000000000HlFneE|S;Wpi|MFLY&dbYEg`Z*+5Xa$#^TaCuNm0Rj{Q6aWAK2mk;8AplLo8ew1v006fg000sI0000000000005+c#DM_-bY*jNb1!sdb97&1VRT_GaCuNm0Rj{Q6aWAK2mk;8Apo4PZvOTH0024%000>P0000000000005+cREhxrbY*jNb1!sdb97&1X<=t>Z*z2MV{<NWc~DCM0u%!j000080000X0B}~zQ1b@>0BIQj02lxO00000000000HlF|jR633Wpi|MFLY&dbYEq5VQh6_bZKvHE^v8JO928D0~7!N00;m803iT3?;M7&1^@s;6951f00000000000001_fvS}O0CZ(@baO9sWpi|2Zf|sDVsCG2E^v8JO928D0~7!N00;m803iTuMdViU1ONc%4gdfj00000000000001_fsLF20CZ(@baO9sWpi|2Zf|sDVsCG2Uu0!$X?A6Dc`k5yP)h*<6ay3h000O8001EX*-!WZTLu6CWflMc82|tP0000000000q=CGl0RVJmb98erbY*jNUvOn|b7^yQWo~0-E^v8JO928D0~7!N00;m803iU_uLBL71^@uM6#xJh00000000000001_fljCa0CZ(@baO9sWpi|2aB^jEVRB_;E^v8JO928D0~7!N00;m803iSn__T@f1poky6951d00000000000001_fgY~`0CZ(@baO9sWpi|2a%FIDa&#_mc~DCM0u%!j000080000X0MK$1fs_XT06iA~02KfL00000000000HlFIwE+NhWpi|MFLY&dbYF6HZggpFWiD`eP)h*<6ay3h000O8001EXXA}K~{to~ES~vgz6#xJL0000000000q=5*%0RVJmb98erbY*jNUvpz=Wo~0-E^v8JO928D0~7!N00;m803iSmPO>Z94FCW&EC2u#00000000000001_fjP|q0CZ(@baO9sWpi|2b98lNc`k5yP)h*<6ay3h000O8001EX#WzWpl@|a2UOfN+9RL6T0000000000q=81;0RVJwZ)|ffVs&Y3WM5-%Y++(wZf|sDVsCG2E^v8JO928D0~7!N00;m803iT6hPxV%2><|A8~^|s00000000000001_ffn-t0CaC}Y;!MeZ**m1Z*OZ~WMyn=c4cyTE^v8JO9ci10000m05AYL4FCY$`~d&}00'}

def restore_source_package(package, checkout, persistent, repo_url):
    """Notebook bootstrap; stdlib only, before installing scientific packages.

    This function is copied by the notebook generator so upload of the notebook
    alone is sufficient. Its authoritative source remains this Python module.
    Existing baseline files cannot be replaced by the private package.
    """
    import base64
    import hashlib
    import io
    import json
    import os
    from pathlib import Path
    import re
    import stat
    import subprocess
    import zipfile

    if package.get("schema") != "tas-private-source-v1" or any(
        re.fullmatch(r"[a-f0-9]{40}", package.get(key, "")) is None
        for key in ("base_commit", "source_commit")
    ):
        raise ValueError("Invalid versioned source package.")
    data = base64.b85decode(package["archive_b85"])
    if hashlib.sha256(data).hexdigest() != package["sha256"]:
        raise ValueError("Private source archive checksum mismatch.")
    checkout, persistent = Path(checkout), Path(persistent)
    base = package["base_commit"]

    def owned_path(name):
        relative = Path(name)
        allowed = len(relative.parts) > 1 and relative.parts[0] in {
            "tas_runtime",
            "experiments",
            "requirements",
            "tests",
            "tools",
        }
        allowed |= name in {"AGENTS.md", "docs/TAS_V7_COLAB.md"}
        return (
            allowed
            and not relative.is_absolute()
            and ".." not in relative.parts
            and (checkout / relative).resolve().is_relative_to(checkout.resolve())
        )

    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = archive.infolist()
        if len(members) != len(package["files"]) or {
            item.filename for item in members
        } != set(package["files"]):
            raise ValueError("Private source file manifest mismatch.")
        if sum(item.file_size for item in members) > 8 * 1024**2:
            raise ValueError("Private source package exceeds its code-only size limit.")
        contents = {}
        for item in members:
            if not owned_path(item.filename) or stat.S_IFMT(
                item.external_attr >> 16
            ) not in {0, stat.S_IFREG}:
                raise ValueError("Unsafe private source member.")
            content = archive.read(item)
            if hashlib.sha256(content).hexdigest() != package["files"][item.filename]:
                raise ValueError("Private source member checksum mismatch.")
            contents[item.filename] = content
    if not (checkout / ".git").exists():
        subprocess.run(
            ["git", "clone", "--no-checkout", str(repo_url), str(checkout)], check=True
        )
    found = subprocess.run(
        ["git", "cat-file", "-e", base + "^{commit}"], cwd=checkout, capture_output=True
    )
    if found.returncode:
        subprocess.run(["git", "fetch", "origin", base], cwd=checkout, check=True)
    for name in contents:
        baseline = subprocess.run(
            ["git", "cat-file", "-e", base + ":" + name],
            cwd=checkout,
            capture_output=True,
        )
        if baseline.returncode == 0:
            raise ValueError(
                "Private delivery cannot overwrite a public baseline file: " + name
            )
    subprocess.run(["git", "checkout", "--detach", base], cwd=checkout, check=True)
    actual = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=checkout, text=True
    ).strip()
    if actual != base:
        raise ValueError("Public baseline checkout mismatch.")
    previous = checkout / ".tas-source.json"
    if previous.exists():
        old = json.loads(previous.read_text())
        for name in set(old.get("files", {})) - set(contents):
            baseline = subprocess.run(
                ["git", "cat-file", "-e", base + ":" + name],
                cwd=checkout,
                capture_output=True,
            )
            if not owned_path(name) or baseline.returncode == 0:
                raise ValueError("Unsafe obsolete source path.")
            (checkout / name).unlink(missing_ok=True)

    def save(path, content):
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".installing")
        temporary.write_bytes(content)
        os.replace(temporary, path)

    for name, content in contents.items():
        save(checkout / name, content)
    receipt = {key: value for key, value in package.items() if key != "archive_b85"}
    receipt_bytes = (json.dumps(receipt, indent=2, sort_keys=True) + "\n").encode()
    save(checkout / ".tas-source.json", receipt_bytes)
    stored = (
        persistent
        / "code"
        / ("source-" + package["source_commit"] + "-" + package["sha256"][:16])
    )
    if (
        not stored.with_suffix(".zip").exists()
        or hashlib.sha256(stored.with_suffix(".zip").read_bytes()).hexdigest()
        != package["sha256"]
    ):
        save(stored.with_suffix(".zip"), data)
    save(stored.with_suffix(".json"), receipt_bytes)
    return receipt


## Connect storage and restore the exact code

Drive authorization may prompt once per runtime. No credentials are saved
in configuration or outputs. Source restoration is automatic and verifies
both the archive and every file before installation. It cannot replace any
file from the frozen public baseline. The same caches work in CPU/GPU sessions.


In [ ]:
try:
    import os, sys, json, subprocess, pathlib, shutil, time
    from IPython.display import display, Markdown, HTML
    from google.colab import drive
    drive.mount("/content/drive")
    persistent = pathlib.Path(PERSISTENT_ROOT)
    persistent.mkdir(parents=True, exist_ok=True)
    checkout = pathlib.Path("/content/tas-v7-checkout")
    if (SOURCE_PACKAGE["base_commit"], SOURCE_PACKAGE["source_commit"]) != (BASELINE_COMMIT, RESEARCH_COMMIT):
        raise ValueError("Regenerate the notebook to change its fixed code versions.")
    source_receipt = restore_source_package(SOURCE_PACKAGE, checkout, persistent, REPO_URL)
    actual_commit = source_receipt["source_commit"]
    os.chdir(checkout)
    if str(checkout) not in sys.path:
        sys.path.insert(0, str(checkout))
    # A new idea notebook may reuse this VM with different private modules.
    for module_name in list(sys.modules):
        if module_name == "tas_runtime" or module_name.startswith("tas_runtime."):
            del sys.modules[module_name]
    import importlib
    importlib.invalidate_caches()
    print("Public baseline:", BASELINE_COMMIT)
    print("Experiment Python source:", actual_commit, "| verified files:", len(source_receipt["files"]))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Isolated, pinned environment

Python 3.10.16 and matched PyTorch 2.4.1 / torchvision 0.19.1 CPU or CUDA
12.1 wheels run in subprocesses. The legacy Transformers 4.35.2 stack is
preserved. No notebook-kernel restart, system Torch replacement or CUDA
extension compilation is needed. GPU mode validates its actual CUDA path.


In [ ]:
try:
    query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    has_gpu = query is not None and query.returncode == 0 and bool(query.stdout.strip())
    effective_phase = ("run" if has_gpu else "prepare") if PHASE == "auto" else PHASE
    mode = "gpu" if effective_phase == "run" else "cpu"
    print("Hardware:", query.stdout.strip() if has_gpu else "CPU")
    print("Phase:", effective_phase, "| environment:", mode)
    if effective_phase == "run" and has_gpu and REQUIRE_A100_80GB and ("A100" not in query.stdout or int(query.stdout.strip().splitlines()[0].split(",")[-1].strip().split()[0]) < 70000):
        raise RuntimeError("This configuration requires A100 80GB; hardware check stopped before environment/model setup.")
    if effective_phase == "run" and not has_gpu:
        raise RuntimeError("Select a GPU runtime, or use PHASE=prepare/score on CPU.")
    if effective_phase == "run" and not (persistent / "preparation.json").is_file():
        raise RuntimeError("Run this notebook once in a CPU runtime first. That downloads/persists models and data before A100 allocation.")
    from tas_runtime.bootstrap import ensure_environment
    python = ensure_environment(mode, "/content/tas-v7-envs", checkout)
    if DATASET == "amber":
        from tas_runtime.bootstrap import ensure_amber
        ensure_amber(python, "/content/tas-v7-envs", checkout)
    env = dict(os.environ, PYTHONHASHSEED="0", TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg",
               NLTK_DATA=str(persistent / "nlp"), YOLO_CONFIG_DIR=str(pathlib.Path(LOCAL_ROOT) / "ultralytics"))
    configuration = {
        "persistent_root":PERSISTENT_ROOT, "local_root":LOCAL_ROOT, "stage":STAGE,
        "seeds":SEEDS, "methods":METHODS, "batch_sizes":BATCH_SIZES,
        "attention_backends":ATTENTION_BACKENDS, "last_position_logits":LAST_POSITION_LOGITS,
        "max_session_minutes":MAX_SESSION_MINUTES, "require_a100":REQUIRE_A100_80GB,
        "run_confirmation":RUN_CONFIRMATION, "run_controls":RUN_CONTROLS,
        "run_diagnostics":RUN_DIAGNOSTICS, "run_final":RUN_FINAL_REMAINDER,
        "reserve_pope":RESERVE_POPE_IMAGES, "split_seed":SPLIT_SEED,
        "coverage_tolerance":COVERAGE_TOLERANCE_PP / 100.0,
        "dataset":DATASET, "external_manifest":EXTERNAL_MANIFEST, "external_image_root":EXTERNAL_IMAGE_ROOT,
        "auto_continue_after_smoke":True, "source_revision":actual_commit,
        "generation":{"max_new_tokens":180,"temperature":1.0,"top_p":1.0,"top_k":None,
                      "precision":"fp16","do_sample":True,"conv_mode":"llava_v1","prompt":"Describe this image."}
    }
    config_path = pathlib.Path("/content/tas-v7-config.json")
    config_path.write_text(json.dumps(configuration, indent=2))
    def run_phase(phase):
        subprocess.run([str(python), "-m", "experiments.tas_v7.cli", "--config",str(config_path),"--phase",phase], cwd=checkout,env=env,check=True)
    if DATASET == "amber":
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp"), "--amber"], cwd=checkout, env=env, check=True)
    run_phase("plan")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Prepare or run the study

First CPU session: approximately 17–18 GB of checkpoints plus annotations,
selected images and NLP resources; existing verified files are reused.
Only 30 train images are fetched, not all COCO train2017.

GPU sessions: lightweight preflight → local asset restoration → one model
load → baseline/view audit → correctness/throughput pilot → configured
stages. Per-image RNG makes retries/resume independent of job order. A
rejected optimization is recorded; baseline failures block larger runs.
Results use immutable chunks and perception caches use shards on Drive.


In [ ]:
try:
    phase_started = time.perf_counter()
    if effective_phase == "prepare":
        # Small NLP assets enable the offline scientific fixtures first.
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp")], cwd=checkout, env=env, check=True)
        run_phase("check")
        run_phase("prepare")
        run_phase("score")
        display(Markdown("**CPU preparation complete.** Files are persistent. Switch to the A100 runtime and Run all again."))
    elif effective_phase == "run":
        run_phase("check")
        run_phase("run")
    elif effective_phase == "score":
        run_phase("score")
    elif effective_phase == "check":
        run_phase("check")
    else:
        raise ValueError("Unknown phase")
    print("Session phase seconds:", round(time.perf_counter() - phase_started, 1))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Inspect saved reports

LEGACY and HARMONIZED are separate protocols; do not infer a matched
algorithm gain across them. CHAIR and annotation category recall do not
establish caption quality or preserved relations. Inspect ambiguous images,
captions, failures, coverage, length, timings and uncertainty together.
Reports include descriptive comparisons for paired captions with exactly
equal word counts. Batch latency and amortized seconds/image are separate.
CPU scoring/report regeneration is available with PHASE="score".


In [ ]:
try:
    import csv, html
    from IPython.display import Image as DisplayImage
    runs_root = persistent / "runs"
    for run_dir in sorted(runs_root.glob("*"))[-3:]:
        display(Markdown("### Run `" + run_dir.name + "`"))
        session = run_dir / "session.json"
        if session.exists():
            display(Markdown("```json\n" + session.read_text() + "\n```"))
        gallery = run_dir / "views.png"
        if gallery.exists():
            display(DisplayImage(filename=str(gallery)))
        for summary in sorted(run_dir.glob("*/summary.csv")):
            rows = list(csv.reader(summary.open()))
            table = "<table>" + "".join("<tr>" + "".join("<td>" + html.escape(value) + "</td>" for value in row) + "</tr>" for row in rows) + "</table>"
            display(Markdown("**" + summary.parent.name + "**"))
            display(HTML(table))
    print("Persistent artifacts:", persistent)
    print("Reuse this folder for later sessions/new idea notebooks. Do not delete frozen manifests or rewrite completed runs.")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Release the session

With `AUTO_SHUTDOWN=True`, the final cell flushes Drive writes and releases
the runtime after preparation, evaluation or a planned budget stop. Errors
also invoke this cleanup before propagating. Reports remain on Drive.
Set it to False only when keeping the runtime for inspection.


In [ ]:
release_runtime()
